In [ ]:
EIR_VERSION = "2.0.16"

# Eir

**Automated bond valence sum analysis, with an integrity verdict.**

Eir reads a directory of CIF files and answers two questions about each one:

1. **Can this entry be assessed at all?** Can the file be read, can the atoms
   be where it says, is there reference data, is the deposited model
   unambiguous.
2. **If so, what does the bond valence model say?**

The order matters. A bond valence sum is computed from bond lengths, so a
length that cannot be right makes every number built on it meaningless — and
often *good-looking*. ICSD 85235 (TbCrO₃) has a chromium octahedron 0.0004 Å
from the mean of every measured Cr³⁺–O bond, a GII of 0.0047, and two oxygens
2.191 Å apart against a family median of 2.750. Judged on its chemistry it is
the finest structure in the 938-entry perovskite set. Judged on its geometry
it is impossible.

---

## The PGRM integrity gates

Four facets, asked in order. The **first failure stops the assessment**.

| | facet | asks |
|---|---|---|
| **P** | Parse | can the file be read? |
| **G** | Geometry | can the atoms be where it says? |
| **R** | Reference | do we have the data to assess it? |
| **M** | Model | is the deposited description unambiguous? |

**P** and **G** are EXPUNGE: no bond valence result should be quoted.
**R** is NOT_ASSESSED: part of the analysis is missing, not wrong.
**M** is FLAG: the analysis stands, with the ambiguity named.

Read `structure_pgrm_tier` for the verdict, `pgrm_reason` for what failed and
`pgrm_evidence` for the measurement — every expunged row carries the number
that condemned it.

This replaces and contains the six-type taxonomy of Reeves-McLaren (2026)
*Dalton Trans.* **55**, 2868 (Type 2 → R1, 3 → M1, 4 → M3, 5 → M2, 6A → M6,
7 → M4). What is added is the **G** facet, which that taxonomy had no category
for and could not have had: it was derived on rock salts, where every atom
sits on a special position and a coordinate error is impossible by
construction. Every family with positional freedom is dominated by geometry
faults.

## The screens

Six independent tests, reported separately because they see different things.

| screen | what it sees |
|---|---|
| radii ratio | shortest cation–anion contact against the Shannon radii |
| bond-valence distance | the same contact against the ion's own parameters |
| **anion–anion packing** | two anions closer than their clouds allow |
| **cation–cation packing** | the same for the cation sublattice |
| coordination | whether bonds are MISSING rather than too short |
| half-lattice shift | whether the cation sits on the wrong origin |

The two packing floors are the only tests that do not look outward from a
cation, and on the perovskites the anion floor has more exclusive catches than
any other screen. Both are set from the measured distributions of three
reference families, not by analogy.

The shift screen **proposes**; nothing is corrected. Each proposal is applied
to a *copy*, the whole analysis re-run, and the outcome fed back into the
verdict, so `cation_on_wrong_origin` means the hypothesis was tested and held.

## Where the numbers come from

**Bond valence parameters:** Gagné & Hawthorne (2015), with Brown (2020) as
fallback. Not Brown & Altermatt (1985) — this matters for any comparison with
published GII values.

**Expected bond lengths:** measured, from Gagné's aggregated survey — 596 ion
configurations, 202 ions, bonded to O²⁻ and N³⁻. Gagné states plainly that
these are more accurate than values obtained by adding ionic radii and should
be used in their stead; `bl_source` records which route each site took, and
Shannon additivity remains the flagged fallback.

**The coordination shell** is defined by bond-valence contribution, not by the
largest gap. `BV_SUM_CUTOFF = 0.02` was arrived at from three independent
directions: the octahedral control on perovskite B sites, the anion
coordination reaching 2 B + 4 A, and Gagné's own longest counted bond sitting
at *s* = 0.019 for the weakly bonded ions where the boundary binds. The gap
shell survives as `cn_gap_shell`, used to detect amputated shells.

Every threshold in Eir carries its measurement in the comment beside it. Where
one does not — `DISTORTED_BAV_MIN`, `DISTORTED_BAUR_MIN` — the comment says so.

## Which instability index to quote

`gii` weights every site equally. `wagii` weights by occupancy.
`gii_site_averaged` combines the sums at a shared site before taking a
deviation.

They are the same quantity **when every occupancy is 1.0** — not merely when
nothing is shared. WAGII weights by occupancy, so a vacancy separates them
exactly as a shared site does: full occupancy gives `gii == wagii` on 869 of
871 perovskites and 355 of 355 pyrochlores, while 1543 partially occupied
spinels sit off the identity line.

`gii_recommended` carries the choice per structure; the summary states the
choice for the dataset, once.

## What a run produces

```
outputs/
  eir_batch_results_v<version>_<stamp>.csv             per site, raw
  ..._quality.csv                                      per site, screened
  ..._structures.csv                                   ONE ROW PER STRUCTURE
  eir_batch_failed_v<version>_<stamp>.csv              parse failures
  eir_summary.txt                                      the narrative
  eir_shift_test_<stamp>.csv  +  shifted_cifs/         the audit trail
  eir_screens / gii / cation_anion / gii_vs_wagii      .pdf and .png
```

**`_structures.csv` filtered to `structure_pgrm_tier == 'EXPUNGE'` is the
reportable-error list.** Each row carries the ICSD code, year, journal, reason
and the measurement.

Figures are single-column at 3.35 in, quantity-calculus axis labels
(`GII / v.u.`, never brackets), University of Sheffield brand palette chosen
so the categories separate on lightness as well as hue.

## Where Huginn begins

Eir answers *is this entry sound, and what is the evidence*. Huginn answers
*what does this pool of entries tell us*: pooling several runs, family-specific
modules, provenance across decades and journals, publication figures.

Reduction to structures is Eir's job and is done once, in
`build_structure_table`, which **asserts** that every column it reduces by
taking the first row is genuinely constant within a structure. Getting that
wrong is not hypothetical: `.first()` on the per-site radii column finds 20 of
48 failing perovskites and misses 28.

## Running it

Run the cells top to bottom. `VALIDATE()` runs sixteen reference tests against
structures with analytically known answers, and **raises** if any fails.
Oxidation states come from a `config.json` in the CIF directory, from the CIF
itself, or interactively.

```python
{"Na": 1, "Ni": 2, "Mn": 4, "O": -2}
```

## Interpreting a result

**Coordination number** is the count of bonds actually summed, from the
bond-valence shell. `cn_gap_shell` beside it is the gap-analysis answer; where
they differ, `cn_fault` says why.

**GII thresholds are family-dependent.** The often-quoted 0.20 v.u. is a rule
of thumb from simple ordered structures. Families whose connectivity
over-constrains the geometry — spinels and many perovskites — routinely exceed
it while being perfectly stable. Read the number against an expectation for
the family, and always alongside the integrity tier.

**A high GII is not evidence of instability until the geometry has passed.**
That is the whole point of the ordering.

## Citation

- **Bond valence parameters:** Gagné, O.C. & Hawthorne, F.C. (2015)
  *Acta Cryst. B* **71**, 562–578
- **Bond lengths:** Gagné, O.C. & Hawthorne, F.C. (2016, 2018a,b, 2020);
  Gagné, O.C. (2018, 2020)
- **The bond valence model:** Brown, I.D. (2002) *The Chemical Bond in
  Inorganic Chemistry*
- **Eir:** Reeves-McLaren, N. (2026) *Dalton Trans.* **55**, 2868

## Version history

- **v2.0.13 (Aug 2026)** — PGRM integrity gates replace the bond-valence-derived
  tier; cation–cation packing floor added; measured bond lengths; adaptive
  shell boundary exported; normaliser translations and origin-choice offsets
  derived from the space group rather than hard-coded; per-structure table;
  four figures; full cell-by-cell audit (every bare `except` narrowed, five
  duplicated definitions consolidated, the Huginn export retired).
- **v2.0.2 (Aug 2026)** — anion-centred BVS engine; anion–anion packing screen;
  WAGII and site-averaged GII; the A-site radii override retired.
- **v2.0.1 (Aug 2026)** — general, physics-based rebuild on a single gemmi
  parse; native per-site geometry schema; pymatgen removed.
- **v1.7 (May 2026)** — data integrity tiers, anti-site handling.
- **v1.6.1 (Apr 2026)** — fractional-valence R₀ interpolation.
- **v1.1 (Dec 2025)** — config files, 3D visualisation, adaptive gap detection.
- **v1.0 (Dec 2025)** — rewrite with proper crystallographic symmetry.


In [ ]:
import sys
print(f"Current Python: {sys.executable}")
print(f"Installing py3Dmol to this environment...\n")

get_ipython().system('{sys.executable} -m pip install py3Dmol')

print(f"\n✓ Installation complete - restart kernel to use py3Dmol")

# Once run, go to Kernel → Restart Kernel (or Restart & Clear Output)
# Then change this cell from Code to Markdown to prevent wasting time reinstalling the library every time.

In [ ]:
# CELL 1: HELPER FUNCTIONS   (Eir 2.0.13)
#
# A NOTE ON THE DATASET SIZES QUOTED IN COMMENTS THROUGHOUT THIS NOTEBOOK
#
# Constants here are set from measurement, and each carries its measurement
# beside it. Many cite '934 ICSD perovskites' or '1886 sites'. Those figures
# are correct AS MEASURED and are deliberately not updated: the reference set
# became 938 structures and 1895 sites at v2.0.3, when the CIF repairs
# (_unquote_atom_site_numbers, normalise_cif_to_ascii) recovered four entries
# that had previously failed to parse - ICSD 118400, 118401, 118459 and
# 124972, all rejected for a double-encoded non-breaking space in an author
# name.
#
# Rewriting the numbers to 938 would falsify the record of when each constant
# was fitted. A measurement quoting 934 predates that repair; one quoting 938
# postdates it. Where a constant would move materially on the four recovered
# structures it has been refitted and says so.
#
# The pyrochlore set is 472 structures and 1133 sites; the spinel set is 1881
# and 6657.
#
# CHANGED IN 2.0.2: anion_anion_contacts() decides which elements are anions
# from the structure rather than from a fixed list. See resolve_anions().
# ============================================================================
# Run this cell before the main Eir functionality cell
#
# VERSION HISTORY:
# - v1.7 (May 2026): Data integrity tier system (ACCEPT/FLAG/EXPUNGE),
#                     minority species detection (I3), implausible BVS detection (I6),
#                     mixed TM occupancy flagging, CIF valence metadata columns,
#                     structure-level tier summary, version string fix (was "v1.5")
# - v1.6.1 (Apr 2026): Fractional valence interpolation for mixed-valence cathodes
# - v1.5 (Mar 2026): Type 6 flagging fix, GII per-label iteration, Type 5 occupancy fix,
#                     formula element parsing fix, version string consolidation
# - v1.4.2 (Feb 2026): Per-atom valence iteration, valence_source tracking
# - v1.4 (29 Jan 2026): Oxidation state extraction from CIF, Type 7 error flags,
#                       anion parameter (u) extraction, enhanced structural parameters
# - v1.2: Temperature and pressure extraction from CIF files
# - v1.1: Config file support, 3D visualisation, adaptive gap detection

import os
import json
import re
import math

# v2.0.15: THIS IMPORT WAS MISSING AND THE CODE WORKED ANYWAY.
#
# load_structure_for_viz and anion_anion_contacts, both defined in this cell,
# call bare `gemmi.`. Nothing in this cell imported it. The only module-level
# `import gemmi` in the notebook is in CELL 6, with a second inside a try in
# CELL 7 - five and six cells below the code that depends on it. Running the
# notebook top to bottom hides this completely, because CELL 6 has run by the
# time anything calls a CELL 1 function.
#
# It surfaced when the self-check below started calling normaliser code
# directly, and it surfaces for anyone who re-runs CELL 1 in a fresh kernel
# and then calls look_at().
#
# It was also invisible for a second reason. Until v2.0.14 anion_anion_contacts
# ended in `except Exception: return None, None`, so on a cold start it did not
# raise NameError - it returned no measurement, and G2 and G5 both silently did
# not fire. Narrowing that except is what let this be found at all, which is
# the argument for narrowing them.
import gemmi

# ============================================================================
# ANION DETECTION
# ============================================================================

# v1.8.3 (B7): anion assignment cannot be a property of the element alone.
# P, S, N, As and H appear in KNOWN_ANIONS, so in a phosphate BOTH P and O
# tested as anions, the pair fell through to "neither/both - skip", and every
# P-O bond was silently dropped. The same killed sulfates, nitrates, borates,
# hydroxides and hydrates. Which species is the anion depends on its partner.

ALWAYS_ANION = {'O': -2, 'F': -1, 'Cl': -1, 'Br': -1, 'I': -1}

# Species that are anionic against an electropositive partner but cationic at
# the centre of a polyanion.
AMPHOTERIC_ANION = {'S': -2, 'Se': -2, 'Te': -2, 'N': -3,
                    'P': -3, 'As': -3, 'H': -1, 'C': -4, 'Si': -4, 'B': -3}

PAULING_EN = {
    'H': 2.20, 'Li': 0.98, 'Be': 1.57, 'B': 2.04, 'C': 2.55, 'N': 3.04,
    'O': 3.44, 'F': 3.98, 'Na': 0.93, 'Mg': 1.31, 'Al': 1.61, 'Si': 1.90,
    'P': 2.19, 'S': 2.58, 'Cl': 3.16, 'K': 0.82, 'Ca': 1.00, 'Sc': 1.36,
    'Ti': 1.54, 'V': 1.63, 'Cr': 1.66, 'Mn': 1.55, 'Fe': 1.83, 'Co': 1.88,
    'Ni': 1.91, 'Cu': 1.90, 'Zn': 1.65, 'Ga': 1.81, 'Ge': 2.01, 'As': 2.18,
    'Se': 2.55, 'Br': 2.96, 'Rb': 0.82, 'Sr': 0.95, 'Y': 1.22, 'Zr': 1.33,
    'Nb': 1.60, 'Mo': 2.16, 'Tc': 1.90, 'Ru': 2.20, 'Rh': 2.28, 'Pd': 2.20,
    'Ag': 1.93, 'Cd': 1.69, 'In': 1.78, 'Sn': 1.96, 'Sb': 2.05, 'Te': 2.10,
    'I': 2.66, 'Cs': 0.79, 'Ba': 0.89, 'La': 1.10, 'Ce': 1.12, 'Pr': 1.13,
    'Nd': 1.14, 'Sm': 1.17, 'Eu': 1.20, 'Gd': 1.20, 'Tb': 1.10, 'Dy': 1.22,
    'Ho': 1.23, 'Er': 1.24, 'Tm': 1.25, 'Yb': 1.10, 'Lu': 1.27, 'Hf': 1.30,
    'Ta': 1.50, 'W': 2.36, 'Re': 1.90, 'Os': 2.20, 'Ir': 2.20, 'Pt': 2.28,
    'Au': 2.54, 'Hg': 2.00, 'Tl': 1.62, 'Pb': 2.33, 'Bi': 2.02,
}

ANION_FORMERS = set(ALWAYS_ANION) | set(AMPHOTERIC_ANION)

# The species treated as ligands BEFORE any valence is known.
#
# Deliberately narrower than ANION_FORMERS, which also holds H, C, Si, B and
# P: in an oxide or chalcogenide those are framework formers rather than
# ligands, and admitting them would pull covalently bound network atoms into
# a coordination sphere. Wherever a valence IS available Eir uses
# resolve_anions(), which decides from the structure rather than from a list.
#
# v2.0.13: ONE definition. This exact nine-element set was written out twice,
# as _ANION_SET in CELL 1b and as ANION_ELEMENTS in CELL 3, with nothing to
# keep them in step - the same fault as the triple JAHN_TELLER_IONS and the
# double CATION_CATION_MIN. Both are now aliases of this.
PROVISIONAL_ANIONS = set(ALWAYS_ANION) | {'S', 'Se', 'Te', 'N'}

# Retained so existing calls keep working. Prefer classify_bond_pair().
KNOWN_ANIONS = dict(ALWAYS_ANION)
KNOWN_ANIONS.update(AMPHOTERIC_ANION)


# Reads that failed, so a file that could not be parsed is distinguishable
# from a file with nothing to report. Every metadata extractor below wrapped
# its whole file read in a bare `except: return result`, which is why
# measurement_pressure_GPa was populated on 10 of 934 perovskites with no way
# to tell absent metadata from a silently failed read. Cleared per batch by
# reset_read_failures().
_EIR_READ_FAILURES = []


def reset_read_failures():
    _EIR_READ_FAILURES.clear()


def report_read_failures():
    """Print anything that could not be read. Call after a batch."""
    if not _EIR_READ_FAILURES:
        return
    print(f"  {len(_EIR_READ_FAILURES)} metadata read(s) failed:")
    for _fn, _where, _err in _EIR_READ_FAILURES[:10]:
        print(f"    {_where}: {_fn} -> {_err}")
    if len(_EIR_READ_FAILURES) > 10:
        print(f"    ... and {len(_EIR_READ_FAILURES) - 10} more")


def _note_read_failure(cif_path, where, err):
    import os as _os
    _EIR_READ_FAILURES.append(
        (_os.path.basename(str(cif_path)), where,
         f"{type(err).__name__}: {err}"))


def is_anion(element, valence=None):
    """
    Whether an element is acting as an anion.

    An explicit valence decides it outright. Without one this falls back to the
    element list, which is only reliable for O and the halogens - use
    classify_bond_pair() wherever a partner is known.
    """
    if valence is not None:
        try:
            return float(valence) < 0
        except (TypeError, ValueError):
            pass
    return element in KNOWN_ANIONS


_SIGN_INVERSIONS = {}


def _valence_signs_are_credible(el1, el2, v1, v2):
    """
    Does the CIF's allocation of negative charge match electronegativity?

    ICSD 651062 (SrS) is deposited as "S2+ 2" and "Sr2- -2": the signs are
    simply the wrong way round in a compound that is unambiguously Sr(2+)S(2-).
    Trusting explicit valences - which is right, and is what makes mixed-valence
    sites work - then makes sulfur the cation, sends the code looking for
    S(2+)-Sr bond valence parameters, and the structure fails with "No M-X
    bonds found" and no indication why.

    Electronegativity settles it: whichever species carries the negative charge
    should be the more electronegative of the two. Where that fails, the CIF is
    wrong rather than the chemistry being unusual.
    """
    e1 = PAULING_EN.get(el1)
    e2 = PAULING_EN.get(el2)
    if e1 is None or e2 is None:
        return True                      # no basis to judge, accept as given
    try:
        f1, f2 = float(v1), float(v2)
    except (TypeError, ValueError):
        return True
    if f1 < 0 <= f2:
        return e1 >= e2
    if f2 < 0 <= f1:
        return e2 >= e1
    return True                          # same sign, nothing to check

def classify_bond_pair(el1, el2, valences=None, elements_present=None):
    """
    Decide which of a bonded pair is the cation and which the anion.
 
    Order of evidence:
        1. explicit valences, if their signs disagree
        2. whether each species is acting as an anion in THIS structure
        3. the more electronegative partner, where both could be anions
 
    Rule 2 is the important one. Whether an amphoteric species acts as an anion
    is not a property of the element, it is a property of the compound. Boron is
    an anion in a boride and a cation in a borate; phosphorus is an anion in a
    phosphide and a cation in a phosphate. Deciding from the pair alone gets
    NaCr(1-x)B(x)O2 wrong, because Na-B and Cr-B contacts within the search
    radius are then classified as cation-anion pairs and Eir goes looking for
    bond valence parameters that should never have been requested.
 
    `elements_present` is the set of element symbols in the structure. Given it,
    an amphoteric species is treated as a cation whenever the structure also
    contains something more electronegative that can act as an anion. Without
    it the old, pair-only behaviour is retained.
 
    Returns:
        (cation, anion), or (None, None) if the pair is not cation-anion.
    """
    valences = valences or {}
 
    # --- 1. explicit valences win outright -----------------------------------
    v1, v2 = valences.get(el1), valences.get(el2)
    if v1 is not None and v2 is not None:
        if not _valence_signs_are_credible(el1, el2, v1, v2):
            # Record and fall through to the electronegativity rules below,
            # which get it right. Reported once per structure by the caller.
            _SIGN_INVERSIONS[tuple(sorted((el1, el2)))] = (
                (el1, v1) if float(v1) < 0 else (el2, v2))
        else:
            try:
                f1, f2 = float(v1), float(v2)
                if f1 < 0 <= f2:
                    return el2, el1
                if f2 < 0 <= f1:
                    return el1, el2
            except (TypeError, ValueError):
                pass
 
    # --- 2. is each species acting as an anion here? -------------------------
    def _acts_as_anion(el):
        if el in ALWAYS_ANION:
            return True
        if el not in AMPHOTERIC_ANION:
            return False
        if not elements_present:
            return True            # no context supplied, fall back to old rule
        e = PAULING_EN.get(el, 0.0)
        for other in elements_present:
            if other == el:
                continue
            if other in ANION_FORMERS and PAULING_EN.get(other, 0.0) > e:
                return False       # something more electronegative is available
        return True
 
    a1, a2 = _acts_as_anion(el1), _acts_as_anion(el2)
    if a1 and not a2:
        return el2, el1
    if a2 and not a1:
        return el1, el2
 
    # --- 3. both could be anions: the more electronegative one is ------------
    if a1 and a2:
        e1 = PAULING_EN.get(el1)
        e2 = PAULING_EN.get(el2)
        if e1 is not None and e2 is not None and abs(e1 - e2) > 1e-6:
            return (el2, el1) if e1 > e2 else (el1, el2)
 
    return None, None

# ============================================================================
# OXIDATION STATE HANDLING (NEW IN v1.4)
# ============================================================================

# Elements that commonly exhibit mixed valence in spinels
VARIABLE_VALENCE_ELEMENTS = {
    'Fe', 'Mn', 'Co', 'Cu', 'Cr', 'V', 'Ni', 'Ti', 'Ce', 'Eu', 'Ru', 'Rh', 'Ir'
}

# ============================================================================
# DATA INTEGRITY TIER SYSTEM (NEW IN v1.7)
# ============================================================================
# Classifies each site (and hence each structure) into one of three tiers
# based on the reliability of the oxidation state assignment and BVS result.
#
# ACCEPT: Valence assignment is unambiguous (integer formal states, or
#         single-element sites with charge-balance-determined non-integer states).
# FLAG:   Two or more TM elements share a crystallographic site AND at least one
#         has a non-integer assigned valence. The valence partitioning is an
#         assumption made by the depositor and cannot be verified from diffraction.
# EXPUNGE: BVS result is demonstrably meaningless (implausible coordinates or
#          minority species on a site dominated by a different cation).
#
# See: Eir 1.7 Specification (NRM, May 2026)
# ============================================================================

TRANSITION_METALS = {
    'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
    'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
    'La', 'Ce', 'Pr', 'Nd', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu',
}

# Thresholds for EXPUNGE tier
MINORITY_OCC_THRESHOLD = 0.05  # Occupancy below which a species is "minority"
MAJORITY_OCC_THRESHOLD = 0.90  # Occupancy above which another species dominates


def is_transition_metal(element):
    """Check if an element is a transition metal (d-block or lanthanide)."""
    return element in TRANSITION_METALS


# v2.0.2: 0.045, not 0.05.
#
# Fractional valences in the ICSD cluster on two decimal places, so a boundary
# at exactly 0.05 sits on a value depositors write. ICSD 215426-215430 is one
# compositional series from one 2013 study, Mn(1+x)Tb(1-x)O3, with Mn valences
# of 3.10, 3.08, 3.07, 3.06 and 3.05. The first four were tiered FLAG and the
# last ACCEPT, on chemistry that is identical across the series, purely because
# 3.05 lands on the threshold. abs(3.05 - 3) is 0.05000000000000071 in binary,
# so the comparison is decided by floating-point representation rather than by
# anything about the structure.
#
# Moving the boundary to 0.045 puts it between the values depositors use, and
# the comparison is now explicit about which side is which. Callers that need
# to see a borderline case should read valence_integer_distance rather than
# rely on the verdict alone.
VALENCE_INTEGER_TOL = 0.045


def valence_integer_distance(valence):
    """How far a valence sits from the nearest integer. None if unstated."""
    if valence is None:
        return None
    try:
        v = float(valence)
    except (TypeError, ValueError):
        return None
    if v != v:
        return None
    return abs(v - round(v))


def is_valence_integer(valence, tolerance=VALENCE_INTEGER_TOL):
    """
    Is a valence effectively integer?

    3.99 is clearly Mn(4+) rounded and counts as integer; 3.313 is a genuine
    charge-balance average and does not. See VALENCE_INTEGER_TOL for why the
    boundary is 0.045 rather than 0.05.
    """
    d = valence_integer_distance(valence)
    if d is None:
        return True  # No valence stated = no non-integer flag
    return d <= tolerance


# RETIRED at v2.0.11: assign_site_integrity_tier, assign_structure_tier and
# assess_flag_severity.
#
# They derived a site's integrity tier FROM its bond valence sum, expunging on
# EXPUNGE_BVS_HIGH = 10 v.u. and EXPUNGE_BVS_LOW = 0.2. That is the wrong way
# round: the sum is computed from the coordinates, so judging the coordinates
# by the sum asks a broken structure to grade itself. It also cannot see the
# fault class that dominates every family with positional freedom. ICSD 85235
# (TbCrO3) has two oxygens 2.191 A apart against a family median of 2.750, a
# GII of 0.0047 and a chromium octahedron 0.0004 A from the measured mean; the
# old rule tiered it ACCEPT, the highest confidence in its batch.
#
# The tier now comes from the PGRM integrity gates, which ask in order whether
# the file can be read, whether the atoms can be where it says, whether the
# reference data exists, and whether the depositor's model is unambiguous.
# Geometry decides severity and chemistry is reported only for structures that
# survive it. See classify_pgrm.
#
# whatif_analysis and PLAUSIBLE_OXIDATION_STATES are KEPT. Asking which
# integer oxidation state the observed geometry supports is a question about
# the assignment, not a verdict on the structure, and PGRM has no equivalent.

PLAUSIBLE_OXIDATION_STATES = {
    'Cr': [3],
    'Mn': [2, 3, 4],
    'Fe': [2, 3],
    'Co': [2, 3, 4],
    'Ni': [2, 3],
    'Ti': [3, 4],
    'V':  [3, 4, 5],
    'Cu': [1, 2],
    'Mg': [2],
    'Al': [3],
    'Zn': [2],
    'Sc': [3],
    'Zr': [4],
    'Nb': [3, 5],
    'Mo': [4, 6],
    'Ru': [3, 4],
    'Rh': [3, 4],
    'Ir': [3, 4],
    'Ce': [3, 4],
    'Eu': [2, 3],
}


def whatif_analysis(element, bvs_value, expected_valence, anion='O'):
    """
    For FLAG-tier sites, test alternative integer oxidation states and report
    which gives the best BVS match.
    
    This implements the pragmatic position: even if we can't verify the
    depositor's valence partitioning, we can ask "does any plausible integer
    state produce a reasonable BVS?" If yes, the structure is probably fine
    and the CIF metadata just needs correcting. If no, the problem may be
    in the coordinates, not just the valence assignment.
    
    Args:
        element: Element symbol
        bvs_value: Calculated BVS (float)
        expected_valence: CIF-assigned expected valence (float)
        anion: Anion element (default 'O')
    
    Returns:
        dict with:
            best_integer_valence: int or None
            best_integer_deviation: float or None (percentage)
            candidates_tested: str (e.g. "Mn: 2(+108%), 3(+5%), 4(-28%)")
            depositor_deviation: float or None (percentage for CIF-assigned valence)
    """
    result = {
        'best_integer_valence': None,
        'best_integer_deviation': None,
        'candidates_tested': None,
        'depositor_deviation': None,
    }
    
    if bvs_value is None or element is None:
        return result
    
    # Depositor's deviation
    if expected_valence is not None and expected_valence != 0:
        result['depositor_deviation'] = round(
            (bvs_value - abs(expected_valence)) / abs(expected_valence) * 100, 1
        )
    
    candidates = PLAUSIBLE_OXIDATION_STATES.get(element, [])
    if not candidates:
        return result
    
    tested = []
    best_val = None
    best_dev = None
    
    for candidate_v in candidates:
        if candidate_v == 0:
            continue
        dev_pct = (bvs_value - candidate_v) / candidate_v * 100
        tested.append(f"{candidate_v}({dev_pct:+.0f}%)")
        
        abs_dev = abs(dev_pct)
        if best_dev is None or abs_dev < best_dev:
            best_dev = abs_dev
            best_val = candidate_v
    
    result['best_integer_valence'] = best_val
    result['best_integer_deviation'] = round(best_dev, 1) if best_dev is not None else None
    result['candidates_tested'] = f"{element}: {', '.join(tested)}" if tested else None
    
    return result


def _total_occupancy_at_position(site_label, site_occupancies_data, tol=0.001):
    """
    Total occupancy of every species sharing one crystallographic position.

    Used to express the minority/majority test as a fraction of how full the
    position actually is, rather than as an absolute occupancy.
    """
    this = site_occupancies_data.get(site_label, {})
    f0 = (this.get('fract_x'), this.get('fract_y'), this.get('fract_z'))
    if any(v is None for v in f0):
        return None
    total = 0.0
    for _lbl, _d in site_occupancies_data.items():
        f1 = (_d.get('fract_x'), _d.get('fract_y'), _d.get('fract_z'))
        if any(v is None for v in f1):
            continue
        if all(abs(a - b) < tol for a, b in zip(f0, f1)):
            total += _d.get('occupancy', 1.0)
    return total


def check_minority_species(site_label, per_atom_valences, site_occupancies_data):
    """
    Check if a site label represents a minority species on a majority-occupied site.
    
    A minority species is defined as: occupancy < 0.05 AND another element on the
    same Wyckoff position has occupancy > 0.90.
    
    This requires matching sites by fractional coordinates (since Wyckoff symbols
    are not always reliably present in ICSD CIFs).
    
    Args:
        site_label: The site label to check (e.g. 'Ti1')
        per_atom_valences: dict from extract_per_atom_valences()
        site_occupancies_data: dict from extract_site_occupancies()
    
    Returns:
        bool: True if this is a minority species on a majority-occupied site
    """
    if site_label not in per_atom_valences:
        return False
    
    this_occ = per_atom_valences[site_label].get('occupancy', 1.0)

    # v1.8.3 (B12): judge "minority" relative to how full the position is,
    # not against an absolute occupancy. In a layered oxide the alkali layer
    # is typically 0.6-0.7 occupied, so no species there ever reaches the
    # absolute 0.90 majority threshold and the rule could never fire - which
    # left genuinely impossible trace sites sitting in the ACCEPT tier.
    _total_occ = _total_occupancy_at_position(site_label, site_occupancies_data)
    if _total_occ and _total_occ > 0:
        if ge(this_occ / _total_occ, MINORITY_OCC_THRESHOLD):
            return False
    elif ge(this_occ, MINORITY_OCC_THRESHOLD):
        return False
    
    # This site has low occupancy. Check if another element at the same
    # fractional coordinates has high occupancy.
    this_site_data = site_occupancies_data.get(site_label, {})
    this_frac = (this_site_data.get('fract_x'), this_site_data.get('fract_y'), this_site_data.get('fract_z'))
    
    if any(v is None for v in this_frac):
        return False
    
    for other_label, other_data in site_occupancies_data.items():
        if other_label == site_label:
            continue
        other_frac = (other_data.get('fract_x'), other_data.get('fract_y'), other_data.get('fract_z'))
        if any(v is None for v in other_frac):
            continue
        
        # Match by fractional coordinates (within tolerance for rounding)
        if all(abs(a - b) < 0.001 for a, b in zip(this_frac, other_frac)):
            other_occ = other_data.get('occupancy', 1.0)
            _other_rel = (other_occ / _total_occ) if (_total_occ and _total_occ > 0) else other_occ
            if gt(_other_rel, MAJORITY_OCC_THRESHOLD):
                # Anti-site check: if this minority element is a majority
                # species on a DIFFERENT crystallographic site, this is
                # anti-site exchange (e.g. Na/Ti swapping between layers),
                # not a trace dopant. Anti-site disorder produces large but
                # physically meaningful BVS deviations — do not EXPUNGE.
                this_element = per_atom_valences.get(site_label, {}).get('element')
                is_antisite = False
                if this_element:
                    for check_label, check_pav in per_atom_valences.items():
                        if check_label == site_label:
                            continue
                        if check_pav.get('element') == this_element:
                            if gt(check_pav.get('occupancy', 0), MAJORITY_OCC_THRESHOLD):
                                is_antisite = True
                                break
                if not is_antisite:
                    return True
                # else: anti-site exchange — fall through to return False
    
    return False


# An anti-site is a minority ON ITS POSITION, not a trace dopant.
# MINORITY_OCC_THRESHOLD is 0.05 because it exists to catch trace species for
# the EXPUNGE rule; reusing it here found 3 of the 18 anti-site structures in
# the ICSD perovskite set, because the relative occupancies run from 2.5% to
# 29%. Fe at 25% on the La site of ICSD 7796 is unambiguously anti-site
# exchange and takes that structure to a per-species GII of 1.30.
#
# The discrimination is done by the second condition, not this one: the same
# element must be a MAJORITY on another crystallographic site. A 30% Sr on the
# La site of a substituted manganite is not anti-site exchange, because Sr is
# nowhere else in the structure, and it is correctly left alone.
ANTISITE_MINORITY_MAX = 0.50


def antisite_exchange(per_atom_valences, site_occupancies_data,
                      minority=None, majority=None):
    """
    Species that are a minority on one position and a majority on another.

    v2.0.2. check_minority_species already computes this internally, in order
    to decline to EXPUNGE it: anti-site exchange produces large but physically
    meaningful bond valence deviations, so the site is retained. That decision
    is right, but the fact was then thrown away, and it is the fact that tells
    a reader which instability index to quote.

    A 2.4% Mn on a 94.4% Tb site is asked what its sum would be in a cavity
    the majority sizes, and answers 0.83 v.u. against a formal 3. Weighted
    equally with the 97% it sits among, that single phantom takes the
    per-species GII of ICSD 215426 to 1.32; WAGII gives 0.28 and the
    site-averaged value 0.16. Across the perovskite set the two diverge by a
    factor of 3.6 below 5% minority occupancy, 2.3 between 5 and 15%, and 1.6
    between 15 and 30%, while agreeing to three decimals on all 916 ordered
    structures.

    Returns (n_antisite_sites, description) so the structure can carry the
    reason alongside the number.
    """
    minority = ANTISITE_MINORITY_MAX if minority is None else minority
    majority = MAJORITY_OCC_THRESHOLD if majority is None else majority
    found = []
    for label, data in (site_occupancies_data or {}).items():
        pav = (per_atom_valences or {}).get(label)
        if not pav:
            continue
        el = pav.get('element')
        occ = pav.get('occupancy', 1.0)
        if el is None or occ is None:
            continue
        total = _total_occupancy_at_position(label, site_occupancies_data)
        rel = (occ / total) if (total and total > 0) else occ
        # Not the dominant species on its own position.
        if ge(rel, minority):
            continue
        # minority here; is the same element a majority somewhere else?
        for other, opav in (per_atom_valences or {}).items():
            if other == label or opav.get('element') != el:
                continue
            if gt(opav.get('occupancy', 0), majority):
                found.append(f'{el} {100 * rel:.1f}% of {label}')
                break
    return len(found), ('; '.join(found) if found else None)


def check_mixed_tm_site(site_label, per_atom_valences, site_occupancies_data):
    """
    Check if a site label is part of a mixed TM occupancy site, and whether
    any TM on that site has a non-integer valence.
    
    Sites are matched by fractional coordinates.
    
    Args:
        site_label: The site label to check
        per_atom_valences: dict from extract_per_atom_valences()
        site_occupancies_data: dict from extract_site_occupancies()
    
    Returns:
        tuple: (is_mixed_tm, has_noninteger_tm)
    """
    if site_label not in per_atom_valences:
        return False, False
    
    this_elem = per_atom_valences[site_label].get('element')
    if not is_transition_metal(this_elem):
        # Not a TM site -- cannot be FLAG
        return False, False
    
    this_site_data = site_occupancies_data.get(site_label, {})
    this_frac = (this_site_data.get('fract_x'), this_site_data.get('fract_y'), this_site_data.get('fract_z'))
    
    if any(v is None for v in this_frac):
        return False, False
    
    # Find all other elements at the same position
    tm_labels_at_position = [site_label]
    
    for other_label, other_data in site_occupancies_data.items():
        if other_label == site_label:
            continue
        other_frac = (other_data.get('fract_x'), other_data.get('fract_y'), other_data.get('fract_z'))
        if any(v is None for v in other_frac):
            continue
        
        if all(abs(a - b) < 0.001 for a, b in zip(this_frac, other_frac)):
            other_elem = other_data.get('element')
            if is_transition_metal(other_elem):
                tm_labels_at_position.append(other_label)
    
    is_mixed_tm = len(tm_labels_at_position) > 1
    
    if not is_mixed_tm:
        return False, False
    
    # Check if any TM on this shared position has non-integer valence
    has_noninteger = False
    for label in tm_labels_at_position:
        if label in per_atom_valences:
            val = per_atom_valences[label].get('explicit_valence')
            if val is not None and not is_valence_integer(val):
                has_noninteger = True
                break
    
    return True, has_noninteger

def flag_same_source_entries(df):
    """
    Dataset-level pass: identify multiple ICSD entries deposited from the
    same source paper.
    
    When a single paper produces multiple ICSD entries (e.g. different
    refinement methods, different samples, phase-separated components),
    the database stores them with equal apparent authority. This function
    groups entries by source paper and flags groups with >1 structure,
    so downstream analysis can identify redundant or subsidiary depositions.
    
    Adds columns:
        source_group: identifier string (first_author|year|journal)
        source_group_size: number of ICSD entries from that paper
        source_group_codes: comma-separated list of ICSD codes in the group
        flag_multi_entry_source: True if >1 entry from the same paper
    
    Args:
        df: pandas DataFrame from Eir batch processing (site-level rows)
    
    Returns:
        df: DataFrame with new columns added
    """
    # Build structure-level lookup: one row per ICSD code
    struct_cols = ['icsd_code', 'authors', 'year', 'journal']
    available = [c for c in struct_cols if c in df.columns]
    if 'icsd_code' not in available or 'authors' not in available:
        # Not enough metadata to group — add empty columns and return
        df['source_group'] = None
        df['source_group_size'] = 1
        df['source_group_codes'] = df['icsd_code'] if 'icsd_code' in df.columns else None
        df['flag_multi_entry_source'] = False
        return df
    
    other_cols = [c for c in available if c != 'icsd_code']
    struct_df = df.groupby('icsd_code')[other_cols].first().reset_index()
    
    # Build source group key from first author surname + year + journal
    def make_source_key(row):
        authors = row.get('authors', '') or ''
        year = row.get('year', '') or ''
        journal = row.get('journal', '') or ''
        # Extract first author surname (before first comma or space)
        first_author = authors.split(',')[0].split(' ')[0].strip().lower()
        # Normalise journal (lowercase, strip whitespace)
        journal_norm = journal.strip().lower()[:40]  # truncate for matching
        return f"{first_author}|{year}|{journal_norm}"
    
    struct_df['source_group'] = struct_df.apply(make_source_key, axis=1)
    
    # Count entries per source group
    group_counts = struct_df.groupby('source_group')['icsd_code'].agg(['count', list]).reset_index()
    group_counts.columns = ['source_group', 'source_group_size', 'group_codes_list']
    group_counts['source_group_codes'] = group_counts['group_codes_list'].apply(
        lambda x: ','.join(str(c) for c in sorted(x))
    )
    group_counts['flag_multi_entry_source'] = group_counts['source_group_size'] > 1
    group_counts = group_counts.drop(columns=['group_codes_list'])
    
    # Merge back to structure level, then to site level
    struct_df = struct_df.merge(group_counts, on='source_group', how='left')
    
    # Map to original df via icsd_code
    merge_cols = ['icsd_code', 'source_group', 'source_group_size', 
                  'source_group_codes', 'flag_multi_entry_source']
    df = df.merge(struct_df[merge_cols], on='icsd_code', how='left')
    
    return df

def extract_cif_oxidation_states(cif_path):
    """
    Extract oxidation states from CIF file's _atom_type_oxidation_number loop.

    ICSD CIFs typically contain this information in a loop like:
    loop_
    _atom_type_symbol
    _atom_type_oxidation_number
    Mg2+ 2
    Al3+ 3
    O2- -2

    For mixed-valence compounds, ICSD may give a non-integer value, e.g.:
    Mn4+ 3.5     (LiMn2O4 — symbol rounded up, true value in second column)

    v1.6 change: non-integer values are preserved as floats rather than truncated.
    A third return value (type_symbol -> oxidation_number map) is added so that
    extract_per_atom_valences can override symbol-parsed integers with the true
    fractional value.

    Returns:
        tuple: (oxidation_states_dict, quality, error_message, type_map)
        - oxidation_states_dict: {element: charge} e.g., {'Li': 1, 'Mn': 3.5, 'O': -2}
        - quality: 'explicit' | 'inferred' | 'missing' | 'ambiguous'
        - error_message: None or description of issue
        - type_map: {type_symbol: oxidation_number} e.g., {'Mn4+': 3.5}
    """
    oxidation_states = {}
    type_map = {}
    quality = 'missing'
    error_message = None

    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except Exception as e:
        return {}, 'missing', f"Could not read CIF: {e}", {}

    in_atom_type_loop = False
    columns = []
    symbol_col = None
    oxidation_col = None

    for line in lines:
        line_stripped = line.strip()

        if line_stripped == 'loop_':
            in_atom_type_loop = False
            columns = []
            symbol_col = None
            oxidation_col = None
            continue

        if line_stripped.startswith('_atom_type_'):
            columns.append(line_stripped)
            col_idx = len(columns) - 1
            if line_stripped == '_atom_type_symbol':
                symbol_col = col_idx
            elif line_stripped == '_atom_type_oxidation_number':
                oxidation_col = col_idx
                in_atom_type_loop = True
            continue

        if in_atom_type_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_type_loop = False
                continue

            parts = line_stripped.split()

            if symbol_col is not None and symbol_col < len(parts):
                symbol_raw = parts[symbol_col]
                element = re.sub(r'[0-9+-]+', '', symbol_raw)
                element = element.capitalize() if len(element) > 1 else element.upper()

                if oxidation_col is not None and oxidation_col < len(parts):
                    ox_str = parts[oxidation_col]
                    if ox_str not in ['?', '.']:
                        try:
                            # v1.6: preserve fractional values (e.g. 3.5 for Mn in LiMn2O4)
                            ox_float = float(ox_str)
                            if ox_float == int(ox_float):
                                oxidation = int(ox_float)
                            else:
                                oxidation = ox_float

                            type_map[symbol_raw] = oxidation

                            if element in oxidation_states:
                                if oxidation_states[element] != oxidation:
                                    quality = 'ambiguous'
                                    error_message = f"Multiple oxidation states for {element}"
                            else:
                                oxidation_states[element] = oxidation
                                if quality != 'ambiguous':
                                    quality = 'explicit'
                        except ValueError:
                            pass

    # Fallback: infer from atom_type_symbol pattern if loop values missing
    if not oxidation_states:
        for line in lines:
            line_stripped = line.strip()
            matches = re.findall(r'([A-Z][a-z]?)(\d+)([+-])', line_stripped)
            for elem, charge, sign in matches:
                charge_val = int(charge) * (1 if sign == '+' else -1)
                if elem not in oxidation_states:
                    oxidation_states[elem] = charge_val
                    quality = 'inferred'

    if not oxidation_states:
        quality = 'missing'
        error_message = "No oxidation state information found in CIF"

    return oxidation_states, quality, error_message, type_map


def parse_chemical_formula(formula):
    """
    Parse chemical formula string into element counts.
    
    Handles formats like:
    - "Mg1 Al2 O4" (space-separated)
    - "MgAl2O4" (no spaces)
    - "Fe3O4"
    - "Li0.5Mn1.5O4"
    
    Returns:
        dict: {element: count} e.g., {'Mg': 1.0, 'Al': 2.0, 'O': 4.0}
    """
    if not formula:
        return {}
    
    # Clean up formula
    formula = formula.strip().replace("'", "").replace('"', '')
    
    # Pattern: Element followed by optional number (including decimals)
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    
    elements = {}
    for match in re.finditer(pattern, formula):
        element = match.group(1)
        count_str = match.group(2)
        
        if element:  # Skip empty matches
            count = float(count_str) if count_str else 1.0
            elements[element] = elements.get(element, 0) + count
    
    return elements


def detect_mixed_valence_requirement(formula, oxidation_states):
    """
    Detect if a formula requires mixed valence to achieve charge balance.
    
    For example:
    - Fe3O4 with Fe3+ and O2-: 3(+3) + 4(-2) = +1 != 0 -> needs Fe2+/Fe3+ mix
    - MgAl2O4 with Mg2+, Al3+, O2-: +2 + 2(+3) + 4(-2) = 0 -> OK
    
    Returns:
        dict with:
        - requires_mixed_valence: bool
        - charge_balance_error: float (should be 0 for valid structure)
        - problematic_elements: list of elements that might need mixed valence
        - suggested_average_charges: dict of {element: required_average_charge}
    """
    result = {
        'requires_mixed_valence': False,
        'charge_balance_error': 0.0,
        'problematic_elements': [],
        'suggested_average_charges': {}
    }
    
    elements = parse_chemical_formula(formula)
    if not elements or not oxidation_states:
        return result
    
    # Separate cations and anions
    anion_elements = {e for e in elements if e in KNOWN_ANIONS}
    cation_elements = {e for e in elements if e not in KNOWN_ANIONS}
    
    # Calculate total anion charge
    anion_charge = sum(
        elements.get(e, 0) * KNOWN_ANIONS.get(e, 0)
        for e in anion_elements
    )
    
    # Calculate total cation charge based on provided oxidation states
    cation_charge = 0
    for elem in cation_elements:
        if elem in oxidation_states:
            cation_charge += elements.get(elem, 0) * oxidation_states[elem]
    
    # Check charge balance
    total_charge = cation_charge + anion_charge

    # v1.8.8: round before storing and comparing. The inputs are occupancies
    # quoted to two or three decimals and valences to three, so digits beyond
    # the sixth are float noise, and reporting an error of
    # 0.010000000000000675 is worse than useless - it invites the reader to
    # believe a precision that is not there.
    total_charge = round(total_charge, 6)
    result['charge_balance_error'] = total_charge

    if gt(abs(total_charge), CHARGE_BALANCE_TOLERANCE):  # Not balanced
        result['requires_mixed_valence'] = True
        
        # Identify which elements might need mixed valence
        for elem in cation_elements:
            if elem in VARIABLE_VALENCE_ELEMENTS:
                result['problematic_elements'].append(elem)
                
                # Calculate what average charge would be needed
                # If we remove this element's contribution and solve for it
                other_cation_charge = sum(
                    elements.get(e, 0) * oxidation_states.get(e, 0)
                    for e in cation_elements if e != elem
                )
                
                # Required charge from this element to balance
                required_charge = -anion_charge - other_cation_charge
                elem_count = elements.get(elem, 1)
                if elem_count > 0:
                    required_avg = required_charge / elem_count
                    result['suggested_average_charges'][elem] = round(required_avg, 2)
    
    return result


def flag_oxidation_state_issues(cif_path, formula):
    """
    Comprehensive check for oxidation state issues (Type 7 errors).
    
    Returns:
        dict with:
        - has_type7_error: bool
        - error_code: str or None
        - error_message: str or None
        - oxidation_states: dict from CIF
        - quality: str
        - mixed_valence_info: dict
    """
    result = {
        'has_type7_error': False,
        'error_code': None,
        'error_message': None,
        'oxidation_states': {},
        'quality': 'unknown',
        'mixed_valence_info': {}
    }
    
    # Extract oxidation states from CIF
    ox_states, quality, error_msg, _ = extract_cif_oxidation_states(cif_path)
    result['oxidation_states'] = ox_states
    result['quality'] = quality
    
    # Check for missing oxidation states
    if quality == 'missing':
        result['has_type7_error'] = True
        result['error_code'] = 'TYPE7_NO_OXIDATION_STATES'
        result['error_message'] = error_msg or "CIF lacks oxidation state information"
        return result
    
    # Check for ambiguous (multiple) oxidation states
    if quality == 'ambiguous':
        result['has_type7_error'] = True
        result['error_code'] = 'TYPE7_EXPLICIT_MIXED_VALENCE'
        result['error_message'] = error_msg or "CIF specifies multiple oxidation states for same element"
        return result
    
    # Check if formula requires mixed valence for charge balance
    mixed_valence_info = detect_mixed_valence_requirement(formula, ox_states)
    result['mixed_valence_info'] = mixed_valence_info
    
    if mixed_valence_info['requires_mixed_valence']:
        result['has_type7_error'] = True
        
        if abs(mixed_valence_info['charge_balance_error']) > 0.5:
            result['error_code'] = 'TYPE7_CHARGE_IMBALANCE'
            result['error_message'] = (
                f"Charge imbalance of {mixed_valence_info['charge_balance_error']:+.1f} "
                f"cannot be resolved with given oxidation states"
            )
        else:
            result['error_code'] = 'TYPE7_REQUIRES_MIXED_VALENCE'
            problematic = ', '.join(mixed_valence_info['problematic_elements'])
            result['error_message'] = (
                f"Charge balance requires mixed valence for: {problematic}"
            )
    
    return result

def flag_all_error_types(cif_path, formula, site_data):
    """
    Flag all detectable error types (2-7) for a structure.
    Based on taxonomy from Sherwood (2026) Dalton Trans.
    
    Types:
        2: Missing BVS parameters
        3: Vacancy disorder / nonstoichiometric
        4: Heterovalent disorder ΔV ≥ 2
        5: Homovalent / site disorder
        6A: Pre-modern techniques (pre-1980)
        7: Mixed valence (handled separately)
    
    Returns:
        dict with flag_type2 through flag_type6 and error descriptions
    """
    flags = {
        'flag_type2': False,
        'flag_type3': False,
        'flag_type4': False,
        'flag_type5': False,
        'flag_type6': False,
        'flag_type8': False,
        'error_types_detected': [],
        'error_type_details': None,
    }
    
    details = []
    
    # ========================================================================
    # TYPE 2: Missing BVS parameters
    # ========================================================================
    if site_data:
        for site in site_data:
            if site.get('bvs') is None and site.get('element') is not None:
                elem = site.get('element')
                valence = site.get('valence')
                if elem and valence:
                    # Check if parameters exist
                    R0, B = get_bv_parameters_fractional(elem, valence, 'O')
                    if R0 is None:
                        flags['flag_type2'] = True
                        if 'TYPE2' not in flags['error_types_detected']:
                            flags['error_types_detected'].append('TYPE2')
                            details.append(f"T2:No BVS params {elem}{valence:+g}-O")
    
    # ========================================================================
    # TYPE 3: Vacancy disorder / nonstoichiometric
    # ========================================================================
    if formula:
        try:
            parsed = parse_chemical_formula(formula)
            for elem, count in parsed.items():
                if count != int(count) and abs(count - round(count)) > 0.05:
                    flags['flag_type3'] = True
                    if 'TYPE3' not in flags['error_types_detected']:
                        flags['error_types_detected'].append('TYPE3')
                        details.append(f"T3:Nonstoich {elem}={count:.2f}")
                    break
        except (ValueError, TypeError, AttributeError, KeyError) as _e:
            # v2.0.11: a crash here used to drop the flag and leave the
            # structure looking clean. Under PGRM that silently downgrades
            # M1 to ACCEPT.
            flags['error_types_detected'].append('TYPE3_UNTESTED')
            details.append(f"T3:not tested ({type(_e).__name__})")
    
    # ========================================================================
    # TYPE 5: Significant site disorder (sub-unity total occupancy = vacancies)
    # ========================================================================
    if site_data:
        for site in site_data:
            occ_str = site.get('occupancy_string', '')
            if occ_str and occ_str != 'unknown':
                try:
                    # Parse "Fe:0.800,Ni:0.200" format → sum individual occupancies
                    total_occ = sum(
                        float(part.split(':')[1]) 
                        for part in occ_str.split(',') 
                        if ':' in part
                    )
                    if lt(total_occ, 0.90):
                        flags['flag_type5'] = True
                        if 'TYPE5' not in flags['error_types_detected']:
                            flags['error_types_detected'].append('TYPE5')
                            elem = site.get('element', '?')
                            details.append(f"T5:Disorder {elem} total_occ={total_occ:.2f}")
                        break
                except (ValueError, IndexError):
                    pass
    
    # ========================================================================
    # TYPE 6A: Pre-modern techniques (pre-1980)
    # ========================================================================
    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
        
        year = None
        # Try multiple year fields
        for pattern in [r'_citation_year\s+(\d{4})', 
                        r'_journal_year\s+(\d{4})',
                        r"primary\s+'[^']+'\s+(\d{4})"]:
            match = re.search(pattern, content)
            if match:
                year = int(match.group(1))
                break
        
        if year and year < 1980:
            flags['flag_type6'] = True
            if 'TYPE6' not in flags['error_types_detected']:
                flags['error_types_detected'].append('TYPE6')
                details.append(f"T6:Pre-modern ({year})")
    except (ValueError, TypeError, AttributeError, IndexError) as _e:
        flags['error_types_detected'].append('TYPE6_UNTESTED')
        details.append(f"T6:not tested ({type(_e).__name__})")
    
    # ========================================================================
    # TYPE 8: Shannon radii violation (>7% deviation for pure-occupancy sites)
    # Literature basis: <5% = accurate (ML radii papers), 7% catches errors
    # while allowing natural variation from polyhedral distortion
    # ========================================================================
    SHANNON_DEVIATION_THRESHOLD = 7.0  # percent
    
    if site_data:
        type8_sites = []
        for site in site_data:
            # Only flag pure-occupancy sites (single cation, no mixing)
            has_mixed = site.get('has_mixed_occupancy', False)
            n_elements = site.get('n_elements_at_site', 1)
            
            if not has_mixed and n_elements == 1:
                shannon_diff = site.get('shannon_diff')
                if shannon_diff is not None and abs(shannon_diff) > SHANNON_DEVIATION_THRESHOLD:
                    elem = site.get('element', '?')
                    type8_sites.append(f"{elem}:{shannon_diff:+.1f}%")
        
        if type8_sites:
            flags['flag_type8'] = True
            if 'TYPE8' not in flags['error_types_detected']:
                flags['error_types_detected'].append('TYPE8')
                details.append(f"T8:Shannon violation {','.join(type8_sites)}")
    
    # ========================================================================
    # Compile results
    # ========================================================================
    flags['error_types_detected'] = ','.join(flags['error_types_detected']) if flags['error_types_detected'] else None
    flags['error_type_details'] = '; '.join(details) if details else None
    
    return flags

def parse_atom_type_with_valence(atom_type_raw):
    """
    Parse atom type symbol to extract element AND explicit valence.
    
    Examples:
        "Fe3+" -> ('Fe', 3)
        "Fe2+" -> ('Fe', 2)
        "O2-"  -> ('O', -2)
        "Mn"   -> ('Mn', None)  # No explicit valence
    
    Returns:
        tuple: (element_symbol, explicit_valence or None)
    """
    if not atom_type_raw:
        return None, None
    
    atom_type_raw = atom_type_raw.strip()
    
    # Pattern: Element + number + charge sign (Fe3+, O2-, Mn2+)
    match = re.match(r'^([A-Z][a-z]?)(\d+)?([+-])$', atom_type_raw)
    
    if match:
        element = match.group(1)
        number = match.group(2)
        sign = match.group(3)
        
        if number and sign:
            valence = int(number) if sign == '+' else -int(number)
            return element, valence
        elif sign:
            valence = 1 if sign == '+' else -1
            return element, valence
    
    # Alternative: charge before number (Fe+3, O-2)
    match = re.match(r'^([A-Z][a-z]?)([+-])(\d+)$', atom_type_raw)
    if match:
        element = match.group(1)
        sign = match.group(2)
        number = match.group(3)
        valence = int(number) if sign == '+' else -int(number)
        return element, valence
    
    # No explicit valence - just extract element
    element = re.sub(r'[0-9+-]+$', '', atom_type_raw)
    element = element.capitalize() if len(element) > 1 else element.upper()
    
    return element, None


def extract_per_atom_valences(cif_path):
    """
    Extract explicit valences from each atom row in the CIF.

    For mixed-valence CIFs like jacobsite:
        Fe2 Fe2+ 16 d 0.625 0.625 0.625 . 0.375
        Fe3 Fe3+ 16 d 0.625 0.625 0.625 . 0.56

    v1.6 change: when the _atom_type loop contains a more precise (fractional)
    oxidation number than the type symbol suggests — as in LiMn2O4 where the
    symbol "Mn4+" labels a value of 3.5 — the fractional value is preferred.

    Returns:
        dict: {site_label: {'element': str, 'explicit_valence': int|float|None,
                           'occupancy': float, 'type_symbol': str}}
    """
    result = {}

    # v1.6: Read the _atom_type loop first so we have the true valence map
    try:
        _, _, _, type_map = extract_cif_oxidation_states(cif_path)
    except Exception:
        type_map = {}

    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
    except (OSError, UnicodeDecodeError) as _e:
        # v2.0.11: recorded, not swallowed. A file that could not be read is
        # not a file with nothing in it.
        _note_read_failure(cif_path, 'extract_per_atom_valences', _e)
        return result

    lines = content.split('\n')

    in_atom_loop = False
    columns = []
    label_col = type_col = occ_col = None

    for line in lines:
        line_stripped = line.strip()

        if line_stripped.startswith('loop_'):
            in_atom_loop = False
            columns = []
            label_col = type_col = occ_col = None
            continue

        if line_stripped.startswith('_atom_site_'):
            col_idx = len(columns)
            columns.append(line_stripped)

            if line_stripped == '_atom_site_label':
                label_col = col_idx
            elif line_stripped == '_atom_site_type_symbol':
                type_col = col_idx
            elif line_stripped == '_atom_site_occupancy':
                occ_col = col_idx

            in_atom_loop = True
            continue

        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue

            parts = line_stripped.split()

            label = parts[label_col] if label_col is not None and label_col < len(parts) else None

            type_symbol = None
            element, explicit_valence = None, None
            if type_col is not None and type_col < len(parts):
                type_symbol = parts[type_col]
                element, explicit_valence = parse_atom_type_with_valence(type_symbol)

                # v1.6: if the _atom_type loop has a more precise value, use it.
                # This is the key fix for LiMn2O4 "Mn4+" where the true value is 3.5.
                if type_symbol in type_map:
                    loop_valence = type_map[type_symbol]
                    # Prefer the loop value whenever it differs from the symbol-parsed one
                    # (handles both fractional overrides and any other mismatches)
                    if explicit_valence is None or loop_valence != explicit_valence:
                        explicit_valence = loop_valence

            occupancy = 1.0
            if occ_col is not None and occ_col < len(parts):
                try:
                    occ_str = parts[occ_col]
                    if occ_str not in ['?', '.']:
                        occupancy = float(occ_str.split('(')[0])
                except (ValueError, TypeError, IndexError):
                    pass

            if label and element:
                result[label] = {
                    'element': element,
                    'explicit_valence': explicit_valence,
                    'occupancy': occupancy,
                    'type_symbol': type_symbol,
                }

    return result

def get_oxidation_states_for_analysis(cif_path, formula, config_states=None):
    """
    Get oxidation states with priority: config > CIF explicit > CIF inferred > None.
    
    This is the main entry point for oxidation state determination in Eir 1.4.
    
    Args:
        cif_path: Path to CIF file
        formula: Chemical formula string
        config_states: Dict of {element: charge} from config.json (highest priority)
    
    Returns:
        tuple: (oxidation_states, source, type7_flag_result)
        - oxidation_states: dict or None
        - source: 'config' | 'cif_explicit' | 'cif_inferred' | None
        - type7_flag_result: dict from flag_oxidation_state_issues
    """
    # Always check for Type 7 issues
    type7_result = flag_oxidation_state_issues(cif_path, formula)
    
    # Priority 1: Config file (user override)
    if config_states:
        return config_states, 'config', type7_result
    
    # Priority 2: CIF explicit oxidation states
    if type7_result['quality'] == 'explicit':
        return type7_result['oxidation_states'], 'cif_explicit', type7_result
    
    # Priority 3: CIF inferred oxidation states
    if type7_result['quality'] == 'inferred':
        return type7_result['oxidation_states'], 'cif_inferred', type7_result
    
    # No oxidation states available
    return None, None, type7_result


# ============================================================================
# ENHANCED STRUCTURAL PARAMETER EXTRACTION (NEW IN v1.4)
# ============================================================================

def _read_it_number(cif_path):
    """Space-group IT number from a CIF, or None."""
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='replace') as f:
            txt = f.read()
    except Exception:
        return None
    m = (re.search(r'_space_group_IT_number\s+(\d+)', txt) or
         re.search(r'_symmetry_Int_Tables_number\s+(\d+)', txt))
    return int(m.group(1)) if m else None


def extract_anion_parameter(cif_path, formula=None):
    """
    Extract anion positional parameters from a spinel CIF file.
    
    v1.7.3: Generalised anion parameter framework for all spinel space groups.
    
    Cubic Fd-3m:     32e (x, x, x)        -> 1 free parameter (u)
    Ordered P4332:   24e (x, 0, 0) + 8c   -> 2 sites, 2 parameters
    Tetragonal I41/amd: 16h (0, y, z)      -> 2 free parameters
    Orthorhombic Fddd:  32h (x, y, z)      -> 3 free parameters
    Orthorhombic Imma:  8h (0, y, z)       -> 2 free parameters
    
    New metrics:
        u_eff:   Effective anion parameter (comparable across all space groups)
        delta_u: Anion sublattice anisotropy (0 for cubic, >0 for distorted)
    
    For Fd-3m, detects origin choice and standardises to origin 2.
    
    Returns:
        dict with: u_parameter, u_parameter_raw, u_deviation, origin_choice,
                   anion_x, anion_y, anion_z, anion_wyckoff, anion_multiplicity,
                   u_eff, delta_u, anion_site_count, anion_x_2/y_2/z_2/wyckoff_2
    """
    result = {
        'u_parameter': None,           # Cubic Fd-3m only: standardised to origin 2
        'u_parameter_raw': None,       # As extracted from CIF (before origin correction)
        'pyrochlore_48f_x': None,      # Fd-3m 48f free coordinate (pyrochlore)
        'pyrochlore_fluorite_limit': None,  # True at x = 3/8, where ordering vanishes
        'u_deviation': None,           # Deviation from ideal (0.25)
        'origin_choice': None,         # 1 or 2 (detected, cubic only)
        'anion_x': None,              # Primary anion site fractional x
        'anion_y': None,              # Primary anion site fractional y
        'anion_z': None,              # Primary anion site fractional z
        'anion_wyckoff': None,         # Full Wyckoff symbol e.g. "32e", "16h"
        'anion_multiplicity': None,    # Just the number e.g. 32
        'anion_wyckoff_letter': None,  # Just the letter e.g. "e"
        # v1.7.3: Generalised anion parameters
        'u_eff': None,                 # Effective u (all space groups)
        'delta_u': None,              # Anion sublattice anisotropy
        'anion_site_count': None,      # Number of distinct anion Wyckoff sites
        'anion_x_2': None,            # Second anion site x (P4332 only)
        'anion_y_2': None,            # Second anion site y
        'anion_z_2': None,            # Second anion site z
        'anion_wyckoff_2': None,       # Second anion Wyckoff symbol
    }
    
    # Determine anion elements. Fourth hand-written copy of the same nine
    # species until v2.0.13; now the single PROVISIONAL_ANIONS set, narrowed
    # below when the formula says which anion this structure actually has.
    anion_elements = set(PROVISIONAL_ANIONS)
    if formula:
        formula_elements = set(re.findall(r'[A-Z][a-z]?', formula))
        if formula_elements & {'S', 'Se', 'Te'}:
            anion_elements = {'S', 'Se', 'Te'}
        elif 'O' in formula_elements:
            anion_elements = {'O'}
    
    try:
        with open(cif_path, 'r', errors='ignore') as f:
            content = f.read()
    except (OSError, UnicodeDecodeError) as _e:
        # v2.0.11: recorded, not swallowed. A file that could not be read is
        # not a file with nothing in it.
        _note_read_failure(cif_path, 'extract_anion_parameter', _e)
        return result

    lines = content.split('\n')
    
    # Parse atom_site loop
    in_atom_loop = False
    columns = []
    label_col = type_col = x_col = y_col = z_col = None
    wyckoff_col = multiplicity_col = None
    
    for line in lines:
        line_stripped = line.strip()
        
        if line_stripped.startswith('loop_'):
            in_atom_loop = False
            columns = []
            label_col = type_col = x_col = y_col = z_col = None
            wyckoff_col = multiplicity_col = None
            continue
        
        if line_stripped.startswith('_atom_site_'):
            col_idx = len(columns)
            columns.append(line_stripped)
            
            if line_stripped == '_atom_site_label':
                label_col = col_idx
            elif line_stripped == '_atom_site_type_symbol':
                type_col = col_idx
            elif line_stripped == '_atom_site_fract_x':
                x_col = col_idx
            elif line_stripped == '_atom_site_fract_y':
                y_col = col_idx
            elif line_stripped == '_atom_site_fract_z':
                z_col = col_idx
            elif line_stripped == '_atom_site_Wyckoff_symbol':
                wyckoff_col = col_idx
            elif line_stripped == '_atom_site_symmetry_multiplicity':
                multiplicity_col = col_idx
            
            in_atom_loop = True
            continue
        
        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue
            
            parts = line_stripped.split()
            required_cols = [c for c in [label_col, type_col, x_col, y_col, z_col] if c is not None]
            if not required_cols or len(parts) < max(required_cols) + 1:
                continue
            
            # Get element type
            elem = None
            if type_col is not None and type_col < len(parts):
                elem_raw = parts[type_col]
                elem = re.sub(r'[0-9+-]+', '', elem_raw)
                elem = elem.capitalize() if len(elem) > 1 else elem.upper()
            elif label_col is not None and label_col < len(parts):
                label = parts[label_col]
                elem_match = re.match(r'([A-Z][a-z]?)', label)
                elem = elem_match.group(1) if elem_match else None
            
            if elem not in anion_elements:
                continue
            
            # ---- Found an anion site ----
            
            def parse_coord(val_str):
                """Parse coordinate value, handling ESDs in parentheses."""
                if not val_str or val_str in ['?', '.']:
                    return None
                match = re.match(r'^([+-]?\d*\.?\d+)', val_str)
                if match:
                    try:
                        return float(match.group(1))
                    except (ValueError, TypeError):
                        return None
                return None
            
            this_x = parse_coord(parts[x_col]) if x_col is not None and x_col < len(parts) else None
            this_y = parse_coord(parts[y_col]) if y_col is not None and y_col < len(parts) else None
            this_z = parse_coord(parts[z_col]) if z_col is not None and z_col < len(parts) else None
            
            # Extract Wyckoff info for this site
            this_wyckoff_letter = parts[wyckoff_col] if wyckoff_col is not None and wyckoff_col < len(parts) else None
            this_multiplicity = None
            if multiplicity_col is not None and multiplicity_col < len(parts):
                try:
                    this_multiplicity = int(parts[multiplicity_col])
                except (ValueError, TypeError, IndexError):
                    pass
            
            # Build full Wyckoff symbol
            if this_multiplicity is not None and this_wyckoff_letter is not None:
                this_wyckoff = f"{this_multiplicity}{this_wyckoff_letter}"
            elif this_wyckoff_letter is not None:
                # Fallback multiplicity lookup (common spinel space groups)
                wyckoff_multiplicities = {
                    'a': 8, 'b': 8, 'c': 16, 'd': 16, 'e': 32, 'f': 48,
                    'g': 48, 'h': 96, 'i': 96, 'j': 192
                }
                mult = wyckoff_multiplicities.get(this_wyckoff_letter.lower())
                if mult:
                    this_wyckoff = f"{mult}{this_wyckoff_letter}"
                    this_multiplicity = mult
                else:
                    this_wyckoff = this_wyckoff_letter
            else:
                this_wyckoff = None
            
            # Store to primary or secondary anion site
            if result['anion_x'] is None:
                # First anion site
                result['anion_x'] = this_x
                result['anion_y'] = this_y
                result['anion_z'] = this_z
                result['anion_wyckoff'] = this_wyckoff
                result['anion_wyckoff_letter'] = this_wyckoff_letter
                result['anion_multiplicity'] = this_multiplicity
                result['anion_site_count'] = 1
            elif result['anion_x_2'] is None:
                # Second anion site (e.g. P4332: 24e + 8c)
                result['anion_x_2'] = this_x
                result['anion_y_2'] = this_y
                result['anion_z_2'] = this_z
                result['anion_wyckoff_2'] = this_wyckoff
                result['anion_site_count'] = 2
            else:
                break  # Two anion sites found, stop
            continue  # Keep looking for additional anion sites
    
    # =========================================================================
    # Generalised anion parameter calculation (v1.7.3)
    # =========================================================================
    ax = result['anion_x']
    ay = result['anion_y']
    az = result['anion_z']
    # v2.0.12: choose which anion site carries the FREE parameter, rather
    # than whichever the CIF happened to list first.
    #
    # In Fd-3m the 48f anion has one free coordinate and the 8b anion is
    # fixed at (3/8,3/8,3/8) by symmetry, so 8b's coordinate carries no
    # information at all. Taking the first-listed site reported
    # u_parameter_raw = 0.375 and u_deviation = 0.000 for every pyrochlore
    # tested, whether its 48f x was 0.375 (ICSD 26799, at the fluorite
    # limit) or -0.0788 (ICSD 14242 and 42702, ordinary). The one entry
    # where 0.375 means something was indistinguishable from the two where
    # it means nothing.
    #
    # The site with the higher multiplicity is the one with positional
    # freedom in every structure type this function handles: 48f over 8b,
    # 32e over 8b, 24e over 8c.
    _m1 = result.get('anion_multiplicity')
    _m2 = None
    _wy2 = result.get('anion_wyckoff_2') or ''
    _mm = re.match(r'(\d+)', _wy2)
    if _mm:
        _m2 = int(_mm.group(1))
    if _m1 is not None and _m2 is not None and _m2 > _m1:
        for _a, _b in (('anion_x', 'anion_x_2'), ('anion_y', 'anion_y_2'),
                       ('anion_z', 'anion_z_2'),
                       ('anion_wyckoff', 'anion_wyckoff_2')):
            result[_a], result[_b] = result[_b], result[_a]
        result['anion_multiplicity'] = _m2
        _lm = re.match(r'\d+([a-z])', result['anion_wyckoff'] or '')
        result['anion_wyckoff_letter'] = _lm.group(1) if _lm else None
        result['anion_site_swapped'] = True
    else:
        result['anion_site_swapped'] = False

    # The swap above rewrote result; ax/ay/az were read from it earlier and
    # must be refreshed or the derivation below silently uses the old site.
    ax = result.get('anion_x')
    ay = result.get('anion_y')
    az = result.get('anion_z')

    wy = result.get('anion_wyckoff', '') or ''

    # v1.8.3 (B2): only assign an anion parameter for structure types where one
    # is defined. On an O3 layered oxide the cubic branch could latch onto an
    # anion at (0, 0, z) and report u = 0.0000 with u_deviation = -0.2500,
    # discarding the meaningful z. The site coordinates are still recorded;
    # only the spinel-specific parameters are withheld.
    _spinel_it_numbers = {227, 212, 213, 141, 70, 74}
    _it = _read_it_number(cif_path)
    result['u_parameter_applicable'] = (_it in _spinel_it_numbers) if _it else None
    if _it is not None and _it not in _spinel_it_numbers:
        result['u_note'] = (f'not a spinel-type space group (IT {_it}); '
                            'anion parameter not defined')
        return result

    if ax is not None:
        
        # ----- Cubic Fd-3m: 32e (x, x, x) -> single u parameter -----
        if '32e' in wy or (ay is not None and az is not None 
                           and abs(ax - ay) < 0.001 and abs(ax - az) < 0.001
                           and '16h' not in wy and '8h' not in wy):
            u_raw = ax
            result['u_parameter_raw'] = u_raw
            
            # Detect origin choice based on u value
            # Origin 1 (S): u ~ 0.375 (range 0.35-0.42)
            # Origin 2 (Z): u ~ 0.250 (range 0.23-0.30)
            if u_raw > 0.32:
                result['origin_choice'] = 1
                u_standardised = u_raw - 0.125
            else:
                result['origin_choice'] = 2
                u_standardised = u_raw
            
            result['u_parameter'] = round(u_standardised, 4)
            result['u_deviation'] = round(u_standardised - 0.25, 4)
            result['u_eff'] = result['u_parameter']
            result['delta_u'] = 0.0
        
        # ----- Tetragonal I41/amd: 16h (0, y, z) -> two parameters -----
        elif '16h' in wy or '8h' in wy:
            if ay is not None and az is not None:
                result['u_eff'] = round((ay + az) / 2, 4)
                result['delta_u'] = round(abs(ay - az), 4)
                # u_parameter left as None (cubic u not defined for tetragonal)
                # u_deviation computed from u_eff for cross-system comparison
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
        
        # ----- Ordered cubic P4332: 24e + 8c (two distinct anion sites) -----
        elif '24e' in wy or '8c' in wy:
            ax2 = result.get('anion_x_2')
            ay2 = result.get('anion_y_2')
            az2 = result.get('anion_z_2')
            wy2 = result.get('anion_wyckoff_2', '') or ''
            
            # 8c site: coords (x, x, x), so u = x
            # 24e site: coords (x, 0, 0), so the free param is x
            u_8c = None
            u_24e = None
            
            if '8c' in wy:
                u_8c = ax
                if ax2 is not None and '24e' in wy2:
                    u_24e = ax2
                elif ax2 is not None:
                    u_24e = ax2  # Assume second site is 24e
            elif '24e' in wy:
                u_24e = ax
                if ax2 is not None and '8c' in wy2:
                    u_8c = ax2
                elif ax2 is not None:
                    u_8c = ax2  # Assume second site is 8c
            
            # Effective u: weighted by multiplicity (24e = 75%, 8c = 25%)
            if u_24e is not None and u_8c is not None:
                result['u_eff'] = round((3 * u_24e + u_8c) / 4, 4)
                result['delta_u'] = round(abs(u_24e - u_8c), 4)
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
            elif u_8c is not None:
                # Only 8c found (unusual but handle gracefully)
                result['u_eff'] = round(u_8c, 4)
                result['u_parameter'] = round(u_8c, 4)
                result['u_deviation'] = round(u_8c - 0.25, 4)
            elif u_24e is not None:
                result['u_eff'] = round(u_24e, 4)
                result['u_deviation'] = round(u_24e - 0.25, 4)
        
        # ----- Orthorhombic Fddd: 32h (x, y, z) -> three parameters -----
        elif '32h' in wy:
            if ay is not None and az is not None:
                result['u_eff'] = round((ax + ay + az) / 3, 4)
                result['delta_u'] = round(max(abs(ax - ay), abs(ay - az), abs(ax - az)), 4)
                result['u_deviation'] = round(result['u_eff'] - 0.25, 4)
        
        # ----- Fd-3m 48f (x, 1/8, 1/8): PYROCHLORE, not a spinel u -----
        # This function is a spinel anion-parameter extractor and Fd-3m is
        # shared by both types, so a pyrochlore used to fall through to the
        # cubic fallback below and have its 48f x reported as a spinel u.
        # They are different quantities: the spinel u describes how far the
        # 32e anion sits from the ideal cubic close packing, the pyrochlore
        # 48f x describes the trigonal distortion of the B-site octahedron.
        # Standardising one against 0.25 says nothing about the other.
        #
        # x = 0.375 is the ideal DEFECT FLUORITE value, at which the
        # pyrochlore ordering vanishes: every A-O and B-O distance becomes
        # equal. ICSD 26799 (Nd2Zr2O7) is deposited at x = 0.375(7), which is
        # a phase-boundary statement rather than a fault, and it is why that
        # entry alone was symmetric under the origin-choice test.
        elif '48f' in wy:
            result['pyrochlore_48f_x'] = ax
            result['pyrochlore_fluorite_limit'] = (
                ax is not None and abs(ax - 0.375) < 0.005)
            result['u_note'] = ('Fd-3m 48f: pyrochlore anion parameter, not the spinel u; reported as pyrochlore_48f_x')

        # ----- Fallback: attempt cubic assumption -----
        else:
            u_raw = ax
            result['u_parameter_raw'] = u_raw
            if u_raw > 0.32:
                result['origin_choice'] = 1
                u_standardised = u_raw - 0.125
            else:
                result['origin_choice'] = 2
                u_standardised = u_raw
            result['u_parameter'] = round(u_standardised, 4)
            result['u_deviation'] = round(u_standardised - 0.25, 4)
            result['u_eff'] = result['u_parameter']
    
    return result


def extract_site_occupancies(cif_path):
    """
    Extract site occupancies with full detail.
    
    Returns:
        dict: {site_label: {element, occupancy, wyckoff, fract_x, fract_y, fract_z}}
    """
    result = {}
    
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except (OSError, UnicodeDecodeError) as _e:
        # v2.0.11: recorded, not swallowed. A file that could not be read is
        # not a file with nothing in it.
        _note_read_failure(cif_path, 'extract_site_occupancies', _e)
        return result

    in_atom_loop = False
    columns = []
    col_indices = {}
    
    for line in lines:
        line_stripped = line.strip()
        
        if line_stripped == 'loop_':
            in_atom_loop = False
            columns = []
            col_indices = {}
            continue
        
        if line_stripped.startswith('_atom_site_'):
            columns.append(line_stripped)
            col_idx = len(columns) - 1
            tag_map = {
                '_atom_site_label': 'label',
                '_atom_site_type_symbol': 'type',
                '_atom_site_occupancy': 'occupancy',
                '_atom_site_fract_x': 'fract_x',
                '_atom_site_fract_y': 'fract_y',
                '_atom_site_fract_z': 'fract_z',
                '_atom_site_Wyckoff_symbol': 'wyckoff',
                '_atom_site_symmetry_multiplicity': 'multiplicity',
            }
            for tag, name in tag_map.items():
                if line_stripped == tag:
                    col_indices[name] = col_idx
            in_atom_loop = True
            continue
        
        if in_atom_loop and columns and not line_stripped.startswith('_'):
            if not line_stripped or line_stripped.startswith('#') or line_stripped.startswith('loop_'):
                in_atom_loop = False
                continue
            
            parts = line_stripped.split()
            
            def get_val(name):
                if name in col_indices and col_indices[name] < len(parts):
                    return parts[col_indices[name]]
                return None
            
            def parse_numeric(val_str):
                if not val_str or val_str in ['?', '.']:
                    return None, None
                match = re.match(r'^([+-]?\d+\.?\d*)\((\d+)\)$', val_str)
                if match:
                    val = float(match.group(1))
                    if '.' in match.group(1):
                        dec_places = len(match.group(1).split('.')[1])
                        esd = int(match.group(2)) * (10 ** -dec_places)
                    else:
                        esd = float(match.group(2))
                    return val, esd
                try:
                    return float(val_str), None
                except (ValueError, TypeError):
                    return None, None
            
            label = get_val('label')
            if label:
                type_raw = get_val('type')
                elem = None
                if type_raw:
                    elem = re.sub(r'[0-9+-]+', '', type_raw)
                    elem = elem.capitalize() if len(elem) > 1 else elem.upper()
                
                occ, _ = parse_numeric(get_val('occupancy'))
                fx, _ = parse_numeric(get_val('fract_x'))
                fy, _ = parse_numeric(get_val('fract_y'))
                fz, _ = parse_numeric(get_val('fract_z'))
                wyckoff = get_val('wyckoff')
                mult_str = get_val('multiplicity')
                mult = int(mult_str) if mult_str and mult_str.isdigit() else None
                
                result[label] = {
                    'element': elem,
                    'occupancy': occ if occ is not None else 1.0,
                    'wyckoff': wyckoff,
                    'multiplicity': mult,
                    'fract_x': fx, 'fract_y': fy, 'fract_z': fz,
                }
    
    return result


# ============================================================================
# CONFIG FILE SUPPORT
# ============================================================================

def _coerce_oxidation_state(v):
    """
    Parse an oxidation state from a config, preserving fractional values.

    v1.8.3 (B4): the previous int(v) silently truncated, so a config entry of
    Mn: 3.5 became Mn: 3 and every downstream fractional-valence code path was
    bypassed. Integers are returned as int so existing formatting is unchanged.
    """
    f = float(v)
    return int(f) if f == int(f) else f


def load_oxidation_states_from_config(config_path=None, config_dict=None):
    """
    Load oxidation states from config file or dictionary.
    """
    if config_dict is not None:
        if not isinstance(config_dict, dict):
            print("Error: config_dict must be a dictionary")
            return None
        
        try:
            # v1.8.3 (B4): keep fractional oxidation states. Casting
            # to int defeated the fractional-valence support
            # whenever a config was supplied.
            validated = {str(k): _coerce_oxidation_state(v)
                         for k, v in config_dict.items()}
            print(f"Loaded oxidation states from provided dictionary:")
            for elem in sorted(validated.keys()):
                print(f"    {elem}: {validated[elem]:+g}")
            return validated
        except (ValueError, TypeError) as e:
            print(f"Error validating config_dict: {e}")
            return None
    
    if config_path is not None:
        if not os.path.exists(config_path):
            print(f"Config file not found: {config_path}")
            return None
        
        try:
            with open(config_path, 'r') as f:
                data = json.load(f)
            
            # v1.8.3 (B4): keep fractional oxidation states. Casting
            # to int defeated the fractional-valence support
            # whenever a config was supplied.
            validated = {str(k): _coerce_oxidation_state(v)
                         for k, v in data.items()}
            print(f"Loaded oxidation states from {config_path}:")
            for elem in sorted(validated.keys()):
                print(f"    {elem}: {validated[elem]:+g}")
            return validated
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON in {config_path}: {e}")
            return None
        except (ValueError, TypeError) as e:
            print(f"Error validating oxidation states in {config_path}: {e}")
            return None
    
    return None


def auto_detect_config_file(directory):
    """Automatically search for config.json in the target directory."""
    config_path = os.path.join(directory, "config.json")
    if os.path.exists(config_path):
        print(f"Found config.json in {directory}")
        return config_path
    return None


# ============================================================================
# 3D VISUALISATION OF PROBLEMATIC SITES
# ============================================================================
# v1.8.4 rewrite. The previous version loaded the whole CIF, replicated it
# 2x2x2 and zoomed to the lot, so the one site you actually wanted to look at
# was an anonymous sphere among a hundred identical ones, with no bonds drawn
# to it and no way to tell which it was. It also coloured the central cation
# red and oxygen blue, which inverts the convention every crystallographer
# reads by reflex.
#
# This version builds the coordination polyhedron explicitly: the central atom,
# its ligands, one cylinder per bond, and the distance written on each bond.
# Nothing else is drawn, because nothing else is being discussed.

# CPK-ish, so the picture reads the way a chemist expects
VIZ_ELEMENT_COLOURS = {
    'O': '#ff2200', 'F': '#90e050', 'Cl': '#1ff01f', 'S': '#ffff30',
    'N': '#3050f8', 'Na': '#ab5cf2', 'Li': '#cc80ff', 'K': '#8f40d4',
    'Mg': '#8aff00', 'Ca': '#3dff00', 'Cr': '#8a99c7', 'Mn': '#9c7ac7',
    'Fe': '#e06633', 'Co': '#f090a0', 'Ni': '#50d050', 'Cu': '#c88033',
    'Zn': '#7d80b0', 'Ti': '#bfc2c7', 'V': '#a6a6ab', 'Al': '#bfa6a6',
    'Sb': '#9e63b5', 'Nb': '#73c2c9', 'Ru': '#248f8f', 'Sr': '#00ff00',
    'Ba': '#00c900', 'B': '#ffb5b5', 'Si': '#f0c8a0', 'P': '#ff8000',
}
VIZ_DEFAULT_COLOUR = '#c0c0c0'

# A cation-anion contact below this fraction of the sum of the ionic radii is
# taken as an error in the deposited coordinates rather than as chemistry. The
# 15% margin allows for normal bond-length variation and thermal motion while
# still catching genuine overlaps. Same value the batch post-processor uses.
IMPLAUSIBLE_CONTACT_FRACTION = 0.85

# Second, independent geometry test, expressed in the tool's own terms.
#
# The Shannon test above needs one threshold per structural role: sodium's
# shortest contacts have a 5th percentile of 0.763 while framework cations sit
# at 0.890, so a single cut either misses bad transition-metal sites or
# condemns good alkali ones. Hardcoding an "alkali vs framework" split would
# work for layered oxides and mean nothing in a phosphate.
#
# R0 and b already encode both the size and the SOFTNESS of a bond, which is
# what the radii sum omits. The distance at which a cation would reach its
# expected valence at its observed coordination is
#     d_ideal = R0 - b*ln(V/N)
# and d_observed/d_ideal needs no element classes at all. On the reference set
# the 5th percentiles of the two populations agree to 2.6% on this measure
# against 14% on the radii ratio, so one threshold serves both.
#
# Kept ALONGSIDE the Shannon test rather than replacing it. This measure is
# circular in a way the other is not: it uses the bond valence parameters to
# judge whether the geometry is consistent with the bond valence parameters.
# That is fine for catching a misplaced atom and weaker as independent
# evidence, so having one test that depends on the parameter set and one that
# does not is worth the extra column.
IMPLAUSIBLE_BVS_DISTANCE_FRACTION = 0.90


# =============================================================================
# TOLERANT COMPARISON
# =============================================================================
# Occupancies, valences and the thresholds tested against them are all drawn
# from the same small set of round decimals, so a computed value landing
# exactly on a threshold is common rather than rare. On the 142-structure
# reference set: ten sites have occupancy exactly 0.90, four have a valence
# spread of exactly 0.30, and 98 of 142 structures have a charge-balance error
# that is an exact multiple of 0.01.
#
# A bare > or < then decides the outcome on the last bit of a float. The two
# structures that exposed this both balance to 0.01, but summed in the order
# their formulae are written one gives 0.010000000000000675 and the other
# 0.009999999999999787, so they fell on opposite sides of `> 0.01` and were
# flagged differently for no chemical reason.
#
# These make a tie resolve the same way every time. The epsilon is far below
# any physically meaningful difference and far above float noise.

COMPARISON_EPS = 1e-9

# A charge-balance error below this is taken as rounding in the published
# formula rather than a real imbalance. Na0.67Ni0.33Ti0.67O2 is
# Na(2/3)Ni(1/3)Ti(2/3)O2 and balances exactly, but as quoted it computes to
# 4.0100. On the reference set the error is bimodal - 131 structures below
# 0.041, eleven above 0.067, nothing between - so 0.05 sits in the gap.
CHARGE_BALANCE_TOLERANCE = 0.05


def gt(value, threshold, eps=COMPARISON_EPS):
    """value > threshold, with a tie counting as not greater."""
    return (value - threshold) > eps


def lt(value, threshold, eps=COMPARISON_EPS):
    """value < threshold, with a tie counting as not less."""
    return (threshold - value) > eps


def ge(value, threshold, eps=COMPARISON_EPS):
    """value >= threshold, with a tie counting as satisfying it."""
    return (value - threshold) > -eps


def le(value, threshold, eps=COMPARISON_EPS):
    """value <= threshold, with a tie counting as satisfying it."""
    return (threshold - value) > -eps
VIZ_CENTRAL_HALO = '#ffd700'      # gold ring so the central atom is unmistakable


def _viz_local_environment(cif_path, element, site_label, rmax=4.2, dedup_tol=0.02):
    """
    Cartesian coordinates of one site and the anions around it.

    Returns (centre_xyz, [(element, xyz, distance), ...]) or (None, []).
    Positions come from a symmetry expansion of the asymmetric unit, the same
    way the bond valence sum gets them, so the picture and the number agree.
    """
    try:
        import numpy as _np
        st = load_structure_for_viz(cif_path)
        if st is None or st.spacegroup is None:
            return None, []

        target = None
        for s in st.sites:
            if s.label == site_label or (target is None and s.element.name == element):
                target = s
                if s.label == site_label:
                    break
        if target is None:
            return None, []

        pts = []
        for s in st.sites:
            for op in st.spacegroup.operations():
                f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
                for i in (-1, 0, 1):
                    for j in (-1, 0, 1):
                        for k in (-1, 0, 1):
                            p = st.cell.orthogonalize(
                                gemmi.Fractional(f[0] + i, f[1] + j, f[2] + k))
                            pts.append((s.element.name, p.x, p.y, p.z))

        o = target.orth(st.cell)
        c = _np.array([o.x, o.y, o.z])
        P = _np.array([[p[1], p[2], p[3]] for p in pts])
        els = [p[0] for p in pts]
        d = _np.linalg.norm(P - c, axis=1)
        order = _np.argsort(d)

        ligands, kept = [], []
        for idx in order:
            if d[idx] <= 0.01 or d[idx] > rmax:
                continue
            if els[idx] not in ALWAYS_ANION and els[idx] not in AMPHOTERIC_ANION:
                continue
            if any(_np.linalg.norm(P[idx] - k) <= dedup_tol for k in kept):
                continue
            kept.append(P[idx])
            ligands.append((els[idx], P[idx], float(d[idx])))
        return c, ligands
    except Exception:
        return None, []


def load_structure_for_viz(cif_path):
    """gemmi SmallStructure from a CIF, reusing Eir's cleaning."""
    try:
        import tempfile as _tf
        cleaned = clean_cif_for_gemmi(cif_path)
        if cleaned is None:
            return None
        with _tf.NamedTemporaryFile('w', suffix='.cif', delete=False,
                                    encoding='utf-8') as t:
            t.write(cleaned)
            tp = t.name
        try:
            st = gemmi.make_small_structure_from_block(
                gemmi.cif.read(tp).sole_block())
        finally:
            os.unlink(tp)
        if st.spacegroup is None:
            m = (re.search(r'_space_group_IT_number\s+(\d+)', cleaned) or
                 re.search(r'_symmetry_Int_Tables_number\s+(\d+)', cleaned))
            if m:
                st.spacegroup = gemmi.find_spacegroup_by_number(int(m.group(1)))
        return st
    except Exception:
        return None

# ============================================================================
# ANION-ANION CONTACT SCREEN
# ============================================================================
# The two existing screens both look outward from a cation. Neither can see a
# structure whose ANIONS are too close to each other, and on the reference set
# seven structures fail on that alone: 9992, 19652, 147267, 175658, 175668,
# 203321 and 203322, with a mean GII near 1.0 and no cation-based warning.
#
# This is the oldest test in the book. Pauling's first rule is sphere packing:
# for a regular octahedron the O-O edge is sqrt(2) times the M-O distance, so
# oxygens 2.2 A apart have their electron clouds 0.6 A inside one another. Real
# oxides do compress below the 2.80 A hard-sphere contact, routinely to 2.5 or
# 2.6 A in edge-sharing frameworks, but 2.35 A is outside anything observed.
#
# It uses no bond valence parameters and no ionic radii for the cation, so it
# is independent of the quantities it is being used to validate.

ANION_ANION_MIN = 2.35        # A, below this the contact is not credible

# v2.0.14: WHAT THIS FLOOR DOES NOT COVER, stated because it was silent.
#
# 2.35 A was measured on perovskites, pyrochlores and spinels, all oxides.
# Run against the chalcogenide sets it is inert: across 505 sulfide, selenide
# and telluride spinels the shortest anion-anion contact is 2.469 A, so G2
# never fires on any of them and the anion sublattice of half the spinel work
# goes unscreened. The measured minima are S-S 2.998, Se-Se 2.469, Te-Te
# 2.517 A, first percentiles 3.002, 2.586, 2.524 - the low tails there are
# candidates for the gate rather than evidence about the floor, so the
# non-oxide values cannot be set from these numbers alone. They need the
# screened population, the same way the oxide value was set. Until that run
# exists the floor stays oxide-only and this comment says so.

# The search window for anion_anion_contacts, in angstroms.
#
# The default of 4.0 A is an oxide-scaled number and was never stated as one.
# Ten of 505 chalcogenide spinels have no contact of either kind inside it and
# returned (None, None); both packing gates read that as nothing to test and
# skipped, so the entry reported as screened when it was not. A further 24
# selenides had an anion contact inside 4.0 A but no cation contact, so G5
# alone skipped. Widened once, on demand, all of them resolve, with
# anion-anion minima out to 4.586 A. 6.5 A clears the largest observed
# minimum with room, and is only ever reached by the structures that need it.
ANION_CONTACT_RMAX_WIDE = 6.5

# The summation cut-off for bond valence, in valence units, applied at BOTH
# ends of every bond.
#
# s = exp((R0 - d)/b) is a property of the bond, not of the end you view it
# from, so a contact admitted to the cation sum must be admitted to the anion
# sum. Eir previously used gap analysis for cations and 0.04 v.u. for anions,
# which counted bonds in 0.02 <= s < 0.04 at one end only. That breaks the
# premise of the cation-anion comparison in the summary, which is that a
# discrepancy cancelling at the shared oxygen is a property of the parameters
# rather than of the structure.
#
# DIRECTION: sourced. Gagne & Hawthorne (2015), Acta Cryst B71, 562-578, sec 4,
# state that the longer bonds should be included when using the parameters
# derived in that work, having tested both practices and found inclusion gives
# better sums. They specify this for elements of periods 4-6 and note it is
# typically not the transition metals, which is the perovskite A site and not
# the B site. Gap analysis excludes those bonds, so every A-site sum Eir
# produced before v2.0.3 used these parameters against their authors' stated
# instruction.
#
# VALUE: a reading of that direction, not a quantity from the source. G&H give
# no numeric cut-off; they judged each polyhedron and discarded the ambiguous
# ones (their selection criterion 8), which a tool cannot reproduce. Measured
# on 830 clean ICSD perovskites, this value is the loosest cut that leaves the
# octahedral B site intact at CN 6.0. The B site is the control: it has no
# shell ambiguity, so any cut that moves it is wrong whatever it does
# elsewhere. 0.01 admits a seventh B-site bond; 0.04 and above truncate the A
# site. On 429 pyrochlores nothing moves under any cut-off between 0.01 and
# 0.08, because neither pyrochlore site is ambiguous, so the choice costs
# nothing where the structure is unambiguous and corrects the sum where it is
# not.
#
# Equivalent distance: s >= cut  <=>  d <= R0 + b*ln(1/cut). At b = 0.37 that
# is R0 + 1.447 A, an ABSOLUTE distance per ion pair, not a multiple of the
# shortest bond.
BV_SUM_CUTOFF = 0.02


# Cut-off as a function of Lewis acidity. NOT YET CANONICAL: exported beside
# the fixed-cut-off result so the change can be measured before it moves a
# verdict, the same staging as 4a/4b.
#
# A fixed s puts the boundary at R0 + b*ln(1/s), and since R0 and b vary little
# across the table that is the same ABSOLUTE distance for every cation: 3.1 to
# 3.6 A whether the ion is Li(1+) or S(6+). BV_SUM_CUTOFF = 0.02 survived the
# perovskites and the pyrochlores because neither structure type has an anion
# between the first shell and 3.2 A, not because it was right. Spinel does, and
# the same constant gave Li(1+) coordination numbers of 8, 10, 14 and 16 on 312
# of 323 sites with cn_gap_shell reading 4 or 6 beside them, which put 271 Li
# sites through the bond-valence distance screen and produced seven lithium
# compositions flagged at 100%.
#
# Fitted over 147 configurations with n >= 30 in Gagne's bond-length data,
# regressing s at the LONGEST observed bond on L = valence / CN:
#
#     s_cut = 0.4167 * L^1.391        R^2 = 0.731 (log-log)
#
#     L band     n   measured median   fitted
#     <0.20     31        0.0189       0.0196
#     0.20-0.30 21        0.0527       0.0606
#     0.30-0.40 22        0.1090       0.0904
#     0.40-0.60 23        0.1867       0.1589
#     0.60-0.90 24        0.2443       0.2793
#     0.90-1.40 16        0.7392       0.4167
#     >1.40     10        1.2347       0.7323
#
# It underfits above L ~ 0.9 and that is harmless: those cations have
# unambiguous shells and the boundary never binds. The band that matters,
# L < 0.3, fits to within 15%. Note that 0.0189 at L < 0.2 is where the fixed
# 0.02 came from, so the constant was right for the ions it was calibrated on
# and wrong for everything else.
BV_CUT_A = 0.4167
BV_CUT_N = 1.391
BV_CUT_MIN, BV_CUT_MAX = 0.01, 0.60


def bv_cutoff_for(valence, cn):
    """The summation cut-off this ion configuration implies."""
    try:
        v, c = abs(float(valence)), float(cn)
    except (TypeError, ValueError):
        return BV_SUM_CUTOFF
    if c <= 0 or v <= 0 or v != v or c != c:
        return BV_SUM_CUTOFF
    return min(BV_CUT_MAX, max(BV_CUT_MIN, BV_CUT_A * (v / c) ** BV_CUT_N))


def bv_shell_boundary(element, valence, cn, anion='O', r0=None, b=None,
                      db=None):
    """
    The distance at which this ion's coordination shell ends.

    MEASURED where possible. Gagne's longest observed bond for the ion
    configuration IS the boundary: it is, by construction, the furthest contact
    anyone counted as bonded in that coordination across the survey. Using it
    needs no fit and no threshold of ours.

    The Lewis-acidity fit is the fallback where the sample is thin. It is only
    a fallback: measured against fitted, Li(1+) CN 4 is 2.444 A against 2.862
    (n = 419) and Zn(2+) CN 4 is 2.076 against 2.389 (n = 227). A 0.4 A error
    in a shell boundary is a coordination number or two.

    Returns (distance, source).
    """
    db = db or db_manager
    try:
        v = int(round(float(valence)))
        c = int(round(float(cn)))
    except (TypeError, ValueError):
        return None, 'unavailable'
    if c <= 0:
        return None, 'unavailable'

    bl = db.load_bond_lengths() or {}

    hit = bl.get((element, v, anion, c))
    if hit and (hit.get('n_bonds') or 0) >= BL_MIN_SAMPLE and hit.get('max'):
        return float(hit['max']), f"measured_n{hit['n_bonds']}"

    # v2.0.8: interpolate across coordination, as expected_bond_length already
    # does. Without this every configuration measured at CN 6 and 8 but not 7,
    # and every rare-earth A site above the tabulated range, fell through to
    # the fit: 30 of 77 sites on the test batch, and the fit is the part of
    # this that is out by 0.4 A on Li and 0.3 A on Zn.
    known = sorted(k[3] for k in bl
                   if k[:3] == (element, v, anion)
                   and (bl[k].get('n_bonds') or 0) >= BL_MIN_SAMPLE
                   and bl[k].get('max'))
    if known:
        lo = max([x for x in known if x < c], default=None)
        hi = min([x for x in known if x > c], default=None)
        if lo is not None and hi is not None:
            a = bl[(element, v, anion, lo)]['max']
            z = bl[(element, v, anion, hi)]['max']
            f = (c - lo) / (hi - lo)
            return float(a + f * (z - a)), f'measured_interp_CN{lo}_{hi}'
        near = min(known, key=lambda x: abs(x - c))
        # Extrapolating an extreme is worse than interpolating one, so this is
        # named differently and should be read with the same caution as tier 3
        # in expected_bond_length.
        return float(bl[(element, v, anion, near)]['max']), \
            f'measured_extrap_CN{near}'

    if r0 is None or not b:
        return None, 'unavailable'
    cut = bv_cutoff_for(v, c)
    return r0 + b * math.log(1.0 / cut), 'lewis_fit'


def bv_shell_cn_adaptive(sorted_distances, r0, b, valence, cn_seed,
                         element=None, anion='O', max_iter=4):
    """
    Coordination number from the ion's own shell boundary.

    The boundary depends on CN and CN depends on the boundary, so this
    iterates from the gap shell, which Eir already has and which is right in
    magnitude even where it is wrong in detail.

    The iteration is clamped so it can only TIGHTEN. Left free it runs away:
    a boundary loose enough to admit the second shell raises CN, which lowers
    Lewis acidity, which loosens the boundary further. On a synthetic Li 16d
    site that took the count to 18. The failure mode being fixed here is
    over-counting, so the safe direction is the only one allowed.
    """
    ds = sorted(float(d) for d in sorted_distances if d and d > 0)
    ev = {'cn_method_adaptive': 'bond_valence_shell_adaptive',
          'bv_boundary_adaptive': None, 'bv_boundary_source': None,
          'bv_cut_iterations': 0}
    if not ds or r0 is None or not b:
        ev['cn_method_adaptive'] = 'unavailable'
        return 0, ev

    try:
        cn = max(1, int(cn_seed))
    except (TypeError, ValueError):
        cn = SHIFT_SHELL_SIZE

    d_cap = None
    for it in range(max_iter):
        d_max, src = bv_shell_boundary(element, valence, cn, anion, r0, b)
        if d_max is None:
            ev['cn_method_adaptive'] = 'unavailable'
            return 0, ev
        # Clamp: never loosen beyond the tightest boundary seen so far.
        d_cap = d_max if d_cap is None else min(d_cap, d_max)
        new = sum(1 for d in ds if d <= d_cap)
        ev['bv_boundary_adaptive'] = round(d_cap, 4)
        ev['bv_boundary_source'] = src
        ev['bv_cut_iterations'] = it + 1
        if new == cn or new == 0:
            break
        cn = new
    return int(cn), ev


def bv_shell_cn(sorted_distances, r0, b, cutoff=BV_SUM_CUTOFF):
    """
    Coordination number by bond-valence contribution.

    Every neighbour whose contribution reaches `cutoff` is in the shell, taken
    in ascending distance order so the result is a shell rather than a set.

    Unlike gap analysis this makes no assumption that the polyhedron is
    compact, which is why it behaves on a tilted perovskite A site where the
    gap cut lands inside the coordination sphere. Unlike a multiple of the
    shortest bond it does not scale with compression: the boundary is
    R0 + b*ln(1/cutoff), fixed for the ion pair.

    Returns (cn, evidence). cn is 0 when no neighbour reaches the cut-off,
    which is a failure to be reported, not a shell of size zero.
    """
    ds = sorted(float(d) for d in sorted_distances if d and d > 0)
    ev = {'cn_method': 'bond_valence_shell',
          'bv_shell_cutoff': cutoff,
          'bv_shell_max_distance': None,
          'bv_shell_last_s': None,
          'bv_shell_next_s': None}

    if not ds or r0 is None or not b:
        ev['cn_method'] = 'unavailable'
        return 0, ev

    d_max = r0 + b * math.log(1.0 / cutoff)
    ev['bv_shell_max_distance'] = round(d_max, 4)

    cn = 0
    for d in ds:
        if d > d_max:
            break
        cn += 1

    if cn:
        ev['bv_shell_last_s'] = round(math.exp((r0 - ds[cn - 1]) / b), 4)
    if cn < len(ds):
        # What the next bond out would have contributed. A site sitting just
        # inside the boundary is a different thing from one well clear of it,
        # and the report should be able to tell them apart.
        ev['bv_shell_next_s'] = round(math.exp((r0 - ds[cn]) / b), 4)

    return int(cn), ev


def econ_weighted_bvs(distances, r0, b, max_iter=25, tol=1e-9):
    """
    Bond valence summed with Hoppe's ECoN weights instead of a cut-off.

    Each neighbour contributes s_i * w_i, with w_i = exp(1 - (d_i/d_av)^6) and
    d_av the ECoN weighted mean bond length. There is no boundary, so there is
    no convention to argue about and no discontinuity as a bond crosses one.
    That is the point of it, and also its limit.

    NOT the GII, and not comparable with any published bond valence sum. G&H
    fitted R0 and b against UNWEIGHTED sums over their chosen shells, so a
    weighted sum does not satisfy the equation those parameters were fitted to.
    Quote it as a second diagnostic, never as the validation figure, and never
    against the 50.8% rock salt pass rate or anything else in the literature.

    Pass a generous neighbour list: the weights fall off as the sixth power, so
    including too much costs nothing and truncating too early defeats the
    purpose.
    """
    d = np.asarray([float(x) for x in distances if x and x > 0], float)
    if d.size == 0 or r0 is None or not b:
        return None, None

    d_av = d.min()
    for _ in range(max_iter):
        w = np.exp(1.0 - (d / d_av) ** 6)
        new = (d * w).sum() / w.sum()
        if abs(new - d_av) < tol:
            break
        d_av = new

    w = np.exp(1.0 - (d / d_av) ** 6)
    s = np.exp((r0 - d) / b)
    return float((s * w).sum()), float(w.sum())


def VALIDATE_SHELL(verbose=True):
    """
    The shell rule against cases with known answers.

    Test 1 is the control that decided the cut-off and must not be quietly
    weakened: an octahedron has one shell however you cut it.
    """
    results = []

    def report(name, ok, detail=''):
        results.append(ok)
        if verbose:
            print(f"  {'PASS' if ok else 'FAIL'}  {name}")
            if detail:
                print(f"        {detail}")

    # 1. A regular octahedron must give CN 6 and nothing else.
    r0, b = 1.759, 0.37                       # Fe3+-O
    oct_d = [1.98] * 6 + [3.42] * 8           # 6 bonds, then the A cations
    cn, ev = bv_shell_cn(oct_d, r0, b)
    report('1. regular octahedron returns CN 6', cn == 6,
           f'boundary {ev["bv_shell_max_distance"]} A, got CN {cn}')

    # 2. A tilted A site: 8 short and 4 long, all of which G&H say to include.
    r0, b = 2.172, 0.37                       # La3+-O
    a_site = [2.42, 2.44, 2.51, 2.55, 2.61, 2.67, 2.72, 2.78,
              3.02, 3.09, 3.15, 3.21, 4.10]
    cn, ev = bv_shell_cn(a_site, r0, b)
    report('2. tilted A site keeps the long bonds', cn == 12,
           f'boundary {ev["bv_shell_max_distance"]} A, got CN {cn}')

    # 3. The boundary is absolute, not relative. Compressing every bond by 5%
    #    must NOT change which bonds are in the shell in the same way a
    #    multiple-of-shortest rule would.
    cn_c, _ = bv_shell_cn([d * 0.95 for d in a_site], r0, b)
    report('3. shell does not scale with compression', cn_c >= cn,
           f'{cn} bonds at nominal, {cn_c} at 0.95x')

    # 4. No neighbour inside the boundary is CN 0, reported, not defaulted.
    cn_z, ev_z = bv_shell_cn([4.5, 4.8], r0, b)
    report('4. nothing in range returns CN 0', cn_z == 0)

    # 5. Missing parameters are declared, not silently treated as a pass.
    _, ev_n = bv_shell_cn([2.0, 2.1], None, 0.37)
    report('5. absent parameters report cn_method unavailable',
           ev_n['cn_method'] == 'unavailable')

    # 6. The ECoN-weighted sum is close to, and below, the unweighted one on a
    #    regular polyhedron, where the weights are all near unity.
    r0, b = 1.759, 0.37
    tot, econ = econ_weighted_bvs([1.98] * 6, r0, b)
    plain = 6 * math.exp((r0 - 1.98) / 0.37)
    report('6. ECoN-weighted sum tracks the plain sum when regular',
           tot is not None and abs(tot - plain) / plain < 0.02
           and abs(econ - 6.0) < 0.05,
           f'weighted {tot:.3f} against plain {plain:.3f}, ECoN {econ:.2f}')

    ok = all(results)
    print(f"\n  {'ALL SHELL TESTS PASSED' if ok else 'SOMETHING FAILED'}"
          f"  ({sum(results)}/{len(results)})")
    return ok
# The cation sublattice floor, set from measurement.
#
# cation_cation_min has been computed since v2.0.2 and nothing compared it
# against anything. Set from the measured distributions rather than by
# analogy with the anion floor: cations vary far more than anions, and
# edge- and face-sharing polyhedra put them legitimately close - 2.96 A
# for Ti-Ti in rutile, 2.65 A for Al-Al in corundum.
#
# Shortest cation-cation contact, structures passing every other screen:
#
#     family        n     min    1st pct   median
#     perovskite  938   1.912     2.787     3.148
#     pyrochlore  472   3.216     3.405     3.627
#     spinel     1877   1.648     2.363     2.915
#
# 2.40 sits below the 1st percentile of all three while catching 59
# structures, 19 of which no other screen sees - including ICSD 133663
# (NdNiO3, Nd-Ni 1.912 A) and 13793 (SrSnO3, Sn-Sr 2.019 A), both of which
# sat in the triage's UNEXPLAINED bucket. 2.6 adds two; 2.0 loses the
# spinel cases.
CATION_CATION_MIN = 2.40      # A, measured; see above

# v2.0.15: WHAT THIS FLOOR DOES NOT APPLY TO.
#
# All three families above are hydrogen-free, and the floor is meaningless for
# hydrogen. The H-H distance inside a single water molecule is 1.51 A; two
# hydroxyls in a hydrogen-bonded chain sit closer still. Run against 101 ICSD
# borosulfates the floor condemned 31 of them on an H contact - 20 H-H, 6 H-S,
# 5 H-N - and 12 on anything else. A proton is not a packing sphere and its
# position in an X-ray refinement is usually placed rather than found.
#
# H is therefore excluded from the cation-cation measurement in
# anion_anion_contacts, which means cation_cation_min never carries an H
# contact and this floor never judges one. Nothing in the three reference
# families changes: none of them contains hydrogen.
#
# The cost is stated: a genuinely impossible H position is now invisible to
# G5. It was never visible to a floor measured without hydrogen in it.
CATION_FLOOR_EXCLUDED_ELEMENTS = frozenset({'H', 'D'})
SPLIT_SITE_MAX = 1.20         # A, closer than this with occupancies summing to
                              # <= 1 means one atom written as two positions

# No cation-oxygen bond in any oxide is shorter than this. B-O runs to about
# 1.37 A, C-O in carbonate to 1.28, Si-O to 1.60; 1.20 leaves clearance below
# all of them. Like ANION_ANION_MIN this is a single constant with no
# element-specific input, so a contact below it is not a short bond, it is not
# a bond. The all_cations variant of ICSD 11616 produced a Dy-O contact of
# 1.181 A, from which Eir built a shell of one and reported the site as
# overbonded at 8.258 v.u.

CATION_ANION_ABSOLUTE_MIN = 1.20

def resolve_anions(elements, valences=None):
    """
    Which elements are acting as anions IN THIS structure.

    AMPHOTERIC_ANION exists because its members are anions in some compounds
    and cations in others: Si is an anion in Mg2Si and a cation in MgSiO3, P
    is an anion in a phosphide and a cation in a phosphate. Treating the set
    as unconditionally anionic is what the name warns against.

    anion_anion_contacts did exactly that, so in MgSiO3 every Si-O bond was
    tested against the 2.35 A anion-anion floor and failed. All eleven ICSD
    entries of that composition were flagged, and the reported
    anion_anion_min was identical to the Si-O bond length in every one, which
    is the signature: 1.7715, 1.7581, 1.7877 A.

    The rule here is the ordinary chemical one. An amphoteric element is an
    anion only if it is the most electronegative species present. Where the
    CIF states an oxidation state, that is used instead, because a depositor
    saying Si4+ settles the question.

    Returns the set of element symbols acting as anions.
    """
    els = {e for e in elements if e}
    anions = {e for e in els if e in ALWAYS_ANION}
    for e in els & set(AMPHOTERIC_ANION):
        v = (valences or {}).get(e)
        if v is not None:
            try:
                if float(v) < 0:
                    anions.add(e)
                continue
            except (TypeError, ValueError):
                pass
        # No stated oxidation state: anionic only if nothing more
        # electronegative is present to take that role.
        if not (els & set(ALWAYS_ANION)):
            anions.add(e)
    return anions


def anion_anion_contacts(cif_path, rmax=4.0, dedup_tol=0.02, valences=None):
    """
    Shortest anion-anion and cation-cation contact in the structure.

    Split sites are excluded. A refinement that models one oxygen as two
    partly-occupied positions 0.18 A apart is not reporting a 0.18 A contact,
    and ICSD 194733 does exactly that. Pairs closer than SPLIT_SITE_MAX whose
    occupancies sum to no more than unity are therefore skipped.

    Returns (anion_anion_min, cation_cation_min), either possibly None.

    v2.0.14: THREE FAULTS, EACH FOUND BY RUNNING THIS ON REAL FILES.

    1. THE PAIR FILTER AND THE BUCKET DISAGREED.
       same_class decided which pairs to consider from resolve_anions(),
       which reads the structure. The line that chose which bucket to put
       the distance in asked static membership of AMPHOTERIC_ANION instead.
       Wherever an amphoteric element is acting as a CATION - Si in a
       silicate, S in a sulfate, B in a borate, H in a hydrate - its
       cation-cation contacts were written to anion_anion_min and tested
       against the anion floor.

       Measured on 101 ICSD borosulfates: anion_anion_min wrong in 34,
       cation_cation_min wrong in 92. Four structures were condemned by G2
       on a distance that is not an anion-anion contact at all - ICSD 428004
       reported 1.000 A as anion packing impossible when its shortest O-O is
       2.363 A - and 35 had a cation-cation contact below the floor, down to
       0.923 A, that G5 never saw because the distance had gone into the
       other bucket. The three reference families are untouched: 0 of 27
       oxide and sulfide spinels change.

       This is the same fault resolve_anions() was written to fix. It was
       fixed at the filter and not at the bucket, which is the recurring
       shape: one idea, two implementations, drifting.

    2. rmax = 4.0 A IS AN OXIDE-SCALED WINDOW.
       Ten of 505 chalcogenide spinels have no contact of either kind inside
       4.0 A and returned (None, None). Both packing gates read that as
       'nothing to test' and skip, so the entry reports as screened when it
       was not. Re-run at 6.0 A all ten resolve, with anion-anion minima out
       to 4.586 A. The window is now widened once, on demand, and the number
       of lattice images follows the window rather than being fixed at one.

    3. except Exception: return None, None
       made a failure indistinguishable from 'nothing found', and both mean
       the gates do not fire. Narrowed. NameError, TypeError and
       AttributeError re-raise: those are defects in this function, not
       properties of the file.
    """
    import numpy as _np
    import math as _math

    def _scan(st, anion_set, rmax):
        # v2.0.14: the image range follows the search radius. At one image a
        # 6.5 A search in a 5 A cell cannot reach the neighbour it is looking
        # for, so widening rmax alone would have been a half fix.
        _abc = [st.cell.a, st.cell.b, st.cell.c]
        _abc = [x for x in _abc if x and x > 0.1] or [rmax]
        n_img = max(1, min(3, int(_math.ceil(rmax / min(_abc)))))
        rng = range(-n_img, n_img + 1)

        pts = []
        for s in st.sites:
            occ = getattr(s, 'occ', 1.0)
            occ = 1.0 if occ is None else float(occ)
            for op in st.spacegroup.operations():
                f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
                for i in rng:
                    for j in rng:
                        for k in rng:
                            p = st.cell.orthogonalize(
                                gemmi.Fractional(f[0] + i, f[1] + j, f[2] + k))
                            pts.append((s.element.name, p.x, p.y, p.z, occ))

        els = [p[0] for p in pts]
        occs = _np.array([p[4] for p in pts])
        P = _np.array([[p[1], p[2], p[3]] for p in pts])
        is_anion = _np.array([e in anion_set for e in els])

        aa = cc = None
        for s in st.sites:
            el = s.element.name
            occ0 = getattr(s, 'occ', 1.0)
            occ0 = 1.0 if occ0 is None else float(occ0)
            o = s.orth(st.cell)
            p0 = _np.array([o.x, o.y, o.z])
            d = _np.linalg.norm(P - p0, axis=1)
            m = _np.where((d > dedup_tol) & (d <= rmax))[0]
            el_is_anion = el in anion_set
            # v2.0.15: see CATION_FLOOR_EXCLUDED_ELEMENTS.
            if not el_is_anion and el in CATION_FLOOR_EXCLUDED_ELEMENTS:
                continue
            same_class = (is_anion == el_is_anion)
            for idx in m:
                if not same_class[idx]:
                    continue
                if (not el_is_anion
                        and els[idx] in CATION_FLOOR_EXCLUDED_ELEMENTS):
                    continue                      # v2.0.15: not a packing sphere
                dist = float(d[idx])
                if dist < SPLIT_SITE_MAX and (occ0 + occs[idx]) <= 1.02:
                    continue                      # one atom, two positions
                # v2.0.14: the bucket now asks the same question the filter
                # asked, and asks it of the structure rather than of a list.
                if el_is_anion:
                    aa = dist if aa is None else min(aa, dist)
                else:
                    cc = dist if cc is None else min(cc, dist)
        return aa, cc

    try:
        st = load_structure_for_viz(cif_path)
        if st is None or st.spacegroup is None:
            return None, None
        # v2.0.2: decided from the structure, not from a fixed list.
        _els = {s.element.name for s in st.sites}
        _anion_set = resolve_anions(_els, valences)
        _has_anion = bool(_anion_set & _els)
        _has_cation = bool(_els - _anion_set)

        aa, cc = _scan(st, _anion_set, rmax)

        # v2.0.14: one widened retry, and only where a bucket that SHOULD
        # have a member came back empty. A structure with no cations has no
        # cation-cation contact at any radius and is not a failure.
        if ((_has_anion and aa is None) or (_has_cation and cc is None)) \
                and rmax < ANION_CONTACT_RMAX_WIDE:
            aa2, cc2 = _scan(st, _anion_set, ANION_CONTACT_RMAX_WIDE)
            aa = aa if aa is not None else aa2
            cc = cc if cc is not None else cc2
        return aa, cc
    except (NameError, TypeError, AttributeError):
        # A defect in this function, not a property of the file. Fail loudly:
        # a silent (None, None) here disables BOTH packing gates for the
        # entry and nothing downstream can tell that apart from a clean pass.
        raise
    except (OSError, UnicodeDecodeError, ValueError, KeyError, IndexError,
            RuntimeError) as _e:
        _note_read_failure(cif_path, 'anion_anion_contacts', _e)
        return None, None

def visualize_problematic_site_3d(cif_path, element, site_label, bonds_info,
                                  deviation, formal_charge, bvs_value,
                                  filename=None, expected_distance=None):
    """
    Draw the coordination polyhedron of one site, with every bond labelled.

    Bonds shorter than 85% of the Shannon radii sum are drawn in red, since a
    contact that short is the usual reason a site ends up here at all.
    """
    try:
        import py3Dmol
        from IPython.display import display
    except ImportError:
        print("  [3D visualisation unavailable - py3Dmol not installed]")
        return

    try:
        centre, ligands = _viz_local_environment(cif_path, element, site_label)

        file_id = ""
        if filename:
            m = re.search(r'CollCode(\d+)', filename)
            file_id = (f" in CollCode{m.group(1)}" if m
                       else f" ({filename.replace('.cif', '')})")

        try:
            fc = f"{int(formal_charge):+d}"
        except (TypeError, ValueError):
            fc = f"{formal_charge:+.2f}" if formal_charge is not None else "?"
        print(f"\n  3D VISUALISATION: {element} at site {site_label}{file_id}")
        print(f"  BVS: {bvs_value:.3f}  |  Expected: {fc}  |  "
              f"Delta: {deviation:+.3f} v.u.")

        if not ligands:
            print("  [could not rebuild the local environment for this site]")
            return

        # Show the shell Eir actually counted, not everything within the search
        # radius. bonds_info is Eir's own coordination determination, so
        # matching against it guarantees the picture and the bond valence sum
        # describe the same bonds. Anything beyond it is drawn faintly and
        # without a bond, which is what makes a truncated coordination obvious:
        # you can see the gap the site is sitting in.
        shell, outer = [], []
        if bonds_info:
            wanted = [b['distance'] for b in bonds_info
                      for _ in range(b.get('count', 1))]
            used = [False] * len(wanted)
            for lig in ligands:
                hit = next((i for i, w in enumerate(wanted)
                            if not used[i] and abs(w - lig[2]) < 0.005), None)
                if hit is None:
                    outer.append(lig)
                else:
                    used[hit] = True
                    shell.append(lig)

        if not shell:
            # v1.8.6: with nothing to match against, take the first
            # coordination shell by the relative-distance rule rather than
            # everything inside the search radius, which drew a dozen bonds
            # and made the picture as unreadable as the one this replaced.
            # Indices, not values: a ligand tuple holds a numpy array, so
            # `l not in shell` raises "truth value of an array is ambiguous".
            _cut = ligands[0][2] * 1.30
            _keep = [i for i, l in enumerate(ligands) if l[2] <= _cut]
            if not _keep:
                _keep = list(range(min(6, len(ligands))))
            shell = [ligands[i] for i in _keep]
            outer = [l for i, l in enumerate(ligands) if i not in _keep]

        outer = [o for o in outer if o[2] <= (shell[-1][2] * 2.2)][:14]

        # what a credible shortest contact would be, for colouring
        if expected_distance is None:
            rc = SHANNON_CN6_VIZ.get(element)
            ra = SHANNON_CN6_VIZ.get(ligands[0][0])
            expected_distance = (rc + ra) if (rc and ra) else None

        view = py3Dmol.view(width=680, height=520)

        ccol = VIZ_ELEMENT_COLOURS.get(element, VIZ_DEFAULT_COLOUR)
        view.addSphere({'center': {'x': float(centre[0]), 'y': float(centre[1]),
                                   'z': float(centre[2])},
                        'radius': 0.55, 'color': ccol})
        view.addSphere({'center': {'x': float(centre[0]), 'y': float(centre[1]),
                                   'z': float(centre[2])},
                        'radius': 0.62, 'color': VIZ_CENTRAL_HALO,
                        'opacity': 0.35})
        view.addLabel(f"{site_label} ({element})",
                      {'position': {'x': float(centre[0]), 'y': float(centre[1]),
                                    'z': float(centre[2])},
                       'backgroundColor': 'black', 'backgroundOpacity': 0.7,
                       'fontColor': 'white', 'fontSize': 12,
                       'inFront': True})

        n_short = 0
        for el, pos, dist in shell:
            short = (expected_distance is not None
                     and dist < 0.85 * expected_distance)
            n_short += short
            view.addSphere({'center': {'x': float(pos[0]), 'y': float(pos[1]),
                                       'z': float(pos[2])},
                            'radius': 0.40,
                            'color': VIZ_ELEMENT_COLOURS.get(el, VIZ_DEFAULT_COLOUR)})
            view.addCylinder({'start': {'x': float(centre[0]), 'y': float(centre[1]),
                                        'z': float(centre[2])},
                              'end': {'x': float(pos[0]), 'y': float(pos[1]),
                                      'z': float(pos[2])},
                              'radius': 0.10,
                              'color': '#e41a1c' if short else '#666666'})
            mid = (centre + pos) / 2.0
            view.addLabel(f"{dist:.3f}",
                          {'position': {'x': float(mid[0]), 'y': float(mid[1]),
                                        'z': float(mid[2])},
                           'backgroundColor': '#e41a1c' if short else 'white',
                           'backgroundOpacity': 0.85,
                           'fontColor': 'white' if short else 'black',
                           'fontSize': 10, 'inFront': True})

        # context: the next neighbours out, no bonds, no labels
        for el, pos, dist in outer:
            view.addSphere({'center': {'x': float(pos[0]), 'y': float(pos[1]),
                                       'z': float(pos[2])},
                            'radius': 0.22, 'opacity': 0.30,
                            'color': VIZ_ELEMENT_COLOURS.get(el, VIZ_DEFAULT_COLOUR)})

        view.zoomTo()
        view.zoom(0.85)

        print(f"  Coordination shell: {len(shell)} ligands"
              + (f", {n_short} shorter than 85% of the expected "
                 f"{expected_distance:.2f} A" if n_short else ""))
        for el, _pos, dist in shell:
            flag = "  <-- implausibly short" if (
                expected_distance and dist < 0.85 * expected_distance) else ""
            print(f"    {element}-{el}  {dist:.4f} A{flag}")
        if outer:
            gap = outer[0][2] - shell[-1][2]
            print(f"  Next neighbours begin at {outer[0][2]:.4f} A, "
                  f"a gap of {gap:.4f} A (drawn faint, no bonds)")

        display(view)
        print()

    except Exception as e:
        print(f"  [3D visualisation failed: {e}]")


# Radii used only for colouring the picture, not for any calculation
SHANNON_CN6_VIZ = {
    'Li': 0.76, 'Na': 1.02, 'K': 1.38, 'Mg': 0.72, 'Ca': 1.00, 'Al': 0.535,
    'Sc': 0.745, 'Ti': 0.605, 'V': 0.58, 'Cr': 0.615, 'Mn': 0.645,
    'Fe': 0.645, 'Co': 0.61, 'Ni': 0.60, 'Cu': 0.73, 'Zn': 0.74,
    'Sb': 0.60, 'Nb': 0.64, 'Ru': 0.62, 'Sr': 1.18, 'Ba': 1.35,
    'O': 1.40, 'F': 1.33, 'Cl': 1.81, 'S': 1.84, 'N': 1.46,
}


# ============================================================================
# JAHN-TELLER ION DETECTION
# ============================================================================

# ============================================================================
# JAHN-TELLER OCTAHEDRAL CATIONS (element-gated)
# ============================================================================
# Strong (e_g-degenerate) octahedral JT cations form 4+2 elongated octahedra:
#   d4 HS (Cr2+, Mn3+, Fe4+), d7 LS (Ni3+), d9 (Cu2+, Ag2+).
# The gap analysis in auto_analyze_coordination (Cell 4) cuts at the largest
# bond gap, which for these ions is the equatorial->axial gap, wrongly
# returning CN=4. The fix there restores the axial pair, gated on ELEMENT
# only: at that point in Cell 4 the valence is not yet assigned, so central_type
# is all we have. Safety comes from geometry, not oxidation state — the fix only
# fires on a genuine 4+2 within MAX_JT_AXIAL_RATIO, so non-JT members of this
# set (Cr3+, Fe3+) are never touched: a clean octahedron is already CN=6 and a
# real tetrahedron's axials sit far beyond the ratio.
JT_OCTAHEDRAL_ELEMENTS = {'Cr', 'Mn', 'Fe', 'Ni', 'Cu', 'Ag'}
MAX_JT_AXIAL_RATIO = 1.30  # axial/equatorial-mean ceiling for a bonding axial

# Ions whose ground-state d-electron count gives a degenerate e_g or t_2g
# occupation and therefore a first-order Jahn-Teller distortion. High-spin d4
# and d9 are the strong cases; low-spin d7 is included because Ni(III) and
# Co(II) in octahedral oxide environments do distort. Weak (t_2g) cases are
# deliberately excluded: their distortions are far too small to widen a
# coordination-shell gap.
JAHN_TELLER_IONS = {
    ('Cr', 2), ('Mn', 3), ('Fe', 4),            # high-spin d4
    ('Cu', 2), ('Ag', 2), ('Ni', 1),            # d9
    ('Ni', 3), ('Co', 2), ('Rh', 2), ('Pd', 3), # low-spin d7
    ('Mn', 6), ('Cr', 4),                       # d2 tetrahedral, occasionally
}


def is_jahn_teller_ion(element, valence):
    """
    Whether an ion is expected to show a first-order Jahn-Teller distortion.

    v1.8.4: this was called by detect_coordination_adaptive() but had never
    been defined anywhere, so any call raised NameError. The function was
    unreachable in practice, which is why it was never noticed.

    Fractional valences are rounded to the nearest integer, since a
    site-averaged valence of 3.3 on manganese still contains Mn(III).
    """
    if element is None or valence is None:
        return False
    try:
        v = int(round(float(valence)))
    except (TypeError, ValueError):
        return False
    return (element, v) in JAHN_TELLER_IONS

# ============================================================================
# ADAPTIVE GAP DETECTION FOR COORDINATION
# ============================================================================

def detect_coordination_adaptive(bond_distances, element=None, valence=None, min_gap=0.05):
    """Adaptive coordination number detection with special handling for Jahn-Teller ions."""
    if len(bond_distances) < 2:
        return len(bond_distances)
    
    is_jt = False
    if element and valence:
        is_jt = is_jahn_teller_ion(element, valence)
    
    effective_min_gap = min_gap * 2.0 if is_jt else min_gap
    
    gaps = []
    for i in range(len(bond_distances) - 1):
        gap = bond_distances[i+1] - bond_distances[i]
        if gap >= effective_min_gap:
            gaps.append((gap, i+1))
    
    if not gaps:
        return sum(1 for d in bond_distances if d <= 1.30 * bond_distances[0])
    
    largest_gap, cn = max(gaps, key=lambda x: x[0])
    return cn


# ============================================================================
# ICSD METADATA EXTRACTION WITH TEMPERATURE & PRESSURE
# ============================================================================

def extract_icsd_metadata(cif_path):
    """
    Extract publication metadata AND measurement conditions from ICSD CIF file.
    """
    metadata = {
        'icsd_code': None,
        'creation_date': None,
        'year': None,
        'journal': None,
        'authors': [],
        'r_factor': None,
        'density': None,
        'measurement_temperature_K': None,
        'measurement_pressure_Pa': None,
        'measurement_pressure_GPa': None,
        'measurement_conditions': 'unknown'
    }
    
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        
        in_citation_loop = False
        in_author_loop = False
        citation_columns = []
        author_columns = []
        
        for i, line in enumerate(lines):
            line_stripped = line.strip()
            
            if line_stripped.startswith('_database_code_ICSD'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    metadata['icsd_code'] = parts[1]
            
            elif line_stripped.startswith('_audit_creation_date'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    metadata['creation_date'] = parts[1]
            
            elif line_stripped.startswith('_exptl_crystal_density_diffrn'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        metadata['density'] = float(parts[1])
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_diffrn_ambient_temperature'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        temp_str = parts[1].split('(')[0].strip()
                        metadata['measurement_temperature_K'] = float(temp_str)
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_cell_measurement_temperature'):
                parts = line_stripped.split()
                if len(parts) > 1 and metadata['measurement_temperature_K'] is None:
                    try:
                        temp_str = parts[1].split('(')[0].strip()
                        metadata['measurement_temperature_K'] = float(temp_str)
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('_diffrn_ambient_pressure'):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        pressure_str = parts[1].split('(')[0].strip()
                        pressure_kpa = float(pressure_str)
                        metadata['measurement_pressure_Pa'] = pressure_kpa * 1000
                        metadata['measurement_pressure_GPa'] = pressure_kpa / 1e6
                    except ValueError:
                        pass
            
            elif any(tag in line_stripped for tag in ['_refine_ls_R_factor', '_refine_ls_wR_factor']):
                parts = line_stripped.split()
                if len(parts) > 1:
                    try:
                        r_val = float(parts[1])
                        if metadata['r_factor'] is None or r_val < metadata['r_factor']:
                            metadata['r_factor'] = r_val
                    except ValueError:
                        pass
            
            elif line_stripped.startswith('loop_'):
                in_citation_loop = False
                in_author_loop = False
                citation_columns = []
                author_columns = []
            
            elif line_stripped.startswith('_citation_') and not line_stripped.startswith('_citation_author'):
                in_citation_loop = True
                citation_columns.append(line_stripped)
            
            elif line_stripped.startswith('_citation_author'):
                in_author_loop = True
                in_citation_loop = False
                author_columns.append(line_stripped)
            
            elif in_citation_loop and citation_columns and not line_stripped.startswith('_') and not line_stripped.startswith('#') and line_stripped:
                # Robust extraction handling semicolon-delimited multiline fields
                # (common in ICSD CIFs for long journal names).
                # Collects all field values for the primary citation row,
                # then looks up year and journal by column index.
                import shlex as _shlex
                fields = []
                in_semi = False
                semi_buf = []
                j = i

                while j < len(lines):
                    jline = lines[j].strip()
                    # Stop at next loop_ or column definition (but only after
                    # we've started collecting, to avoid stopping immediately)
                    if j > i and (jline.startswith('loop_') or
                                  (jline.startswith('_') and fields)):
                        break
                    if jline.startswith(';'):
                        if in_semi:
                            fields.append(' '.join(semi_buf))
                            semi_buf = []
                            in_semi = False
                            rest = jline[1:].strip()
                            if rest:
                                try:
                                    fields.extend(_shlex.split(rest))
                                except ValueError:
                                    pass
                        else:
                            in_semi = True
                            after = jline[1:].strip()
                            if after:
                                semi_buf.append(after)
                    elif in_semi:
                        semi_buf.append(jline)
                    elif jline:
                        try:
                            fields.extend(_shlex.split(jline))
                        except ValueError:
                            pass
                    j += 1

                if fields and fields[0] == 'primary':
                    for col_idx, col_name in enumerate(citation_columns):
                        if col_idx < len(fields):
                            if '_citation_year' in col_name:
                                try:
                                    metadata['year'] = int(fields[col_idx])
                                except ValueError:
                                    pass
                            elif '_citation_journal_full' in col_name:
                                metadata['journal'] = fields[col_idx]

                in_citation_loop = False
            
            elif in_author_loop and author_columns and not line_stripped.startswith('_') and not line_stripped.startswith('#') and not line_stripped.startswith('loop_') and line_stripped:
                import shlex
                try:
                    values = shlex.split(line_stripped)
                    for col_idx, col_name in enumerate(author_columns):
                        if '_citation_author_name' in col_name and col_idx < len(values):
                            author = values[col_idx]
                            if author and author not in metadata['authors']:
                                metadata['authors'].append(author)
                except ValueError:
                    pass
            
            elif in_author_loop and (line_stripped.startswith('_') or line_stripped.startswith('loop_')):
                in_author_loop = False
        
        if len(metadata['authors']) > 3:
            metadata['authors'] = metadata['authors'][:3] + ['et al.']
        
        # Classify measurement conditions
        temp_K = metadata['measurement_temperature_K']
        pressure_GPa = metadata['measurement_pressure_GPa']
        
        TEMP_AMBIENT_MAX = 350
        TEMP_HIGH = 500
        PRESSURE_AMBIENT_MAX = 0.0002
        PRESSURE_HIGH = 0.5
        
        if temp_K is not None or pressure_GPa is not None:
            is_high_temp = False
            is_high_pressure = False
            is_ambient_temp = True
            is_ambient_pressure = True
            
            if temp_K is not None:
                if temp_K > TEMP_HIGH:
                    is_high_temp = True
                    is_ambient_temp = False
                elif temp_K > TEMP_AMBIENT_MAX:
                    is_ambient_temp = False
            
            if pressure_GPa is not None:
                if pressure_GPa > PRESSURE_HIGH:
                    is_high_pressure = True
                    is_ambient_pressure = False
                elif pressure_GPa > PRESSURE_AMBIENT_MAX:
                    is_ambient_pressure = False
            
            if is_high_temp and is_high_pressure:
                metadata['measurement_conditions'] = 'high_T_high_P'
            elif is_high_temp:
                metadata['measurement_conditions'] = 'high_T'
            elif is_high_pressure:
                metadata['measurement_conditions'] = 'high_P'
            elif is_ambient_temp and is_ambient_pressure:
                metadata['measurement_conditions'] = 'ambient'
            else:
                metadata['measurement_conditions'] = 'elevated'
        else:
            metadata['measurement_conditions'] = 'unknown'
        
    except Exception as e:
        print(f"  [Warning: Could not extract all metadata: {e}]")
    
    return metadata


# ============================================================================
# GEOMETRY VALIDATION
# ============================================================================

# ============================================================================
# MIXED OCCUPANCY DETECTION & ANALYSIS
# ============================================================================

def extract_site_occupancy_info(data_block, site_label):
    """Extract occupancy information for a specific crystallographic site."""
    try:
        labels = data_block.get('_atom_site_label', [])
        types = data_block.get('_atom_site_type_symbol', [])
        occupancies = data_block.get('_atom_site_occupancy', [])
        
        if not labels or not types:
            return None
        
        site_data = []
        for i, label in enumerate(labels):
            if label.strip() == site_label:
                element = types[i].strip().rstrip('+-0123456789')
                element = element.capitalize() if len(element) > 1 else element.upper()
                
                occ = 1.0
                if occupancies and i < len(occupancies):
                    try:
                        occ_str = occupancies[i].strip()
                        occ = float(occ_str.split('(')[0])
                    except (ValueError, AttributeError):
                        occ = 1.0
                
                site_data.append({
                    'element': element,
                    'occupancy': occ
                })
        
        return site_data if site_data else None
    
    except Exception:
        return None


def analyze_mixed_occupancy_at_site(site_coords, site_data, final_corrected_data, block_name, data_block=None):
    """Analyse mixed occupancy at a crystallographic site."""
    result = {
        'has_mixed_occupancy': False,
        'occupancy_string': 'unknown',
        'n_elements_at_site': 1,
        'disorder_type': 'pure',
        'valence_difference': 0.0,
        'max_valence_at_site': None,
        'min_valence_at_site': None
    }
    
    elements = site_data.get('elements', [])
    if not elements:
        return result
    
    valences = []
    
    if block_name in final_corrected_data:
        for coords, site_info in final_corrected_data[block_name]['sites_with_bonds'].items():
            if coords == site_coords:
                confirmed_valences = site_info.get('confirmed_valences', {})
                for elem in elements:
                    val = confirmed_valences.get(elem)
                    if val is not None:
                        valences.append(abs(val))
                break
    
    if not valences:
        for elem in elements:
            elem_data = site_data.get('element_data', {}).get(elem, {})
            val = elem_data.get('valence')
            if val is not None:
                valences.append(abs(val))
    
    result['n_elements_at_site'] = len(elements)
    
    if valences:
        # v1.8.3 (B5): do not truncate. int() on a fractional valence shifts
        # it downwards, and these feed the Type 4 flag and the
        # SPECTROSCOPY_REQUIRED tier, so Mn3.67+ was judged as Mn3+.
        result['max_valence_at_site'] = max(valences)
        result['min_valence_at_site'] = min(valences)
        result['valence_difference'] = float(max(valences) - min(valences))
    
    if len(elements) > 1:
        result['has_mixed_occupancy'] = True
        
        labels = site_data.get('labels', [])
        actual_occupancies = {}
        
        if data_block is not None and labels:
            for elem in elements:
                found = False
                for site_label in labels:
                    occ_info = extract_site_occupancy_info(data_block, site_label)
                    if occ_info:
                        for entry in occ_info:
                            if entry['element'] == elem:
                                actual_occupancies[elem] = entry['occupancy']
                                found = True
                                break
                    if found:
                        break
                
                if not found and labels:
                    match = re.search(r'(\d+)$', labels[0])
                    if match:
                        suffix = match.group(1)
                        constructed_label = f"{elem}{suffix}"
                        occ_info = extract_site_occupancy_info(data_block, constructed_label)
                        if occ_info:
                            for entry in occ_info:
                                if entry['element'] == elem:
                                    actual_occupancies[elem] = entry['occupancy']
                                    break
        
        occ_parts = []
        if actual_occupancies:
            for elem in elements:
                occ = actual_occupancies.get(elem, 0.0)
                occ_parts.append(f"{elem}:{occ:.3f}")
        else:
            for elem in elements:
                occ = 1.0 / len(elements)
                occ_parts.append(f"{elem}:{occ:.2f}")
        
        result['occupancy_string'] = ','.join(occ_parts)
    else:
        result['occupancy_string'] = f"{elements[0]}:1.00"
    
    if len(elements) == 1:
        result['disorder_type'] = 'pure'
        result['valence_difference'] = 0.0
    elif not valences:
        result['disorder_type'] = 'heterovalent'
        result['valence_difference'] = 0.0
    else:
        if len(set(valences)) == 1:
            result['disorder_type'] = 'homovalent'
            result['valence_difference'] = 0.0
        else:
            result['disorder_type'] = 'heterovalent'
            result['valence_difference'] = float(max(valences) - min(valences))
    
    return result


def calculate_discrepancy_factor(bvs_value, formal_valence):
    """Calculate d1 = (S_i - V_i) from Bosi (2014)."""
    if bvs_value is None or formal_valence is None:
        return None, None, 'unknown'
    
    d1 = bvs_value - abs(formal_valence)
    abs_d1 = abs(d1)
    
    if d1 > 0.05:
        bonding_state = 'overbonded'
    elif d1 < -0.05:
        bonding_state = 'underbonded'
    else:
        bonding_state = 'balanced'
    
    return d1, abs_d1, bonding_state


def assign_confidence_flag(disorder_type, valence_diff, abs_d1, deviation):
    """
    Assign confidence level for BVS result based on disorder and strain.

    v2.0.3: returns NOT_ASSESSED when there is no bond valence sum to be
    confident about. abs_d1 is None exactly when the sum could not be computed,
    and coercing that None to zero put it inside the tightest band: Bi(4+) in
    ICSD 138851, Pd(3+) in 93099 and Gd in 22964 were all reported HIGH while
    carrying no result at all. In 138851 the Sr site, which WAS evaluated and
    came out underbonded, was correctly marked LOW, so the flag was exactly
    inverted on the one page a reader would compare them.
    """
    if abs_d1 is None:
        # No sum, so no confidence. Review is required by definition: something
        # is missing that the site needs.
        return 'NOT_ASSESSED', False, True

    abs_dev = abs(deviation) if deviation is not None else 0
    abs_d1_val = abs_d1

    requires_review = False
    bosi_artifact_likely = False

    if disorder_type == 'heterovalent' and valence_diff >= 3:
        confidence = 'SPECTROSCOPY_REQUIRED'
        bosi_artifact_likely = True
        requires_review = True
        return confidence, bosi_artifact_likely, requires_review

    if abs_dev > 0.30 or abs_d1_val > 0.20:
        requires_review = True

    if disorder_type == 'pure':
        if abs_d1_val < 0.10:
            confidence = 'HIGH'
        elif abs_d1_val < 0.20:
            confidence = 'MEDIUM'
        else:
            confidence = 'LOW'
            requires_review = True

    elif disorder_type == 'homovalent':
        if abs_d1_val < 0.15:
            confidence = 'MEDIUM'
        elif abs_d1_val < 0.25:
            confidence = 'LOW'
        else:
            confidence = 'LOW'
            requires_review = True

    elif disorder_type == 'heterovalent':
        if valence_diff < 2:
            confidence = 'MEDIUM'
        elif valence_diff < 3:
            confidence = 'LOW'
            requires_review = True
        else:
            confidence = 'SPECTROSCOPY_REQUIRED'
            bosi_artifact_likely = True
            requires_review = True

    else:
        confidence = 'LOW'
        requires_review = True

    return confidence, bosi_artifact_likely, requires_review


# ============================================================================
# INITIALISATION MESSAGE
# ============================================================================

print("=" * 80)
print(f"Eir v{EIR_VERSION} helper functions loaded successfully")
print("=" * 80)
print("  - Multi-anion support enabled (O, F, Cl, Br, I, S, Se, Te, etc.)")
print("  - Config file support enabled")
print("  - 3D visualisation enabled (if py3Dmol installed)")
print("  - Adaptive gap detection enabled")
print("  - Temperature/Pressure extraction enabled")
print("  - ICSD metadata extraction with conditions classification")
print("  - Geometry validation enabled")
print("  - Mixed occupancy analysis enabled")
print("  NEW IN v1.4:")
print("  - CIF oxidation state extraction")
print("  - Type 7 error flagging (mixed valence detection)")
print("  - Anion parameter (u) extraction")
print("  - Enhanced structural parameter extraction")
print("=" * 80)

In [ ]:
# CELL 1c: STRUCTURAL SELF-CHECK   (Eir 2.0.12)
#
# Run immediately after CELL 1. Two independent checks: the live objects in
# this session, and the notebook source on disk.
#
# This began as a one-off confirmation that the Stage 1 excisions had removed
# three duplicated definitions. It is now the standing structural check,
# because every subsequent fault in this notebook has been of the same kind
# and none of them was caught:
#
#   - JAHN_TELLER_IONS defined three times and KNOWN_ANIONS twice, with the
#     later copy silently winning
#   - bv_shell_cn and econ_weighted_bvs deleted as a side effect of inserting
#     the adaptive shell beside them; syntax was valid, no name was
#     duplicated, and the only test that exercised them was never called
#   - extract_all_structural_parameters dead, and hiding the fact that four
#     extractors beneath it were dead too
#   - propose_half_lattice_shift's result tuple gaining an element while one
#     of its three readers kept the old index, which a per-site try swallowed
#     into "no proposals across 938 structures"
#
# The source scan needs the file on disk, so SAVE THE NOTEBOOK before running
# this cell. It finds the notebook itself rather than requiring a path: the
# old version defaulted to NOTEBOOK = None and skipped the scan silently,
# which is the same fault it exists to detect.

import json as _json, re as _re
from pathlib import Path as _Path

NOTEBOOK = None      # leave None to auto-detect; set a path to override

# Names the pipeline calls and must therefore be able to find. Add to this
# when a function becomes load-bearing; the cost of a missing entry is a
# deletion that passes every check, which has happened once.
_CORE = [
    'bv_shell_cn', 'bv_shell_cn_adaptive', 'bv_shell_boundary', 'bv_cutoff_for',
    'econ_weighted_bvs', 'expected_bond_length', 'radii_contact_threshold',
    'bvs_distance_threshold', 'normaliser_translations', 'origin_choice_offset',
    'propose_half_lattice_shift', 'anion_anion_contacts', 'resolve_anions',
    'clean_cif_for_gemmi', 'assert_finite_sites', 'normalise_cif_to_ascii',
    '_unquote_atom_site_numbers', 'group_bond_counts', 'is_jahn_teller_ion',
    'whatif_analysis', 'extract_anion_parameter', 'classify_pgrm',
    'provisional_shell_cn', 'assess_site_geometry',
    'may_visualise', 'reset_visualisations',
    'look_at', 'look_at_family',
    'formula_from_cif',
    'apply_pgrm_suppression', '_note_read_failure', 'report_read_failures',
]

# Tripwires. If one of these fails because the change was DELIBERATE, update
# the expected value here in the same commit; that is the point of it.
_EXPECTED_JT = {
    ('Cr', 2), ('Mn', 3), ('Fe', 4),
    ('Cu', 2), ('Ag', 2), ('Ni', 1),
    ('Ni', 3), ('Co', 2), ('Rh', 2), ('Pd', 3),
    ('Mn', 6), ('Cr', 4),
}
_EXPECTED_ANIONS = {
    'O': -2, 'F': -1, 'Cl': -1, 'Br': -1, 'I': -1,
    'S': -2, 'Se': -2, 'Te': -2, 'N': -3, 'P': -3,
    'As': -3, 'H': -1, 'C': -4, 'Si': -4, 'B': -3,
}

_fail, _warn = [], []

# ---- 1. the live objects ---------------------------------------------------
if JAHN_TELLER_IONS != _EXPECTED_JT:
    _fail.append(
        f"JAHN_TELLER_IONS changed: missing {_EXPECTED_JT - JAHN_TELLER_IONS}, "
        f"unexpected {JAHN_TELLER_IONS - _EXPECTED_JT}. If deliberate, update "
        f"_EXPECTED_JT in this cell.")
if KNOWN_ANIONS != _EXPECTED_ANIONS:
    _fail.append(
        f"KNOWN_ANIONS changed: missing {set(_EXPECTED_ANIONS) - set(KNOWN_ANIONS)}, "
        f"unexpected {set(KNOWN_ANIONS) - set(_EXPECTED_ANIONS)}. If deliberate, "
        f"update _EXPECTED_ANIONS in this cell.")
if ANION_FORMERS != set(ALWAYS_ANION) | set(AMPHOTERIC_ANION):
    _fail.append("ANION_FORMERS is no longer the union of ALWAYS_ANION "
                 "and AMPHOTERIC_ANION")

# behaviour, not just content
if not is_jahn_teller_ion('Mn', 3):
    _fail.append("is_jahn_teller_ion('Mn', 3) should be True")
if is_jahn_teller_ion('Ti', 4):
    _fail.append("is_jahn_teller_ion('Ti', 4) should be False")
if not is_jahn_teller_ion('Mn', 3.3):
    _fail.append("is_jahn_teller_ion no longer rounds fractional valences")

# ---- 2. the notebook source ------------------------------------------------
_path = _Path(NOTEBOOK) if NOTEBOOK else None
if _path is None:
    _here = _Path.cwd()
    _cands = [p for p in sorted(_here.glob('*.ipynb'))
              if 'EIR_VERSION' in p.read_text(encoding='utf-8', errors='ignore')]
    _path = max(_cands, key=lambda p: p.stat().st_mtime) if _cands else None

if _path is None or not _path.exists():
    _fail.append("Source scan could not run: no saved notebook found in "
                 f"{_Path.cwd()}. Save the notebook, or set NOTEBOOK. A check "
                 "that silently does not run is worse than no check.")
else:
    _nb = _json.loads(_path.read_text(encoding='utf-8'))
    _cells = [(c['source'] if isinstance(c.get('source'), str)
               else ''.join(c.get('source', [])))
              for c in _nb.get('cells', []) if c.get('cell_type') == 'code']
    _src = '\n'.join(_cells)
    print(f"  Source scanned: {_path.name}  "
          f"({len(_cells)} code cells, {len(_src) / 1024:.0f} kB)")

    # 2a. duplicate top-level definitions
    _seen = {}
    for _ci, _cs in enumerate(_cells):
        for _m in _re.finditer(r'^(?:def (\w+)\s*\(|([A-Z][A-Z0-9_]{2,})\s*=)',
                               _cs, _re.M):
            _n = _m.group(1) or _m.group(2)
            _seen.setdefault(_n, []).append(_ci)
    _dups = {k: v for k, v in _seen.items() if len(v) > 1}
    if _dups:
        for _k, _v in sorted(_dups.items()):
            _fail.append(f"{_k} defined {len(_v)} times, cells {_v}")

    # 2b. core names that must exist
    for _n in _CORE:
        if f'def {_n}(' not in _src:
            _fail.append(f"{_n} is called by the pipeline but no longer defined")

    # 2c. called but never defined, anywhere
    _defs = set(_re.findall(r'^def (\w+)', _src, _re.M))
    _builtin = set(dir(__builtins__)) if not isinstance(__builtins__, dict) \
        else set(__builtins__)
    _called = set(_re.findall(r'(?<![\w.])(_?[a-z]\w{3,})\s*\(', _src))
    _unknown = sorted(n for n in _called - _defs - _builtin
                      if n in _CORE)
    for _n in _unknown:
        _fail.append(f"{_n} called but never defined")

    # 2d. defined and never called: reported, not failed. Entry points and
    #     validators legitimately have no caller inside the notebook.
    # A function passed as an OBJECT - in a dispatch tuple, as a callback -
    # is used without ever being followed by '('. Counting only call syntax
    # reported the four figure functions as dead when eir_figures dispatches
    # over them by reference.
    _orphans = sorted(
        n for n in _defs
        if len(_re.findall(r'(?<!def )' + _re.escape(n) + r'\s*\(', _src)) == 0
        and len(_re.findall(r'(?<!def )(?<![\w.])' + _re.escape(n)
                            + r'(?!\s*\()(?![\w])', _src)) == 0)
    if _orphans:
        _warn.append(f"{len(_orphans)} function(s) defined and never called: "
                     f"{', '.join(_orphans[:8])}"
                     + (' ...' if len(_orphans) > 8 else ''))

    # 2e. error swallowing
    _bare = len(_re.findall(r'\n\s*except:\s*\n', _src))
    _pass = len(_re.findall(r'except Exception:\s*\n\s*pass', _src))
    if _bare:
        _fail.append(f"{_bare} bare `except:` clause(s): a defect in the code "
                     f"becomes an empty result with no message")
    if _pass:
        _warn.append(f"{_pass} `except Exception: pass` clause(s)")

# ---- verdict ---------------------------------------------------------------
print()
if _fail:
    print("STRUCTURAL SELF-CHECK FAILED")
    for _f in _fail:
        print(f"  - {_f}")
else:
    print("STRUCTURAL SELF-CHECK PASSED")
    print(f"  JAHN_TELLER_IONS   {len(JAHN_TELLER_IONS)} ion pairs")
    print(f"  KNOWN_ANIONS       {len(KNOWN_ANIONS)} species")
    print(f"  ANION_FORMERS      {len(ANION_FORMERS)} species")
    print(f"  core names present {len(_CORE)}")
if _warn:
    print()
    for _w in _warn:
        print(f"  note: {_w}")


In [ ]:
# CELL 1b: OUTPUT CONTROL AND GEOMETRY CORE   (Eir 2.0.13)
# ============================================================================
# RENUMBERED IN 2.0.2. This cell and the main functionality cell were both
# labelled "CELL 2", which is how a replacement for one ends up pasted over the
# other. The names below are what the main cell reports missing when that
# happens: _dbg, _ANION_SET, _EIR_GEOMETRY and assess_site_geometry.
#
# Run order: CELL 1 (helpers) -> CELL 1b (this) -> CELL 2 (main) ->
#            CELL 3 (anion BVS) -> ... -> CELL 7 (coordination quality)
#
# Contents:
#   - output control: VERBOSE and _dbg, plus the warning filters
#   - the parser-agnostic assessment core: ECoN, shell CN, geometry
#     classification, distortion indices, polyhedral volume, off-centring,
#     bond-valence vector sum and continuous shape measure
#   - assess_site_geometry, which the main cell calls per site, and the
#     _EIR_GEOMETRY dict it writes into
#
# The coordination shell guard lives in CELL 2, beside the bond valence sum
# it protects.
#
# WHEN THIS RUNS, AND WHAT IT THEREFORE CANNOT KNOW
#
# assess_site_geometry is called from calculate_bonds_from_positions, while
# the neighbour list is being built and BEFORE any valence is assigned. It
# cannot use the canonical coordination number, the bond valence parameters,
# or the per-bond valences. Two consequences used to be silent and are now
# explicit: the shell here is PROVISIONAL, and the vector sum here is
# geometric.
# ----------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Output control: quiet by default. Set VERBOSE=True for the
# per-structure debug stream (asymmetric-unit site lists, supercell neighbour
# dumps, bond-pair/position debug). The per-structure result line, BVS-deviation
# warnings, and 3D visualisations are always shown.
# ---------------------------------------------------------------------------
import warnings as _warnings
_warnings.filterwarnings('ignore', category=DeprecationWarning)
_warnings.filterwarnings('ignore', message='.*occupancies.*')
_warnings.filterwarnings('ignore', message='.*Issues encountered while parsing CIF.*')

VERBOSE = False

# 3D VISUALISATION. Off by default since v2.0.13.
#
# Each viewer is a live WebGL context in the notebook front end, and the
# trigger was |deviation| > 0.30 with no cap: 316 viewers on the perovskite
# set, 501 on the pyrochlores, 3537 on the spinels. Browsers hard-limit live
# WebGL contexts to something like 8 to 16, and a page demanding thousands
# locks up or crashes. Each viewer's geometry is also stored in the notebook's
# output, so the .ipynb grows and every save and scroll gets slower.
#
# Set VISUALISE_SITES = True for a small batch where you want to look at the
# geometry. VISUALISE_MAX caps it regardless, so a run cannot take the front
# end down by accident.
VISUALISE_SITES = False
VISUALISE_MAX = 12
_VISUALISED = [0]         # count for this run; reset by reset_visualisations()


def reset_visualisations():
    _VISUALISED[0] = 0


def may_visualise():
    """True if another 3D viewer may be drawn in this run."""
    if not VISUALISE_SITES:
        return False
    if _VISUALISED[0] >= VISUALISE_MAX:
        if _VISUALISED[0] == VISUALISE_MAX:
            _VISUALISED[0] += 1        # so this prints once
            print(f'  [3D visualisation stopped at {VISUALISE_MAX} viewers. '
                  f'Raise VISUALISE_MAX if you need more, but a browser will '
                  f'not hold many more than this.]')
        return False
    _VISUALISED[0] += 1
    return True
def _dbg(*a, **k):
    if VERBOSE:
        print(*a, **k)

"""
Eir 2.0.x assessment core - PARSER-AGNOSTIC.
Every function takes plain (x,y,z) cartesian tuples, so it runs identically on
gemmi positions (Eir's native parse) or any other source. No pymatgen, no I/O.
The Eir pipeline feeds it `site_orth` as the centre and `unique_neighbors`
(each with a cartesian position) as ligands; it returns the standard per-site
schema.
"""
import numpy as np, itertools

# ---------- coordination ----------
def econ(distances, max_iter=25, tol=1e-9):
    """Hoppe (1979) effective coordination number from bond distances."""
    d = np.asarray([x for x in distances if x > 0], float)
    if len(d) == 0: return 0.0, 0.0
    dmin = d.min(); d_av = dmin
    for _ in range(max_iter):
        w = np.exp(1.0 - (d/d_av)**6)
        new = (d*w).sum()/w.sum()
        if abs(new-d_av) < tol: break
        d_av = new
    w = np.exp(1.0 - (d/d_av)**6)
    return float(w.sum()), float(d_av)

def provisional_shell_cn(distances, factor=1.35, max_cut=3.5):
    """
    PROVISIONAL integer CN by hybrid cutoff, for use before valences exist.

    This is not the canonical coordination number. Eir's canonical shell is
    bond-valence based (bv_shell_cn in CELL 1, boundary from the ion's own
    parameters) and cannot be computed here because the valence is not yet
    known. The two disagree on 706 of 1895 perovskite sites, 37%, with the
    canonical shell larger by 1 to 4.

    That disagreement was invisible while every geo_* column came from this
    rule and no column recorded which rule produced it. Callers that know the
    canonical CN should pass it to assess_site_geometry; geo_cn_rule records
    which was used.

    The max_cut of 3.5 A is the same fixed absolute distance that gave Li(1+)
    a coordination number of 16 under a fixed bond-valence cut-off: one radius
    suits no ion in particular. It survives here only because this shell is
    provisional and is superseded downstream.
    """
    d = np.sort(np.asarray([x for x in distances if x > 0], float))
    if len(d) < 2: return len(d)
    d_ref = d[1] if d[1]/d[0] > 1.25 else d[0]
    cut = min(factor*d_ref, max_cut)
    return max(int((d <= cut).sum()), 2)

# ---------- angles + geometry ----------
def _unit_vecs(centre, ligs):
    c = np.asarray(centre, float)
    v = [np.asarray(p, float)-c for p in ligs]
    return [x/np.linalg.norm(x) for x in v if np.linalg.norm(x) > 0]

def _angles(centre, ligs):
    u = _unit_vecs(centre, ligs)
    return sorted(float(np.degrees(np.arccos(np.clip(np.dot(a,b),-1,1))))
                  for a,b in itertools.combinations(u,2))

def classify_geometry(centre, ligs):
    cn = len(ligs); ang = _angles(centre, ligs)
    if not ang: return 'unknown', None
    nt = sum(1 for a in ang if a > 150)
    if cn == 2:  return ('linear' if nt else 'bent'), (180.0 if nt else None)
    if cn == 3:  return ('trigonal planar' if max(ang) >= 115 else 'trigonal pyramidal'), 120.0
    if cn == 4:  return ('square planar', 90.0) if nt >= 1 else ('tetrahedral', 109.47)
    if cn == 5:  return ('trigonal bipyramidal' if nt >= 1 else 'square pyramidal'), None
    if cn == 6:  return ('octahedral', 90.0) if nt >= 2 else ('trigonal prismatic', None)
    if cn == 7:  return 'capped octahedral/prismatic', None
    if cn == 8:  return ('cubic/hex-bipyramidal' if nt >= 3 else 'square antiprism/dodecahedral'), None
    if cn == 12: return 'cuboctahedral/icosahedral', None
    return 'CN%d polyhedron' % cn, None

# ---------- distortion indices ----------
def distortion_indices(bonds, centre, ligs, ideal_cis):
    b = np.asarray(bonds, float); bm = b.mean()
    baur = float(np.mean(np.abs(b-bm)/bm))
    qe = float(np.mean((b/bm)**2))
    srt = np.sort(b); spread = float(srt[-1]-srt[0])
    iso = 0.0 if (len(srt) < 3 or spread < 1e-6) else float(max(srt[1]-srt[0], srt[-1]-srt[-2])/spread)
    bav = None
    if ideal_cis is not None:
        cis = [a for a in _angles(centre, ligs) if a < 150]
        if len(cis) > 1:
            bav = float(sum((a-ideal_cis)**2 for a in cis)/(len(cis)-1))
    return {'baur_index': baur, 'quadratic_elongation': qe,
            'bond_angle_variance': bav, 'within_site_isolation': iso}

# ---------- NEW metrics ----------
def polyhedral_volume(ligs):
    """Convex-hull volume of the ligand vertices (A^3)."""
    if len(ligs) < 4: return None
    try:
        from scipy.spatial import ConvexHull
        return float(ConvexHull(np.asarray(ligs, float)).volume)
    except Exception:
        return None

def volume_distortion(centre, ligs, geometry):
    """
    Balic-Zunic/Makovicky-style: (V_ideal - V_real)/V_ideal, where V_ideal is
    the ideal polyhedron of the same mean centre-vertex radius. Uses the
    classified geometry's ideal vertex directions.
    """
    ideal = _ideal_vertices(geometry, len(ligs))
    if ideal is None or len(ligs) < 4: return None
    v_real = polyhedral_volume(ligs)
    c = np.asarray(centre, float)
    r = np.mean([np.linalg.norm(np.asarray(p,float)-c) for p in ligs])
    v_ideal = polyhedral_volume([c + r*np.asarray(u) for u in ideal])
    if not v_real or not v_ideal: return None
    return float((v_ideal - v_real)/v_ideal)

def off_centering(centre, ligs):
    """Distance from the cation to the centroid of its ligands (A).
    Captures 2nd-order Jahn-Teller / ferroelectric off-centre displacement."""
    c = np.asarray(centre, float)
    centroid = np.mean(np.asarray(ligs, float), axis=0)
    return float(np.linalg.norm(c - centroid))

def bv_vector_sum(centre, ligs, valences):
    """
    Magnitude of the bond-valence vector sum |Sum s_i * u_i|. Near zero for a
    symmetric site; large for a lone-pair / off-centre cation. `valences` are
    the per-bond bond valences s_i (from Eir's BVS). Falls back to unit vectors
    (geometric eccentricity) if valences are None.
    """
    c = np.asarray(centre, float)
    us = []
    for p, s in zip(ligs, (valences if valences is not None else [1.0]*len(ligs))):
        v = np.asarray(p, float) - c; n = np.linalg.norm(v)
        if n > 0: us.append((s if s is not None else 1.0) * v/n)
    if not us: return None
    return float(np.linalg.norm(np.sum(us, axis=0)))

def _ideal_vertices(geometry, cn):
    """Unit vertex directions for the common ideal polyhedra."""
    import math
    g = (geometry or '').split('/')[0]
    if g == 'octahedral' or (cn == 6 and g == 'octahedral'):
        return [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
    if g == 'tetrahedral':
        a = 1/math.sqrt(3); return [(a,a,a),(-a,-a,a),(-a,a,-a),(a,-a,-a)]
    if g == 'square planar':
        return [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0)]
    if g == 'trigonal prismatic':
        import math
        h = 0.5; out=[]
        for z in (h,-h):
            for k in range(3):
                th = 2*math.pi*k/3
                out.append((math.cos(th)*math.sqrt(1-h*h), math.sin(th)*math.sqrt(1-h*h), z))
        return out
    return None

def cshm(centre, ligs, geometry, iters=6):
    """
    Continuous shape measure vs the classified ideal polyhedron:
    100 * min over rotation+permutation of mean|q_i - p_i|^2 / mean|q_i|^2.
    0 = perfect; grows with distortion. Assignment-based (Hungarian + Kabsch,
    iterated) - identical values to the brute permutation search, ~80x faster.
    """
    ideal = _ideal_vertices(geometry, len(ligs))
    if ideal is None: return None
    c = np.asarray(centre, float)
    Q = np.asarray([np.asarray(p, float) - c for p in ligs], float)
    if len(Q) != len(ideal): return None
    try:
        from scipy.optimize import linear_sum_assignment
    except Exception:
        return None
    P0 = np.asarray(ideal, float)
    Q = Q / np.sqrt((Q**2).sum(1).mean())
    P0 = P0 / np.sqrt((P0**2).sum(1).mean())
    R = np.eye(3); best = None
    for _ in range(iters):
        P = P0 @ R.T
        D = ((Q[:, None, :] - P[None, :, :])**2).sum(2)
        _, ci = linear_sum_assignment(D)
        Pm = P0[ci]
        H = Pm.T @ Q; U, _, Vt = np.linalg.svd(H)
        d = np.sign(np.linalg.det(Vt.T @ U.T))
        R = Vt.T @ np.diag([1, 1, d]) @ U.T
        s = 100.0 * ((Q - Pm @ R.T)**2).sum() / (Q**2).sum()
        best = s if best is None else min(best, s)
    return float(best)

# ===========================================================================
# Eir 2.0.x native geometry assessment - fed from gemmi neighbours in
# calculate_bonds_from_positions, read back when site_result is built.
# ===========================================================================
# Alias of PROVISIONAL_ANIONS (CELL 1), where the reasoning lives.
_ANION_SET = PROVISIONAL_ANIONS

# Thresholds for is_distorted. NOT MEASURED: inherited from v1.x and never
# tested against a dataset, unlike ECON_REGULAR_MIN, ANION_ANION_MIN and
# CATION_CATION_MIN, each of which carries its measurement beside it. Treat
# is_distorted as indicative until these are set the same way.
DISTORTED_BAV_MIN = 25.0      # bond angle variance, deg^2
DISTORTED_BAUR_MIN = 0.02     # Baur index
_EIR_GEOMETRY = {}

def assess_site_geometry(centre, anion_neighbors, element, cn=None,
                         valences=None):
    """
    Geometry of one coordination polyhedron.

    cn:       the canonical coordination number, if the caller knows it. When
              None a provisional shell is used and geo_cn_rule says so.
    valences: per-bond bond valences s_i, if known. When None the vector sum
              is geometric and is reported as geometric_eccentricity rather
              than as a bond-valence quantity.
    """
    an = sorted(anion_neighbors, key=lambda t: t[0])
    dists = [d for d, _ in an]
    if cn is not None and 2 <= int(cn) <= len(dists):
        cn = int(cn)
        cn_rule = 'canonical'
    else:
        cn = provisional_shell_cn(dists)
        cn_rule = 'provisional'
    ligs = [p for _, p in an[:cn]]
    bonds = dists[:cn]
    if len(ligs) < 2:
        return {}
    geometry, ideal = classify_geometry(centre, ligs)
    di = distortion_indices(bonds, centre, ligs, ideal)
    ec, _ = econ(bonds)
    ang = _angles(centre, ligs)
    bmin, bmax = min(bonds), max(bonds)
    pv = polyhedral_volume(ligs); vd = volume_distortion(centre, ligs, geometry)
    cs = cshm(centre, ligs, geometry); bav = di['bond_angle_variance']
    return {
        'geo_cn': cn, 'geo_econ': round(ec, 3), 'geometry': geometry,
        'is_distorted': bool((bav or 0) > DISTORTED_BAV_MIN
                             or di['baur_index'] > DISTORTED_BAUR_MIN),
        'geo_bond_min': round(bmin, 4), 'geo_bond_max': round(bmax, 4),
        'geo_bond_mean': round(sum(bonds)/len(bonds), 4),
        'geo_bond_spread': round(bmax/bmin, 4),
        'baur_index': round(di['baur_index'], 5),
        'quadratic_elongation': round(di['quadratic_elongation'], 5),
        'bond_angle_variance': round(bav, 2) if bav is not None else None,
        'collapse_ratio': round(bmin/bmax, 4),
        'within_site_isolation': round(di['within_site_isolation'], 4),
        'angle_min': round(min(ang), 2) if ang else None,
        'angle_max': round(max(ang), 2) if ang else None,
        'polyhedral_volume': round(pv, 4) if pv is not None else None,
        'volume_distortion': round(vd, 5) if vd is not None else None,
        'off_centering': round(off_centering(centre, ligs), 4),
        'geo_cn_rule': cn_rule,
        # v2.0.13: this was exported as 'bv_vector_sum' on every site while
        # only ever being called with valences=None, so the column named for
        # the bond-valence vector sum held the geometric eccentricity - the
        # magnitude of a sum of UNIT vectors. Both are useful; they are not
        # the same quantity, and a sum of unit vectors is not a bond valence.
        'geometric_eccentricity': round(bv_vector_sum(centre, ligs, None), 4),
        'bv_vector_sum': (round(bv_vector_sum(centre, ligs, valences[:cn]), 4)
                          if valences is not None else None),
        'cshm': round(cs, 4) if cs is not None else None,
    }


# ============================================================================
# SELF-CHECK
# ============================================================================
# Reaching this proves the cell completed. The main cell checks for exactly
# these four names, so a mismatch here and there is caught at the source
# rather than several cells downstream.

def _eir_geometry_core_self_check(verbose=True):
    required = ['VERBOSE', '_dbg', '_ANION_SET', '_EIR_GEOMETRY',
                'assess_site_geometry', 'econ', 'provisional_shell_cn',
                'classify_geometry',
                'distortion_indices', 'polyhedral_volume', 'volume_distortion',
                'off_centering', 'bv_vector_sum', 'cshm']
    missing = [n for n in required if n not in globals()]
    if missing:
        raise NameError(
            f"This cell did not finish: {', '.join(missing)} undefined. "
            f"Scroll up for the real error and re-run it in full.")

    # A regular octahedron is the one case where every measure has an exact
    # known answer, so it is worth asserting rather than assuming.
    c = (0.0, 0.0, 0.0)
    ligs = [(2.0, 0, 0), (-2.0, 0, 0), (0, 2.0, 0),
            (0, -2.0, 0), (0, 0, 2.0), (0, 0, -2.0)]
    g = assess_site_geometry(c, [(2.0, p) for p in ligs], 'Mg')
    problems = []
    if g.get('geo_cn') != 6:
        problems.append(f"geo_cn {g.get('geo_cn')} (expected 6)")
    # `x or default` is wrong here: a correct result of 0.0 is falsy and would
    # be replaced by the default, failing the very case it is meant to pass.
    def _near(key, target, tol=1e-6):
        v = g.get(key)
        if v is None:
            problems.append(f'{key} missing (expected {target})')
        elif abs(float(v) - target) > tol:
            problems.append(f'{key} {v} (expected {target})')

    _near('geo_econ', 6.0)
    _near('baur_index', 0.0, 1e-9)
    _near('off_centering', 0.0, 1e-9)
    _near('quadratic_elongation', 1.0, 1e-9)
    _near('geometric_eccentricity', 0.0, 1e-9)
    if g.get('geo_cn_rule') != 'provisional':
        problems.append('geo_cn_rule should be provisional when no cn given')
    if g.get('bv_vector_sum') is not None:
        problems.append('bv_vector_sum should be None when no valences given')
    _g2 = assess_site_geometry(c, [(2.0, p) for p in ligs], 'Mg', cn=6,
                               valences=[0.5] * 6)
    if _g2.get('geo_cn_rule') != 'canonical':
        problems.append('passing cn did not set geo_cn_rule to canonical')
    if _g2.get('geo_cn') != 6:
        problems.append('geo_cn wrong with cn=6 passed')
    if _g2.get('bv_vector_sum') is None:
        problems.append('bv_vector_sum still None when valences supplied')
    elif abs(_g2['bv_vector_sum']) > 1e-9:
        problems.append('bv_vector_sum should be 0 on a regular octahedron')
    if g.get('geometry') != 'octahedral':
        problems.append(f"geometry {g.get('geometry')!r} (expected 'octahedral')")
    if problems:
        raise AssertionError(
            'geometry core is loaded but wrong on a regular octahedron: '
            + '; '.join(problems))

    if verbose:
        print('Eir 2.0.13 CELL 1b loaded and self-checked.')
        print(f'  output control: VERBOSE = {VERBOSE}')
        print('  geometry core verified against a regular octahedron '
              '(CN 6, ECoN 6.000, zero distortion).')
        print('  Next: CELL 2 (main functionality).')
    return True


_eir_geometry_core_self_check()

In [ ]:
# CELL 2: MAIN EIR FUNCTIONALITY   (Eir 2.0.13)
# ============================================================================
# This replaces the cell beginning "# CELL 2: MAIN EIR FUNCTIONALITY".
#
# It does NOT replace the other cell that was also labelled CELL 2, the one
# beginning "# CELL 2: OUTPUT CONTROL". That cell defines VERBOSE, _dbg,
# _ANION_SET, _EIR_GEOMETRY and assess_site_geometry, all of which this cell
# needs, and it is renumbered CELL 1b in 2.0.2 so the collision cannot recur.
#
# Run order: CELL 1 (helpers) -> CELL 1b (output control and geometry core) ->
#            CELL 2 (this) -> CELL 3 (anion BVS) -> ... ->
#            CELL 7 (coordination quality)
#
# CHANGED IN 2.0.2 - COORDINATION SHELL DETERMINATION
# ---------------------------------------------------
# `auto_analyze_coordination` set the coordination number from the largest gap
# anywhere in the sorted bond list. For a compact polyhedron that gap is the
# shell boundary and the rule is right. For a broad, distorted shell it is not:
# the twelve A-O distances of a GdFeO3-type perovskite span roughly 2.4-3.2 A,
# and an internal gap can exceed the gap at the boundary. The cut then
# amputates the shell, and because bond valence sums over the shell, the result
# is a spuriously LOW BVS indistinguishable from genuine underbonding.
#
# Measured on 934 ICSD perovskites: 330 sites (322 structures) had cn below
# cn_relative and 472 below geo_cn, with a median BVS deviation of -10.4%
# against -1.4% elsewhere, and 43% of the structures with GII > 0.20 contained
# one. Eight NaNbO3/NaTaO3 entries were assigned CN = 3 where cn_relative =
# 10-12, geo_cn = 10-12 and ECoN = 8.4-8.7, giving Na BVS of 0.45-0.54 v.u.
# and an apparent -50% deviation that is pure artefact.
#
# The existing Jahn-Teller axial recovery is the same failure caught as a
# special case for one element set and one CN range. It is retained, and
# `guarded_shell_cn` below generalises it: ECoN is bond-length weighted and
# cannot substantially exceed the true coordination number, so it is a valid
# lower bound on shell size. Where the gap cut sits more than
# ECON_SHELL_TOLERANCE below the ECoN of the full candidate shell, the cut is
# extended by the minimum needed.
#
# Nothing is changed silently: cn_gap, cn_revised, cn_revision_reason,
# econ_shell and cn_shell_max are exported on every site.
#
# Also in 2.0.2:
#   - structure-level core_geometry_plausible is now exported. It was computed
#     but never written to the CSV, so the column of that name held the
#     PER-SITE value while structure_bvs_geometry_plausible and
#     anion_packing_plausible were per structure. Anything reading all three
#     the same way was wrong: groupby('filename').first() misses the failure
#     whenever the bad site is not row one.
#   - `cn_fault` is defined here as a module-level function and made symmetric.
#     Replace the `_cn_fault` closure in CELL 7 (add_coordination_quality) with
#     a call to it; see the note beside the definition.
#
# Run VALIDATE_CN() after pasting this cell.

"""
Eir Batch Processor - Automated Bond Valence Sum Analysis
================================================================

NEW IN v1.7:
- Data integrity tier system: ACCEPT/FLAG/EXPUNGE classification per site and structure
- EXPUNGE (I6): Detects implausible BVS values (>10 v.u. or <0.2 v.u. for valence >1)
- EXPUNGE (I3): Detects minority species (<5% occupancy on majority-occupied sites)
- FLAG: Detects mixed TM occupancy with non-integer valence partitioning
- New output columns: integrity_tier, flag_reason, occupancy, is_minority_species,
  is_mixed_occupancy_tm, cif_valence, formal_valence_from_symbol, symbol_number_agreement,
  structure_tier, n_sites_flagged, n_sites_expunged
- Fixed version banner (was stuck at "v1.5")

PREVIOUS (v1.5/v1.6.1):

NEW IN v1.4:
- CIF oxidation state extraction (automatic from _atom_type_oxidation_number)
- Type 7 error flags for mixed valence detection
- Anion parameter (u) extraction for spinels
- Enhanced structural parameter extraction
- Priority: config.json > CIF explicit > CIF inferred > interactive prompt

NEW IN v1.3:
- Improved handling of framework formers (Ge, Si, B, P)
- Parameter-based bond selection replaces hardcoded metal lists
- Auto-detects ALL cation-anion pairs with available BVS parameters
- Future-proof against new elements/parameter additions

NEW IN v1.2:
- Multi-anion support: Handles O, F, Cl, Br, I, S, Se, Te, etc.
- Auto-detects M-X bonds (not just M-O)
- Merged BVS parameters (Gagne & Hawthorne 2015 + Brown 2020)
- 1,385 M-X parameter pairs vs 168 M-O pairs (8x expansion)
- Temperature/Pressure extraction from CIF files
- Automatic classification of measurement conditions

NEW IN v1.1:
- Config file support (config.json) for unattended batch processing
- Automatic 3D visualisation of problematic sites (|deviation| > 0.3 v.u.)
- Adaptive gap detection for Jahn-Teller active ions
- Enhanced output with deviation column

OUTPUT:
All results saved to ./outputs/ subfolder:
- eir_batch_results_v<version>_YYYYMMDD_HHMMSS.csv        per site, raw
- oxidation_states_used_YYYYMMDD_HHMMSS.json
- eir_batch_results_v<version>_..._quality.csv            per site, screened
        - eir_batch_results_v<version>_..._structures.csv         per structure
        - eir_batch_failed_v<version>_YYYYMMDD_HHMMSS.csv         parse failures

CITATIONS:
Gagné & Hawthorne (2015) Acta Cryst. B71, 561-578
Brown (2020) IUCr Bond Valence Parameters
Brown & Altermatt (1985) Acta Cryst. B41, 244-247
Brese & O'Keeffe (1991) Acta Cryst. B47, 192-197
"""

import os
import sys
import json
import re
import math
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# CIF parsing
try:
    from CifFile import ReadCif
except ImportError as _e:
    # v2.0.2: was sys.exit(1). In a notebook that raises SystemExit, which
    # terminates the CELL rather than the process, leaving every function
    # below this line undefined. The failure then surfaces several cells later
    # as a NameError on something unrelated, pointing at the wrong place
    # entirely. Raise instead, so the error names its own cause.
    raise ImportError(
        "PyCifRW is required by this cell. Install it with "
        "`pip install pycifrw`, then re-run this cell in full. Until it is "
        "installed nothing below this line is defined, and later cells will "
        "fail with NameError on names that live here.") from _e

# gemmi handles ICSD CIF parsing and crystallographic symmetry natively
# PYMATGEN_AVAILABLE removed at v2.0.13: the flag was set False and never
# read. The native pipeline uses gemmi throughout.

# py3Dmol for 3D visualisation
try:
    import py3Dmol
    from IPython.display import display
    PY3DMOL_AVAILABLE = True
    print("✓ py3Dmol available - 3D visualisation enabled for problematic sites")
except ImportError as e:
    PY3DMOL_AVAILABLE = False
    print("⚠ py3Dmol not available - Install with: pip install py3Dmol")
    print(f"   Import error: {e}")
except Exception as e:
    PY3DMOL_AVAILABLE = False
    print("⚠ py3Dmol import failed - Unexpected error")
    print(f"   Error: {e}")

# ============================================================================
# DATABASE MANAGER (From Skuld 1.5 - UNCHANGED)
# ============================================================================

class DatabaseManager:
    """Manages external JSON databases for ionic radii and bond valence parameters"""
    
    def __init__(self, data_dir="./Data"):
        self.data_dir = data_dir
        self.shannon_radii = None
        self.bond_valence_params = None
        self.oxidation_states = None
        self.element_categories = None
        
        if not os.path.exists(self.data_dir):
            print(f"Warning: Data directory '{self.data_dir}' not found!")
            print("Please ensure your JSON database files are in the ./Data folder")
    
    def load_shannon_radii(self):
        """Load Shannon-Prewitt ionic radii from JSON file"""
        if self.shannon_radii is not None:
            return self.shannon_radii
            
        filepath = os.path.join(self.data_dir, "shannon_radii.json")
        
        if not os.path.exists(filepath):
            print(f"Shannon radii file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.shannon_radii = {}
                for element, valence_dict in data.items():
                    for valence_str, cn_dict in valence_dict.items():
                        valence = int(valence_str)
                        for cn_str, radius in cn_dict.items():
                            cn = int(cn_str)
                            self.shannon_radii[(element, valence, cn)] = radius
                
            print(f"✓ Loaded Shannon radii database with {len(self.shannon_radii)} entries")
        except Exception as e:
            print(f"✗ Error loading Shannon radii: {e}")
            return None
        
        return self.shannon_radii
    
    def load_bond_lengths(self):
        """
        Load measured mean bond lengths from Gagne's aggregated bond-length data.

        596 ion configurations, 202 ions, bonded to O2- and N3-, from the largest
        bond-length dispersion analysis published for inorganic crystals. These
        are MEASURED distances, not radii to be added: Gagne states plainly that
        they are more accurate and comprehensive than values obtained by adding
        ionic radii and should be used in their stead wherever possible, and
        G&H (2020) sec 7 makes the same argument from their own data.

        Sources, by the ref key on each entry:
          a  G&H (2016) Acta Cryst B72, 602-625   alkali and alkaline earths
          b  Gagne (2018) Acta Cryst B74, 49-62   lanthanides and actinides
          c  G&H (2018a) Acta Cryst B74, 63-78    metalloids, post-transition
          d  G&H (2018b) Acta Cryst B74, 79-96    non-metals
          e  G&H (2020) IUCrJ 7, 581-629          transition metals
          f  Gagne (2020) ChemRxiv 11626974       nitrides

        Keyed (element, valence, anion, cn) to match the other two databases.
        """
        if getattr(self, 'bond_lengths', None) is not None:
            return self.bond_lengths

        filepath = os.path.join(self.data_dir, "gagne_bond_lengths.json")
        if not os.path.exists(filepath):
            print(f"Bond-length file not found at {filepath}")
            print("  Falling back to Shannon additivity throughout. Expect a "
                  "coordination-dependent bias on large, weakly bonded cations; "
                  "see "
                  "RADII_CN_SLOPE.")
            self.bond_lengths = {}
            return self.bond_lengths

        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
            self.bond_lengths = {}
            for element, vd in data.items():
                if element.startswith('_'):
                    continue
                for valence_str, ad in vd.items():
                    for anion, cnd in ad.items():
                        for cn_str, rec in cnd.items():
                            self.bond_lengths[(element, int(valence_str),
                                               anion, int(cn_str))] = rec
            print(f"\u2713 Loaded measured bond lengths with "
                  f"{len(self.bond_lengths)} configurations")
        except Exception as e:
            print(f"\u2717 Error loading bond lengths: {e}")
            self.bond_lengths = {}
        return self.bond_lengths

    def load_bond_valence_params(self):
        """Load Gagné & Hawthorne bond valence parameters from JSON file"""
        if self.bond_valence_params is not None:
            return self.bond_valence_params
            
        filepath = os.path.join(self.data_dir, "bvs_parameters.json")
        
        if not os.path.exists(filepath):
            print(f"BVS parameters file not found at {filepath}")
            return None
        
        try:
            import ast
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.bond_valence_params = {}
                for key, params in data.items():
                    # Keys are string representations of tuples: "('Na', 1, 'O')"
                    try:
                        cation, cation_valence, anion = ast.literal_eval(key)
                        self.bond_valence_params[(cation, cation_valence, anion)] = {
                            'R0': params['r0'],  # Note lowercase in JSON
                            'B': params.get('b', 0.37)  # Note lowercase in JSON
                        }
                    except (ValueError, SyntaxError) as e:
                        # Skip malformed keys
                        continue
            
            print(f"✓ Loaded BVS parameters database with {len(self.bond_valence_params)} entries")
        except Exception as e:
            print(f"✗ Error loading BVS parameters: {e}")
            import traceback
            traceback.print_exc()
            return None
        
        return self.bond_valence_params
    
    def load_common_oxidation_states(self):
        """Load common oxidation states from JSON file"""
        if self.oxidation_states is not None:
            return self.oxidation_states
            
        filepath = os.path.join(self.data_dir, "common_oxidation_states.json")
        
        if not os.path.exists(filepath):
            print(f"Oxidation states file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.oxidation_states = json.load(f)
            
            print(f"✓ Loaded oxidation states database with {len(self.oxidation_states)} elements")
        except Exception as e:
            print(f"✗ Error loading oxidation states: {e}")
            return None
        
        return self.oxidation_states
    
    def load_element_categories(self):
        """Load element categories from JSON file"""
        if self.element_categories is not None:
            return self.element_categories
            
        filepath = os.path.join(self.data_dir, "element_categories.json")
        
        if not os.path.exists(filepath):
            print(f"Element categories file not found at {filepath}")
            return None
        
        try:
            with open(filepath, 'r') as f:
                self.element_categories = json.load(f)
            
            print(f"✓ Loaded element categories database")
        except Exception as e:
            print(f"✗ Error loading element categories: {e}")
            return None
        
        return self.element_categories
    
    def get_ionic_radius(self, element, valence, cn):
        """Get ionic radius for given element, valence, and coordination number"""
        if self.shannon_radii is None:
            self.load_shannon_radii()
        
        if self.shannon_radii is None:
            return None
        
        return self.shannon_radii.get((element, valence, cn))
    
    def get_bv_parameters(self, cation, cation_valence, anion):
        """Get R0 and B parameters for BVS calculation"""
        if self.bond_valence_params is None:
            self.load_bond_valence_params()
        
        if self.bond_valence_params is None:
            return None, None
        
        params = self.bond_valence_params.get((cation, cation_valence, anion))
        if params:
            return params['R0'], params['B']
        return None, None
    
    def get_common_oxidation_states(self, element):
        """Get list of common oxidation states for an element"""
        if self.oxidation_states is None:
            self.load_common_oxidation_states()
        
        if self.oxidation_states is None:
            return []
        
        return self.oxidation_states.get(element, [])
    
    def get_element_category(self, element):
        """Get category for an element (alkali, transition, etc.)"""
        if self.element_categories is None:
            self.load_element_categories()
        
        if self.element_categories is None:
            return "unknown"
        
        for category, elements in self.element_categories.items():
            if element in elements:
                return category
        return "other"

# Initialise global database manager
db_manager = DatabaseManager()
db_manager.load_shannon_radii()
db_manager.load_bond_valence_params()
db_manager.load_bond_lengths()
db_manager.load_common_oxidation_states()
db_manager.load_element_categories()

# ============================================================================
# BVS PARAMETER LOOKUP WITH FRACTIONAL VALENCE SUPPORT (NEW IN v1.6)
# ============================================================================

def get_bv_parameters_fractional(cation, cation_valence, anion):
    """
    Look up BVS parameters (R0, b) for a cation, supporting non-integer valences
    via linear interpolation between adjacent integer states.

    For integer valences this is equivalent to db_manager.get_bv_parameters().
    For fractional valences (e.g. Mn 3.5 in LiMn2O4), parameters are interpolated
    linearly between floor(v) and ceil(v). This is the most defensible treatment
    when the underlying valence is a charge-averaged mean on the diffraction
    timescale — which is the usual assumption for electron-hopping systems at
    ambient temperature.

    v1.6 NEW: enables BVS analysis of mixed-valence cathode materials (LiMn2O4,
    LiNi0.5Mn1.5O4, NCA/NMC, etc.) without the user having to pre-assign a
    nominal integer valence.

    Args:
        cation: Element symbol
        cation_valence: Formal valence (int or float)
        anion: Anion element symbol

    Returns:
        tuple (R0, b) or (None, None) if parameters cannot be determined.
    """
    if cation_valence is None:
        return None, None

    # Integer valence (or float-encoded integer): direct lookup
    if float(cation_valence) == int(cation_valence):
        return db_manager.get_bv_parameters(cation, int(cation_valence), anion)

    # Fractional valence: interpolate
    v_lower = int(math.floor(cation_valence))
    v_upper = int(math.ceil(cation_valence))
    fraction = cation_valence - v_lower

    R0_lower, b_lower = db_manager.get_bv_parameters(cation, v_lower, anion)
    R0_upper, b_upper = db_manager.get_bv_parameters(cation, v_upper, anion)

    if R0_lower is not None and R0_upper is not None:
        R0 = (1.0 - fraction) * R0_lower + fraction * R0_upper
        b  = (1.0 - fraction) * b_lower  + fraction * b_upper
        return R0, b

    # Only one neighbour available — fall back to whichever exists
    if R0_lower is not None:
        return R0_lower, b_lower
    if R0_upper is not None:
        return R0_upper, b_upper

    return None, None

# ============================================================================
# SHANNON RADIUS LOOKUP WITH FRACTIONAL VALENCE SUPPORT (NEW IN v1.8.3)
# ============================================================================

# Sample size below which the observed MINIMUM bond length is not a reliable
# floor. The minimum is an extreme-value statistic and converges far more slowly
# than the mean: G&H (2020) Table 2 report the grand mean stabilising to
# +/-0.005 A somewhere between 20 and 200 coordination polyhedra depending on
# bond strength, and an extreme needs more than a mean. Below this the
# distribution is described but its tail is not.
BL_MIN_SAMPLE = 30


def expected_bond_length(element, valence, cn, anion='O', db=None):
    """
    The distance a bond from this ion, at this coordination, to this anion is
    expected to have. MEASURED where possible, derived where not.

    Returns a dict with mean, sd, min, n_bonds, source and tier, or None if
    nothing can be said. Never returns a number without saying where it came
    from: a verdict that cannot be traced to its source cannot be checked.

    Tiers, in order:
      1 measured               exact ion configuration in Gagne's data
      2 measured_interpolated  CN between two tabulated values, linear in CN
                               (G&H 2016 report mean bond length rising linearly
                               with CN for the larger cations)
      3 measured_extrapolated  CN outside the tabulated range; the nearest is
                               used and named. Treat with care.
      4 shannon_additive       adding ionic radii, the old behaviour, kept as a
                               fallback and FLAGGED as such because it is the
                               construction G&H's own data argues against.
    """
    db = db or db_manager
    try:
        v = int(round(float(valence)))
        c = int(round(float(cn)))
    except (TypeError, ValueError):
        return None
    if c <= 0:
        return None

    bl = db.load_bond_lengths() or {}

    hit = bl.get((element, v, anion, c))
    if hit:
        return {'mean': hit['mean'], 'sd': hit['sd'],
                'min': hit['min'] if (hit.get('n_bonds') or 0) >= BL_MIN_SAMPLE
                       else None,
                'n_bonds': hit.get('n_bonds'), 'ref': hit.get('ref'),
                'source': 'measured', 'tier': 1, 'cn_used': c}

    known = sorted(k[3] for k in bl if k[:3] == (element, v, anion))
    if known:
        lo = max([x for x in known if x < c], default=None)
        hi = min([x for x in known if x > c], default=None)
        if lo is not None and hi is not None:
            a, b = bl[(element, v, anion, lo)], bl[(element, v, anion, hi)]
            f = (c - lo) / (hi - lo)
            return {'mean': round(a['mean'] + f * (b['mean'] - a['mean']), 4),
                    'sd': None, 'min': None,
                    'n_bonds': (a.get('n_bonds') or 0) + (b.get('n_bonds') or 0),
                    'ref': a.get('ref'),
                    'source': f'measured_interpolated_CN{lo}_{hi}',
                    'tier': 2, 'cn_used': c}
        near = min(known, key=lambda x: abs(x - c))
        a = bl[(element, v, anion, near)]
        return {'mean': a['mean'], 'sd': a['sd'], 'min': None,
                'n_bonds': a.get('n_bonds'), 'ref': a.get('ref'),
                'source': f'measured_extrapolated_from_CN{near}',
                'tier': 3, 'cn_used': near}

    r_cat, _ = get_ionic_radius_fractional(element, v, c)
    r_an, _ = get_ionic_radius_fractional(anion, -2 if anion == 'O' else -3, c)
    if r_cat is None or r_an is None:
        return None
    return {'mean': round(r_cat + r_an, 4), 'sd': None, 'min': None,
            'n_bonds': None, 'ref': 'Shannon (1976) Acta Cryst A32, 751',
            'source': 'shannon_additive', 'tier': 4, 'cn_used': c}


def get_ionic_radius_fractional(element, valence, cn):
    """
    Shannon-Prewitt ionic radius, tolerant of non-integer valence and of
    coordination numbers that are not tabulated.

    The direct lookup keys on (element, valence, cn) and therefore misses
    every fractional valence, which on a survey of mixed-occupancy layered
    oxides removed the Shannon comparison from a quarter of all sites - and
    from *all* of the disordered ones, which are exactly the sites where an
    independent geometric check is most wanted.

    Resolution order:
        1. exact (element, valence, cn)
        2. fractional valence: interpolate between floor(v) and ceil(v) at
           the same CN, matching get_bv_parameters_fractional
        3. CN not tabulated: use the nearest tabulated CN for that valence
        4. give up

    Interpolating rather than snapping to the nearer integer valence is the
    same choice made for the bond valence parameters, and for the same
    reason: snapping makes the result discontinuous at half-integer valence,
    where a large part of a mixed-valence dataset sits.

    Returns:
        (radius, source) where source is one of 'exact', 'valence_interp',
        'cn_nearest', 'valence_interp+cn_nearest', or (None, 'unavailable').
    """
    if valence is None or cn is None:
        return None, 'unavailable'

    def _exact(v, c):
        return db_manager.get_ionic_radius(element, v, c)

    def _nearest_cn(v, c):
        """Nearest tabulated CN for this (element, valence)."""
        # v1.8.4: trigger the lazy load rather than relying on an earlier
        # get_ionic_radius call having already done it. It always had, but
        # only because of the order the branches happen to run in.
        if db_manager.shannon_radii is None:
            db_manager.load_shannon_radii()
        if db_manager.shannon_radii is None:
            return None, None
        cns = [k[2] for k in db_manager.shannon_radii
               if k[0] == element and k[1] == v]
        if not cns:
            return None, None
        best = min(cns, key=lambda x: (abs(x - c), x))
        return db_manager.shannon_radii.get((element, v, best)), best

    is_integer = float(valence) == int(valence)

    # 1. exact
    if is_integer:
        r = _exact(int(valence), cn)
        if r is not None:
            return r, 'exact'

    # 2. fractional valence, same CN
    if not is_integer:
        v_lo, v_hi = int(math.floor(valence)), int(math.ceil(valence))
        f = valence - v_lo
        r_lo, r_hi = _exact(v_lo, cn), _exact(v_hi, cn)
        if r_lo is not None and r_hi is not None:
            return (1.0 - f) * r_lo + f * r_hi, 'valence_interp'
        if r_lo is not None:
            return r_lo, 'valence_interp'
        if r_hi is not None:
            return r_hi, 'valence_interp'

    # 3. CN not tabulated
    if is_integer:
        r, used = _nearest_cn(int(valence), cn)
        if r is not None:
            return r, 'cn_nearest'
    else:
        v_lo, v_hi = int(math.floor(valence)), int(math.ceil(valence))
        f = valence - v_lo
        r_lo, c_lo = _nearest_cn(v_lo, cn)
        r_hi, c_hi = _nearest_cn(v_hi, cn)
        if r_lo is not None and r_hi is not None:
            return (1.0 - f) * r_lo + f * r_hi, 'valence_interp+cn_nearest'
        if r_lo is not None:
            return r_lo, 'valence_interp+cn_nearest'
        if r_hi is not None:
            return r_hi, 'valence_interp+cn_nearest'

    return None, 'unavailable'


# ============================================================================
# UTILITY FUNCTIONS (From Skuld 1.5 - UNCHANGED)
# ============================================================================

def parse_value_with_error(value_string):
    """
    Parse CIF value strings that may contain errors in parentheses.
    Example: "1.2345(67)" -> 1.2345, 0.0067
    """
    if not value_string or value_string == '.' or value_string == '?':
        return None, None
    
    # v2.0.2: the mantissa was \d+\.?\d*, which requires a digit BEFORE the
    # decimal point. CIFs are written both ways, so a value with a leading dot
    # AND a standard uncertainty fell through to the float() fallback, which
    # cannot parse the parentheses either, and the function returned
    # (None, None) -- silently, so the caller sees a missing value rather than
    # a parse failure and the bond simply leaves the analysis.
    #
    #     2.0947(3)   parsed before and now
    #     1.          float fallback, fine before and now
    #     .4890       float fallback, fine before and now
    #     .4890(12)   LOST before, parsed now
    #     -.0181(3)   LOST before, parsed now
    #
    # The same fault in a different place cost ICSD 420711 its shift test: a
    # coordinate written '1.' was rejected by an equivalent pattern in CELL
    # 13b. Worth checking any pattern that reads a number out of a CIF.
    match = re.match(
        r'^([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\((\d+)\)$',
        value_string.strip())
    if match:
        value_str, error_str = match.groups()
        value = float(value_str)
        
        if '.' in value_str:
            decimal_places = len(value_str.split('.')[1].split('e')[0].split('E')[0])
        else:
            decimal_places = 0
        
        error = float(error_str) * (10 ** (-decimal_places))
        return value, error
    else:
        try:
            value = float(value_string.strip())
            return value, None
        except ValueError:
            return None, None

def validate_element_combination(cation, anion):
    """Check if cation-anion combination is chemically reasonable"""
    cation_category = db_manager.get_element_category(cation)
    anion_category = db_manager.get_element_category(anion)
    
    if cation_category == "other" and anion_category == "other":
        return False
    
    return True

# ============================================================================
# BOND DISTANCE CALCULATION FOR ICSD CIF FILES
# ============================================================================

# A numeric CIF value is not supposed to be quoted, but the ICSD exporter
# quotes any field carrying trailing whitespace, e.g. ICSD 22964 (GdFeO3),
# whose Gd y coordinate is written '0.06269'. gemmi does not raise on this:
# it returns NaN, the site keeps its label and loses its position, and every
# distance to it compares False against both the minimum and the maximum bond
# cut-off. Gd emerged with CN 0, no bond valence sum, confidence HIGH, and the
# structure took the lowest GII in the 934-entry perovskite dataset, computed
# over its iron alone.
_QUOTED_NUMBER = re.compile(
    r"""(?<![\w.])(['"])\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:\(\d+\))?)\s*\1(?![\w.])""")


def _unquote_atom_site_numbers(content):
    """
    Strip quotes from numeric values inside the atom_site loop.

    Confined to that loop, and to data rows within it, so a genuinely quoted
    label such as 'Gd1' and a quoted value elsewhere in the file are both left
    alone. Standard uncertainties are preserved: '0.4655(3) ' becomes
    0.4655(3), which gemmi reads as it always did.

    Returns (content, n_lines_repaired).
    """
    out, n, in_loop = [], 0, False
    for line in content.split('\n'):
        s = line.strip()
        if s.startswith('_atom_site_'):
            in_loop = True
        elif in_loop and (s.startswith('loop_') or s.startswith('data_')
                          or s.startswith(';')
                          or (s.startswith('_') and not s.startswith('_atom_site_'))):
            in_loop = False
        if in_loop and s and s[0] not in '_#;' and not s.startswith('loop_'):
            new = _QUOTED_NUMBER.sub(r'\2', line)
            if new != line:
                n += 1
                line = new
        out.append(line)
    return '\n'.join(out), n

def assert_finite_sites(small_struct, cif_file_path=''):
    """
    Refuse a parsed structure whose coordinates are not all finite.

    gemmi returns NaN rather than raising when an _atom_site_fract_* value
    cannot be converted, so a site survives parsing with a label and no
    position. This is the only place that catches it: NaN passes every
    downstream comparison silently, and the site is not so much rejected as
    never considered.

    Returns (ok, message). The caller decides what to do; this only refuses to
    let it through unremarked.
    """
    bad = []
    for s in small_struct.sites:
        f = s.fract
        for axis, v in (('x', f.x), ('y', f.y), ('z', f.z)):
            if not math.isfinite(v):
                bad.append(f'{s.label}.{axis}')
    if not bad:
        return True, ''
    return False, ('non-finite fractional coordinates on ' + ', '.join(bad)
                   + '; the CIF parsed but the positions did not')

# ICSD exports occasionally carry a non-breaking space that has been encoded
# twice, U+00C2 followed by U+00A0, in a citation author name. CIF 1.1 permits
# only ASCII 32-126 in an unquoted value, so gemmi refuses the whole file and
# the structure is lost with the message 'Failed to parse CIF file', which
# points nowhere near the actual line. Four perovskites in the ICSD set fail
# this way; one of them, ICSD 124972, on a single character.
_MANGLED_NBSP = '\u00c2\u00a0'
_INVISIBLE = '\u00a0\u2007\u202f\u200b\u200c\u200d\ufeff'


def normalise_cif_to_ascii(content):
    """
    Bring a CIF within the CIF 1.1 character set without changing its tokens.

    Invisible characters are DELETED rather than replaced with a space. That
    distinction is the whole point: 'Akamatsu,<NBSP>Hirofumi' is one token, and
    substituting a space would split it in two, turning a character error into
    a loop row-length error in an unquoted field. Deleting gives
    'Akamatsu,Hirofumi', which is one token and still legible.

    Everything else above 127 is folded to its ASCII base where one exists and
    dropped where none does, so accented author names survive readably.

    Returns (content, n_characters_repaired).
    """
    import unicodedata

    n = 0
    if _MANGLED_NBSP in content:
        n += content.count(_MANGLED_NBSP)
        content = content.replace(_MANGLED_NBSP, '')
    for ch in _INVISIBLE:
        if ch in content:
            n += content.count(ch)
            content = content.replace(ch, '')
    if not content.isascii():
        out = []
        for ch in content:
            if ord(ch) < 128:
                out.append(ch)
                continue
            n += 1
            out.append(''.join(c for c in unicodedata.normalize('NFKD', ch)
                               if ord(c) < 128 and not unicodedata.combining(c)))
        content = ''.join(out)
    return content, n

def clean_cif_for_gemmi(cif_file_path):
    """
    Pre-process CIF file to remove lines that cause gemmi parsing errors.
    
    Removes _chemical_name_structure_type which often has UTF-8 encoding issues in ICSD files.
    Replaces non-standard reverse rhombohedral symops ('HR' suffix) with standard obverse
    centering, which gemmi requires for space group determination.

    v2.0.3: also unquotes numeric values inside the atom_site loop. A quoted
    number is not legal in that position and gemmi returns NaN rather than
    raising, so the site keeps its label and loses its position. See
    _unquote_atom_site_numbers.
    
    Returns cleaned CIF as string for gemmi to parse.
    """
    try:
        import re
        
        # Read file with UTF-8 encoding, replacing errors
        with open(cif_file_path, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        
        content = ''.join(lines)

        # v2.0.3 (rule 0). Both repairs must run before anything else looks at
        # the file, and before gemmi sees it at all. ASCII first, because it is
        # about bytes; unquoting second, because it is about tokens.
        content, _n_ascii = normalise_cif_to_ascii(content)
        content, _n_unquoted = _unquote_atom_site_numbers(content)
        
        # v1.8.3 (B13): the two repairs below are load-bearing, not cosmetic.
        # Without the _chemical_name_structure_type strip, 22 of the 142
        # structures in the reference set do not parse at all, including most
        # of the NaCrO2 entries; without the HR rewrite several reverse-
        # rhombohedral space groups fail to resolve. Both log when they fire so
        # a change in parsing behaviour is visible in the run output.
        _repairs = []

        if _n_ascii:
            _repairs.append(f'normalised {_n_ascii} non-ASCII character'
                            f'{"s" if _n_ascii > 1 else ""}')
        if _n_unquoted:
            _repairs.append(f'unquoted {_n_unquoted} atom_site numeric value'
                            f'{"s" if _n_unquoted > 1 else ""}')

        # Fix non-standard reverse rhombohedral centering ('HR' suffix)
        # Some older ICSD CIFs encode symops in the reverse setting, which gemmi
        # cannot match to a tabulated space group. Replace the HM label and
        # regenerate symops from the IT number using standard obverse centering.
        if ' HR' in content:
            sg_tags = ['_space_group_name_H-M_alt', '_symmetry_space_group_name_H-M']
            has_hr = any(tag in content and ' HR' in content for tag in sg_tags)
            
            if has_hr:
                try:
                    import gemmi as _gemmi
                    
                    # Fix HM label
                    _repairs.append('HR->H symmetry setting')
                    content = content.replace(" HR'", " H'")
                    content = content.replace(' HR\r', ' H\r')
                    content = content.replace(' HR\n', ' H\n')
                    
                    # Get IT number and regenerate standard symops
                    it_match = re.search(r'_space_group_IT_number\s+(\d+)', content)
                    if not it_match:
                        it_match = re.search(r'_symmetry_Int_Tables_number\s+(\d+)', content)
                    
                    if it_match:
                        sg = _gemmi.find_spacegroup_by_number(int(it_match.group(1)))
                        if sg:
                            ops = sg.operations()
                            new_loop = 'loop_\n_space_group_symop_id\n_space_group_symop_operation_xyz\n'
                            for i, op in enumerate(ops):
                                new_loop += f"{i+1} '{op.triplet()}'\n"
                            
                            # Replace old symops loop
                            pattern = r"loop_\s*\n_space_group_symop_id\s*\n_space_group_symop_operation_xyz\s*\n((?:\d+\s+'[^']+'\s*[\r\n]+)+)"
                            content = re.sub(pattern, new_loop, content)
                            print(f"  Note: Replaced reverse rhombohedral symops with standard obverse (SG {sg.hm})")
                except Exception as e:
                    print(f"  Warning: Could not fix HR symops: {e}")
        
        # Re-split into lines for structure_type removal
        lines = content.split('\n')
        
        cleaned_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # Check if this line contains the problematic tag
            if '_chemical_name_structure_type' in line:
                if 'structure_type strip' not in _repairs:
                    _repairs.append('structure_type strip')
                # Skip this line
                # Also skip the next line if it doesn't start with _ or # (it's the value)
                i += 1
                if i < len(lines):
                    next_line = lines[i].strip()
                    # If next line is a value (not a tag), skip it too
                    if next_line and not next_line.startswith('_') and not next_line.startswith('#') and not next_line.startswith('loop_') and not next_line.startswith('data_'):
                        i += 1
                continue
            
            cleaned_lines.append(line)
            i += 1
        
        cleaned = '\n'.join(cleaned_lines)
        if _repairs:
            _dbg(f"    [CIF repair] {', '.join(_repairs)} (B13)")
        return cleaned

    except (NameError, AttributeError, TypeError, SyntaxError, IndexError):
        # v2.0.3: these are defects in this function, not in the CIF. Swallowed,
        # they surface as 'Could not read CIF file' on every structure in the
        # run and send the reader to look at their data for a fault that is in
        # the code. Let them out.
        raise
    except Exception as e:
        print(f"  Warning: Error cleaning CIF file "
              f"({type(e).__name__}: {e}): {cif_file_path}")
        return None


def formula_from_cif(cif_path):
    """
    _chemical_formula_sum, read from the CLEANED text.

    v2.0.13. Two faults sat here. The formula was read with PyCifRW straight
    off the RAW file, while every other read in Eir goes through
    clean_cif_for_gemmi - so a file the repairs fix parsed for gemmi and
    raised here. ICSD 118400 and its three siblings carry a double-encoded
    non-breaking space in an author name and are exactly that case.

    And the exception list did not include PyCifRW's own StarError, so one
    unparseable file ended a whole batch rather than being recorded and
    skipped.

    A regex on the cleaned text is used first: this is one tag, and building
    a full STAR parse tree to read it is both slower and more fragile than
    the string it came from.
    """
    try:
        text = clean_cif_for_gemmi(cif_path)
    except Exception as _e:
        _note_read_failure(cif_path, 'formula/clean', _e)
        text = None

    if text:
        m = re.search(r"_chemical_formula_sum\s+(?:'([^']*)'|\"([^\"]*)\""
                      r"|;\s*\n(.*?)\n;|(\S.*))", text)
        if m:
            val = next((g for g in m.groups() if g), '')
            return ' '.join(val.split())

    # Fall back to the parser, on the cleaned text where there is one.
    tmp_path = None
    try:
        if text:
            with tempfile.NamedTemporaryFile(mode='w', suffix='.cif',
                                             delete=False,
                                             encoding='utf-8') as tmp:
                tmp.write(text)
                tmp_path = tmp.name
        cf = ReadCif(tmp_path or str(cif_path))
        return cf[list(cf.keys())[0]].get('_chemical_formula_sum', '')
    except Exception as _e:
        # Deliberately broad. PyCifRW raises StarError, which belongs to no
        # standard hierarchy, and a formula that cannot be read must not end
        # the batch: the file is recorded and analysis continues without it.
        _note_read_failure(cif_path, 'formula', _e)
        return ''
    finally:
        if tmp_path:
            try:
                os.unlink(tmp_path)
            except OSError:
                pass


def calculate_bonds_from_positions(cif_file_path, max_bond_distance=4.2):
    """
    Calculate bond distances from CIF using proper crystallographic symmetry operations.
    
    Uses gemmi library for robust handling of:
    - Partial occupancy (e.g., Na site 2/3 occupied)  
    - Mixed occupancy (e.g., Cr0.8Ti0.2 on same site)
    - All space groups (R-3m, P63/mmc, C2/m, etc.)
    - Proper symmetry code generation
    
    CRITICAL: ICSD files are small molecule/inorganic structures, not macromolecular.
    Uses SmallStructure, not Structure.
    
    Args:
        cif_file_path: Path to CIF file
        max_bond_distance: Maximum bond distance to consider (Ångströms).
            v1.8.1: raised 3.5 -> 4.2. At 3.5 the true coordination-shell
            boundary in layered oxides often fell outside the candidate
            list, so the gap cut fired on intra-shell distortion instead
            and returned CN 2/3/4 for octahedral sites.
    
    Returns:
        dict with bond data formatted like GSAS-II _geom_bond output, or None if failed

    v2.0.3: refuses a structure whose parsed coordinates are not all finite,
    and rejects a non-finite distance explicitly rather than by accident.
    gemmi returns NaN, not an error, when an _atom_site_fract_* value cannot be
    converted, and the old candidate filter tested `distance < 0.01 or distance
    > max_bond_distance`, both of which are False for NaN. A site with no
    position therefore acquired no bonds, reached CN 0, produced no bond
    valence sum, and was reported with confidence HIGH. ICSD 22964 (GdFeO3)
    took the lowest GII in the 934-entry perovskite dataset that way, computed
    over its iron alone, because the Gd y coordinate is deposited as the quoted
    string '0.06269 '.
    """
    try:
        import gemmi
    except ImportError:
        print("  ✗ gemmi not available - install with: pip install gemmi")
        print("    gemmi is required for proper crystallographic bond calculation")
        return None
    
    import tempfile
    
    try:
        # Read file with UTF-8 error handling
        cleaned_cif = clean_cif_for_gemmi(cif_file_path)
        if cleaned_cif is None:
            print("  ✗ Could not read CIF file")
            return None
        
        # Write to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False, encoding='utf-8') as tmp:
            tmp.write(cleaned_cif)
            tmp_path = tmp.name
        
        try:
            # Read CIF document
            doc = gemmi.cif.read(tmp_path)
            block = doc.sole_block()
            
            # CRITICAL: Use SmallStructure for ICSD (small molecule/inorganic) CIFs
            # NOT Structure (macromolecular)
            small_struct = gemmi.make_small_structure_from_block(block)

            
        finally:
            # Clean up temp file
            import os
            # v2.0.3: narrowed from a bare except. Failing to delete a temp file
            # is not a reason to swallow a KeyboardInterrupt or a NameError.
            try:
                os.unlink(tmp_path)
            except OSError:
                pass

        # v1.8.6: a CIF may declare its symmetry with only
        # _space_group_IT_number and no H-M name or symop loop. That is legal
        # CIF and several programs emit it, but gemmi leaves .spacegroup as
        # None and the supercell loop below then raises AttributeError, which
        # reaches the user as "No M-X bonds found". Four other functions in
        # this file already recover from this; this one did not.
        if small_struct is not None and small_struct.spacegroup is None:
            _itm = (re.search(r'_space_group_IT_number\s+(\d+)', cleaned_cif) or
                    re.search(r'_symmetry_Int_Tables_number\s+(\d+)', cleaned_cif))
            if _itm:
                _sg = gemmi.find_spacegroup_by_number(int(_itm.group(1)))
                if _sg is not None:
                    small_struct.spacegroup = _sg
                    print(f"  Note: space group taken from IT number "
                          f"{_itm.group(1)} ({_sg.hm})")
        if small_struct is not None and small_struct.spacegroup is None:
            print("  ✗ No usable space group: the CIF gives no symmetry "
                  "operations, no H-M name and no IT number.")
            return None

        
        if not small_struct or len(small_struct.sites) == 0:
            print("  ✗ No atomic sites found in CIF")
            return None

        # v2.0.3: a site can survive parsing with a label and no position. That
        # is the one failure mode nothing downstream can see, because NaN
        # compares False against every threshold it meets, so the site is never
        # rejected, merely never considered.
        _ok, _why = assert_finite_sites(small_struct, cif_file_path)
        if not _ok:
            print(f"  ✗ {_why}")
            return None
        
        # Build bond data in GSAS-II format
        bond_data = {
            'bond_label_1': [],
            'bond_label_2': [],
            'bond_distance': [],
            'bond_symmetry_2': []
        }
        
        bond_count = 0
        atom_type_pairs = set()
        
        # Debug: Print all sites found
        _dbg(f"  Debug: Found {len(small_struct.sites)} sites in asymmetric unit:")
        _EIR_GEOMETRY.clear()
        for site in small_struct.sites:
            frac = site.fract
            _dbg(f"    {site.label}: {site.element.name} at ({frac.x:.4f}, {frac.y:.4f}, {frac.z:.4f}) occ={site.occ:.2f}")
        
        # CRITICAL: Generate full supercell for proper coordination detection
        # Gemmi's neighbor search works with asymmetric units, but we need all atoms
        # Generate 3x3x3 supercell to ensure we find all neighbors within cutoff
        _dbg(f"  Generating 3×3×3 supercell for coordination analysis...")
        
        supercell_atoms = []
        
        # For each site in asymmetric unit
        for site in small_struct.sites:
            elem_type = site.element.name
            site_label = site.label
            
            # Apply all symmetry operations
            for op in small_struct.spacegroup.operations():
                # Transform fractional coordinates
                transformed_frac = op.apply_to_xyz([site.fract.x, site.fract.y, site.fract.z])
                
                # Apply lattice translations for 3x3x3 supercell
                for i in range(-1, 2):  # -1, 0, 1
                    for j in range(-1, 2):
                        for k in range(-1, 2):
                            # Add lattice translation
                            final_frac = [
                                transformed_frac[0] + i,
                                transformed_frac[1] + j,
                                transformed_frac[2] + k
                            ]
                            
                            # Convert to Cartesian
                            cart_pos = small_struct.cell.orthogonalize(gemmi.Fractional(*final_frac))
                            
                            supercell_atoms.append({
                                'label': site_label,
                                'element': elem_type,
                                'position': cart_pos,
                                'frac': final_frac,
                                'original_site': site
                            })
        
        _dbg(f"  Generated {len(supercell_atoms)} atoms in supercell")
        
        # Now find neighbors by simple distance calculation
        for site in small_struct.sites:
            site_orth = site.orth(small_struct.cell)
            elem1 = site.label
            elem1_type = site.element.name
            
            neighbors_for_site = []
            
            # Check distance to every atom in supercell
            for atom_dict in supercell_atoms:
                distance = atom_dict['position'].dist(site_orth)
                
                # Keep only atoms inside the candidate shell. Written as a
                # positive test rather than as two exclusions: for a finite
                # distance the two are identical, and for NaN the exclusions
                # are both False while this is False, so a distance that is not
                # a number is rejected here instead of silently admitted.
                if not (0.01 <= distance <= max_bond_distance):
                    continue
                
                neighbors_for_site.append({
                    'label': atom_dict['label'],
                    'element': atom_dict['element'],
                    'position': atom_dict['position'],
                    'distance': distance
                })
            
            # Deduplicate by position with 0.05Å tolerance  
            # Looser tolerance handles general positions in R-3m better
            # v1.8.9: dedup by true separation, not by snapped coordinate.
            # Grid snapping is not a tolerance: two images of the same atom
            # straddling a 0.05 A grid line got different keys and both
            # survived, inflating CN by an exact integer factor.
            DEDUP_TOL = 0.05
            unique_neighbors = []
            for neighbor in sorted(neighbors_for_site, key=lambda n: n['distance']):
                p = neighbor['position']
                if any((p.x - q.x)**2 + (p.y - q.y)**2 + (p.z - q.z)**2 <= DEDUP_TOL**2
                       for q in (n['position'] for n in unique_neighbors)):
                    continue
                unique_neighbors.append(neighbor)
            
            _dbg(f"  Debug: Site {site.label} ({site.element.name}) has {len(unique_neighbors)} neighbors in supercell")
            if site.element.name not in _ANION_SET:
                _c = site_orth
                _an = [(n['distance'], (n['position'].x, n['position'].y, n['position'].z))
                       for n in unique_neighbors if n['element'] in _ANION_SET]
                if len(_an) >= 2:
                    # Provisional: no valence exists yet, so no canonical CN
                    # and no per-bond valences. The inputs are kept so the
                    # same assessment can be redone once they do - see the
                    # re-assessment where site_result is built.
                    _EIR_GEOMETRY[site.label] = assess_site_geometry(
                        (_c.x, _c.y, _c.z), _an, site.element.name)
                    _EIR_GEOMETRY[site.label]['_geo_centre'] = (
                        _c.x, _c.y, _c.z)
                    _EIR_GEOMETRY[site.label]['_geo_neighbours'] = _an
                    _EIR_GEOMETRY[site.label]['_geo_element'] = site.element.name
            
            # v2.0.3: enumerate rather than unique_neighbors.index(neighbor),
            # which was an O(n^2) scan returning the index of the first EQUAL
            # element rather than of this one.
            for _ui, neighbor in enumerate(unique_neighbors):
                elem2 = neighbor['label']
                elem2_type = neighbor['element']
                distance = neighbor['distance']
                atom_type_pairs.add(tuple(sorted([elem1_type, elem2_type])))
                
                position_key = f"u{_ui}"
                symmetry_code = "supercell"
                
                _dbg(f"    -> {elem2} ({elem2_type}) dist={distance:.4f}")
                
                # Store bond
                bond_data['bond_label_1'].append(elem1)
                bond_data['bond_label_2'].append(elem2)
                bond_data['bond_distance'].append(f"{distance:.5f}")
                bond_data['bond_symmetry_2'].append(f"{symmetry_code}|{position_key}")
                bond_count += 1
        
        print(f"  ✓ Calculated {bond_count} bonds using crystallographic symmetry (gemmi)")
        _dbg(f"  ✓ Bond types found: {sorted(atom_type_pairs)}")
        print(f"  ✓ Space group: {small_struct.spacegroup_hm}")
        
        return bond_data
        
    except Exception as e:
        print(f"  ✗ Error calculating bonds with gemmi: {e}")
        import traceback
        traceback.print_exc()
        return None

def is_bond_data_present(data_block):
    """
    Check if a CIF data block contains pre-calculated bond distance data.
    
    Args:
        data_block: PyCifRW data block dictionary
    
    Returns:
        True if bond data present (GSAS-II style), False if missing (ICSD style)
    """
    required_keys = [
        '_geom_bond_atom_site_label_1',
        '_geom_bond_atom_site_label_2',
        '_geom_bond_distance',
        '_geom_bond_site_symmetry_2'
    ]
    
    return all(key in data_block for key in required_keys)

# ============================================================================
# DIRECTORY SELECTION DIALOG
# ============================================================================

def select_directory():
    """
    Opens a tkinter dialog to select a directory containing CIF files.
    Returns the selected directory path or None if cancelled.
    """
    import tkinter as tk
    from tkinter import filedialog
    
    root = tk.Tk()
    root.overrideredirect(True)
    root.attributes('-topmost', True)
    root.withdraw()
    
    print("\n" + "="*80)
    print("SELECT DIRECTORY CONTAINING CIF FILES")
    print("="*80)
    print("A file dialog will open. Please select the directory containing your CIF files.")
    print("(If the dialog appears behind other windows, check your taskbar)")
    
    directory_path = filedialog.askdirectory(
        title="Select Directory Containing CIF Files",
        mustexist=True
    )
    
    root.destroy()
    
    if not directory_path:
        print("\n✗ No directory selected. Exiting.")
        return None
    
    print(f"\n✓ Selected directory: {directory_path}")
    
    # Validate directory contains CIF files
    cif_files = list(Path(directory_path).glob("*.cif"))
    if not cif_files:
        print(f"\n✗ ERROR: No CIF files found in {directory_path}")
        print("Please select a directory containing .cif files")
        return None
    
    print(f"✓ Found {len(cif_files)} CIF files")
    _n=len(cif_files)
    print(f"  Rough estimate: ~{_n*0.9/60:.0f} min for large cells "
          f"(spinels/pyrochlores); simple oxides run several times faster.")
    
    return directory_path

# ============================================================================
# CIF PARSING
# ============================================================================

# v2.0.3: the last parse failure, so process_single_cif can record WHY rather
# than only THAT. A single mutable dict rather than a global rebind, so the
# reference survives a cell re-run.
_LAST_PARSE_ERROR = {'reason': None}

def parse_cif_file(file_path):
    """
    Parse a CIF file and return the CifFile object.
    For ICSD CIFs without bond data, calculates bonds from atomic positions.

    v2.0.3: reads the CLEANED text rather than the file on disk. Eir has two
    CIF readers, PyCifRW here and gemmi in calculate_bonds_from_positions, and
    until now only the second was protected by clean_cif_for_gemmi. That had
    two consequences. Four perovskites carrying a double-encoded non-breaking
    space in an author name were rejected by PyCifRW before the repair could
    run, and reported as 'Failed to parse CIF file' with no line number. And
    the two readers could disagree about the same file without anything
    noticing: on ICSD 22964 PyCifRW read the quoted coordinate '0.06269 '
    correctly while gemmi returned NaN, so the coords column was right and the
    geometry built on it was not.
    """
    import tempfile

    try:
        cleaned = clean_cif_for_gemmi(file_path)
        if cleaned is None:
            _LAST_PARSE_ERROR['reason'] = 'could not read or clean the file'
            print(f"  ✗ Could not read {os.path.basename(file_path)}")
            return None

        with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False,
                                         encoding='utf-8') as tmp:
            tmp.write(cleaned)
            tmp_path = tmp.name
        try:
            cf = ReadCif(tmp_path)
        finally:
            try:
                os.unlink(tmp_path)
            except OSError:
                pass

        # Check if bonds need to be calculated (ICSD CIFs)
        for block_name, data_block in cf.items():
            if not is_bond_data_present(data_block):
                _dbg(f"  ⚠ No bond data in CIF - calculating from crystallographic symmetry (ICSD mode)...")

                # Calculate bonds using ASE
                bond_data = calculate_bonds_from_positions(file_path)

                if bond_data is not None and bond_data.get('bond_label_1'):
                    # Add calculated bonds to the data block
                    data_block['_geom_bond_atom_site_label_1'] = bond_data['bond_label_1']
                    data_block['_geom_bond_atom_site_label_2'] = bond_data['bond_label_2']
                    data_block['_geom_bond_distance'] = bond_data['bond_distance']
                    data_block['_geom_bond_site_symmetry_2'] = bond_data['bond_symmetry_2']
                elif bond_data is not None:
                    # v1.8.6: zero bonds is not a parse failure. Writing an empty
                    # list into the CIF block makes PyCifRW raise "not enough
                    # values to unpack", which reaches the user as "Failed to
                    # parse CIF file" and sends them hunting for a syntax error
                    # that does not exist.
                    print(f"  ✗ No bonds found within the search radius. The cell "
                          f"may be far larger than the contents, or the coordinates "
                          f"may be wrong.")
                else:
                    print(f"  ✗ Failed to calculate bonds - structure will be skipped")

        return cf

    except (NameError, AttributeError, TypeError, SyntaxError):
        raise
    except Exception as e:
        # v2.0.3: keep the reason. 'Failed to parse CIF file' in the failed
        # files CSV names no line and no cause, and four entries sat behind it
        # for the whole of this dataset.
        _LAST_PARSE_ERROR['reason'] = f'{type(e).__name__}: {e}'
        print(f"  ERROR parsing {os.path.basename(file_path)}: {e}")
        return None

def extract_initial_site_data(cif_object):
    """
    EXACT COPY from Skuld 1.5 (lines 808-938)
    Extracts initial atom site data and all bond information,
    including proper handling of symmetry operations for bonds beyond unit cell.
    """
    if not cif_object:
        return {}

    processed_data = {}

    for block_name, data_block in cif_object.items():
        block_data = {
            'unit_cell': {},
            'atom_sites_data': [],
            'all_bond_pairs_found': set(),
            'sites_with_bonds': {}
        }
        processed_data[block_name] = block_data

        # --- General Data Items ---
        block_data['publ_author_name'] = data_block.get("_publ_contact_author_name")
        block_data['gsasii_version'] = data_block.get("_gsas_GSASII_version")
        block_data['pd_block_id'] = data_block.get("_pd_block_id")

        # --- Atom Site Data Processing ---
        label_to_canonical_coords = {} 
        label_to_original_coord_strs = {}
        label_to_type = {}
        
        try:
            labels = data_block['_atom_site_label']
            types = data_block['_atom_site_type_symbol']
            xs_orig = data_block['_atom_site_fract_x'] 
            ys_orig = data_block['_atom_site_fract_y']
            zs_orig = data_block['_atom_site_fract_z']

            if not (labels and types and xs_orig and ys_orig and zs_orig and \
                    len(labels) == len(types) == len(xs_orig) == len(ys_orig) == len(zs_orig)):
                print(f"  No complete _atom_site_loop data found in block {block_name}. Skipping atom site processing for this block.")
                block_data['atom_sites_data'] = [] 
            else:
                for i in range(len(labels)):
                    label = labels[i].strip()
                    atom_type_raw = types[i].strip().rstrip('+-0123456789')
                    atom_type = atom_type_raw.capitalize() if len(atom_type_raw) > 1 else atom_type_raw.upper()
                    
                    try:
                        x = float(xs_orig[i].split('(')[0].strip())
                        y = float(ys_orig[i].split('(')[0].strip())
                        z = float(zs_orig[i].split('(')[0].strip())
                        coords_tuple = (x, y, z)
                        
                        block_data['atom_sites_data'].append({'label': label, 'type': atom_type, 'coords': coords_tuple})
                        label_to_canonical_coords[label] = coords_tuple 
                        label_to_original_coord_strs[label] = (xs_orig[i], ys_orig[i], zs_orig[i])
                        label_to_type[label] = atom_type 
                    except (ValueError, IndexError):
                        print(f"  Warning: Could not parse coordinates for atom {label}. Skipping.")
                        continue
        except KeyError:
            print(f"  No _atom_site_label or related loop data found in block {block_name}. Skipping atom site processing for this block.")
            block_data['atom_sites_data'] = []

        # --- Bond Distances (CRITICAL: includes symmetry code handling) ---
        all_raw_bonds_per_central_site = {} 
        site_metadata = {} 

        bond_label_1s = data_block.get("_geom_bond_atom_site_label_1")
        bond_label_2s = data_block.get("_geom_bond_atom_site_label_2")
        bond_distance_raw_strs = data_block.get("_geom_bond_distance")
        bond_symmetry_2s = data_block.get("_geom_bond_site_symmetry_2")

        if bond_label_1s and bond_label_2s and bond_distance_raw_strs and bond_symmetry_2s and \
           all(len(lst) == len(bond_label_1s) for lst in [bond_label_2s, bond_distance_raw_strs, bond_symmetry_2s]):
            
            for i in range(len(bond_label_1s)):
                atom1_label = bond_label_1s[i].strip()
                atom2_label = bond_label_2s[i].strip()
                dist_str = bond_distance_raw_strs[i].strip()
                atom2_symmetry_code = bond_symmetry_2s[i].strip()  # CRITICAL: preserves symmetry info

                dist_val, dist_err = parse_value_with_error(dist_str)
                
                if dist_val is None:
                    continue 

                atom1_canonical_coords = label_to_canonical_coords.get(atom1_label)
                atom2_canonical_coords = label_to_canonical_coords.get(atom2_label) 

                atom1_type = label_to_type.get(atom1_label)
                atom2_type = label_to_type.get(atom2_label)

                # CRITICAL FIX for ICSD CIFs: If type not found in label_to_type,
                # extract element from label (e.g., "Na1" -> "Na", "Cr1" -> "Cr")
                if not atom1_type and atom1_label:
                    # Extract element symbol from label (strip trailing digits)
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom1_label)
                    if match:
                        atom1_type = match.group(1)
                
                if not atom2_type and atom2_label:
                    import re
                    match = re.match(r'^([A-Z][a-z]?)', atom2_label)
                    if match:
                        atom2_type = match.group(1)

                if not atom1_type or not atom2_type:
                    continue 
                
                # Validation
                if not validate_element_combination(atom1_type, atom2_type):
                    print(f"  Warning: Unusual element combination {atom1_type}-{atom2_type}, but including in analysis")
                
                block_data['all_bond_pairs_found'].add(tuple(sorted((atom1_type, atom2_type))))

                central_atom_coords = atom1_canonical_coords
                central_atom_element = atom1_type
                
                if central_atom_coords:
                    if central_atom_coords not in all_raw_bonds_per_central_site:
                        all_raw_bonds_per_central_site[central_atom_coords] = []
                        site_metadata[central_atom_coords] = {
                            'elements': set(),
                            'labels_at_site': set(),
                        }

                    site_metadata[central_atom_coords]['elements'].add(central_atom_element)
                    site_metadata[central_atom_coords]['labels_at_site'].add(atom1_label)
                    
                    # CRITICAL: Store complete bond tuple including symmetry code
                    all_raw_bonds_per_central_site[central_atom_coords].append(
                        (atom1_label, atom2_label, atom1_type, atom2_type, atom2_symmetry_code, dist_val, dist_err)
                    )
        else:
            print(f"  No geometric bond data found or incomplete in block {block_name}.")

        block_data['sites_with_bonds'] = {
            coords: {
                'elements': sorted(list(site_metadata[coords]['elements'])),
                'labels': sorted(list(site_metadata[coords]['labels_at_site'])),
                'raw_bonds_list': raw_bonds_list
            } for coords, raw_bonds_list in all_raw_bonds_per_central_site.items()
        }
    
    return processed_data

# ============================================================================
# AUTOMATED BOND SELECTION (v1.2 - Multi-anion support)
# ============================================================================

def has_bvs_parameters_for_pair(cation, anion, valence_dict=None):
    """
    Check if BVS parameters exist for a given cation-anion pair.
    
    Args:
        cation: Cation element symbol (e.g., 'Ge', 'Si', 'Ti')
        anion: Anion element symbol (e.g., 'O', 'F', 'S')
        valence_dict: Optional dict of {element: valence} to check specific oxidation states
    
    Returns:
        bool: True if parameters exist for this pair
    """
    # Check common oxidation states for the cation
    if valence_dict and cation in valence_dict:
        # User specified oxidation state
        cation_valences = [abs(valence_dict[cation])]
    else:
        # Try common oxidation states from database
        cation_valences = db_manager.get_common_oxidation_states(cation)
        if not cation_valences:
            # If not in database, try common defaults
            common_defaults = {
                'Ge': [4, 2], 'Si': [4], 'B': [3], 'P': [5, 3],
                'As': [5, 3], 'Sb': [5, 3], 'Te': [6, 4, 2]
            }
            cation_valences = common_defaults.get(cation, [])
    
    if not cation_valences:
        return False
    
    # CORRECTED: Use bond_valence_params, not bvs_params
    # Also handle the fact that keys are tuples
    for val in cation_valences:
        param_key = (cation, val, anion)
        if param_key in db_manager.bond_valence_params:
            return True
    
    return False

def auto_select_bond_types(processed_data, valences=None):
    """
    Automatically select cation-anion bond types for analysis based on BVS parameter availability.
    
    EIR v1.3 ENHANCEMENT: Now checks BVS parameter database directly instead of hardcoded metal lists.
    This ensures ALL analyzable elements are included (Ge, Si, B, P, etc.) without manual updates.
    
    Args:
        processed_data: Data structure from previous processing steps
    
    Returns: processed_data with 'selected_bond_types_for_analysis' added
    """
    for block_name, block_content in processed_data.items():
        unique_bond_pairs = sorted(list(block_content['all_bond_pairs_found']))
        
        _dbg(f"  DEBUG: Found {len(unique_bond_pairs)} unique bond pairs: {unique_bond_pairs}")
        
        if not unique_bond_pairs:
            block_content['selected_bond_types_for_analysis'] = {}
            continue
        
        # Auto-select ALL cation-anion bonds where BVS parameters exist
        selected_bond_types_map = {}
        
        # v1.8.3 (B7): the whole element set decides whether an amphoteric
        # species is acting as an anion, so it is gathered once here.
        _elements_present = {e for _pair in unique_bond_pairs for e in _pair}

        for type1, type2 in unique_bond_pairs:
            # Identify which element is the anion
            anion = None
            cation = None
            
            # v1.8.3 (B7): decide by context, not by element alone, so that
            # polyanion centres (P in a phosphate, S in a sulfate) are treated
            # as cations against oxygen instead of the pair being discarded.
            cation, anion = classify_bond_pair(
                type1, type2, valences=valences,
                elements_present=_elements_present)
            if cation is None or anion is None:
                continue
            
            # EIR v1.3: Check if BVS parameters exist for this pair
            has_params = has_bvs_parameters_for_pair(cation, anion)
            
            _dbg(f"  DEBUG: Bond {cation}-{anion}: has_bvs_params={has_params}")
            
            if has_params:
                # Parameters exist - include this bond type for analysis
                selected_bond_types_map[(cation, anion)] = {
                    'central_atom_type': cation,
                    'ligand_type': anion
                }
            else:
                print(f"  WARNING: No BVS parameters for {cation}-{anion}, excluding from analysis")
        
        block_content['selected_bond_types_for_analysis'] = selected_bond_types_map
        
        if selected_bond_types_map:
            _dbg(f"  ✓ Auto-selected {len(selected_bond_types_map)} cation-anion bond types: {list(selected_bond_types_map.keys())}")
        else:
            print(f"  ✗ No cation-anion bonds with BVS parameters found")
    
    return processed_data

# ============================================================================
# AUTOMATED COORDINATION ANALYSIS (Replaces interactive_review_and_correct)
# ============================================================================


# ============================================================================
# COORDINATION SHELL GUARD (NEW IN v2.0.2)
# ============================================================================

# How far ECoN must exceed the gap cut before the shell is extended.
#
# Tuned on 934 ICSD perovskites by comparing the bond valence deviation before
# and after every revision. At 0.5 the guard revised 94 sites, improving 81 and
# WORSENING 13; every one of those 13 was a single-bond extension with an ECoN
# excess below 1.25, where the cut was already close to right. At 1.25 it
# revises 65 sites, improves 65, worsens none, and the total improvement is
# unchanged. Raising it further only discards good revisions.
ECON_TRIGGER_TOLERANCE = 1.25

# Ceiling on shell extension, as a multiple of the shortest bond. Matches
# CN_RELATIVE_CUTOFF in CELL 7 so the two agree by construction.
SHELL_EXTENSION_CUTOFF = 1.30


def econ_bound(distances, max_iter=25, tol=1e-9):
    """
    Hoppe (1979) effective coordination number, returning only the count.

    Defined here rather than reusing `econ` from CELL 1 so that this cell can
    be pasted over on its own without a load-order dependency.
    """
    d = np.asarray([float(x) for x in distances if x and x > 0], float)
    if d.size == 0:
        return 0.0
    d_av = d.min()
    for _ in range(max_iter):
        w = np.exp(1.0 - (d / d_av) ** 6)
        new = (d * w).sum() / w.sum()
        if abs(new - d_av) < tol:
            break
        d_av = new
    w = np.exp(1.0 - (d / d_av) ** 6)
    return float(w.sum())


def guarded_shell_cn(sorted_distances, gap_cn,
                     cutoff=SHELL_EXTENSION_CUTOFF,
                     tolerance=ECON_TRIGGER_TOLERANCE):
    """
    Validate a gap-analysis coordination number against ECoN, and extend it
    where the gap cut has amputated the coordination shell.

    Parameters
    ----------
    sorted_distances : ascending unique cation-anion distances
    gap_cn           : the coordination number from gap analysis
    cutoff           : shell ceiling as a multiple of the shortest bond
    tolerance        : how far ECoN must exceed the cut before it is revised

    Returns
    -------
    (cn, evidence). The evidence dict records what was tested and why, so a
    revision is always traceable and never silent.
    """
    ds = sorted(float(d) for d in sorted_distances if d and d > 0)
    ev = {'cn_gap': int(gap_cn), 'cn_revised': False,
          'cn_revision_reason': None, 'econ_shell': None, 'cn_shell_max': None}

    if len(ds) < 2 or gap_cn < 1:
        return int(gap_cn), ev

    # Candidate shell: everything within `cutoff` of the shortest bond. This is
    # the largest set the cut could plausibly have been drawn from.
    d0 = ds[0]
    cn_max = sum(1 for d in ds if d <= cutoff * d0)
    ev['cn_shell_max'] = int(cn_max)

    if cn_max <= gap_cn:
        # The cut already spans the whole candidate shell; nothing to do.
        ev['econ_shell'] = round(econ_bound(ds[:gap_cn]), 3)
        return int(gap_cn), ev

    e_shell = econ_bound(ds[:cn_max])
    ev['econ_shell'] = round(e_shell, 3)

    if e_shell <= gap_cn + tolerance:
        # ECoN is close enough to the cut that intervening does more harm than
        # good; see the note on ECON_TRIGGER_TOLERANCE.
        return int(gap_cn), ev

    # ECoN is counting bonds the cut discarded. Extend to the smallest CN
    # consistent with the ECoN measured over the FULL candidate shell.
    # Measuring it over the truncated set instead would be self-fulfilling:
    # ECoN of the first three bonds of a twelve-bond shell is always about 3.
    # Target the nearest integer to ECoN, NOT gap_cn + tolerance: the trigger
    # decides whether to intervene, the shell size decides how far.
    cn = int(min(cn_max, max(gap_cn, np.ceil(e_shell - 0.5))))
    ev['cn_revised'] = True
    ev['cn_revision_reason'] = (
        f'gap cut at {gap_cn} left ECoN {e_shell:.2f} over the '
        f'{cn_max}-bond shell; extended to {cn}')
    return int(cn), ev


# Slope of the bond-valence distance threshold with coordination number.
#
# The screen compares the SHORTEST contact against d_ideal, which is the MEAN
# bond length implied by the ion's own parameters. In any distorted polyhedron
# the shortest bond sits below the mean, and by more as the coordination rises,
# so a single fixed threshold is biased against high-coordination sites.
#
# Measured on sites in sound structures (GII < 0.25) in the 934-structure ICSD
# perovskite set:
#
#     CN         6      8      9     10     12
#     n        646    505     68     46     20
#     median  0.994  0.945  0.938  0.906  0.877
#     2nd pct 0.945  0.895  0.860  0.870  0.789
#
# At a fixed 0.90 the screen failed 1.1% of six-coordinate sites and 6.9% of
# A-site cations in structures that are otherwise sound, and roughly half of
# all twelve-coordinate sites. The trend is close to linear in CN, so the
# threshold now follows it, staying just below the 2nd percentile at every
# coordination. This is the same bias the retired A-site advisory override in
# CELL 8 was written for; that override was correct about this screen and
# wrong about the radii screen, where the two distributions are identical.
BVS_DISTANCE_CN_SLOPE = 0.018
BVS_DISTANCE_CN_REF = 6
BVS_DISTANCE_CN_CLAMP = (4, 12)


def bvs_distance_threshold(cn, base=None, slope=BVS_DISTANCE_CN_SLOPE):
    """
    Threshold for the bond-valence distance screen at a given coordination.

    Falls with CN because the shortest bond of a high-coordination site is
    legitimately further below its mean. Clamped, so an absurd CN from a
    broken structure cannot loosen the test that is meant to catch it.
    """
    base = IMPLAUSIBLE_BVS_DISTANCE_FRACTION if base is None else base
    try:
        c = float(cn)
    except (TypeError, ValueError):
        return base
    if c != c:
        return base
    lo, hi = BVS_DISTANCE_CN_CLAMP
    c = min(max(c, lo), hi)
    return base - slope * (c - BVS_DISTANCE_CN_REF)


# Slope of the radii-ratio threshold with coordination number.
#
# Same bias, same shape, same remedy as BVS_DISTANCE_CN_SLOPE above. The note on
# that constant records that the retired CELL 8 override "was correct about this
# screen and wrong about the radii screen, where the two distributions are
# identical". That was measured when gap analysis was reporting perovskite A
# sites at CN 8 or 9. Under the bond-valence shell (v2.0.3, 4b) they are at 12,
# and the two distributions are no longer identical: the conclusion expired with
# the shell rule that produced it.
#
# The screen compares the SHORTEST contact against the sum of the Shannon radii
# AT THE SITE'S OWN CN. Shannon radii rise with coordination, so raising the CN
# raises the expected distance while the shortest bond does not move. Across the
# 874 sites whose CN changed at 4b the mean cation radius rose from 1.100 to
# 1.203 A with no change to any observed contact.
#
# Measured on core_shortest_contact_ratio in structures with GII < 0.25,
# 938-structure ICSD perovskite set:
#
#     CN          6      9     10     12
#     n         701     74    279    340
#     median  0.976  0.904  0.907  0.881
#     2nd pct 0.918  0.861  0.856  0.828
#
# At a fixed 0.85 the screen failed 12.4% of twelve-coordinate sites against
# 0.6% of six-coordinate ones, and 3.72% of all sites in sound structures. At
# 0.015 per CN the false-positive rate is 0.43% overall and flat with
# coordination: 0.58% at CN 6, 0.00% at CN 8, 9 and 12, 0.36% at CN 10. The
# slope holds the threshold a constant ~0.068 below the 2nd percentile at every
# coordination, which is the same spacing the fixed value gave at CN 6. Slopes
# of 0.013 and 0.015 are indistinguishable on this dataset; steeper values buy
# almost nothing and only loosen the test.
#
# v2.0.4: this constant is a correction for a systematically wrong expected
# distance. With measured bond lengths available (expected_bond_length, below)
# the correction should become unnecessary; a refitted slope on measured
# distances is the test of whether it can be retired.
RADII_CN_SLOPE = 0.015
RADII_CN_REF = 6
RADII_CN_CLAMP = (4, 12)


def radii_contact_threshold(cn, base=None, slope=RADII_CN_SLOPE):
    """
    Threshold for the radii-ratio screen at a given coordination.

    Falls with CN for the same reason as bvs_distance_threshold, and clamped for
    the same reason: an absurd CN from a broken structure must not loosen the
    test that exists to catch it.
    """
    base = IMPLAUSIBLE_CONTACT_FRACTION if base is None else base
    try:
        c = float(cn)
    except (TypeError, ValueError):
        return base
    if c != c:
        return base
    lo, hi = RADII_CN_CLAMP
    c = min(max(c, lo), hi)
    return base - slope * (c - RADII_CN_REF)


def full_shell_cn(distances, factor=1.35, max_cut=3.5):
    """
    Integer coordination number by distance cutoff, robust to a lone short
    bond. Deliberately the same rule as `shell_cn` in CELL 1, so that
    cn_full_shell and geo_cn agree by construction and any difference between
    them is informative rather than definitional.
    """
    d = sorted(float(x) for x in distances if x and x > 0)
    if len(d) < 2:
        return len(d)
    d_ref = d[1] if d[1] / d[0] > 1.25 else d[0]
    cut = min(factor * d_ref, max_cut)
    return max(sum(1 for x in d if x <= cut), 2)


def group_bond_counts(sorted_cn_bonds, cn):
    """
    Group the shortest `cn` bonds into the {count, distance, error,
    ligand_label} records the bond valence sum consumes.

    Two ligand labels at the same distance are distinct species sharing a
    position and are kept apart, so their occupancies can be applied
    separately; this mirrors STEP 5 of auto_analyze_coordination.
    """
    if cn <= 0 or not sorted_cn_bonds:
        return []
    cn = min(int(cn), len(sorted_cn_bonds))
    cutoff = sorted_cn_bonds[cn - 1][0]
    sel = [(round(d, 5), e, d, l) for (d, e, l) in sorted_cn_bonds if d <= cutoff]
    cmap = Counter((r, e, l) for r, e, _, l in sel)
    out = []
    for (r, e, l) in sorted(cmap.keys(), key=lambda k: (k[0], str(k[2]))):
        orig = next((od for rr, ee, od, ll in sel
                     if rr == r and ee == e and ll == l), r)
        out.append({'count': cmap[(r, e, l)], 'distance': orig,
                    'error': e, 'ligand_label': l})
    return out


def cn_fault(cn, cn_rel, geo_cn=None, tol=0.5, under_tol=1.5):
    """
    Symmetric coordination fault detection.

    Replaces the `_cn_fault` closure in add_coordination_quality (CELL 7),
    which tested only `cn > cn_rel + 0.5` and so caught duplicate symmetry
    images but never an amputated shell. Under-counting is the more damaging
    direction: it lowers the bond valence sum and imitates genuine
    underbonding, whereas over-counting raises it and is usually obvious.

    A shell short by one bond is normal ambiguity between shell definitions,
    hence the wider tolerance in that direction; short by two or more is an
    amputation.

    To use in CELL 7, replace

        df['cn_fault'] = [_cn_fault(c, r) for c, r in
                          zip(df['cn'], df['cn_relative'])]

    with

        df['cn_fault'] = [cn_fault(c, r, g) for c, r, g in
                          zip(df['cn'], df['cn_relative'],
                              df['geo_cn'] if 'geo_cn' in df.columns
                              else [None] * len(df))]
    """
    if cn is None or cn != cn:
        return None
    refs = [r for r in (cn_rel, geo_cn) if r is not None and r == r and r]
    if not refs:
        return None

    if cn_rel is not None and cn_rel == cn_rel and cn_rel and cn > cn_rel + tol:
        m = cn / cn_rel
        if abs(m - round(m)) < 1e-6 and round(m) > 1:
            return f'duplicate_images_x{int(round(m))}'
        return 'cn_exceeds_relative'

    # Reported against the most generous reference: a shell is amputated
    # relative to its true size, not relative to the smallest estimate.
    ref = max(refs)
    if cn < ref - under_tol:
        return ('cn_below_shell_severe' if (ref - cn) >= 3
                else 'cn_below_shell_mild')
    return None


def VALIDATE_CN(verbose=True):
    """
    Checks for the shell guard. Half of these exist to confirm the fix does
    NOT disturb correct behaviour, which is the part that matters.
    """
    cases = [
        # the failure: a distorted twelve-coordinate A site cut at three
        ('distorted A site, gap cut at 3',
         [2.42, 2.45, 2.51, 2.72, 2.75, 2.78, 2.80, 2.83, 2.90, 2.95, 3.02, 3.05],
         3, lambda cn: cn >= 8),
        ('distorted A site, gap cut at 12',
         [2.42, 2.45, 2.51, 2.72, 2.75, 2.78, 2.80, 2.83, 2.90, 2.95, 3.02, 3.05],
         12, lambda cn: cn == 12),
        ('regular octahedron',
         [1.97, 1.97, 1.97, 1.98, 1.98, 1.98, 3.30, 3.31], 6, lambda cn: cn == 6),
        # Eir's own JT axial recovery restores 4 -> 6 before the guard sees
        # this, and the guard's trigger now sits above the ECoN excess of a
        # 4+2 polyhedron (0.64), so it correctly declines to act twice.
        ('Jahn-Teller 4+2 cut to 4',
         [1.94, 1.95, 1.95, 1.96, 2.35, 2.36, 3.40], 4, lambda cn: cn == 4),
        ('genuine square planar',
         [1.94, 1.94, 1.95, 1.95, 2.85, 2.86, 3.50], 4, lambda cn: cn == 4),
        ('regular tetrahedron',
         [1.62, 1.62, 1.63, 1.63, 3.10, 3.15], 4, lambda cn: cn == 4),
        ('Wyckoff-error site',
         [1.60, 1.63, 2.95, 3.10, 3.20], 2, lambda cn: cn <= 3),
        ('single contact', [2.10], 1, lambda cn: cn == 1),
        ('no contacts', [], 0, lambda cn: cn == 0),
        # a regular polyhedron must be recovered in full, not one short:
        # the target is the nearest integer to ECoN, which here is exactly 12
        ('regular cuboctahedron cut to 6', [2.72] * 12 + [3.85], 6,
         lambda cn: cn == 12),
        ('cut at 3 with a real boundary',
         [2.00, 2.02, 2.04, 2.60, 2.61, 2.62, 4.00], 3, lambda cn: cn <= 6),
    ]
    fault_cases = [
        ((3, 12.0, 12.0), 'cn_below_shell_severe'),
        ((8, 9.0, 9.0), None),            # short by one: normal ambiguity
        ((11, 12.0, 12.0), None),
        ((12, 6.0, 6.0), 'duplicate_images_x2'),
        ((7, 6.0, 6.0), 'cn_exceeds_relative'),
        ((4, 6.0, 6.0), 'cn_below_shell_mild'),
        ((6, None, None), None),
    ]

    passed = 0
    for name, ds, gap_cn, check in cases:
        cn, ev = guarded_shell_cn(ds, gap_cn)
        ok = check(cn)
        passed += ok
        if verbose:
            print(f"  [{'pass' if ok else 'FAIL'}] {name:<38} gap {gap_cn:>2} -> "
                  f"cn {cn:>2}   {ev['cn_revision_reason'] or 'unchanged'}")
    for args, expect in fault_cases:
        got = cn_fault(*args)
        ok = got == expect
        passed += ok
        if verbose:
            print(f"  [{'pass' if ok else 'FAIL'}] cn_fault{args} -> {got!r} "
                  f"(expected {expect!r})")
    total = len(cases) + len(fault_cases)
    print(f'\n  VALIDATE_CN: {passed}/{total} checks passed')
    return passed == total




# ============================================================================
# RADII-RATIO SCREEN (REWRITTEN IN v2.0.2)
# ============================================================================
# The screen compares the shortest cation-anion contact with the sum of the
# ionic radii. It used SHANNON_CN6_VIZ, a 25-element table built for the 3D
# visualiser, and `continue`d silently whenever the element was absent from
# it. That table contains no lanthanides at all, and no Y, Zr, Sn, Bi, Ta, Rh,
# In, Ge, Si, Pb, Mo or W.
#
# The consequence was that the screen had never tested a rare-earth site:
#
#                    sites testable   silently skipped   structures with
#                                                        no testable site
#     perovskites      1008 (53%)        879 (47%)             73
#     pyrochlores       312 (28%)        821 (72%)            222
#
# A LaCrO3 entry (ICSD 220506) with a 1.008 A La-O contact, 41% of the sum of
# the radii, passed the screen because La was not in the table. So did YVO3
# 121777 at 1.403 A and NdFeO3 74087 at 1.507 A. For the pyrochlores the
# reported "radii ratio: 0 structures flagged" did not mean nothing failed; it
# meant almost nothing was tested.
#
# The screen now takes the cation radius from the full Shannon database at the
# site's own valence and coordination number, which is the same source
# geometry_plausible() in CELL 7 already used, and falls back through the
# smaller tables only if that returns nothing. Where no radius can be found at
# all the site is recorded as UNTESTED rather than skipped, so a null is
# visibly different from a pass.

_CORE_ANION_RADIUS = {'O': 1.40, 'F': 1.33, 'Cl': 1.81, 'Br': 1.96, 'I': 2.20,
                      'S': 1.84, 'Se': 1.98, 'Te': 2.21, 'N': 1.46}


def core_radii_ratio(element, ligand, d_shortest, valence=None, cn=None):
    """
    Shortest contact as a fraction of the sum of the ionic radii.

    Returns (ratio, expected_distance, source). `source` is one of
    'shannon_db', 'shannon_cn6', 'viz_table' or 'untested', so the provenance
    of every screening decision is recorded rather than assumed.
    """
    r_c, source = None, None

    # 1. the full Shannon tabulation, at this site's own valence and CN
    if valence is not None and cn:
        try:
            v = float(valence)
            if v == v:
                r_c, _ = get_ionic_radius_fractional(element, v, int(cn))
                if r_c is not None:
                    source = 'shannon_db'
        except (TypeError, ValueError):
            r_c = None

    # 2. the CN6 table from CELL 7, if that cell has been run
    if r_c is None:
        try:
            r_c = SHANNON_CN6.get(element)
        except NameError:
            r_c = None
        if r_c is not None:
            source = 'shannon_cn6'

    # 3. the visualiser table, last. Wrapped like the one above: if this is
    # ever reached before CELL 1 has run, an untested verdict is the right
    # answer, not a NameError raised from inside a screening loop.
    if r_c is None:
        try:
            r_c = SHANNON_CN6_VIZ.get(element)
        except NameError:
            r_c = None
        if r_c is not None:
            source = 'viz_table'

    lig = ligand or 'O'
    r_a = _CORE_ANION_RADIUS.get(lig)
    if r_a is None:
        try:
            r_a, _ = get_ionic_radius_fractional(
                lig, -2 if lig in ('O', 'S', 'Se', 'Te') else -1, 6)
        except (TypeError, ValueError):
            r_a = None

    if not (d_shortest and r_c and r_a):
        return None, None, 'untested'
    expected = r_c + r_a
    return d_shortest / expected, expected, source



# ============================================================================
# COORDINATION PLAUSIBILITY SCREEN (NEW IN v2.0.2)
# ============================================================================
# The three geometry screens all ask the same question from different angles:
# is some contact too SHORT. None of them asks whether bonds are MISSING, and
# that is a distinct and common failure. A cation deposited at the wrong site
# keeps normal bond lengths to the few anions it does reach, so every
# short-contact test passes while half its coordination sphere is absent.
#
# Measured over 1877 cation sites in the 929-structure ICSD perovskite set,
# median GII against the coordination number found:
#
#     CN        1     2     3     4     5     6     7     8     9    10    12
#     n         8    28    12    20     5   912     7   554   108   140    75
#     GII    1.17  0.80  1.15  0.83  1.32  0.18  0.42  0.16  0.25  0.14  0.21
#
# Every ordinary coordination sits near 0.15-0.20; every unusual one is three
# to eight times worse. Thirteen of those sites were in structures that passed
# all three geometry screens, including NaOsO3 with Os(5+) at CN 3 and a GII
# of 1.71, and LaYO3 with Y(3+) at CN 3 and a GII of 1.41.
#
# Coordination number alone is not the test, though: ICSD 79873 has Cu at CN 4
# with a GII of 0.085 and a bond valence sum 3.8% from formal, which is an
# ordinary square-planar Cu(2+). The question is whether THIS ion is known to
# adopt THIS coordination, and Shannon's tabulation already answers it. An ion
# is tabulated at the coordinations it is observed in, so a coordination well
# outside that range is a statement about the deposited structure.
#
# This screen therefore uses no new data and no new threshold: it reports the
# span of coordination numbers Shannon lists for the ion, and whether the
# observed value falls inside it.

# The tolerance is deliberately ASYMMETRIC, because the two directions mean
# different things.
#
# Under-coordination is the diagnostic case and the tabulated minimum is a
# real floor: an ion is not observed below the smallest coordination Shannon
# lists for it, so any excursion below is evidence. A symmetric tolerance of 1
# was tried first and let four of the seven known-bad cases through, including
# Ti(4+) at CN 3 against a tabulated floor of 4, and Sn(4+) at CN 3 likewise.
#
# Over-coordination is weaker evidence. Shannon does not tabulate every ion at
# every high coordination it adopts, and an inflated count is more often a
# symmetry-expansion artefact, which cn_fault in CELL 7 already names.
# Under-coordination is the diagnostic direction and the tabulated minimum
# is a real floor: an ion is not observed below the smallest coordination
# Shannon lists for it, so any excursion below is evidence.
CN_UNDER_TOLERANCE = 0

# Over-coordination is OFF. It was tried at a tolerance of 1 and proved
# actively misleading: of 64 over-coordinated sites in the 934-structure ICSD
# perovskite set, 26 sat in structures with GII below 0.25, and the median GII
# of the whole over-coordinated group was 0.285 against 1.048 for the
# under-coordinated group and 0.19 for the dataset. It is barely distinguish-
# able from the background.
#
# The cause is that Shannon does not tabulate every ion at every coordination
# it adopts in a perovskite A site. ScAlO3 has a GII of 0.029, about as good
# as a perovskite gets, and all four ICSD entries were flagged because Sc(3+)
# is tabulated only to CN 8 while its A site is 10-coordinate. Yb, Ho, Tm and
# Lu behave the same way.
#
# Duplicate symmetry images, the one over-counting case that does matter, are
# already named by cn_fault in CELL 7. Set this to an integer to re-enable.
CN_OVER_TOLERANCE = None


def shannon_cn_range(element, valence):
    """
    The coordination numbers Shannon tabulates for an ion.

    Returns (cn_min, cn_max, n_entries), or (None, None, 0) if the ion is not
    tabulated. Fractional valences take the union of the bracketing integers,
    matching how get_bv_parameters_fractional and get_ionic_radius_fractional
    treat them.
    """
    if db_manager.shannon_radii is None:
        db_manager.load_shannon_radii()
    table = db_manager.shannon_radii
    if not table or valence is None:
        return None, None, 0
    try:
        v = float(valence)
    except (TypeError, ValueError):
        return None, None, 0
    if v != v:
        return None, None, 0
    vs = ({int(v)} if v == int(v)
          else {int(math.floor(v)), int(math.ceil(v))})
    cns = [k[2] for k in table if k[0] == element and k[1] in vs]
    if not cns:
        return None, None, 0
    return min(cns), max(cns), len(cns)


def coordination_plausible(element, valence, cn,
                           under_tol=CN_UNDER_TOLERANCE,
                           over_tol=CN_OVER_TOLERANCE):
    """
    Is the coordination number one this ion is known to adopt?

    Returns (verdict, cn_min, cn_max) where verdict is True, False, or None
    when the ion is not tabulated and no judgement is possible. None is
    returned rather than True so that an untested site is never mistaken for a
    clean one.

    Under-coordination is the diagnostic case: it means bonds are missing,
    which the short-contact screens cannot see. Over-coordination is reported
    too, but is more often a symmetry-expansion artefact than a bad structure,
    and cn_fault in CELL 7 already names that case specifically.

    v2.0.3: a coordination number of zero is a FAILURE, not an absence of
    evidence. It was previously folded in with the untabulated ions and
    returned None, which every screen reads as a pass. ICSD 22964 (GdFeO3)
    reached the lowest GII in the perovskite set that way, with its Gd site
    carrying no bonds, no sum, and no flag.
    """
    if cn is None or cn != cn:
        return None, None, None
    if cn <= 0:
        # No shell at all. Nothing to compare against a tabulated range, and
        # nothing that could make this acceptable.
        return False, None, None
    lo, hi, n = shannon_cn_range(element, valence)
    if lo is None:
        return None, None, None
    # A single tabulated coordination is the STRONGEST evidence, not the
    # weakest: Os(5+) is listed only at CN 6, so finding it at CN 3 is exactly
    # the case worth flagging. An earlier version treated n == 1 as untestable
    # and returned None, which excused two of the worst entries in the set.
    if cn < lo - under_tol:
        return False, lo, hi
    if over_tol is not None and cn > hi + over_tol:
        return False, lo, hi
    return True, lo, hi



# ============================================================================
# HALF-LATTICE SHIFT CONSISTENCY (NEW IN v2.0.2)
# ============================================================================
# The failure mode that explains most low cn_relative values. A cation
# deposited half a lattice vector from where the anion sublattice expects it
# keeps a full complement of neighbours, so nothing is missing and nothing is
# impossibly short; the coordination sphere is SPLIT, some contacts too short
# and some too long. Every geometry screen passes it, and the bond valence sum
# often passes too, because the over- and under-bonded halves cancel.
#
# ICSD 209078 (CaTiO3, Prodjosantoso et al. 2017) is the type case. Ti is
# deposited at 4a (0,0,0) in Pbnm while the oxygens were refined for 4b:
#
#     as deposited : Ti-O 1.65, 2.20, 2.97 A   BVS 3.89, only 2.8% from formal
#     half-shifted : Ti-O 1.92, 1.97 A         textbook CaTiO3
#
# The bond valence sum is nearly right at the wrong position, because the short
# and long halves cancel. Only the geometry gives it away.
#
# The Wyckoff letter is NOT the test. ICSD 403584 (PbNiO3) also has its B
# cation at 4a and is entirely sound: its A site sits at y = 0.458 against
# 0.031 in CaTiO3, so the two are described on origins differing by (0,1/2,0),
# under which 4a and 4b exchange. Both conventions are legal, and a screen
# keyed on the label would condemn a correct structure. ICSD 79873 (NdCuO2.93)
# is a third control: its Cu first shell spans 1.82 to 2.50 A, which is a
# Jahn-Teller 4+2 and must also pass.
#
# This screen DETECTS and PROPOSES. It never rewrites coordinates. A shift that
# improves the geometry is evidence of a mixed-origin deposition, not proof of
# one, and it cannot be proven from a single deposited CIF.

# Ratio of longest to shortest bond in the first shell, above which the
# coordination is judged split rather than merely distorted. A regular
# octahedron sits near 1.02, a Jahn-Teller 4+2 near 1.37 (ICSD 79873, which
# must NOT be flagged), a strongly tilted A site near 1.5.
SHIFT_SPREAD_SUSPECT = 1.50

# How much better the alternative must be before it is worth proposing.
# Deliberately demanding: the cost of a false proposal on a sound structure is
# higher than the cost of missing a marginal one.
SHIFT_IMPROVEMENT = 1.30

# Bonds compared at each candidate position. Six is the octahedral first shell
# in these families; comparing the same count at both positions is what makes
# the two numbers comparable, so the value matters less than its consistency.
SHIFT_SHELL_SIZE = 6

# Above this many candidate translations the space group has too little
# symmetry for the test to mean anything. P1 admits every translation, so the
# screen would "improve" any site by sliding it onto the nearest anion.
NORMALISER_MAX_CANDIDATES = 15

_NORMALISER_CACHE = {}
_ORIGIN_CHOICE_CACHE = {}


def normaliser_translations(sg, denoms=(2, 3, 4, 6)):
    """
    The pure translations under which a structure description is equivalent.

    A translation t maps the space group onto itself iff (I - W)t is a lattice
    translation for every rotation part W in the group. Conjugating (W, w) by t
    gives (W, w + (I - W)t), so the condition is exactly that the operation
    list is unchanged modulo the lattice.

    v2.0.6: this REPLACES a hard-coded list of the seven half-lattice vectors.
    Those were not arbitrary and not perovskite-specific, but nor were they
    general: they are the normaliser translations of Pnma, which follow from
    inversion alone (W = -I gives 2t in Z^3, hence t in {0, 1/2}^3). The
    derivation here returns exactly those seven for space group 62, and returns
    the RIGHT answer elsewhere, which the constant could not. For Fd-3m, the
    pyrochlore group, the cubic rotations force t = (1/2, 1/2, 1/2) and nothing
    else; the seven half-lattice vectors are not normaliser translations of that
    group at all, which is why the screen fired zero times on 472 pyrochlores.

    Candidates equal to a centring vector, or differing from one already found
    by a centring vector, are dropped: they describe the same structure.
    """
    key = sg.xhm() if hasattr(sg, 'xhm') else str(sg.number)
    if key in _NORMALISER_CACHE:
        return _NORMALISER_CACHE[key]

    from fractions import Fraction as _Fr
    import itertools as _it

    ops = sg.operations()
    rots = {tuple(tuple(r) for r in op.rot) for op in ops}
    cen = [tuple(_Fr(int(x), 24) for x in c) for c in ops.cen_ops]
    vals = sorted({_Fr(n, d) for d in denoms for n in range(d)})

    found = []
    for t in _it.product(vals, repeat=3):
        if all(v == 0 for v in t):
            continue
        ok = True
        for R in rots:
            # gemmi stores rotation parts scaled by 24
            for i in range(3):
                d = t[i] - sum(_Fr(R[i][j], 24) * t[j] for j in range(3))
                if d.denominator != 1:
                    ok = False
                    break
            if not ok:
                break
        if not ok:
            continue
        if any(all((t[i] - c[i]).denominator == 1 for i in range(3)) for c in cen):
            continue
        if any(all(((t[i] - u[i]) - c[i]).denominator == 1 for i in range(3))
               for u in found for c in cen):
            continue
        found.append(t)

    if len(found) > NORMALISER_MAX_CANDIDATES:
        found = []
    out = [tuple(float(x) for x in t) for t in found]
    _NORMALISER_CACHE[key] = out
    return out


def origin_choice_offset(sg):
    """
    The offset between the two origin choices of a space group, or None.

    Twenty-four space groups are tabulated with two origins, one at a centre of
    symmetry and one at a site of higher point symmetry. A CIF whose symmetry
    operations belong to one setting while its coordinates were refined on the
    other is mis-registered as a WHOLE STRUCTURE, and no normaliser translation
    describes that: the two settings have different operation lists.

    Derived rather than tabulated. The offset is the t for which conjugating
    setting 1 by t reproduces setting 2, found by search over eighths, quarters
    and thirds. For Fd-3m this returns (3/8, 3/8, 3/8), the tabulated value.
    """
    key = sg.number
    if key in _ORIGIN_CHOICE_CACHE:
        return _ORIGIN_CHOICE_CACHE[key]

    import gemmi as _g
    from fractions import Fraction as _Fr
    import itertools as _it

    def _opset(g):
        return {(tuple(tuple(r) for r in op.rot),
                 tuple(_Fr(int(x), 24) % 1 for x in op.tran))
                for op in g.operations()}

    res = None
    try:
        base = sg.short_name()
        s1 = _g.find_spacegroup_by_name(base + ':1')
        s2 = _g.find_spacegroup_by_name(base + ':2')
        if s1 is not None and s2 is not None:
            A, B = _opset(s1), _opset(s2)
            if A != B:
                vals = sorted({_Fr(n, d) for d in (2, 4, 8, 3, 6)
                               for n in range(d)})
                for t in _it.product(vals, repeat=3):
                    moved = set()
                    for R, w in A:
                        moved.add((R, tuple(
                            (w[i] + t[i]
                             - sum(_Fr(R[i][j], 24) * t[j] for j in range(3))) % 1
                            for i in range(3))))
                    if moved == B:
                        res = tuple(float(x) for x in t)
                        break
    except Exception:
        res = None
    _ORIGIN_CHOICE_CACHE[key] = res
    return res


def _shell_at(cell, anion_pts, frac, n=SHIFT_SHELL_SIZE, rmax=4.0):
    # v2.0.6: n defaults to six because an octahedron has six, which was a
    # perovskite assumption. Callers now pass the site's OWN coordination, so
    # the same count is compared at both positions whatever the family.
    """Sorted distances to the n nearest anions from a fractional position."""
    import gemmi
    c = cell.orthogonalize(gemmi.Fractional(*frac))
    d = sorted(p.dist(c) for p in anion_pts if 0.1 < p.dist(c) < rmax)
    return d[:n]


def _spread(d):
    """Longest over shortest in the shell. 1.0 is a regular polyhedron."""
    if len(d) < 2 or d[0] <= 0:
        return None
    return d[-1] / d[0]


def propose_half_lattice_shift(cell, anion_pts, frac, r_expected=None,
                               spread_suspect=SHIFT_SPREAD_SUSPECT,
                               improvement=SHIFT_IMPROVEMENT,
                               candidates=None, shell_size=None):
    """
    Test a cation position against the alternatives its space group allows.

    v2.0.6: `candidates` is the normaliser translation set for the structure's
    own space group, from normaliser_translations(). When None the seven
    half-lattice vectors are used, which is correct for Pnma and wrong for
    anything else; the caller should always pass them.

    `shell_size` is the site's own coordination number. The comparison is only
    meaningful if the same number of bonds is counted at both positions, and
    fixing that number at six was an assumption about octahedra.

    Returns a dict describing the deposited geometry and, where one is
    warranted, the best alternative. `proposed_shift` is None when the
    deposited position is the better of the candidates, which is the usual and
    expected outcome.

    r_expected is the sum of the ionic radii, used only to reject an
    alternative whose shortest contact is impossible; the screen never
    proposes a shift into a worse geometry.
    """
    n_shell = int(shell_size) if shell_size and shell_size > 1 else SHIFT_SHELL_SIZE
    shifts = candidates if candidates is not None else [
        (0.5, 0, 0), (0, 0.5, 0), (0, 0, 0.5), (0.5, 0.5, 0),
        (0.5, 0, 0.5), (0, 0.5, 0.5), (0.5, 0.5, 0.5)]
    d0 = _shell_at(cell, anion_pts, frac, n=n_shell)
    s0 = _spread(d0)
    out = {'shift_shell_spread': round(s0, 4) if s0 else None,
           'shift_n_candidates': len(shifts),
           'shift_shell_size': n_shell,
           'shift_shell_min': round(d0[0], 4) if d0 else None,
           'proposed_shift': None, 'proposed_spread': None,
           'proposed_shell_min': None, 'shift_evidence': None}
    if s0 is None:
        return out

    best = None
    for sh in shifts:
        f2 = tuple((frac[i] + sh[i]) % 1.0 for i in range(3))
        d1 = _shell_at(cell, anion_pts, f2, n=n_shell)
        s1 = _spread(d1)
        if s1 is None or not d1:
            continue
        # Never propose a move into an impossible contact.
        if r_expected and d1[0] < 0.85 * r_expected:
            continue
        if s1 > spread_suspect:
            continue
        if best is None or s1 < best[1]:
            best = (sh, s1, d1[0])

    # Only propose when the deposited geometry is itself suspect AND the
    # alternative is markedly better. Both conditions matter: the first stops
    # the screen second-guessing sound structures, the second stops it acting
    # on noise.
    if best and s0 > spread_suspect and best[1] * improvement < s0:
        out['proposed_shift'] = '%g,%g,%g' % best[0]
        out['proposed_spread'] = round(best[1], 4)
        out['proposed_shell_min'] = round(best[2], 4)
        out['shift_evidence'] = (
            f'first-shell spread {s0:.2f} as deposited, {best[1]:.2f} at '
            f'{best[0]}; suspected mixed-origin deposition, NOT corrected')
    return out


# ---------------------------------------------------------------------------
def _anion_points(st, anion_elements=('O', 'F', 'S', 'Cl', 'N', 'Se', 'Br', 'I')):
    """Symmetry-expanded anion positions over the cells needed for a 4 A reach."""
    import gemmi
    n = max(1, int(math.ceil(4.0 / min(st.cell.a, st.cell.b, st.cell.c))))
    span = range(-n, n + 1)
    pts = []
    for s in st.sites:
        if s.element.name not in anion_elements:
            continue
        for op in st.spacegroup.operations():
            f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
            for i in span:
                for j in span:
                    for k in span:
                        pts.append(st.cell.orthogonalize(
                            gemmi.Fractional(f[0] + i, f[1] + j, f[2] + k)))
    return pts


def _bvs_at_proposal(cell, anion_pts, frac, shift_txt, element, valence,
                     anion='O'):
    """
    Bond valence sum the site WOULD have at the proposed position.

    Computed for reporting only; nothing is written back. Uses the same
    parameters and the same shell size as the sum at the deposited position,
    so the two are comparable, and reports the deviation from the formal
    valence at each so the proposal can be judged rather than taken on trust.

    On ICSD 138222 (DyCrO3) the deposited Cr sums to 8.36 v.u. against a
    formal 3; at the proposed position it sums to close to 3. That is the
    evidence for the proposal, and it belongs beside it.
    """
    out = {'bvs_if_shifted': None, 'bvs_deviation_if_shifted': None,
           'shift_improves_bvs': None}
    try:
        v = float(valence)
        if v == 0 or v != v:
            return out
        R0, b = get_bv_parameters_fractional(element, v, anion)
        if R0 is None:
            return out
        sh = [float(x) for x in str(shift_txt).split(',')]
        f2 = tuple((frac[i] + sh[i]) % 1.0 for i in range(3))
        d_now = _shell_at(cell, anion_pts, frac)
        d_new = _shell_at(cell, anion_pts, f2)
        if not d_new:
            return out
        s_now = sum(math.exp((R0 - d) / b) for d in d_now)
        s_new = sum(math.exp((R0 - d) / b) for d in d_new)
        out['bvs_if_shifted'] = round(s_new, 4)
        out['bvs_deviation_if_shifted'] = round(s_new - abs(v), 4)
        out['shift_improves_bvs'] = bool(
            abs(s_new - abs(v)) < abs(s_now - abs(v)))
    except (TypeError, ValueError, ZeroDivisionError):
        return out
    return out


def half_lattice_screen(cif_path, sites, verbose=False):
    """
    Run the shift consistency test over every cation site of one structure.

    Mutates each site dict in place and returns the number of sites for which
    a shift is proposed. Loads the CIF once; failures are non-fatal, because a
    proposal is an enhancement to the reporting and must never cost the
    structure its analysis.
    """
    try:
        import gemmi, tempfile, os as _os
        cleaned = clean_cif_for_gemmi(cif_path)
        if cleaned is None:
            return 0
        with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False,
                                         encoding='utf-8') as tmp:
            tmp.write(cleaned); tmp_path = tmp.name
        try:
            st = gemmi.make_small_structure_from_block(
                gemmi.cif.read(tmp_path).sole_block())
        finally:
            try:
                _os.unlink(tmp_path)
            except OSError:
                pass
        if st is None or st.spacegroup is None or not st.sites:
            return 0
        pts = _anion_points(st)
        if not pts:
            return 0
        # v2.0.6: derived once per structure from its own symmetry, not
        # assumed. An empty set means the group admits too many translations
        # for the test to discriminate (P1 and the like), and the screen
        # stands down rather than proposing something meaningless.
        _cand_shifts = normaliser_translations(st.spacegroup)
        _orig = origin_choice_offset(st.spacegroup)
        if verbose:
            print(f'  [half-lattice screen: {len(_cand_shifts)} candidate '
                  f'translation(s) for {st.spacegroup.hm}'
                  + (f'; origin-choice offset {_orig}' if _orig else '') + ']')
        if not _cand_shifts:
            return 0
    except (NameError, AttributeError, TypeError, IndexError, KeyError):
        # As below: these are defects in the screen, not properties of the
        # CIF, and the derivation of the candidate translations runs inside
        # this block. A broken derivation must not present as a clean run.
        raise
    except Exception as _e:
        # Not conditional on verbose. A structure the screen could not read
        # is a structure the screen did not test, and that should be visible.
        print(f'  [half-lattice screen unavailable: {type(_e).__name__}: {_e}]')
        return 0

    n_proposed = 0
    _skipped = []
    for _s in sites:
        coords = _s.get('coords')
        if not coords or len(coords) != 3:
            continue
        try:
            res = propose_half_lattice_shift(
                st.cell, pts, tuple(float(c) for c in coords),
                r_expected=_s.get('core_expected_contact'),
                candidates=_cand_shifts,
                shell_size=_s.get('cn'))
            # v2.0.2: say what the proposal would GIVE. A candidate nobody
            # evaluates is an assertion; the consequence is computable without
            # acting on it, and it is what makes the proposal falsifiable.
            # Nothing is written back to the structure.
            if res.get('proposed_shift'):
                res.update(_bvs_at_proposal(
                    st.cell, pts,
                    tuple(float(c) for c in coords),
                    res['proposed_shift'],
                    _s.get('element'), _s.get('valence')))
        except (NameError, AttributeError, TypeError, IndexError, KeyError):
            # v2.0.13: a defect in this code, not in the structure. Swallowed,
            # it reports zero proposals across an entire dataset without a
            # word: a positional index in propose_half_lattice_shift went out
            # of step and the screen silently stopped working on all 938
            # perovskites while every other number stayed correct.
            raise
        except Exception as _e:
            # A genuine per-site failure skips the site, not the run, but it
            # is counted and named rather than passed over.
            _skipped.append(f"{_s.get('site_label')}: {type(_e).__name__}: {_e}")
            continue
        _s.update(res)
        if res.get('proposed_shift'):
            n_proposed += 1
            print(f"\n  SPLIT COORDINATION: {_s.get('element')} at "
                  f"{_s.get('site_label')} has a first shell spanning "
                  f"{res['shift_shell_min']:.3f} to "
                  f"{res['shift_shell_min'] * res['shift_shell_spread']:.3f} A "
                  f"(ratio {res['shift_shell_spread']:.2f}).")
            print(f"  Shifting it by ({res['proposed_shift']}) gives a spread of "
                  f"{res['proposed_spread']:.2f}, which suggests the cation and "
                  f"anion sublattices were deposited on different origins.")
            if res.get('bvs_if_shifted') is not None:
                print(f"  Its bond valence sum would go from "
                      f"{_s.get('bvs')} to {res['bvs_if_shifted']} v.u. against a "
                      f"formal {_s.get('valence')}.")
            print(f"  Nothing has been changed. This is a candidate to check "
                  f"against the source, not a correction: it cannot be proven "
                  f"from one deposited CIF.")
    if _skipped:
        print(f'  [half-lattice screen skipped {len(_skipped)} site(s): '
              f"{'; '.join(_skipped[:3])}"
              f"{' ...' if len(_skipped) > 3 else ''}]")
    return n_proposed

def auto_analyze_coordination(processed_data):
    """
    Automatically analyse coordination numbers and bond lengths.
    Uses the EXACT gap analysis logic from Skuld 1.5 but accepts results automatically.
    
    CRITICAL: Preserves lines 1083-1160 from Skuld 1.5 for proper coordination detection.
    """
    final_data = processed_data

    for block_name, block_content in final_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content['selected_bond_types_for_analysis']:
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            central_atom_elements_at_site = site_info['elements']
            
            bonds_for_analysis = {}
            for (central_type_selected, ligand_type_selected) in block_content['selected_bond_types_for_analysis'].keys():
                if central_type_selected in central_atom_elements_at_site:
                    filtered_raw_bonds = [
                        b for b in site_info['raw_bonds_list'] 
                        if b[2] == central_type_selected and b[3] == ligand_type_selected
                    ]
                    if filtered_raw_bonds:
                        bonds_for_analysis[(central_type_selected, ligand_type_selected)] = filtered_raw_bonds

            if not bonds_for_analysis:
                continue

            site_info['cn_and_bonds_by_type'] = {}

            for (central_type, ligand_type), raw_bonds_list_for_pair in bonds_for_analysis.items():
                
                # Report site coordinates for validation
                print(f"\n    Analyzing site at {coords}: {central_type}")
                
                # ======================================================================
                # CRITICAL SECTION: EXACT COPY from Skuld 1.5 (lines 1083-1160)
                # This handles symmetry operations and proper coordination detection
                # ======================================================================
                
                # STEP 1: Identify unique geometric positions using (label, position)
                # For ICSD files, symmetry_code now includes |x_y_z position encoding
                unique_geometric_positions_to_shortest_bond = {}
                
                # DEBUG: Show first few bonds to verify position encoding
                if len(raw_bonds_list_for_pair) <= 10:
                    _dbg(f"  DEBUG: Raw bonds for {central_type}-{ligand_type}:")
                    for bond_tuple in raw_bonds_list_for_pair[:10]:
                        _dbg(f"    {bond_tuple[0]}-{bond_tuple[1]}: {bond_tuple[5]:.4f}Å symm='{bond_tuple[4]}'")
                
                # DEBUG: Track unique positions
                position_debug = set()
                
                for bond_tuple in raw_bonds_list_for_pair:
                    ligand_label = bond_tuple[1]
                    ligand_symmetry_code = bond_tuple[4]  # May include |x_y_z position
                    dist_val = bond_tuple[5]
                    dist_err = bond_tuple[6]

                    # Use Cartesian position if available, else use symmetry code
                    if '|' in ligand_symmetry_code:
                        # ICSD mode: extract position part for uniqueness
                        symm_part, pos_part = ligand_symmetry_code.split('|', 1)
                        geo_position_id = (ligand_label, pos_part)  # Use 3D position
                        position_debug.add(pos_part)
                    else:
                        # GSAS-II mode: use symmetry code
                        geo_position_id = (ligand_label, ligand_symmetry_code)
                    
                    if geo_position_id not in unique_geometric_positions_to_shortest_bond or \
                       dist_val < unique_geometric_positions_to_shortest_bond[geo_position_id][0]:  # Compare with OLD distance [0], not error [1]
                        unique_geometric_positions_to_shortest_bond[geo_position_id] = (dist_val, dist_err)
                
                # DEBUG: Show unique positions found
                if position_debug:
                    _dbg(f"  DEBUG: Found {len(position_debug)} unique positions: {sorted(list(position_debug)[:6])}")

                # STEP 2: Create list of all bonds within coordination sphere
                cn_bonds_list = []
                for _geo_id, (dist_val, dist_err) in unique_geometric_positions_to_shortest_bond.items():
                    # v1.8.3 (A8): keep the ligand label so its occupancy can
                    # weight the bond valence sum further downstream.
                    cn_bonds_list.append((dist_val, dist_err, _geo_id[0]))
                
                sorted_cn_bonds = sorted(cn_bonds_list, key=lambda x: x[0])

                # STEP 3: Gap analysis to find primary coordination sphere
                # Add minimum gap threshold to avoid splitting coordination due to numerical precision
                MIN_GAP_THRESHOLD = 0.05  # Å - ignore gaps smaller than this
                initial_cn = 0
                primary_sphere_cutoff_dist = 0.0 
                
                if sorted_cn_bonds:
                    max_gap = 0
                    cutoff_index = None 
                    
                    if len(sorted_cn_bonds) > 1:
                        for i in range(len(sorted_cn_bonds) - 1):
                            gap = sorted_cn_bonds[i+1][0] - sorted_cn_bonds[i][0]
                            # Only consider gaps larger than threshold
                            if gap > MIN_GAP_THRESHOLD and gap > max_gap:
                                max_gap = gap
                                cutoff_index = i + 1
                    if cutoff_index is None:
                        # no resolvable shell boundary: bound by the first
                        # shell rather than by the search radius
                        d0 = sorted_cn_bonds[0][0]
                        cutoff_index = sum(1 for b in sorted_cn_bonds
                                           if b[0] <= 1.30 * d0)
                    
                    initial_cn = cutoff_index

                    # v1.7.4: Jahn-Teller axial recovery.
                    # For strong-JT octahedral cations the largest gap in the
                    # bond list is the equatorial->axial gap, so the naive cut
                    # above amputates the two long axial bonds and returns CN=4.
                    # If the site is a JT ion cut to 4 or 5, try to restore the
                    # axial pair when it sits within MAX_JT_AXIAL_RATIO of the
                    # equatorial mean (genuine 4+2), but leave true square-planar
                    # / square-pyramidal and broken low-CN sites untouched.
                    if (central_type in JT_OCTAHEDRAL_ELEMENTS
                            and initial_cn in (4, 5)
                            and len(sorted_cn_bonds) >= 6):
                        eq_mean = sum(b[0] for b in sorted_cn_bonds[:4]) / 4.0
                        restored_cn = initial_cn
                        for j in range(initial_cn, 6):
                            if sorted_cn_bonds[j][0] <= eq_mean * MAX_JT_AXIAL_RATIO:
                                restored_cn = j + 1
                            else:
                                break
                        initial_cn = restored_cn
                        cutoff_index = restored_cn

                    if cutoff_index > 0:
                        primary_sphere_cutoff_dist = sorted_cn_bonds[cutoff_index - 1][0]

                # STEP 4: Extract bonds within primary sphere
                initial_bonds_for_display = []
                for bond_tuple in sorted_cn_bonds: 
                    dist_val, dist_err, _lig_lbl = bond_tuple
                    if dist_val <= primary_sphere_cutoff_dist:
                        initial_bonds_for_display.append(
                            {'distance': dist_val, 'error': dist_err,
                             'ligand_label': _lig_lbl})
                
                # STEP 5: Count bond multiplicities
                initial_detailed_bond_counts = []
                if initial_bonds_for_display:
                    grouped_bonds_info = []
                    for bond_dict in initial_bonds_for_display:
                        rounded_dist = round(bond_dict['distance'], 5)
                        original_error = bond_dict['error']
                        grouped_bonds_info.append((rounded_dist, original_error,
                                                   bond_dict['distance'],
                                                   bond_dict.get('ligand_label')))

                    # Group by (distance, error, ligand label). Two ligand
                    # labels at the same distance are distinct species sharing
                    # a position and must not be merged, or their occupancies
                    # cannot be applied separately.
                    bond_counts_map = Counter((r_d, o_e, lbl)
                                              for r_d, o_e, _, lbl in grouped_bonds_info)
                    sorted_unique_keys = sorted(bond_counts_map.keys(),
                                                key=lambda k: (k[0], str(k[2])))

                    for (dist_val_rounded, dist_err_val, lig_lbl) in sorted_unique_keys:
                        count = bond_counts_map[(dist_val_rounded, dist_err_val, lig_lbl)]
                        original_dist_val = next((orig_d for r_d, e_v, orig_d, l in grouped_bonds_info 
                                                  if r_d == dist_val_rounded and e_v == dist_err_val
                                                  and l == lig_lbl), 
                                                 dist_val_rounded)

                        initial_detailed_bond_counts.append({
                            'count': count, 
                            'distance': original_dist_val, 
                            'error': dist_err_val,
                            'ligand_label': lig_lbl
                        })

                # ======================================================================
                # END CRITICAL SECTION
                # ======================================================================
                
                # AUTOMATED DECISION (v2.0.2): accept the gap CN only where
                # ECoN agrees the shell is complete. See guarded_shell_cn.
                _all_d = [b[0] for b in sorted_cn_bonds]
                confirmed_cn, _cn_ev = guarded_shell_cn(_all_d, initial_cn)

                if confirmed_cn != initial_cn:
                    initial_detailed_bond_counts = group_bond_counts(
                        sorted_cn_bonds, confirmed_cn)
                    print(f"    {central_type}-{ligand_type}: CN revised "
                          f"{initial_cn} -> {confirmed_cn} "
                          f"({_cn_ev['cn_revision_reason']})")

                confirmed_bond_counts = initial_detailed_bond_counts

                # v2.0.2: report the number of bonds actually summed.
                # The shell boundary is a DISTANCE, so when the cut lands
                # inside a set of symmetry-equivalent bonds, every one of them
                # is included. A pyrochlore A site is two short 8b contacts
                # plus six 48f contacts that are equal by symmetry; a cut at
                # seven sums eight bonds. The sum was always right, but `cn`
                # was not, and cn_fault and bvs_distance_ratio are both
                # computed from `cn`. Measured on the ICSD pyrochlores this
                # affected 71 sites, and 16 in the perovskites.
                _n_bonds = sum(b['count'] for b in confirmed_bond_counts)
                if _n_bonds and _n_bonds != confirmed_cn:
                    _cn_ev['cn_tie_extended'] = True
                    _cn_ev['cn_before_ties'] = int(confirmed_cn)
                    confirmed_cn = int(_n_bonds)

                # v2.0.2 DUAL SHELL. The gap analysis and the distance cutoff
                # disagree systematically for large, distorted, high-coordination
                # cations: a perovskite A site is typically cut to 8 or 9 while
                # the distance shell holds 12, and the four longest bonds carry
                # roughly 0.3 v.u., which is 10% of a trivalent sum. ECoN sides
                # with the shorter shell, because it is bond-length weighted and
                # answers "what is the effective coordination", not "which bonds
                # carry bond valence".
                #
                # Comparing deviations ACROSS structures cannot settle which is
                # right, because coordination number and tolerance factor both
                # track distortion, so the comparison is confounded. The sum is
                # therefore computed over BOTH shells for every site and both
                # are exported, making the comparison within-structure and
                # controlled. Neither is privileged: `bvs` remains the gap-shell
                # value, and `bvs_full_shell` sits beside it.
                _full_bonds = group_bond_counts(sorted_cn_bonds,
                                                full_shell_cn(_all_d))
                # counted the same way, for the same reason
                _full_cn = sum(b['count'] for b in _full_bonds) or None
                site_info.setdefault('_full_shell', {})[(central_type, ligand_type)] = (
                    _full_cn, _full_bonds)
                
                # DIAGNOSTIC OUTPUT: Show bond distances for validation
                if initial_detailed_bond_counts:
                    bond_strs = []
                    for b in initial_detailed_bond_counts:
                        bond_strs.append(f"{b['count']}×{b['distance']:.4f}Å")
                    
                    avg_dist = sum(b['distance']*b['count'] for b in initial_detailed_bond_counts) / sum(b['count'] for b in initial_detailed_bond_counts)
                    
                    # VALIDATION: Check against expected values for different bond types
                    warnings = []
                    
                    # Oxide validation
                    if ligand_type == 'O':
                        if central_type == 'Cr' and confirmed_cn == 6:
                            if avg_dist < 1.95 or avg_dist > 2.05:
                                warnings.append(f"⚠️ Cr-O distance {avg_dist:.3f}Å unusual (expect 1.98-2.01Å)")
                        elif central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.25 or avg_dist > 2.40:
                                warnings.append(f"⚠️ Na-O distance {avg_dist:.3f}Å unusual (expect 2.30-2.35Å)")
                        elif central_type == 'Ni' and confirmed_cn == 6:
                            if avg_dist < 2.00 or avg_dist > 2.15:
                                warnings.append(f"⚠️ Ni-O distance {avg_dist:.3f}Å unusual (expect 2.05-2.10Å)")
                    
                    # Halide validation
                    elif ligand_type == 'Cl':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.75 or avg_dist > 2.90:
                                warnings.append(f"⚠️ Na-Cl distance {avg_dist:.3f}Å unusual (expect 2.80-2.85Å)")
                    
                    elif ligand_type == 'F':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.25 or avg_dist > 2.45:
                                warnings.append(f"⚠️ Na-F distance {avg_dist:.3f}Å unusual (expect 2.30-2.35Å)")
                    
                    # Sulfide validation
                    elif ligand_type == 'S':
                        if central_type == 'Na' and confirmed_cn == 6:
                            if avg_dist < 2.80 or avg_dist > 3.00:
                                warnings.append(f"⚠️ Na-S distance {avg_dist:.3f}Å unusual (expect 2.85-2.95Å)")
                    
                    # Print diagnostic info
                    print(f"    {central_type}-{ligand_type}: CN={confirmed_cn}, bonds=[{', '.join(bond_strs)}], avg={avg_dist:.4f}Å")
                    for warning in warnings:
                        print(f"      {warning}")
                
                site_info['cn_and_bonds_by_type'][(central_type, ligand_type)] = {
                    'confirmed_cn': confirmed_cn,
                    'confirmed_bond_counts': confirmed_bond_counts,
                    'cn_evidence': _cn_ev,
                    # v2.0.3: the full candidate list, so a shell can be chosen
                    # downstream where R0 and b are known. The gap rule was used
                    # at all only because the shell had to be decided before the
                    # parameters were available; this fixes the ordering rather
                    # than passing parameters backwards into the decision.
                    'candidate_bonds': sorted_cn_bonds,
                    'full_shell_cn': site_info['_full_shell'][(central_type, ligand_type)][0],
                    'full_shell_bond_counts': site_info['_full_shell'][(central_type, ligand_type)][1]
                }
    
    return final_data

# ============================================================================
# AUTOMATED VALENCE ASSIGNMENT (Replaces get_valences_interactively)
# ============================================================================

def _cif_valence_is_usable(element, v):
    """
    Is a valence taken from a CIF worth believing?

    Two ways it is not. Zero means the depositor left the oxidation state
    unset - ICSD writes "Na0+ 0" - and a bond valence sum against zero is
    undefined. A negative value on a species that does not form anions means
    the signs have been entered the wrong way round, as in ICSD 651062 where
    SrS is deposited as "S2+ 2" and "Sr2- -2".

    Both are errors in the deposited entry rather than unusual chemistry, and
    in both cases the fallback chain gives the right answer.
    """
    if v is None:
        return False
    try:
        f = float(v)
    except (TypeError, ValueError):
        return False
    if f == 0.0:
        return False
    if f < 0.0 and element not in ANION_FORMERS:
        return False
    return True

def auto_assign_valences(processed_data, user_oxidation_states=None, per_atom_valences=None):
    """
    Assign oxidation states with priority:
    
    1. Per-atom explicit valence from CIF (e.g., "Fe3+" -> 3) [NEW in v1.4.2]
    2. User-provided oxidation states (config or interactive)
    3. Database common states
    4. Hardcoded fallback defaults
    
    Args:
        processed_data: Data structure from previous processing steps
        user_oxidation_states: Optional dict of {element: valence}
        per_atom_valences: Optional dict from extract_per_atom_valences() [NEW]
    """
    for block_name, block_content in processed_data.items():
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue
            
            relevant_elements = set()
            for (central_type, ligand_type) in site_info['cn_and_bonds_by_type'].keys():
                relevant_elements.add(central_type)
                relevant_elements.add(ligand_type)
            
            if not relevant_elements:
                continue

            site_valences = {}
            valence_sources = {}
            
            for element in relevant_elements:
                valence = None
                source = None
                
                # PRIORITY 1 [NEW]: Per-atom explicit valence from CIF
                if per_atom_valences:
                    site_label = site_info.get('site_label', '')
                    if site_label in per_atom_valences:
                        atom_data = per_atom_valences[site_label]
                        _ev = atom_data['explicit_valence']
                        # A stated valence of zero is not an assignment. Taking
                        # it literally means the bond valence sum is computed
                        # against zero, no parameters exist for M(0+), and the
                        # site returns nothing at all.
                        if (atom_data['element'] == element
                                and _cif_valence_is_usable(element, _ev)):
                            valence = _ev
                            source = 'cif_per_atom'
                
                # PRIORITY 2: User-provided oxidation states
                #
                # Zero is rejected here too. batch_process_directory reads the
                # CIF's own oxidation-state loop and passes it in through this
                # argument, so a file declaring "Na0+ 0" arrives as a user
                # override of zero and outranks the database fallback. That is
                # why guarding PRIORITY 1 alone fixed the single-structure path
                # and left the batch unchanged: a direct call passes no
                # override, the batch passes the zeros.
                if (valence is None and user_oxidation_states
                        and element in user_oxidation_states
                        and _cif_valence_is_usable(
                            element, user_oxidation_states[element])):
                    valence = user_oxidation_states[element]
                    source = 'user_override'
                
                # PRIORITY 3: Database common states
                if valence is None:
                    common_states = db_manager.get_common_oxidation_states(element)
                    if common_states:
                        valence = common_states[0]
                        source = 'database'
                    else:
                        # PRIORITY 4: Hardcoded fallback
                        default_valences = {
                            'Na': 1, 'K': 1, 'Li': 1,
                            'O': -2, 'F': -1, 'S': -2,
                            'Ni': 2, 'Mn': 3, 'Fe': 3, 'Cr': 3,
                            'Ti': 4, 'Al': 3, 'Mg': 2, 'Sb': 5
                        }
                        valence = default_valences.get(element, 0)
                        source = 'fallback'
                
                site_valences[element] = valence
                valence_sources[element] = source
            
            site_info['confirmed_valences'] = site_valences
            site_info['valence_sources'] = valence_sources
    
    return processed_data

# ============================================================================
# BVS CALCULATIONS (From Skuld 1.5 - EXACT PRESERVATION)
# ============================================================================

def perform_calculations_and_report(processed_data, b_param_default=0.37,
                                    site_occupancies=None):
    """
    Perform BVS, Shannon-Prewitt, and other crystallochemical calculations.
    PRESERVES EXACT calculation logic from Skuld 1.5 (lines 1307-1537).
    """
    calculated_results = {}
    site_occupancies = site_occupancies or {}

    _occ_clamped = {'any': False}

    def _ligand_occupancy(label):
        """
        Occupancy of a ligand site, defaulting to full.

        Clamped to 1.0. A site cannot hold more than one atom, so a refined
        occupancy above unity is either an unconstrained refinement sitting
        within error of 1 or a transcription error. Either way, letting it
        through would inflate the bond valence sum, and A8's weighting makes
        the calculation sensitive to it in a way it was not before.
        Occurrences are recorded so they are not silently absorbed.
        """
        if not label:
            return 1.0
        d = site_occupancies.get(label)
        if not d:
            return 1.0
        occ = d.get('occupancy', 1.0)
        if occ is None:
            return 1.0
        occ = float(occ)
        if occ > 1.0:
            if occ > 1.02:
                _occ_clamped['any'] = True
            return 1.0
        if occ < 0.0:
            # v1.8.6: a negative occupancy is always a data error. It was being
            # clamped to zero silently, so a site could quietly contribute
            # nothing to the sum with no indication why.
            _occ_clamped['any'] = True
            return 0.0
        return occ

    for block_name, block_content in processed_data.items():
        
        if not block_content['sites_with_bonds'] or not block_content.get('selected_bond_types_for_analysis'):
            continue
        
        calculated_results[block_name] = {}

        for coords, site_info in block_content['sites_with_bonds'].items():
            if 'cn_and_bonds_by_type' not in site_info:
                continue

            site_results = {
                'elements': site_info['elements'],
                'labels': site_info['labels'],
                'element_data': {}
            }

            for (central_type, ligand_type), bond_analysis_data in site_info['cn_and_bonds_by_type'].items():
                confirmed_cn = bond_analysis_data['confirmed_cn']
                confirmed_bond_counts = bond_analysis_data['confirmed_bond_counts']
                
                central_valence = site_info['confirmed_valences'].get(central_type)
                ligand_valence = site_info['confirmed_valences'].get(ligand_type)

                if central_valence is None or ligand_valence is None:
                    continue

                # v2.0.3 (4b): THE CANONICAL SHELL.
                #
                # Everything below this point - Shannon radii, ionic potential,
                # the bond valence sum, the distortion indices, ECoN - consumes
                # confirmed_cn and confirmed_bond_counts. Until now those came
                # from gap analysis, which assumes a compact polyhedron and so
                # amputates a tilted perovskite A site: 356 of 1895 sites in the
                # ICSD perovskite set carried a sum over a truncated shell, and
                # A-type cations showed a systematic -6.4% median deviation that
                # was arithmetic rather than chemistry.
                #
                # The shell is now set by bond-valence contribution, which needs
                # R0 and b. Those are available HERE, three lines after the
                # valences are resolved, and were not available at the point
                # where the gap cut was made; that ordering is the whole reason
                # the gap rule was used at all. See BV_SUM_CUTOFF in CELL 1.
                #
                # The gap shell is retained and reported, so the change is
                # visible per site rather than asserted.
                _gap_cn = confirmed_cn
                _gap_bonds = confirmed_bond_counts
                _cand = bond_analysis_data.get('candidate_bonds') or []
                _r0_c, _b_c = get_bv_parameters_fractional(
                    central_type, central_valence, ligand_type)

                _bv_cn, _bv_ev = bv_shell_cn([_x[0] for _x in _cand],
                                             _r0_c, _b_c)
                if _bv_cn and _cand:
                    _bv_bonds = group_bond_counts(_cand, _bv_cn)
                    # group_bond_counts cuts on the DISTANCE of the cn-th bond,
                    # so symmetry-equivalent ties arrive together and the count
                    # can exceed _bv_cn. That is correct; take the count from
                    # the grouped bonds.
                    confirmed_bond_counts = _bv_bonds
                    confirmed_cn = sum(_x['count'] for _x in _bv_bonds)
                    _cn_method = _bv_ev['cn_method']
                else:
                    # No parameters, or nothing inside the boundary. Fall back
                    # to the gap shell rather than lose the site, and SAY SO:
                    # a site whose shell was chosen by a different rule must be
                    # distinguishable in the output from one that was not.
                    _cn_method = ('gap_fallback_no_parameters' if _r0_c is None
                                  else 'gap_fallback_empty_shell')

                element_calc_data = {'valence': central_valence,
                                     'ligand_valence': ligand_valence,
                                     'cn': confirmed_cn,
                                     'cn_method': _cn_method,
                                     'cn_gap_shell': _gap_cn,
                                     'bv_shell_max_distance': _bv_ev['bv_shell_max_distance'],
                                     'bv_shell_last_s': _bv_ev['bv_shell_last_s'],
                                     'bv_shell_next_s': _bv_ev['bv_shell_next_s']}

                # --- Shannon-Prewitt Ionic Radii Comparison ---
                if confirmed_bond_counts:
                    central_radius, _cr_src = get_ionic_radius_fractional(
                        central_type, central_valence, confirmed_cn)
                    ligand_radius, _lr_src = get_ionic_radius_fractional(
                        ligand_type, ligand_valence, confirmed_cn)

                    if central_radius is not None and ligand_radius is not None:
                        expected_distance = central_radius + ligand_radius
                        
                        element_shannon_diffs = []
                        
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            observed_distance = bond_info['distance']
                            bond_error = bond_info.get('error', np.nan)
                            percent_diff = ((observed_distance - expected_distance) / expected_distance) * 100
                            element_shannon_diffs.append({
                                'bond_count': count, 
                                'observed_distance': observed_distance, 
                                'expected_distance': expected_distance, 
                                'percent_diff': percent_diff,
                                'error': bond_error 
                            })
                        
                        element_calc_data['shannon_diffs'] = element_shannon_diffs
                        # v2.0.13: carried forward so the geometry can be
                        # re-assessed with real per-bond valences. They are
                        # known here and nowhere downstream.
                        element_calc_data['bv_r0'] = _r0_c
                        element_calc_data['bv_b'] = _b_c
                        element_calc_data['cation_radius'] = central_radius
                        element_calc_data['anion_radius'] = ligand_radius
                        element_calc_data['shannon_source'] = (
                            _cr_src if _cr_src == _lr_src
                            else f"{_cr_src}/{_lr_src}")

                # --- Ionic Potential ---
                if element_calc_data.get('cation_radius') and element_calc_data['cation_radius'] > 0:
                    ionic_potential = float(abs(central_valence)) / element_calc_data['cation_radius']
                    element_calc_data['ionic_potential'] = ionic_potential

                # --- Bond Valence Sum (BVS) Calculation ---
                if confirmed_bond_counts:
                    r0_val, b_val = get_bv_parameters_fractional(central_type, central_valence, ligand_type)
                    
                    if r0_val is not None:
                        bvs_total = 0.0
                        bvs_unweighted = 0.0

                        # v1.8.3 (A8): weight each ligand's contribution by its
                        # site occupancy. A split anion site refined as two
                        # partly-occupied positions describes ONE oxygen; summing
                        # both at full weight double-counts it and roughly doubles
                        # the bond valence sum.
                        for bond_info in confirmed_bond_counts:
                            count = bond_info['count']
                            distance = bond_info['distance']
                            occ = _ligand_occupancy(bond_info.get('ligand_label'))

                            s = math.exp((r0_val - distance) / b_val)
                            bvs_total += (s * count * occ)
                            bvs_unweighted += (s * count)

                        element_calc_data['bvs_unweighted'] = bvs_unweighted

                        if central_valence != 0:
                            bvs_percent_diff = ((bvs_total - abs(central_valence)) / abs(central_valence)) * 100
                            
                            element_calc_data['bvs'] = bvs_total
                            element_calc_data['bvs_percent_diff'] = bvs_percent_diff

                        # v2.0.2: the same sum over the full distance shell.
                        # Identical arithmetic, identical occupancy weighting,
                        # identical parameters; only the set of bonds differs.
                        # That is what makes the comparison controlled.
                        _fs_cn = bond_analysis_data.get('full_shell_cn')
                        _fs_bonds = bond_analysis_data.get('full_shell_bond_counts')
                        if _fs_bonds:
                            _fs_total = 0.0
                            for bond_info in _fs_bonds:
                                _occ = _ligand_occupancy(bond_info.get('ligand_label'))
                                _s = math.exp((r0_val - bond_info['distance']) / b_val)
                                _fs_total += (_s * bond_info['count'] * _occ)
                            element_calc_data['cn_full_shell'] = _fs_cn
                            element_calc_data['bvs_full_shell'] = _fs_total
                            if central_valence != 0:
                                element_calc_data['bvs_full_shell_percent_diff'] = (
                                    (_fs_total - abs(central_valence))
                                    / abs(central_valence)) * 100
                            # Positive means the bonds the gap cut discarded
                            # were carrying this much valence.
                            element_calc_data['bvs_shell_delta'] = (
                                _fs_total - bvs_total)

                        # v2.0.3 (4b): the GAP shell, as the comparison.
                        # `bvs` above is now the bond-valence shell, so what is
                        # worth reporting beside it is what the old rule gave.
                        # Identical arithmetic, identical occupancy weighting,
                        # identical parameters; only the set of bonds differs,
                        # which is what makes the comparison controlled.
                        element_calc_data['cn_bv_shell'] = confirmed_cn
                        element_calc_data['bvs_bv_shell'] = bvs_total
                        if central_valence != 0:
                            element_calc_data['bvs_bv_shell_percent_diff'] = (
                                element_calc_data.get('bvs_percent_diff'))
                        if _gap_bonds:
                            _gap_total = 0.0
                            for bond_info in _gap_bonds:
                                _occ = _ligand_occupancy(bond_info.get('ligand_label'))
                                _s = math.exp((r0_val - bond_info['distance']) / b_val)
                                _gap_total += (_s * bond_info['count'] * _occ)
                            element_calc_data['cn_gap_shell'] = sum(
                                _x['count'] for _x in _gap_bonds)
                            element_calc_data['bvs_gap_shell'] = _gap_total
                            if central_valence != 0:
                                element_calc_data['bvs_gap_shell_percent_diff'] = (
                                    (_gap_total - abs(central_valence))
                                    / abs(central_valence)) * 100
                            # Positive means the bonds the gap cut discarded
                            # were carrying this much valence.
                            element_calc_data['bvs_gap_shell_delta'] = (
                                bvs_total - _gap_total)

                        # v2.0.7: the adaptive cut-off, REPORTED not canonical.
                        # Seeded from the gap shell, which is right in
                        # magnitude even where it is wrong in detail. See
                        # BV_CUT_A in CELL 1 for the fit and for why a single
                        # cut-off gave Li(1+) a coordination number of 16.
                        # v2.0.8: seeded from the CANONICAL shell, not the gap
                        # shell. The adaptive rule can only tighten, so a wrong
                        # seed is unrecoverable: seeded from the gap shell, Mn
                        # in ICSD 11616 stayed at CN 2 and returned a bond
                        # valence sum of 7.21, +140%, because the gap shell is
                        # depressed by the very fault being diagnosed. Using a
                        # quantity the fault suppresses to start the diagnosis
                        # is the same circularity as sizing the shift window by
                        # the deposited coordination number.
                        _ad_cn, _ad_ev = bv_shell_cn_adaptive(
                            [_b[0] for _b in _cand], r0_val, b_val,
                            central_valence, confirmed_cn,
                            element=central_type, anion=ligand_type)
                        element_calc_data['bv_boundary_adaptive'] = _ad_ev['bv_boundary_adaptive']
                        element_calc_data['bv_boundary_source'] = _ad_ev['bv_boundary_source']
                        element_calc_data['bv_cut_iterations'] = _ad_ev['bv_cut_iterations']
                        if _ad_cn and _cand:
                            _ad_bonds = group_bond_counts(_cand, _ad_cn)
                            _ad_total = 0.0
                            for bond_info in _ad_bonds:
                                _occ = _ligand_occupancy(bond_info.get('ligand_label'))
                                _s = math.exp((r0_val - bond_info['distance']) / b_val)
                                _ad_total += (_s * bond_info['count'] * _occ)
                            element_calc_data['cn_bv_adaptive'] = sum(
                                _b['count'] for _b in _ad_bonds)
                            element_calc_data['bvs_bv_adaptive'] = _ad_total
                            if central_valence != 0:
                                element_calc_data['bvs_bv_adaptive_percent_diff'] = (
                                    (_ad_total - abs(central_valence))
                                    / abs(central_valence)) * 100

                        # v2.0.3: the convention-free companion. Hoppe weights,
                        # no boundary, no threshold to argue about. NOT the GII
                        # and not comparable with any published sum: G&H fitted
                        # R0 and b against unweighted sums, so a weighted sum
                        # does not satisfy the equation those parameters solve.
                        # Report it beside the conventional figure, never
                        # instead of it.
                        _ew, _ewcn = econ_weighted_bvs(
                            [_b[0] for _b in _cand], r0_val, b_val)
                        if _ew is not None:
                            element_calc_data['bvs_econ_weighted'] = _ew
                            element_calc_data['econ_weighted_cn'] = _ewcn
                            if central_valence != 0:
                                element_calc_data['bvs_econ_weighted_percent_diff'] = (
                                    (_ew - abs(central_valence))
                                    / abs(central_valence)) * 100

                # --- Occupancy-weighted coordination number (C1) ---
                if confirmed_bond_counts:
                    element_calc_data['ligand_occupancy_clamped'] = bool(_occ_clamped['any'])
                    element_calc_data['cn_occupancy_weighted'] = round(sum(
                        b['count'] * _ligand_occupancy(b.get('ligand_label'))
                        for b in confirmed_bond_counts), 3)

                # --- Bond Length Distortion (BLD) ---
                if confirmed_bond_counts and confirmed_cn > 0:
                    total_distance_for_avg = sum(b['distance'] * b['count'] for b in confirmed_bond_counts)
                    total_count_for_avg = sum(b['count'] for b in confirmed_bond_counts)
                    
                    if total_count_for_avg > 0:
                        average_observed_distance = total_distance_for_avg / total_count_for_avg
                        
                        sum_of_squared_deviations = 0.0
                        for bond_info in confirmed_bond_counts:
                            distance = bond_info['distance']
                            count = bond_info['count']
                            sum_of_squared_deviations += count * ((distance - average_observed_distance) / average_observed_distance)**2
                        
                        bld = sum_of_squared_deviations / confirmed_cn
                        element_calc_data['bond_length_distortion'] = bld

                # v1.8.3 (A3): accumulate across ligand types instead of
                # overwriting. A cation bonded to two anion species (an
                # oxyfluoride, say) had its first bond valence sum replaced by
                # the second, so only part of its coordination was counted.
                _prev = site_results['element_data'].get(central_type)
                if _prev is None:
                    element_calc_data['ligand_types'] = [ligand_type]
                    # v1.8.4: the geometry fields below (shannon_diff,
                    # avg_bond_length, anion_radius, bond_length_distortion)
                    # are per ligand type. When a cation bonds to two anion
                    # species only the first survives the merge, so record
                    # which one it describes rather than leaving it implicit.
                    element_calc_data['fields_describe_ligand'] = ligand_type
                    site_results['element_data'][central_type] = element_calc_data
                else:
                    for _k in ('bvs', 'bvs_unweighted', 'cn',
                               'cn_occupancy_weighted', 'bvs_full_shell',
                               'cn_full_shell', 'bvs_shell_delta',
                               'bvs_bv_shell', 'cn_bv_shell',
                               'bvs_gap_shell', 'cn_gap_shell',
                               'bvs_gap_shell_delta', 'bvs_econ_weighted',
                               'bvs_bv_adaptive', 'cn_bv_adaptive'):
                        _a, _b = _prev.get(_k), element_calc_data.get(_k)
                        if _a is not None and _b is not None:
                            _prev[_k] = _a + _b
                        elif _b is not None:
                            _prev[_k] = _b
                    for _k, _v in element_calc_data.items():
                        if _k not in _prev or _prev[_k] is None:
                            _prev[_k] = _v
                    _prev.setdefault('ligand_types', [])
                    if ligand_type not in _prev['ligand_types']:
                        _prev['ligand_types'].append(ligand_type)
                    _prev.setdefault('fields_describe_ligand', ligand_type)
                    _val = _prev.get('valence')
                    if _val:
                        _prev['bvs_percent_diff'] = (
                            (_prev['bvs'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs') is not None else None)
                        _prev['bvs_full_shell_percent_diff'] = (
                            (_prev['bvs_full_shell'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs_full_shell') is not None else None)
                        _prev['bvs_bv_shell_percent_diff'] = (
                            (_prev['bvs_bv_shell'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs_bv_shell') is not None else None)
                        _prev['bvs_gap_shell_percent_diff'] = (
                            (_prev['bvs_gap_shell'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs_gap_shell') is not None else None)
                        _prev['bvs_bv_adaptive_percent_diff'] = (
                            (_prev['bvs_bv_adaptive'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs_bv_adaptive') is not None else None)
                        _prev['bvs_econ_weighted_percent_diff'] = (
                            (_prev['bvs_econ_weighted'] - abs(_val)) / abs(_val) * 100
                            if _prev.get('bvs_econ_weighted') is not None else None)
            
            calculated_results[block_name][coords] = site_results
            
    return calculated_results

def calculate_global_instability_index(calculated_results, cif_path=None, site_occupancies=None, per_atom_valences=None):
    """
    Calculate Global Instability Index (GII) and Weighted-Average GII (WAGII).
 
    v1.7.5 (July 2026): corrected to the published definitions.
 
        GII   = sqrt( (1/N) * SUM (S_i - V_i)^2 )
        WAGII = sqrt( SUM w_i * (S_i - V_i)^2 / SUM w_i )
 
    after Salinas-Sanchez et al. (1992) J. Solid State Chem. 100, 201 and
    Reeves-McLaren (2026). Versions up to and including 1.7.4 returned
    SUM|S_i - V_i| / SUM V_i, which is a valence-normalised mean absolute
    deviation, not the GII. That quantity is retained as 'bvs_discrepancy' for
    comparison with previously generated results.
 
    WAGII weights each contribution by site occupancy, providing a more physically
    meaningful metric for disordered structures where minority species at partially
    occupied sites can disproportionately influence conventional GII.
 
    For fully occupied sites (occ=1.0), WAGII = GII.
    """
    if site_occupancies is None and cif_path is not None:
        site_occupancies = extract_site_occupancies(cif_path)
 
    if site_occupancies is None:
        site_occupancies = {}
 
    gii_results = {}
 
    for block_name, block_results in calculated_results.items():
        # --- legacy accumulators (pre-1.7.5 quantity, kept for comparison) ---
        total_bvs_deviation = 0.0
        total_formal_valence = 0.0
        total_weighted_bvs_deviation = 0.0
        total_weighted_formal_valence = 0.0
        # --- v1.7.5: accumulators for the published RMS definitions ---
        total_sq_deviation = 0.0
        total_weighted_sq_deviation = 0.0
        total_occupancy = 0.0
        n_sites_counted = 0
        n_sites_zero_occupancy = 0
        n_sites_below_floor = 0
        sq_below_floor = 0.0
        site_contributions = []
 
        for site_coords, site_data in block_results.items():
            if 'element_data' not in site_data:
                continue
 
            site_labels = site_data.get('labels', [])
 
            # v1.5 FIX: Iterate by LABEL to handle mixed-valence sites correctly
            # (e.g., Fe2/Fe3 with Fe2+/Fe3+ need separate GII contributions)
            for label in site_labels:
                # Resolve element for this label
                element = None
                # v1.8.3 (A4): longest symbol first. site_data['elements'] is
                # sorted alphabetically, so a one-letter symbol is otherwise
                # tested before a two-letter one beginning with it - C before
                # Cr, N before Na, B before Ba - and "Cr1" resolves to carbon.
                for elem in sorted(site_data['elements'], key=len, reverse=True):
                    if label.upper().startswith(elem.upper()):
                        element = elem
                        break
                if element is None and site_data['elements']:
                    element = site_data['elements'][0]
                if element is None:
                    continue
 
                element_data = site_data['element_data'].get(element, {})
 
                if 'bvs' in element_data and element_data['bvs'] is not None:
                    observed_bvs = element_data['bvs']
 
                    # Use per-atom valence if available (correct for mixed-valence)
                    formal_valence = abs(element_data.get('valence', 0))
                    if per_atom_valences and label in per_atom_valences:
                        pav = per_atom_valences[label]
                        if pav.get('explicit_valence') is not None:
                            formal_valence = abs(pav['explicit_valence'])
 
                    if formal_valence > 0:
                        bvs_deviation = abs(observed_bvs - formal_valence)
 
                        # Find occupancy for this label
                        occupancy = 1.0
                        if label in site_occupancies:
                            occ_data = site_occupancies[label]
                            occupancy = occ_data.get('occupancy', 1.0)

                        # v2.0.2: a site refined to zero occupancy holds no
                        # atom, so its bond valence sum describes something
                        # that is not in the structure. GII counts sites rather
                        # than weighting them, so such a site was contributing
                        # its full squared deviation: ICSD 138129 declares Yb2
                        # at 16c with occupancy 0 to record an anti-site model
                        # in which no ytterbium sits on the titanium site, and
                        # that phantom returned 6.87 v.u. against a formal 3.
                        #
                        # Excluding them takes ICSD 149744 from 1.2168 to
                        # 0.1983, which is its WAGII to four decimal places,
                        # and 149753 from 1.2253 to 0.2027 against 0.2026. The
                        # unweighted and weighted indices agree once an absent
                        # atom stops being counted; that agreement is the
                        # evidence, not a chosen threshold.
                        try:
                            if occupancy is not None and float(occupancy) == 0.0:
                                n_sites_zero_occupancy += 1
                                continue
                        except (TypeError, ValueError):
                            pass

                        # v2.0.15 (4.8): AND NOR MAY A SITE WHOSE SHORTEST
                        # CONTACT IS NOT A BOND.
                        #
                        # CATION_ANION_ABSOLUTE_MIN was tested per site and
                        # recorded in contact_above_floor, but only AFTER the
                        # shell had been built and the sum computed, and the
                        # sum still reached GII. ICSD 220506 is the case: La1
                        # has a 1.008 A contact, from which a shell of eleven
                        # was built and 27.457 v.u. returned against a formal
                        # 3, and the structure reported GII = 17.294 - the
                        # number that put it at the top of the triage. Cr1 in
                        # the same structure is at 3.051 v.u. against 3, which
                        # is a perfectly ordinary octahedron.
                        #
                        # So the index was a statement about one impossible
                        # contact wearing the clothes of a structure-wide
                        # measure. The measurement stays in min_bond_length
                        # and in the G1 evidence; what goes is its
                        # contribution to a number that is meant to describe
                        # the structure.
                        #
                        # Same reasoning and same shape as the zero-occupancy
                        # exclusion above: a sum computed over something that
                        # is not there, or not a bond, is not evidence about
                        # bonding.
                        #
                        # MEASURED: 1 site in 1 structure of the 938
                        # perovskites, already EXPUNGE on G1. It moves no
                        # structure that anyone would quote.
                        _sd = element_data.get('shannon_diffs') or []
                        _dmin = None
                        for _s in _sd:
                            _od = _s.get('observed_distance')
                            if _od is not None and (_dmin is None or _od < _dmin):
                                _dmin = _od
                        if _dmin is not None and _dmin < CATION_ANION_ABSOLUTE_MIN:
                            n_sites_below_floor += 1
                            # Kept, so the index Eir used to report is still
                            # recoverable from the CSV and this change can be
                            # audited rather than believed. ICSD 220506 goes
                            # from 17.294 to 0.051 - which is the point, and
                            # also the danger: 0.051 reads as an excellent
                            # structure. It is not. It is one ordinary CrO6
                            # octahedron in a file whose La is 1.008 A from an
                            # oxygen, and it is EXPUNGE at G1. Quote neither
                            # number for it. See gii_all_sites.
                            sq_below_floor += bvs_deviation ** 2
                            continue

                        # v1.7.5: squared deviations, counted per site
                        total_sq_deviation += bvs_deviation ** 2
                        n_sites_counted += 1
                        total_weighted_sq_deviation += occupancy * (bvs_deviation ** 2)
                        total_occupancy += occupancy
 
                        # legacy quantity, retained for comparison only
                        total_bvs_deviation += bvs_deviation
                        total_formal_valence += formal_valence
                        total_weighted_bvs_deviation += occupancy * bvs_deviation
                        total_weighted_formal_valence += occupancy * formal_valence
 
                        site_contributions.append({
                            'site': site_coords,
                            'element': element,
                            'label': label,
                            'formal_valence': formal_valence,
                            'observed_bvs': observed_bvs,
                            'bvs_deviation': bvs_deviation,
                            'bvs_percent_diff': element_data.get('bvs_percent_diff', 0),
                            'occupancy': occupancy,
                            'element_category': db_manager.get_element_category(element)
                        })
 
        if n_sites_counted > 0:
            # v1.7.5: published definitions
            gii = math.sqrt(total_sq_deviation / n_sites_counted)
            wagii = math.sqrt(total_weighted_sq_deviation / total_occupancy) if total_occupancy > 0 else gii
            # pre-1.7.5 quantity, for comparison with older output
            bvs_discrepancy = total_bvs_deviation / total_formal_valence if total_formal_valence > 0 else None
 
            if gii < 0.05:
                stability_assessment = "Very Stable"
            elif gii < 0.1:
                stability_assessment = "Stable"
            elif gii < 0.2:
                stability_assessment = "Moderately Unstable"
            elif gii < 0.3:
                stability_assessment = "Unstable"
            else:
                stability_assessment = "Highly Unstable"
 
            if n_sites_zero_occupancy:
                print(f"  {n_sites_zero_occupancy} site(s) refined to zero "
                      f"occupancy excluded from the instability index: there is "
                      f"no atom there to be over- or under-bonded.")
            if n_sites_below_floor:
                print(f"  {n_sites_below_floor} site(s) excluded from the "
                      f"instability index: shortest contact below "
                      f"{CATION_ANION_ABSOLUTE_MIN} A, which is not a bond. "
                      f"The contact is still reported and still condemns the "
                      f"structure at G1.")
            gii_results[block_name] = {
                'gii': gii,
                'wagii': wagii,
                'n_sites_zero_occupancy': n_sites_zero_occupancy,
                'n_sites_below_floor': n_sites_below_floor,
                # v2.0.16: RENAMED. v2.0.15 called this gii_all_sites, which
                # is the name CELL 7 has used since well before this session
                # for a different quantity entirely - the index taken over
                # cations AND anions together. Two definitions of one name,
                # which is the fault this project keeps catching, introduced
                # here by the fix for 4.8. CELL 7 wrote last and won, so no
                # exported number was ever wrong; the new one simply never
                # appeared. The name now says what it is.
                'gii_with_floor_failures': (
                    math.sqrt((total_sq_deviation + sq_below_floor)
                              / (n_sites_counted + n_sites_below_floor))
                    if n_sites_below_floor else gii),
                'bvs_discrepancy': bvs_discrepancy,
                'stability_assessment': stability_assessment,
                'site_contributions': site_contributions,
                'total_sites_analyzed': len(site_contributions)
            }
        else:
            gii_results[block_name] = {
                'gii': None,
                'wagii': None,
                'n_sites_below_floor': n_sites_below_floor,
                'bvs_discrepancy': None,
                'stability_assessment': 'Cannot Assess',
                'site_contributions': [],
                'total_sites_analyzed': 0
            }
 
    return gii_results

# ============================================================================
# BATCH PROCESSING ENGINE
# ============================================================================

def process_single_cif(file_path, user_oxidation_states=None):
    """
    Process a single CIF file using validated Eir 1.0 logic.
    NEW in v1.2 continued: Mixed occupancy detection, Bosi (2014) strain analysis,
    and automatic 3D visualisation of problematic sites.
    NEW in v1.4: Structure-type specific analysis (spinel, perovskite, etc.)
    
    Args:
        file_path: Path to CIF file
        user_oxidation_states: Optional dict of {element: valence} overrides
                              If provided, these oxidation states will be used instead of database defaults
    
    Returns:
        dict with processing results including success status, metadata, sites data, and GII
    """
    results = {
        'success': False,
        'error': None,
        'metadata': {},
        'sites': [],
        'gii': None,
        'filename': os.path.basename(file_path)
    }
    
    # Parse CIF
    _LAST_PARSE_ERROR['reason'] = None
    cif_obj = parse_cif_file(file_path)
    if cif_obj is None:
        results['error'] = ("Failed to parse CIF file: "
                            + (_LAST_PARSE_ERROR['reason'] or 'no reason recorded'))
        return results
    
    # Process first data block
    block_name = list(cif_obj.keys())[0]
    data_block = cif_obj[block_name]
    
    # Extract metadata
    results['metadata'] = {
        'block_name': block_name,
        'filename': os.path.basename(file_path),
        'chemical_formula': data_block.get("_chemical_formula_sum", "N/A"),
        'space_group': data_block.get("_space_group_name_H-M_alt", 
                                     data_block.get("_symmetry_space_group_name_H-M", "N/A")),
    }
    
    # Extract ICSD publication metadata
    icsd_metadata = extract_icsd_metadata(file_path)
    results['metadata'].update({
        'icsd_code': icsd_metadata.get('icsd_code'),
        'year': icsd_metadata.get('year'),
        'journal': icsd_metadata.get('journal'),
        'authors': ', '.join(icsd_metadata.get('authors', [])) if icsd_metadata.get('authors') else None,
        'r_factor': icsd_metadata.get('r_factor'),
        'density': icsd_metadata.get('density'),
        'measurement_temperature_K': icsd_metadata.get('measurement_temperature_K'),
        'measurement_pressure_Pa': icsd_metadata.get('measurement_pressure_Pa'),
        'measurement_pressure_GPa': icsd_metadata.get('measurement_pressure_GPa'),
        'measurement_conditions': icsd_metadata.get('measurement_conditions')
    })
    
    # Validated Eir 1.0 workflow
    try:
        # Step 1: Extract site data
        initial_extracted_data = extract_initial_site_data(cif_obj)
        
        if not initial_extracted_data:
            results['error'] = "No data extracted from CIF"
            return results
        
        # Step 1.5 [v1.8.4]: element-level valences from the CIF, so that
        # classify_bond_pair can use its most reliable rule. Previously the
        # explicit-valence branch existed but was never given anything to work
        # with, and every pair fell through to the electronegativity heuristic.
        try:
            _cif_ox, _, _, _ = extract_cif_oxidation_states(file_path)
        except Exception:
            _cif_ox = None
        _pair_valences = dict(_cif_ox or {})
        if user_oxidation_states:
            _pair_valences.update(user_oxidation_states)

        # Step 2: Auto-select bond types
        data_with_selected_bonds = auto_select_bond_types(
            initial_extracted_data, valences=_pair_valences)
        
        if not data_with_selected_bonds[block_name].get('selected_bond_types_for_analysis'):
            results['error'] = "No M-X bonds found"  # Updated to M-X (not just M-O)
            return results
        
        # Step 3: Auto-analyse coordination
        final_corrected_data = auto_analyze_coordination(data_with_selected_bonds)
        
        # Step 3.5 [NEW v1.4.2]: Extract per-atom valences for mixed-valence support
        per_atom_valences = extract_per_atom_valences(file_path)
        
        # Step 3.6 [NEW v1.7]: Extract site occupancies for integrity tier classification
        site_occupancies_data = extract_site_occupancies(file_path)
        
        # Step 4: Auto-assign valences (now with per-atom support)
        final_data_with_valences = auto_assign_valences(
            final_corrected_data, 
            user_oxidation_states,
            per_atom_valences
        )
        
        # Step 5: Perform calculations
        calculated_results = perform_calculations_and_report(
        final_data_with_valences, site_occupancies=site_occupancies_data)
        
        # Step 6: Calculate GII
        gii_results = calculate_global_instability_index(calculated_results, cif_path=file_path, per_atom_valences=per_atom_valences)
        
        # ============================================================================
        # Extract results for export WITH MIXED OCCUPANCY ANALYSIS
        # ============================================================================
        if block_name in calculated_results:
            for coords, site_data in calculated_results[block_name].items():
                labels = site_data.get('labels', ['unknown'])
                elements_at_site = site_data.get('elements', [])
                
                # ================================================================
                # v1.4.2 FIX: Iterate by LABEL, not by element
                # This ensures each site_label (e.g., Fe2, Fe3) gets its own row
                # with the correct element and explicit valence from the CIF
                # ================================================================
                for site_label in labels:
                    
                    # ----------------------------------------------------------
                    # Determine element and valence for this specific label
                    # ----------------------------------------------------------
                    if per_atom_valences and site_label in per_atom_valences:
                        # Best case: found in per_atom_valences with explicit data
                        atom_info = per_atom_valences[site_label]
                        element = atom_info['element']
                        explicit_valence = atom_info['explicit_valence']
                        valence_source = 'cif_per_atom' if explicit_valence is not None else 'cif_inferred'
                    else:
                        # Fallback: try to match label to an element in the site's element list
                        element = None
                        
                        # First try: label starts with element symbol (Fe1 -> Fe)
                        # v1.8.3 (A4): longest symbol first, so Cr is tested
                        # before C and Na before N.
                        for elem in sorted(elements_at_site, key=len, reverse=True):
                            if site_label.upper().startswith(elem.upper()):
                                element = elem
                                break
                        
                        # Second try: use first element at site
                        if element is None and elements_at_site:
                            element = elements_at_site[0]
                            print(f"    ⚠ Warning: Could not match label '{site_label}' to element, using {element}")
                        
                        # Last resort: extract from label string
                        if element is None:
                            element = re.sub(r'[0-9]+$', '', site_label)
                            element = element.capitalize() if len(element) > 1 else element.upper()
                        
                        explicit_valence = None
                        valence_source = 'fallback'
                    
                    # Skip if we couldn't determine element
                    if not element:
                        print(f"    ⚠ Skipping label '{site_label}' - could not determine element")
                        continue
                    
                    # ----------------------------------------------------------
                    # Get element_data (BVS, CN, etc.) for this element
                    # ----------------------------------------------------------
                    elem_data = site_data['element_data'].get(element, {})
                    
                    if not elem_data:
                        # Element wasn't analysed (e.g., no bonds found, or anion)
                        # Skip silently for anions, warn for cations
                        if element not in ['O', 'S', 'Se', 'Te', 'F', 'Cl', 'Br', 'I', 'N']:
                            print(f"    ⚠ No element_data for {element} at {site_label}")
                        continue
                    
                    # ----------------------------------------------------------
                    # Override valence with explicit per-atom valence if available
                    # ----------------------------------------------------------
                    elem_data = elem_data.copy()  # Don't modify original
                    
                    # A stated valence of zero is not an oxidation state
                    # assignment. ICSD entries routinely carry "Fe0+", "Na0+"
                    # or "Ba0+" where the depositor left it unset, and
                    # overwriting the computed valence with zero leaves the
                    # site with no usable deviation: the guard below skips the
                    # percentage, and |d1| is measured against nothing. On the
                    # 613-structure rock salt set this silently removed 73
                    # sites in 40 ordinary structures - NaCl, CaS, BaS - from
                    # the validation statistics. Keep the valence that was
                    # used for the calculation instead.
                    if _cif_valence_is_usable(element, explicit_valence):
                        elem_data['valence'] = explicit_valence
                        elem_data['valence_source'] = valence_source
                    elif explicit_valence is not None:
                        elem_data['valence_source'] = 'cif_valence_ignored'
                        
                        # Recalculate BVS percent diff with correct valence
                        bvs_val = elem_data.get('bvs')
                        if bvs_val is not None and explicit_valence != 0:
                            elem_data['bvs_percent_diff'] = (bvs_val - abs(explicit_valence)) / abs(explicit_valence) * 100
                    else:
                        elem_data['valence_source'] = valence_source
                    
                    # ----------------------------------------------------------
                    # Get bond information for 3D visualisation
                    # ----------------------------------------------------------
                    bond_counts = None
                    full_site_info = None
                    cn_evidence = {}
                    
                    if block_name in final_corrected_data:
                        for site_coords, site_info in final_corrected_data[block_name]['sites_with_bonds'].items():
                            if site_coords == coords:
                                if 'cn_and_bonds_by_type' in site_info:
                                    for (central_type, ligand_type), bond_analysis in site_info['cn_and_bonds_by_type'].items():
                                        if central_type == element:
                                            bond_counts = bond_analysis.get('confirmed_bond_counts', [])
                                            # v2.0.2: how the shell was decided
                                            cn_evidence = bond_analysis.get('cn_evidence') or {}
                                            break
                                full_site_info = site_info
                                break
                    
                    # ========================================================================
                    # Mixed occupancy analysis (from the helper-functions cell)
                    # ========================================================================
                    mixed_occ_data = analyze_mixed_occupancy_at_site(
                        coords, 
                        site_data,
                        final_corrected_data, 
                        block_name,
                        data_block
                    )
                    
                    # ========================================================================
                    # Calculate deviation and d1 (discrepancy factor from Bosi 2014)
                    # ========================================================================
                    bvs_val = elem_data.get('bvs')
                    valence = elem_data.get('valence')
                    
                    if bvs_val is not None and valence is not None:
                        deviation = bvs_val - abs(valence)
                    else:
                        deviation = None
                    
                    d1, abs_d1, bonding_state = calculate_discrepancy_factor(bvs_val, valence)
                    
                    # ========================================================================
                    # Assign confidence flag based on disorder type and strain
                    # ========================================================================
                    confidence, bosi_likely, needs_review = assign_confidence_flag(
                        mixed_occ_data['disorder_type'],
                        mixed_occ_data['valence_difference'],
                        abs_d1,
                        deviation
                    )
                    
                    # ========================================================================
                    # DATA INTEGRITY TIER ASSIGNMENT (NEW IN v1.7)
                    # ========================================================================
                    # Check minority species (I3) and mixed TM occupancy (FLAG)
                    site_occupancy = per_atom_valences[site_label]['occupancy'] if (per_atom_valences and site_label in per_atom_valences) else 1.0
                    is_minority = check_minority_species(site_label, per_atom_valences or {}, site_occupancies_data or {})
                    is_mixed_tm, has_noninteger_tm = check_mixed_tm_site(site_label, per_atom_valences or {}, site_occupancies_data or {})
                    
                    # Get the CIF-assigned valence (before any override) for tier metadata
                    cif_valence = None
                    formal_valence_from_symbol = None
                    symbol_number_agreement = None
                    
                    if per_atom_valences and site_label in per_atom_valences:
                        pav_info = per_atom_valences[site_label]
                        cif_valence = pav_info.get('explicit_valence')
                        # Parse integer from type_symbol for comparison
                        if pav_info.get('type_symbol'):
                            _, symbol_val = parse_atom_type_with_valence(pav_info['type_symbol'])
                            if symbol_val is not None:
                                formal_valence_from_symbol = int(symbol_val) if float(symbol_val) == int(symbol_val) else symbol_val
                            # Check agreement between symbol and oxidation_number
                            if cif_valence is not None and symbol_val is not None:
                                symbol_number_agreement = abs(cif_valence - symbol_val) < 0.05
                    
                    # v2.0.11: no tier is assigned here. This cell computes
                    # EVIDENCE; the PGRM gates classify it in post-processing,
                    # where geometry decides severity before any chemistry is
                    # read. See the retirement note in CELL 1.
                    integrity_tier, flag_reason, flag_severity = None, None, None

                    # "What if": which integer oxidation state does the observed
                    # geometry support? Asked of any site whose assignment the
                    # depositor could not verify from diffraction - a shared
                    # transition-metal position, a non-integer partition, or a
                    # minority species - rather than of sites a bond valence
                    # threshold happened to flag.
                    whatif_result = {
                        'best_integer_valence': None,
                        'best_integer_deviation': None,
                        'candidates_tested': None,
                        'depositor_deviation': None,
                    }
                    if bvs_val is not None and (is_mixed_tm or has_noninteger_tm
                                                or is_minority):
                        whatif_result = whatif_analysis(element, bvs_val, valence)
                    
                    # ========================================================================
                    # Build site result with ALL columns
                    # ========================================================================
                    site_result = {
                        # Original columns
                        'coords': coords,
                        'element': element,
                        'site_label': site_label,
                        'valence': elem_data.get('valence'),
                        'valence_source': elem_data.get('valence_source', 'unknown'),  # NEW in v1.4.2
                        'cn': elem_data.get('cn'),
                        'bvs': round(elem_data.get('bvs'), 4) if elem_data.get('bvs') is not None else None,
                        'bvs_percent_diff': round(elem_data.get('bvs_percent_diff'), 2) if elem_data.get('bvs_percent_diff') is not None else None,
                        'deviation': round(deviation, 4) if deviation is not None else None,
                        
                        # Mixed occupancy columns
                        'has_mixed_occupancy': mixed_occ_data['has_mixed_occupancy'],
                        'occupancy_string': mixed_occ_data['occupancy_string'],
                        'n_elements_at_site': mixed_occ_data['n_elements_at_site'],
                        'disorder_type': mixed_occ_data['disorder_type'],
                        'valence_difference': round(mixed_occ_data['valence_difference'], 1) if mixed_occ_data['valence_difference'] else 0.0,
                        'max_valence_at_site': mixed_occ_data['max_valence_at_site'],
                        'min_valence_at_site': mixed_occ_data['min_valence_at_site'],
                        
                        # Bosi (2014) strain indicators
                        'discrepancy_d1': round(d1, 4) if d1 is not None else None,
                        'abs_discrepancy_d1': round(abs_d1, 4) if abs_d1 is not None else None,
                        'bonding_state': bonding_state,
                        
                        # Confidence & flagging
                        'confidence_flag': confidence,
                        'bosi_artifact_likely': bosi_likely,
                        'requires_manual_review': needs_review,
                        
                        # Data integrity tier (NEW in v1.7)
                        'integrity_tier': integrity_tier,
                        'flag_reason': flag_reason,
                        'flag_severity': flag_severity,
                        'occupancy': round(site_occupancy, 4),
                        'is_minority_species': is_minority,
                        'is_mixed_occupancy_tm': is_mixed_tm,
                        'site_delta_v': round(mixed_occ_data['valence_difference'], 1) if mixed_occ_data['valence_difference'] else 0.0,
                        'cif_valence': cif_valence,
                        # v2.0.2: how far the stated valence sits from an
                        # integer, so a tier decided near the boundary is
                        # visible rather than silently on one side of it.
                        'valence_integer_distance': (
                            round(valence_integer_distance(cif_valence), 4)
                            if valence_integer_distance(cif_valence) is not None
                            else None),
                        'formal_valence_from_symbol': formal_valence_from_symbol,
                        'symbol_number_agreement': symbol_number_agreement,
                        # "What if" analysis (FLAG-tier sites only)
                        'best_integer_valence': whatif_result['best_integer_valence'],
                        'best_integer_deviation': whatif_result['best_integer_deviation'],
                        'candidates_tested': whatif_result['candidates_tested'],
                        'depositor_deviation': whatif_result['depositor_deviation'],
                        
                        # Original columns
                        'ionic_potential': round(elem_data.get('ionic_potential'), 3) if elem_data.get('ionic_potential') is not None else None,
                        'cation_radius': round(elem_data.get('cation_radius'), 3) if elem_data.get('cation_radius') is not None else None,
                        'bld': round(elem_data.get('bond_length_distortion'), 6) if elem_data.get('bond_length_distortion') is not None else None,
                        'bvs_unweighted': round(elem_data.get('bvs_unweighted'), 4) if elem_data.get('bvs_unweighted') is not None else None,
                        'cn_occupancy_weighted': elem_data.get('cn_occupancy_weighted'),
                        'ligand_occupancy_clamped': elem_data.get('ligand_occupancy_clamped'),
                        'ligand_types': ','.join(elem_data.get('ligand_types', [])) or None,
                        'fields_describe_ligand': elem_data.get('fields_describe_ligand'),
                        'bvs_distance_ratio': None,
                        'bvs_geometry_plausible': None,
                        # Coordination shell provenance (NEW in v2.0.2). A CN
                        # revision is never silent: these say what the gap
                        # analysis returned, what ECoN said about it, and
                        # whether the shell was extended.
                        'cn_gap': cn_evidence.get('cn_gap'),
                        'cn_revised': cn_evidence.get('cn_revised'),
                        'cn_revision_reason': cn_evidence.get('cn_revision_reason'),
                        'econ_shell': cn_evidence.get('econ_shell'),
                        'cn_shell_max': cn_evidence.get('cn_shell_max'),
                        'cn_tie_extended': cn_evidence.get('cn_tie_extended', False),
                        'cn_before_ties': cn_evidence.get('cn_before_ties'),
                        # Dual-shell diagnostic (NEW in v2.0.2). The same sum
                        # over the full distance shell, so that the shell
                        # convention can be tested within a structure rather
                        # than by comparing structures that differ in both
                        # coordination number and chemistry.
                        'cn_full_shell': elem_data.get('cn_full_shell'),
                        'bvs_full_shell': (round(elem_data['bvs_full_shell'], 4)
                                           if elem_data.get('bvs_full_shell') is not None else None),
                        'bvs_full_shell_percent_diff': (
                            round(elem_data['bvs_full_shell_percent_diff'], 2)
                            if elem_data.get('bvs_full_shell_percent_diff') is not None else None),
                        'bvs_shell_delta': (round(elem_data['bvs_shell_delta'], 4)
                                            if elem_data.get('bvs_shell_delta') is not None else None),
                        # v2.0.3: the bond-valence shell, and the ECoN-weighted
                        # sum that needs no shell at all.
                        'cn_method': elem_data.get('cn_method'),
                        'cn_bv_shell': elem_data.get('cn_bv_shell'),
                        'bv_shell_max_distance': elem_data.get('bv_shell_max_distance'),
                        'bv_shell_last_s': elem_data.get('bv_shell_last_s'),
                        'bv_shell_next_s': elem_data.get('bv_shell_next_s'),
                        'bvs_bv_shell': (round(elem_data['bvs_bv_shell'], 4)
                                         if elem_data.get('bvs_bv_shell') is not None else None),
                        'bvs_bv_shell_percent_diff': (
                            round(elem_data['bvs_bv_shell_percent_diff'], 2)
                            if elem_data.get('bvs_bv_shell_percent_diff') is not None else None),
                        'cn_gap_shell': elem_data.get('cn_gap_shell'),
                        'bvs_gap_shell': (round(elem_data['bvs_gap_shell'], 4)
                                          if elem_data.get('bvs_gap_shell') is not None else None),
                        'bvs_gap_shell_percent_diff': (
                            round(elem_data['bvs_gap_shell_percent_diff'], 2)
                            if elem_data.get('bvs_gap_shell_percent_diff') is not None else None),
                        'bvs_gap_shell_delta': (round(elem_data['bvs_gap_shell_delta'], 4)
                                                if elem_data.get('bvs_gap_shell_delta') is not None else None),
                        # v2.0.7: the adaptive cut-off, reported not canonical.
                        'cn_bv_adaptive': elem_data.get('cn_bv_adaptive'),
                        'bv_boundary_adaptive': elem_data.get('bv_boundary_adaptive'),
                        'bv_boundary_source': elem_data.get('bv_boundary_source'),
                        'bv_cut_iterations': elem_data.get('bv_cut_iterations'),
                        'bvs_bv_adaptive': (round(elem_data['bvs_bv_adaptive'], 4)
                                            if elem_data.get('bvs_bv_adaptive') is not None else None),
                        'bvs_bv_adaptive_percent_diff': (
                            round(elem_data['bvs_bv_adaptive_percent_diff'], 2)
                            if elem_data.get('bvs_bv_adaptive_percent_diff') is not None else None),
                        'bvs_econ_weighted': (round(elem_data['bvs_econ_weighted'], 4)
                                              if elem_data.get('bvs_econ_weighted') is not None else None),
                        'bvs_econ_weighted_percent_diff': (
                            round(elem_data['bvs_econ_weighted_percent_diff'], 2)
                            if elem_data.get('bvs_econ_weighted_percent_diff') is not None else None),
                        'econ_weighted_cn': (round(elem_data['econ_weighted_cn'], 3)
                                             if elem_data.get('econ_weighted_cn') is not None else None),
                        'site_bvs_averaged': None,
                        'site_expected_averaged': None,
                        'site_deviation_averaged': None,
                    }
                    
                    # Add Shannon-Prewitt data if available (unchanged)
                    if 'shannon_diffs' in elem_data and elem_data['shannon_diffs']:
                        avg_shannon_diff = np.mean([s['percent_diff'] for s in elem_data['shannon_diffs']])
                        site_result['shannon_diff'] = round(avg_shannon_diff, 2)
                        site_result['avg_bond_length'] = round(np.mean([s['observed_distance'] for s in elem_data['shannon_diffs']]), 4)
                        site_result['min_bond_length'] = round(min(s['observed_distance'] for s in elem_data['shannon_diffs']), 4)
                        site_result['shannon_source'] = elem_data.get('shannon_source')

                    # ====================================================
                    # v2.0.5: OBSERVED AGAINST MEASURED, IN ANGSTROMS
                    #
                    # Reported, never flagged. The obvious screen here - 'is
                    # any bond outside the range ever observed for this ion' -
                    # was built and rejected, and the reason is worth keeping.
                    # An extreme-value statistic needs a large sample, and
                    # Gagne's high-CN configurations are precisely the small
                    # ones: La(3+) reaches n >= 30 only at CN 8 and 9, while
                    # the perovskite A site sits at 10 to 12. On the
                    # 938-structure ICSD perovskite set that screen condemned
                    # 224 sites in structures passing every other test, some
                    # with GII below 0.02. The reference tail is missing where
                    # it is most needed and no threshold repairs that.
                    #
                    # What survives is the report. A difference in angstroms
                    # beside a stated sample size is something a reader can
                    # weigh; a ratio against a fitted threshold is not. On
                    # that dataset 96.9% of sites resolve to a measured
                    # expectation at tier 1 or 2.
                    #
                    # bl_ratio_min is the one genuinely parameter-free
                    # quantity: this site's shortest contact against the
                    # shortest ever observed for this exact ion
                    # configuration. Like against like, so it carries no
                    # coordination dependence - measured at 1.014 to 1.049
                    # across CN 6 to 12, against 0.976 falling to 0.881 for
                    # the Shannon ratio the radii screen uses. It is computed
                    # only where the sample supports the tail
                    # (BL_MIN_SAMPLE), which is 66% of B sites and 12% of A
                    # sites here. Null means the tail is not characterised;
                    # that is a fact about the reference data, not a failure.
                    # ====================================================
                    _bl_lig = (elem_data.get('ligand_type')
                               or (str(site_result.get('ligand_types') or 'O')
                                   .split(',')[0]) or 'O')
                    _bl = expected_bond_length(element, valence,
                                               site_result.get('cn'), _bl_lig)
                    if _bl:
                        site_result['bl_expected_mean'] = _bl['mean']
                        site_result['bl_expected_sd'] = _bl['sd']
                        site_result['bl_expected_min'] = _bl['min']
                        site_result['bl_n_bonds'] = _bl['n_bonds']
                        site_result['bl_source'] = _bl['source']
                        site_result['bl_tier'] = _bl['tier']
                        _obs_mean = site_result.get('avg_bond_length')
                        _obs_min = site_result.get('min_bond_length')
                        if _obs_mean is not None:
                            site_result['bl_delta_mean'] = round(
                                _obs_mean - _bl['mean'], 4)
                        if _obs_min is not None and _bl['min']:
                            site_result['bl_ratio_min'] = round(
                                _obs_min / _bl['min'], 4)
                            site_result['bl_delta_min'] = round(
                                _obs_min - _bl['min'], 4)


                    # ========================================================================
                    # 3D visualisation for problematic sites
                    # ========================================================================
                    if (deviation is not None and abs(deviation) > 0.30
                            and PY3DMOL_AVAILABLE and bond_counts
                            and may_visualise()):
                        print(f"\n    ⚠️  Significant BVS deviation detected for {element} site!")
                        print(f"        BVS: {bvs_val:.3f}  │  Expected: {valence:+g}  │  Δ: {deviation:+.3f} v.u.")
                        
                        # NEW: Print diagnostic information
                        if d1 is not None:
                            print(f"        d1: {d1:+.3f} v.u. ({bonding_state})")
                        print(f"        Disorder: {mixed_occ_data['disorder_type']}", end="")
                        if mixed_occ_data['valence_difference'] > 0:
                            print(f" (ΔV={mixed_occ_data['valence_difference']:.0f})")
                        else:
                            print()
                        print(f"        Occupancy: {mixed_occ_data['occupancy_string']}")
                        print(f"        Confidence: {confidence}")
                        if bosi_likely:
                            print(f"        ⚠️  Bosi artifact likely - spectroscopy required!")
                        
                        print(f"    [Generating 3D visualisation...]")
                        
                        try:
                            visualize_problematic_site_3d(
                                cif_path=file_path,
                                element=element,
                                site_label=site_label,
                                bonds_info=bond_counts,
                                deviation=deviation,
                                formal_charge=valence,
                                bvs_value=bvs_val,
                                filename=os.path.basename(file_path)
                            )
                        except Exception as viz_error:
                            print(f"    [3D visualisation failed: {viz_error}]")
                    
                    elif (deviation is not None and abs(deviation) > 0.30
                          and not PY3DMOL_AVAILABLE):
                        print(f"\n    ⚠️  Significant deviation ({deviation:+.3f} v.u.)")
                        if d1 is not None:
                            print(f"        d1: {d1:+.3f} v.u. ({bonding_state})")
                        print(f"        Disorder: {mixed_occ_data['disorder_type']}")
                        print(f"        Confidence: {confidence}")
                        print(f"        Install py3Dmol for 3D visualisation")
                    
                    # v2.0.13: re-assess the geometry with the CANONICAL
                    # coordination number and the real per-bond valences.
                    #
                    # assess_site_geometry runs during bond calculation,
                    # before any valence is assigned, so its first pass uses
                    # a provisional shell (1.35 x d_ref, capped at 3.5 A).
                    # That shell disagrees with the canonical one on 706 of
                    # 1895 perovskite sites, 37%, which meant geo_bond_min,
                    # geo_bond_max, the distortion indices and cshm were all
                    # computed over a DIFFERENT polyhedron from the bond
                    # valence sum beside them - including the two columns
                    # bl_ratio_min is compared against.
                    _geo = dict(_EIR_GEOMETRY.get(site_label, {}))
                    _gc = _geo.pop('_geo_centre', None)
                    _gn = _geo.pop('_geo_neighbours', None)
                    _ge = _geo.pop('_geo_element', None)
                    _gcn = site_result.get('cn')
                    # Per-bond valences s_i = exp((R0 - d)/b), computed here
                    # because R0 and b are not carried past the element loop.
                    # Without them bv_vector_sum would stay None for ever,
                    # which is how the column came to hold the geometric
                    # eccentricity under a bond-valence name in the first
                    # place.
                    _r0, _bb = elem_data.get('bv_r0'), elem_data.get('bv_b')
                    _geo_valences = None
                    if _gn and _r0 is not None and _bb:
                        _geo_valences = [math.exp((_r0 - _d) / _bb)
                                         for _d, _ in sorted(_gn,
                                                             key=lambda t: t[0])]
                    if _gc and _gn and _gcn:
                        try:
                            _geo = assess_site_geometry(
                                _gc, _gn, _ge, cn=_gcn,
                                valences=_geo_valences)
                        except (TypeError, ValueError, IndexError) as _e:
                            _geo['geo_reassess_error'] = (
                                f'{type(_e).__name__}: {_e}')
                    site_result.update(_geo)
                    results['sites'].append(site_result)
        
        # Add GII (unchanged)
            if block_name in gii_results:
                results['gii'] = gii_results[block_name].get('gii')
                results['wagii'] = gii_results[block_name].get('wagii')
                results['n_sites_zero_occupancy'] = gii_results[block_name].get(
                    'n_sites_zero_occupancy')
                # v2.0.15: exported, because a GII computed over fewer sites
                # than the structure has is a different quantity and the
                # reader must be able to see that it happened.
                results['n_sites_below_floor'] = gii_results[block_name].get(
                    'n_sites_below_floor')
                # The index as it was computed before v2.0.15, so the change
                # can be checked against the previous run rather than taken
                # on trust. Equal to gii wherever nothing was excluded.
                results['gii_with_floor_failures'] = gii_results[block_name].get(
                    'gii_with_floor_failures')
                results['bvs_discrepancy'] = gii_results[block_name].get('bvs_discrepancy')
                results['stability_assessment'] = gii_results[block_name].get('stability_assessment')

        # =====================================================================
        # SITE-AVERAGED DEVIATION ON SHARED SITES
        # =====================================================================
        # A site shared by two species is reported once per species, with each
        # species' bond valence sum computed at the SAME geometry using its own
        # R0. So a 1.6% Ti anti-site impurity in NaTiO2 is asked what its sum
        # would be if it owned a sodium-sized site, returns 1.55 against an
        # expected 3.0, and that -1.45 deviation is then weighted EQUALLY with
        # the 98.4% majority. ICSD 85657 comes out at GII 1.02 for a compound
        # that is essentially ordered.
        #
        # Below, each crystallographic site gets one deviation, from the
        # occupancy-weighted combination of the per-species sums that Eir has
        # already computed. That gives 0.17 for the same structure.
        #
        # Note it combines the REPORTED sums rather than recomputing from the
        # mean bond length. Recomputing looks equivalent and is not: a
        # Jahn-Teller site with 4 short and 2 long bonds has a bond valence sum
        # of 2.94, while six bonds evaluated at the mean distance give 2.15,
        # because the exponential is convex. Doing it that way turned three
        # ordered manganates from GII 0.13 into 1.03.
        #
        # Reported alongside gii, not replacing it. The residual gap between
        # ordered and shared sites is not yet explained, so both should stay
        # visible.
        try:
            _by_site = {}
            for _s in results['sites']:
                _by_site.setdefault(str(_s.get('coords')), []).append(_s)

            # `x or 1.0` treats a refined occupancy of 0.0 as full, because
            # 0.0 is falsy. CIFs declare zero-occupancy placeholder sites to
            # describe an anti-site model: ICSD 138129 has Yb2 at 16c with
            # occupancy 0, whose bond valence sum is 6.87 v.u. against a formal
            # 3 because ytterbium does not fit a titanium octahedron. Weighted
            # as if fully present, it took that site's expected valence from
            # 4.0 to 3.4996 and the site-averaged GII from about 0.46 to 1.57,
            # which is how gii_site_averaged came to exceed wagii on a
            # structure it should have agreed with.
            #
            # None still means "not stated", and that does default to full.
            def _occ_of(g):
                v = g.get('occupancy')
                if v is None:
                    return 1.0
                try:
                    v = float(v)
                except (TypeError, ValueError):
                    return 1.0
                return 1.0 if v != v else v

            _site_devs = []
            for _coords, _grp in _by_site.items():
                _w = sum(_occ_of(g) for g in _grp)
                if not _w:
                    continue
                if any(g.get('bvs') is None or g.get('valence') is None for g in _grp):
                    continue
                _bvs_site = sum(_occ_of(g) * g['bvs'] for g in _grp) / _w
                _exp_site = sum(_occ_of(g) * abs(g['valence']) for g in _grp) / _w
                _dev = _bvs_site - _exp_site
                _site_devs.append(_dev)
                for g in _grp:
                    g['site_bvs_averaged'] = round(_bvs_site, 4)
                    g['site_expected_averaged'] = round(_exp_site, 4)
                    g['site_deviation_averaged'] = round(_dev, 4)

            if _site_devs:
                results['gii_site_averaged'] = round(
                    math.sqrt(sum(x * x for x in _site_devs) / len(_site_devs)), 6)
                results['n_crystallographic_sites'] = len(_site_devs)
            else:
                results['gii_site_averaged'] = None
                results['n_crystallographic_sites'] = None
        except Exception as _e:
            results['gii_site_averaged'] = None
            results['n_crystallographic_sites'] = None
            print(f"  [site-averaged GII unavailable: {_e}]")

        # CIF charge-sign errors found while selecting bond pairs
        if _SIGN_INVERSIONS:
            results['cif_sign_inversion'] = True
            results['cif_sign_inversion_detail'] = '; '.join(
                f"{e}{v:+g} in a {'/'.join(p)} pair" for p, (e, v) in _SIGN_INVERSIONS.items())
            print(f"\n  CIF CHARGE SIGNS INVERTED: "
                  f"{results['cif_sign_inversion_detail']}.")
            print(f"  The negative charge is assigned to the less electronegative "
                  f"species, which cannot be right. The CIF's oxidation states have "
                  f"been ignored for these pairs and the assignment made from "
                  f"electronegativity instead. This is an error in the deposited "
                  f"entry, not in the structure.")
        else:
            results['cif_sign_inversion'] = False
            results['cif_sign_inversion_detail'] = None
        
        # anion-anion screen: the only test here that looks at the anion
        # sublattice rather than outward from a cation
        _aa, _cc = anion_anion_contacts(file_path)
        # v2.0.14: `if _aa` discards a measured 0.0 as though it were
        # absent. dedup_tol keeps 0.0 from arising today, so this is a
        # correctness fix rather than a behaviour change - but 'no value'
        # and 'zero' must not be the same test in a function whose None
        # switches a gate off.
        results['anion_anion_min'] = None if _aa is None else round(_aa, 4)
        results['cation_cation_min'] = None if _cc is None else round(_cc, 4)

        # v2.0.12: the anion's free positional parameter, where the structure
        # type defines one. Computed since v1.8.3 and never wired: it was
        # reachable only through extract_all_structural_parameters, which
        # nothing called. Reported, not screened.
        try:
            _ap = extract_anion_parameter(file_path)
        except (OSError, UnicodeDecodeError, ValueError, TypeError,
                IndexError, KeyError) as _e:
            _note_read_failure(file_path, 'anion_parameter', _e)
            _ap = {}
        for _k in ('u_parameter', 'u_deviation', 'u_eff', 'delta_u',
                   'origin_choice', 'anion_wyckoff', 'anion_site_swapped',
                   'pyrochlore_48f_x', 'pyrochlore_fluorite_limit',
                   'u_parameter_applicable', 'u_note'):
            results[_k] = _ap.get(_k)
        results['anion_packing_plausible'] = (
            None if _aa is None else bool(ge(_aa, ANION_ANION_MIN)))
        if results['anion_packing_plausible'] is False:
            print(f"\n  ANION CONTACT TOO SHORT: two anions {_aa:.3f} A apart. "
                  f"Below {ANION_ANION_MIN} A their electron clouds overlap by more "
                  f"than any observed oxide.")
            print(f"  Neither cation-based screen can see this. Check the "
                  f"coordinates and the symmetry expansion.")

        # v2.0.12: the cation sublattice floor. cation_cation_min has been
        # computed and exported since v2.0.2 and nothing ever read it. On the
        # three reference families it catches 59 structures, 19 of which no
        # other screen sees, including two that sat in the triage's
        # UNEXPLAINED bucket: ICSD 133663 (NdNiO3, Nd-Ni 1.912 A) and 13793
        # (SrSnO3, Sn-Sr 2.019 A). See CATION_CATION_MIN for the measurement.
        # v2.0.15: CATION_CATION_MIN does not apply to hydrogen; see the
        # constant. anion_anion_contacts excludes H from the cation bucket,
        # so _cc never carries an H contact and this test is unchanged for
        # every structure in the three reference families, none of which
        # contains hydrogen.
        results['cation_packing_plausible'] = (
            None if _cc is None else bool(ge(_cc, CATION_CATION_MIN)))
        if results['cation_packing_plausible'] is False:
            print(f"\n  CATION CONTACT TOO SHORT: two cations {_cc:.3f} A "
                  f"apart, below the {CATION_CATION_MIN} A floor.")
            print(f"  Cations repel; edge- and face-sharing polyhedra bring "
                  f"them to 2.6-3.0 A and no closer. A contact below this is "
                  f"a coordinate or symmetry fault, not chemistry.")
            print(f"  Split sites are already exempt: a single atom modelled "
                  f"as two partly-occupied positions is not two cations.")
        
        # v1.8.6: geometry screen, in the core path.
        # This test previously lived only in add_coordination_quality, so a
        # single-structure analysis had no sanity check at all: a CIF with a
        # 0.5 A cation-anion contact returned success=True and a GII in the
        # hundreds, with nothing to say the geometry was impossible. Ions have
        # a finite size, so a contact much below the sum of their radii cannot
        # be real, and a bond valence built on it is meaningless.
        # v2.0.2: which species dominates each crystallographic position.
        #
        # The geometry screens ask whether a bond length can be real. A
        # minority species has no geometry of its own: it sits in the cavity
        # the majority defines, so the question becomes whether the WRONG ion
        # fits, and the answer is no -- which is why it is a minority. ICSD
        # 119363 (Lu2Ti2O7) is condemned by Lu2 at 0.8% occupancy on the
        # titanium site, ratio 0.869 against a 0.900 threshold, while every
        # species actually present passes: Lu1 0.929, Ti2 0.984, Ti1 1.055.
        #
        # Across the ICSD pyrochlore set, 32 of the 95 bond-valence distance
        # failures and 8 of the 51 radii failures are structures in which every
        # failing site is below 5% occupancy.
        #
        # The per-site verdicts are unchanged and still exported; only the
        # STRUCTURE-level roll-up is restricted to the dominant species. Where
        # no species reaches half a position, the most abundant one is used, so
        # a three-way split still contributes.
        _pos_max = {}
        for _s in results['sites']:
            _k = str(_s.get('coords'))
            try:
                _o = float(_s.get('occupancy') if _s.get('occupancy') is not None else 1.0)
            except (TypeError, ValueError):
                _o = 1.0
            if _k not in _pos_max or _o > _pos_max[_k]:
                _pos_max[_k] = _o
        for _s in results['sites']:
            try:
                _o = float(_s.get('occupancy') if _s.get('occupancy') is not None else 1.0)
            except (TypeError, ValueError):
                _o = 1.0
            _s['is_dominant_species'] = bool(
                _o >= _pos_max.get(str(_s.get('coords')), _o) - 1e-9)

        _worst_ratio, _worst = None, None
        _worst_bratio, _worst_b = None, None
        # v2.0.2: track the worst MARGIN against each site's own threshold, not
        # the worst raw ratio. Once the threshold varies with coordination the
        # smallest ratio is no longer the worst offender: a twelve-coordinate
        # site at 0.85 is comfortably inside its 0.792 threshold while a
        # six-coordinate site at 0.88 is outside its 0.900.
        _worst_bmargin = None
        _bvs_any_fail = False
        # v2.0.3: the radii screen now has a per-site threshold, so the worst
        # site is the one with the smallest MARGIN, not the smallest ratio.
        _worst_cmargin = None
        _core_any_fail = False
        _cn_implausible = []
        for _s in results['sites']:
            # Coordination plausibility. Independent of every test below,
            # which all ask whether a contact is too short; this asks whether
            # bonds are missing.
            _cnp, _cnlo, _cnhi = coordination_plausible(
                _s.get('element'), _s.get('valence'), _s.get('cn'))
            _s['cn_plausible'] = _cnp
            _s['cn_shannon_min'] = _cnlo
            _s['cn_shannon_max'] = _cnhi
            if _cnp is False and _s.get('is_dominant_species', True):
                _cn_implausible.append(_s)
            _d = _s.get('min_bond_length') or _s.get('avg_bond_length')

            # v2.0.3 (rule 1). The absolute floor, tested before anything that
            # divides by a radius or a bond valence parameter. A contact below
            # CATION_ANION_ABSOLUTE_MIN is not a short bond, it is not a bond,
            # and no sum built on it means anything. The all_cations variant of
            # ICSD 11616 produced a Dy-O contact of 1.181 A, from which a shell
            # of one was built and the site reported as overbonded at 8.258 v.u.
            _s['contact_above_floor'] = (
                None if _d is None
                else bool(_d >= CATION_ANION_ABSOLUTE_MIN))
            if _s['contact_above_floor'] is False:
                _s['cn_plausible'] = False
                _s['bonding_state'] = 'not_a_coordination_environment'
                if (_s.get('is_dominant_species', True)
                        and not any(_x is _s for _x in _cn_implausible)):
                    _cn_implausible.append(_s)

            _lig = (_s.get('fields_describe_ligand')
                    or (_s.get('ligand_types') or 'O').split(',')[0] or 'O')
            _ratio, _expected, _rsrc = core_radii_ratio(
                _s.get('element'), _lig, _d,
                valence=_s.get('valence'), cn=_s.get('cn'))
            _s['core_expected_contact'] = (round(_expected, 4)
                                           if _expected else None)
            _s['core_radii_source'] = _rsrc
            if _ratio is None:
                # UNTESTED, not passed. Recorded so it cannot be mistaken for
                # a clean result downstream.
                _s['core_shortest_contact_ratio'] = None
                _s['core_geometry_plausible'] = None

            # Parameter-based ratio. Runs whether or not a radius was found,
            # because it needs none; that is the point of it.
            #
            # `_d` is tested here explicitly. Until v2.0.2 this block sat below
            # a `continue` that fired when no ionic radius was available, and
            # that continue was doing two jobs: gating this screen on the radii
            # table (the fault it was removed to fix) and, incidentally,
            # guaranteeing `_d` was not None before the division below.
            # Removing it un-gated the screen as intended and dropped the null
            # check with it, which cost five structures with an unhandled
            # TypeError. A guard should never be load-bearing by accident.
            _v, _cn = _s.get('valence'), _s.get('cn')
            if _v and _cn and _d:
                _r0b = get_bv_parameters_fractional(_s.get('element'), _v, 'O')
                if _r0b and _r0b[0] is not None:
                    _R0, _b = _r0b
                    _d_ideal = _R0 - _b * math.log(abs(_v) / _cn)
                    if _d_ideal > 0:
                        _bratio = _d / _d_ideal
                        _s['bvs_distance_ratio'] = round(_bratio, 4)
                        _s['bvs_distance_threshold'] = round(
                            bvs_distance_threshold(_cn), 4)
                        _s['bvs_geometry_plausible'] = bool(
                            ge(_bratio, _s['bvs_distance_threshold']))
                        if not _s['bvs_geometry_plausible'] and                                 _s['is_dominant_species']:
                            _bvs_any_fail = True
                        _bmargin = _bratio - _s['bvs_distance_threshold']
                        if _s['is_dominant_species'] and (
                                _worst_bmargin is None or _bmargin < _worst_bmargin):
                            _worst_bmargin = _bmargin
                            _worst_bratio = _bratio
                            _worst_b = _s
            # v1.8.8: named core_* so they cannot collide with the columns
            # add_coordination_quality writes. Both compute the same quantity
            # by different routes, and the post-processor runs second, so
            # sharing a name meant one silently overwrote the other with no
            # way to tell which you were reading. They currently agree to
            # 0.0001; keeping both visible means a future divergence is
            # informative rather than invisible.
            if _ratio is not None:
                _s['core_shortest_contact_ratio'] = round(_ratio, 4)
                _s['core_contact_threshold'] = round(
                    radii_contact_threshold(_s.get('cn')), 4)
                _s['core_geometry_plausible'] = bool(
                    ge(_ratio, _s['core_contact_threshold']))
                if not _s['core_geometry_plausible'] and _s['is_dominant_species']:
                    _core_any_fail = True
                # structure verdict from the dominant species only, and by
                # MARGIN: with a per-site threshold the lowest ratio is no
                # longer necessarily the site that failed.
                _cmargin = _ratio - _s['core_contact_threshold']
                if _s['is_dominant_species'] and (
                        _worst_cmargin is None or _cmargin < _worst_cmargin):
                    _worst_cmargin, _worst_ratio, _worst = _cmargin, _ratio, _s
        # How many sites could not be tested at all. A structure whose every
        # site is untested is not a structure that passed.
        results['core_sites_untested'] = sum(
            1 for _s in results['sites']
            if _s.get('core_radii_source') == 'untested')
        results['core_sites_tested'] = sum(
            1 for _s in results['sites']
            if _s.get('core_radii_source') not in (None, 'untested'))
        # One implausible coordination condemns the structure, on the same
        # rule as the other screens.
        results['n_sites_cn_implausible'] = len(_cn_implausible)
        _judged = [_s for _s in results['sites']
                   if _s.get('cn_plausible') is not None]
        results['cn_plausible'] = (None if not _judged
                                   else not bool(_cn_implausible))
        if _cn_implausible:
            _w = min(_cn_implausible, key=lambda x: x.get('cn') or 99)
            if _w.get('cn_shannon_min') is None:
                print(f"\n  NO COORDINATION SHELL: {_w.get('element')} at "
                      f"{_w.get('site_label')} has no neighbours within the "
                      f"search radius. No bond valence sum is possible and "
                      f"none should be inferred.")
            else:
                print(f"\n  IMPLAUSIBLE COORDINATION: {_w.get('element')} at "
                      f"{_w.get('site_label')} has CN {_w.get('cn')}, outside the "
                      f"{_w.get('cn_shannon_min')}-{_w.get('cn_shannon_max')} range "
                      f"Shannon tabulates for it.")
                print(f"  Its bonds are of ordinary length, so no short-contact "
                      f"screen can see this; what is wrong is that bonds are "
                      f"MISSING. Treat the bond valence sum as incomplete.")
        results['core_shortest_contact_ratio'] = (round(_worst_ratio, 4)
                                                  if _worst_ratio is not None else None)
        # v2.0.3: one failing site condemns the structure, on the same rule as
        # every other screen. Comparing the worst ratio against the fixed
        # constant is the fault fixed for the bond-valence screen below; with a
        # per-site threshold it becomes wrong rather than merely redundant.
        results['core_geometry_plausible'] = (None if _worst_ratio is None
                                              else not _core_any_fail)
        results['core_contact_threshold'] = (
            _worst.get('core_contact_threshold') if _worst else None)
        results['bvs_distance_ratio'] = (round(_worst_bratio, 4)
                                         if _worst_bratio is not None else None)
        # One failing site condemns the structure, on the same rule as every
        # other screen. This previously compared the single worst RATIO against
        # the fixed constant, so the per-site thresholds were computed, exported
        # and then ignored by the only column anything downstream reads.
        results['bvs_geometry_plausible'] = (
            None if _worst_bratio is None else not _bvs_any_fail)
        results['bvs_distance_threshold'] = (
            _worst_b.get('bvs_distance_threshold') if _worst_b else None)

        if (results['bvs_geometry_plausible'] is False
                and results['core_geometry_plausible'] is not False):
            print(f"\n  SHORT CONTACT (bond valence test): {_worst_b.get('element')} at "
                  f"{_worst_b.get('site_label')} sits at {_worst_bratio:.0%} of the "
                  f"distance its own bond valence parameters imply.")
            print(f"  The radii-based screen did not catch this. Check the "
                  f"coordinates before trusting the result.")
        if results['core_geometry_plausible'] is False:
            print(f"\n  IMPLAUSIBLE GEOMETRY: {_worst.get('element')} at "
                  f"{_worst.get('site_label')} has a shortest contact of "
                  f"{_worst.get('min_bond_length') or _worst.get('avg_bond_length'):.3f} A, "
                  f"{_worst_ratio:.0%} of the "
                  f"{_worst.get('core_expected_contact') or float('nan'):.2f} A "
                  f"expected from ionic radii "
                  f"({_worst.get('core_radii_source')}).")
            print(f"  Treat every bond valence result for this structure as "
                  f"unreliable: it is a problem with the deposited coordinates, "
                  f"not with the chemistry.")

        # Half-lattice shift consistency. Runs after the geometry screens
        # because it uses core_expected_contact to reject a proposal that
        # would move a cation into an impossible contact.
        results['n_sites_shift_proposed'] = half_lattice_screen(
            file_path, results['sites'])

        # A1.1: anion-centred BVS, for the parameter-offset diagnosis
        # v2.0.2: anti-site exchange. Computed inside check_minority_species
        # to decline to expunge it, and then discarded; it is the fact that
        # decides which instability index to quote for this structure.
        results['n_antisite_species'], results['antisite_detail'] = antisite_exchange(
            per_atom_valences or {}, site_occupancies_data or {})
        results['has_antisite_disorder'] = bool(results['n_antisite_species'])
        if results['has_antisite_disorder']:
            print(f"\n  ANTI-SITE EXCHANGE: {results['antisite_detail']}.")
            print(f"  A minority species is being asked what its bond valence sum "
                  f"would be in a site the majority sizes, and the per-species GII "
                  f"weights that answer equally with the majority it sits among.")
            print(f"  Quote wagii or gii_site_averaged for this structure, not gii.")

        results['anion_sites'], results['anion_summary'] = anion_bvs_for_structure(
            file_path, results['sites'], cation_gii=results.get('gii'))
        results['success'] = True
        
    except Exception as e:
        results['error'] = f"Processing error: {str(e)}"
        import traceback
        print(f"  Detailed error: {traceback.format_exc()}")
    
    return results

# ============================================================================
# INTERACTIVE OXIDATION STATE ASSIGNMENT
# ============================================================================

# collect_unique_elements_from_cifs removed at v2.0.13: superseded by
# prompt_for_oxidation_states, which collects the elements it needs
# from the parsed structures rather than by re-reading the CIFs.
def prompt_for_oxidation_states(elements):
    """
    Interactive prompt for oxidation state assignment.
    Shows smart defaults based on oxide chemistry, allows user override.
    
    Args:
        elements: set of element symbols
    
    Returns:
        dict mapping element → oxidation state {e.g., 'Ti': 4, 'Na': 1}
    """
    # Smart defaults for oxide battery materials AND halides/sulfides
    OXIDE_DEFAULTS = {
        # Alkali metals
        'Li': 1, 'Na': 1, 'K': 1, 'Rb': 1, 'Cs': 1,
        # Alkaline earth
        'Mg': 2, 'Ca': 2, 'Sr': 2, 'Ba': 2,
        # Transition metals (oxide battery cathodes)
        'Ti': 4,   # Ti⁴⁺ most common in oxides
        'V': 5,    # V⁵⁺ in V₂O₅
        'Cr': 3,   # Cr³⁺ in chromites
        'Mn': 4,   # Mn⁴⁺ in MnO₂, battery cathodes (can also be 2, 3)
        'Fe': 3,   # Fe³⁺ slightly more common (can also be 2)
        'Co': 3,   # Co³⁺ in LiCoO₂
        'Ni': 2,   # Ni²⁺ in most cathodes
        'Cu': 2,   # Cu²⁺ most common
        'Zn': 2,   # Zn²⁺ only
        'Cd': 2,   # Cd²⁺ only
        # Other common
        'Al': 3, 'Sb': 5, 'Sn': 4, 'Zr': 4, 'Nb': 5, 'Mo': 6, 'W': 6,
        # Anions (oxides, halides, chalcogenides)
        'O': -2, 'F': -1, 'Cl': -1, 'Br': -1, 'I': -1,
        'S': -2, 'Se': -2, 'Te': -2,
        'N': -3, 'P': -3, 'As': -3, 'H': -1,
    }
    
    print("\n" + "="*80)
    print("OXIDATION STATE ASSIGNMENT")
    print("="*80)
    print("Please confirm oxidation states for elements found in your CIF files.")
    print("Defaults are shown in [brackets] - press ENTER to accept, or type new value.")
    print("="*80 + "\n")
    
    oxidation_states = {}
    
    # Sort: anions last, then alphabetical
    anions = sorted([e for e in elements if e in ['O', 'F', 'S', 'Cl', 'Br', 'I']])
    cations = sorted([e for e in elements if e not in anions])
    sorted_elements = cations + anions
    
    for element in sorted_elements:
        # Get default suggestion
        default = OXIDE_DEFAULTS.get(element)
        
        if default is not None:
            prompt_text = f"  {element:>2s} oxidation state [{default:+d}]: "
        else:
            # Check database
            common_states = db_manager.get_common_oxidation_states(element)
            if common_states:
                # For elements with multiple common states, show options
                states_str = ', '.join([f"{s:+d}" for s in common_states])
                default = common_states[0]  # Suggest first
                prompt_text = f"  {element:>2s} oxidation state (common: {states_str}) [{default:+d}]: "
            else:
                prompt_text = f"  {element:>2s} oxidation state [?]: "
                default = None
        
        while True:
            try:
                user_input = input(prompt_text).strip()
                
                if user_input == '' and default is not None:
                    # Accept default
                    oxidation_states[element] = default
                    break
                elif user_input != '':
                    # Parse user input
                    valence = int(user_input)
                    oxidation_states[element] = valence
                    break
                else:
                    # No default and no input
                    print("    ERROR: Please enter an oxidation state")
            
            except ValueError:
                print("    ERROR: Please enter a valid integer (e.g., 4, -2)")
    
    # Show summary
    print("\n" + "="*80)
    print("OXIDATION STATES CONFIRMED:")
    print("="*80)
    for element in sorted_elements:
        valence = oxidation_states[element]
        print(f"  {element:>2s}: {valence:+g}")
    print("="*80 + "\n")
    
    # Final confirmation
    confirm = input("Proceed with these oxidation states? [Y/n]: ").strip().lower()
    if confirm and confirm not in ['y', 'yes', '']:
        print("\nAborted by user")
        return None
    
    return oxidation_states

def batch_process_directory(directory_path):
    """
    Process all CIF files in a directory using EXACT Skuld 1.5 logic.
    Includes interactive oxidation state checkpoint for user control.
    Automatically creates /outputs subfolder for results.
    
    NEW IN v1.4: Attempts CIF oxidation state extraction before interactive prompt
    
    Args:
        directory_path: Path to directory containing CIF files
    
    Returns:
        pandas DataFrame with results, or None if failed
    """
    # Create outputs subdirectory
    output_dir = os.path.join(directory_path, "outputs")
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n✓ Output directory: {output_dir}")
    
    cif_files = list(Path(directory_path).glob("*.cif"))
    
    if not cif_files:
        print(f"No CIF files found in {directory_path}")
        return None
    
    # ============================================================================
    # STEP 1: Config file provides OVERRIDES (highest priority)
    # ============================================================================
    config_path = auto_detect_config_file(directory_path)
    override_oxidation_states = None
    
    if config_path:
        override_oxidation_states = load_oxidation_states_from_config(config_path=config_path)
        print(f"✓ Config file loaded - these will override CIF values")
    
    # ============================================================================
    # STEP 2: Scan all CIFs to see what we're dealing with
    # ============================================================================
    print(f"\nScanning {len(cif_files)} CIF files for oxidation states...")
    
    cif_ox_summary = {'explicit': 0, 'inferred': 0, 'ambiguous': 0, 'missing': 0, 'per_atom_explicit': 0}
    elements_needing_fallback = set()
    ambiguous_files = []
    missing_files = []
    per_atom_explicit_files = []  # NEW: Track well-resolved mixed-valence files
    
    for cif_file in cif_files:
        try:
            formula = formula_from_cif(cif_file)
            
            # First check for per-atom explicit valences
            per_atom_valences = extract_per_atom_valences(str(cif_file))
            has_per_atom_explicit = any(
                v['explicit_valence'] is not None 
                for v in per_atom_valences.values()
            )
            
            # Check for mixed valence at element level
            element_valences = {}
            for label, data in per_atom_valences.items():
                elem = data['element']
                val = data['explicit_valence']
                if val is not None:
                    if elem not in element_valences:
                        element_valences[elem] = set()
                    element_valences[elem].add(val)
            
            has_mixed_valence = any(len(vals) > 1 for vals in element_valences.values())
            
            # Classify the file
            if has_per_atom_explicit and has_mixed_valence:
                # Mixed valence WITH per-atom resolution - this is GOOD data!
                quality = 'per_atom_explicit'
                cif_ox_summary['per_atom_explicit'] += 1
                per_atom_explicit_files.append(cif_file.name)
                # NO fallback needed - we have explicit valences per atom
            
            elif has_per_atom_explicit:
                # Single valence per element with explicit data
                quality = 'explicit'
                cif_ox_summary['explicit'] += 1
            
            else:
                # Fall back to original flag_oxidation_state_issues logic
                result = flag_oxidation_state_issues(str(cif_file), formula)
                quality = result.get('quality', 'missing')
                cif_ox_summary[quality] = cif_ox_summary.get(quality, 0) + 1
                
                if quality == 'ambiguous':
                    ambiguous_files.append(cif_file.name)
                    parsed = parse_chemical_formula(formula)
                    for elem in parsed.keys():
                        if not is_anion(elem):
                            elements_needing_fallback.add(elem)
                elif quality == 'missing':
                    missing_files.append(cif_file.name)
                    parsed = parse_chemical_formula(formula)
                    for elem in parsed.keys():
                        if not is_anion(elem):
                            elements_needing_fallback.add(elem)
        except Exception as e:
            cif_ox_summary['missing'] += 1
            missing_files.append(cif_file.name)
    
    # Print summary
    print(f"  Explicit oxidation states:    {cif_ox_summary['explicit']} files")
    print(f"  Per-atom explicit (mixed):    {cif_ox_summary['per_atom_explicit']} files")  # NEW
    print(f"  Inferred oxidation states:    {cif_ox_summary['inferred']} files")
    print(f"  Ambiguous (needs fallback):   {cif_ox_summary['ambiguous']} files")
    print(f"  Missing oxidation states:     {cif_ox_summary['missing']} files")
    
    # NEW: Report per-atom explicit files (these are GOOD, not problematic)
    if per_atom_explicit_files:
        print(f"\n  ✓ Per-atom explicit files (mixed valence resolved):")
        for fn in per_atom_explicit_files[:10]:
            print(f"    - {fn}")
        if len(per_atom_explicit_files) > 10:
            print(f"    ... and {len(per_atom_explicit_files) - 10} more")
    
    # List problematic files (only truly ambiguous/missing)
    if ambiguous_files:
        print(f"\n  ⚠ Ambiguous files (will use fallback):")
        for fn in ambiguous_files[:20]:
            print(f"    - {fn}")
        if len(ambiguous_files) > 20:
            print(f"    ... and {len(ambiguous_files) - 20} more")
    
    if missing_files:
        print(f"\n  ⚠ Missing oxidation state files (will use fallback):")
        for fn in missing_files[:20]:
            print(f"    - {fn}")
        if len(missing_files) > 20:
            print(f"    ... and {len(missing_files) - 20} more")
    
    # Export review CSV (only for truly problematic files)
    if ambiguous_files or missing_files or per_atom_explicit_files:
        review_path = os.path.join(output_dir, "oxidation_state_review.csv")
        with open(review_path, 'w') as f:
            f.write("filename,status,notes\n")
            for fn in per_atom_explicit_files:
                f.write(f"{fn},per_atom_explicit,Mixed valence resolved - no action needed\n")
            for fn in ambiguous_files:
                f.write(f"{fn},ambiguous,Needs fallback oxidation states\n")
            for fn in missing_files:
                f.write(f"{fn},missing,No oxidation states found\n")
        print(f"\n  Review list exported to: {review_path}")
    
    # ============================================================================
    # STEP 3: Prompt for FALLBACK values only if needed
    # ============================================================================
    fallback_oxidation_states = {}
    
    if elements_needing_fallback and override_oxidation_states is None:
        print(f"\n{len(elements_needing_fallback)} elements need fallback oxidation states:")
        print(f"  {', '.join(sorted(elements_needing_fallback))}")
        print(f"\nThese will be used ONLY when CIF values are ambiguous/missing.")
        
        fallback_oxidation_states = prompt_for_oxidation_states(elements_needing_fallback)
    
    # ============================================================================
    # STEP 4: Process each file with per-file oxidation state resolution
    # ============================================================================
    all_results = []
    failed_files = []
    
    for _prog_i, cif_file in enumerate(cif_files, 1):
        if _prog_i == 1 or _prog_i % max(1, len(cif_files)//50) == 0 or _prog_i == len(cif_files):
            print(f"  Processing {_prog_i}/{len(cif_files)} ({100*_prog_i//len(cif_files)}%)...", flush=True)
        # Get formula for THIS file
        this_formula = formula_from_cif(cif_file)
        
        # NEW v1.4.2: Extract per-atom valences for mixed-valence detection
        per_atom_valences = extract_per_atom_valences(str(cif_file))
        
        # Check if this CIF has per-atom explicit valences (mixed valence resolved)
        has_per_atom_explicit = any(
            v['explicit_valence'] is not None 
            for v in per_atom_valences.values()
        )
        
        # Try CIF first (only pass config override, NOT fallback)
        ox_states, source, type7_result = get_oxidation_states_for_analysis(
            str(cif_file), 
            this_formula,
            override_oxidation_states
        )
        
        # NEW v1.4.2: If we have per-atom explicit valences, override the type7 flag
        # Mixed valence with per-atom resolution is NOT an error - it's good data!
        if has_per_atom_explicit and type7_result:
            if type7_result.get('error_code') == 'TYPE7_EXPLICIT_MIXED_VALENCE':
                type7_result['has_type7_error'] = False
                type7_result['error_code'] = None
                type7_result['quality'] = 'per_atom_explicit'
        
        # If CIF failed AND no per-atom valences, use fallback
        if ox_states is None and not has_per_atom_explicit and fallback_oxidation_states:
            ox_states = fallback_oxidation_states
            source = 'fallback'
        
        # Process with resolved oxidation states
        # NOTE: process_single_cif extracts per_atom_valences internally for BVS calc
        result = process_single_cif(str(cif_file), ox_states)
        
        if result['success']:
            gii_str = f"{result['gii']:.4f}" if result['gii'] is not None else "N/A"
            source_str = f"[{source}]" if source else "[no ox]"
            print(f"✓ {source_str} ({len(result['sites'])} sites, GII={gii_str})")
            
            # ================================================================
            # NEW: Flag all error types (2-6) for this structure
            # ================================================================
            error_flags = flag_all_error_types(
                str(cif_file), 
                this_formula, 
                result['sites'],
            )
            
            # v2.0.11: no structure tier here either. The demotions that used
            # to happen below - partial instability index, unscreened cation -
            # are the R facet of PGRM and are derived there from the same
            # evidence: a site with no bond valence sum is R1, a site with no
            # tabulated radius is R2. Computing them twice, in two places, with
            # two names, is how CELL 14 came to print two contradictory
            # recommendations on one page.
            structure_tier = None
            n_flagged = n_expunged = 0

            # v2.0.3 (rule 2). A site with no bond valence parameters never
            # reaches the instability index, and left no trace of having been
            # dropped. ICSD 93099 (LaPdO3) returned 0.0103, the second lowest
            # GII in the perovskite set, computed over its lanthanum alone.
            _dom = []
            for _s in result['sites']:
                if not _s.get('is_dominant_species', True):
                    continue
                _o = _s.get('occupancy')
                if _o is not None and float(_o) == 0.0:
                    continue
                _dom.append(_s)
            _uncovered = [_s for _s in _dom if _s.get('bvs') is None]
            result['gii_sites_total'] = len(_dom)
            result['gii_sites_covered'] = len(_dom) - len(_uncovered)
            result['gii_coverage_note'] = None
            if _uncovered:
                result['gii_coverage_note'] = '; '.join(
                    f"{_s.get('element')}({_s.get('valence')}) at "
                    f"{_s.get('site_label')}: no bond valence parameters"
                    for _s in _uncovered)
                print(f"  PARTIAL INSTABILITY INDEX: "
                      f"{result['gii_sites_covered']} of {len(_dom)} cation "
                      f"sites contributed. {result['gii_coverage_note']}. "
                      f"The GII is not a whole-structure figure.")

            # The geometry side of the same rule. core_sites_untested has been
            # computed and exported since v1.8.8 and read by nothing; a site
            # with no Shannon radius is invisible to two of the five screens.
            # Five Fe(4+) sites in CaFeO3 and EuFeO3 sit here, all ACCEPT.
            if result.get('core_sites_untested'):
                print(f"  UNSCREENED CATION: "
                      f"{result['core_sites_untested']} site(s) have no "
                      f"tabulated radius, so the radii-ratio and coordination "
                      f"screens could not test them.")
            
            for site in result['sites']:
                row = {
                    **result['metadata'], 
                    **site,
                    'gii': round(result['gii'], 4) if result['gii'] is not None else None,
                    'wagii': round(result['wagii'], 4) if result.get('wagii') is not None else None,
                    'gii_sites_total': result.get('gii_sites_total'),
                    'gii_sites_covered': result.get('gii_sites_covered'),
                    'gii_coverage_note': result.get('gii_coverage_note'),
                    'n_sites_zero_occupancy': result.get('n_sites_zero_occupancy'),
                    # v2.0.16: these two are the audit trail for the 4.8
                    # change and in v2.0.15 neither reached the CSV, because
                    # this row is an explicit whitelist and they were not on
                    # it. A diagnostic that is computed, printed and not
                    # exported is the same fault as cation_packing_plausible
                    # in v2.0.12: the triage said one thing and the batch log
                    # said another.
                    'n_sites_below_floor': result.get('n_sites_below_floor'),
                    'gii_with_floor_failures': (
                        round(result['gii_with_floor_failures'], 4)
                        if result.get('gii_with_floor_failures') is not None
                        else None),
                    'bvs_discrepancy': round(result['bvs_discrepancy'], 4) if result.get('bvs_discrepancy') is not None else None,
                    'gii_site_averaged': (round(result['gii_site_averaged'], 4)
                                          if result.get('gii_site_averaged') is not None else None),
                    'n_crystallographic_sites': result.get('n_crystallographic_sites'),
                    'cif_sign_inversion': result.get('cif_sign_inversion'),
                    'cif_sign_inversion_detail': result.get('cif_sign_inversion_detail'),
                    'anion_anion_min': result.get('anion_anion_min'),
                    'cation_cation_min': result.get('cation_cation_min'),
                    'anion_packing_plausible': result.get('anion_packing_plausible'),
        # v2.0.13a: computed and printed since v2.0.12 and never
        # exported, so the triage and summary both reported it as 0
        # structures flagged while the batch log said otherwise.
        'cation_packing_plausible': result.get('cation_packing_plausible'),
                    'structure_bvs_distance_ratio': result.get('bvs_distance_ratio'),
                    'structure_bvs_geometry_plausible': result.get('bvs_geometry_plausible'),
                    # v2.0.2: the structure-level radii-ratio verdict was
                    # computed but never exported, so the core_geometry_plausible
                    # column held the PER-SITE value while the two above were per
                    # structure. Anything reading all three the same way was
                    # wrong: groupby('filename').first() misses the failure
                    # whenever the bad site is not the first row.
                    'structure_core_geometry_plausible': result.get('core_geometry_plausible'),
                    'structure_core_shortest_contact_ratio': result.get('core_shortest_contact_ratio'),
                    'core_sites_tested': result.get('core_sites_tested'),
                    'core_sites_untested': result.get('core_sites_untested'),
                    'structure_cn_plausible': result.get('cn_plausible'),
                    'n_sites_cn_implausible': result.get('n_sites_cn_implausible'),
                    'n_sites_shift_proposed': result.get('n_sites_shift_proposed'),
                    'has_antisite_disorder': result.get('has_antisite_disorder'),
                    'n_antisite_species': result.get('n_antisite_species'),
                    'antisite_detail': result.get('antisite_detail'),
                    # Which instability index to quote for THIS structure.
                    # gii is per species and is misleading wherever a site is
                    # shared; see antisite_exchange() in CELL 1.
                    'gii_recommended': ('wagii' if result.get('has_antisite_disorder')
                                        else 'gii'),
                    # Type 7 flags (oxidation state issues)
                    'flag_type7': type7_result.get('has_type7_error') if type7_result else None,
                    'type7_error_code': type7_result.get('error_code') if type7_result else None,
                    'oxidation_quality': type7_result.get('quality') if type7_result else None,
                    # NEW v1.4.2: Track where valence came from
                    'valence_source': site.get('valence_source', 'unknown'),
                    # NEW v1.4.2: Types 2-6 flags
                    'flag_type2': error_flags.get('flag_type2'),
                    'flag_type3': error_flags.get('flag_type3'),
                    'flag_type4': error_flags.get('flag_type4'),
                    'flag_type5': error_flags.get('flag_type5'),
                    'flag_type6': error_flags.get('flag_type6'),
                    'flag_type8': error_flags.get('flag_type8'),
                    'error_types_detected': error_flags.get('error_types_detected'),
                    'error_type_details': error_flags.get('error_type_details'),
                    # NEW v1.7: Structure-level integrity tier
                    'structure_tier': structure_tier,
                    'n_sites_flagged': n_flagged,
                    'n_sites_expunged': n_expunged,
                    # A1.1: anion-centred BVS summary
                    **(result.get('anion_summary') or {}),
                }
                all_results.append(row)
        
        else:
            print(f"✗ ({result['error']})")
            failed_files.append({
                'filename': cif_file.name,
                'error': result['error']
            })
    
    # ============================================================================
    # STEP 5: Save results to CSV files
    # ============================================================================
    if all_results:
        df = pd.DataFrame(all_results)
        
        # NEW IN v1.4: Ensure Type 7 columns exist
        type7_columns = ['flag_type7', 'type7_error_code', 'oxidation_quality']
        for col in type7_columns:
            if col not in df.columns:
                df[col] = None

        # ============================================================================
        # DATASET-LEVEL PASS: Flag multiple entries from same source paper (v1.7.2)
        # ============================================================================
        if 'icsd_code' in df.columns and 'authors' in df.columns:
            df = flag_same_source_entries(df)
            
            # Print summary of multi-entry sources
            multi = df[df['flag_multi_entry_source'] == True]
            if len(multi) > 0:
                multi_groups = multi.groupby('source_group').first().reset_index()
                print(f"\n⚠ Dataset-level check: {len(multi_groups)} source papers have multiple ICSD entries:")
                for _, grp in multi_groups.iterrows():
                    codes = grp['source_group_codes']
                    n = grp['source_group_size']
                    # Get one example formula per code
                    code_list = codes.split(',')
                    examples = []
                    for code in code_list[:4]:
                        formula = df[df['icsd_code'] == code]['chemical_formula'].iloc[0] if len(df[df['icsd_code'] == code]) > 0 else '?'
                        examples.append(f"{code}({formula})")
                    suffix = f" +{len(code_list)-4} more" if len(code_list) > 4 else ""
                    print(f"  {grp['source_group']}: {n} entries — {', '.join(examples)}{suffix}")
                print(f"  Note: multiple entries from one paper may represent different samples,")
                print(f"  refinement methods, or phase-separated components. Check the original")
                print(f"  paper to assess whether entries are independent determinations.")
        
        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_file = os.path.join(output_dir,
                                    f"eir_batch_results_v{EIR_VERSION}_{timestamp}.csv")
        df.to_csv(results_file, index=False)
        print(f"\n✓ Results saved to: {results_file}")
        print(f"  Total sites analysed: {len(df)}")
        print(f"  Total columns: {len(df.columns)}")
        
        # ============================================================================
        # PRINT STRUCTURE-TYPE ANALYSIS SUMMARY (Eir v1.4+)
        # ============================================================================
        if 'spinel_classification' in df.columns:
            spinel_count = df['spinel_classification'].notna().sum()
            if spinel_count > 0:
                spinel_structures = df[df['spinel_classification'].notna()]['filename'].nunique()
                print(f"\n  Spinel structures detected: {spinel_structures} files, {spinel_count} sites")
                print(f"    Classifications:")
                for classification in df['spinel_classification'].dropna().unique():
                    count = (df['spinel_classification'] == classification).sum()
                    print(f"      - {classification}: {count} sites")
                
                if 'inversion_parameter' in df.columns:
                    lambda_values = df['inversion_parameter'].dropna()
                    if len(lambda_values) > 0:
                        print(f"    Inversion parameters:")
                        print(f"      - λ = 0 (normal): {(lambda_values == 0.0).sum()} sites")
                        print(f"      - λ = 1 (inverse): {(lambda_values == 1.0).sum()} sites")
                        print(f"      - 0 < λ < 1 (random): {((lambda_values > 0.0) & (lambda_values < 1.0)).sum()} sites")
        
        # Build merged oxidation states dict for logging
        # Priority: override (config) > fallback (interactive prompt) > empty
        effective_oxidation_states = {}
        if fallback_oxidation_states:
            effective_oxidation_states.update(fallback_oxidation_states)
        if override_oxidation_states:
            effective_oxidation_states.update(override_oxidation_states)
        
        # Save oxidation states used for this batch
        oxidation_states_file = os.path.join(output_dir, f"oxidation_states_used_{timestamp}.json")
        with open(oxidation_states_file, 'w') as f:
            import json
            json.dump(effective_oxidation_states, f, indent=2, sort_keys=True)
        print(f"\n✓ Oxidation states saved to: {oxidation_states_file}")
        
        # Save failed files log if any
        if failed_files:
            failed_df = pd.DataFrame(failed_files)
            # v2.0.13: the version was hard-coded at v1.7, so a 2.0.x run
            # wrote its failures under a 1.7 filename and the two could not
            # be told apart in a directory holding both.
            failed_file = os.path.join(
                output_dir,
                f"eir_batch_failed_v{EIR_VERSION}_{timestamp}.csv")
            failed_df.to_csv(failed_file, index=False)
            print(f"✓ Failed files log saved to: {failed_file}")
            print(f"  ({len(failed_files)} files failed)")
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print(f"EIR BATCH PROCESSOR v{EIR_VERSION} - Summary Statistics")
        print(f"{'='*80}")
        
        # Show oxidation states used
        print(f"\nOxidation states used:")
        for element in sorted(effective_oxidation_states.keys()):
            valence = effective_oxidation_states[element]
            print(f"  {element:>2s}: {valence:+g}")
        
        # Show GII statistics if available
        if 'gii' in df.columns and df['gii'].notna().any():
            print(f"\nGlobal Instability Index (GII) statistics:")
            gii_by_structure = df.groupby('filename')['gii'].first()
            print(f"  Mean GII: {gii_by_structure.mean():.4f}")
            print(f"  Min GII:  {gii_by_structure.min():.4f} ({gii_by_structure.idxmin()})")
            print(f"  Max GII:  {gii_by_structure.max():.4f} ({gii_by_structure.idxmax()})")
            
            stable_structures = (gii_by_structure < 0.1).sum()
            unstable_structures = (gii_by_structure >= 0.2).sum()
            print(f"  Stable (GII < 0.1): {stable_structures} structures")
            print(f"  Unstable (GII ≥ 0.2): {unstable_structures} structures")
        
        # The v1.7 ACCEPT/FLAG/EXPUNGE summary was removed at v2.0.13a. The
        # tier it counted was derived FROM the bond valence sum and was
        # retired at v2.0.11; the counters survived it and printed three
        # zeros at the end of every run. The verdict now comes from the PGRM
        # gates, reported by the post-processing cell and the triage.
        print(f"{'='*80}\n")
        
        return df
    else:
        print("\n✗ No successful analyses")
        if failed_files:
            print(f"All {len(failed_files)} files failed. Check error messages above.")
        return None

# ============================================================================
# CONVENIENCE FUNCTIONS
# ============================================================================

def run_batch_analysis():
    """
    Convenience function for easy use in Jupyter notebooks or scripts.
    Opens directory selection dialog and processes all CIF files.
    
    Usage:
        from Eir_v1_1 import run_batch_analysis
        df = run_batch_analysis()
    
    Returns:
        pandas DataFrame with results, or None if failed/cancelled
    """
    print("\n" + "="*80)
    print("Running Eir v1.8.1 Batch Analysis...")
    print("="*80)
    
    # Select directory with tkinter dialog
    directory = select_directory()
    
    if directory is None:
        return None
    
    # Process all CIF files
    results_df = batch_process_directory(directory)
    
    if results_df is not None:
        print("\n" + "="*80)
        print("BATCH PROCESSING COMPLETE")
        print("="*80)
        print(f"Successfully analysed: {results_df['filename'].nunique()} structures")
        print(f"Total sites analysed: {len(results_df)}")
        output_dir = os.path.join(directory, "outputs")
        print(f"\nAll results saved to: {output_dir}")
        print("\nNext steps:")
        print("  1. Review the results CSV file")
        print("  2. Check for any failed files in the failed log")
        print("  3. Analyse results with statistical tools or custom scripts")
        
        return results_df
    
    return None

# ============================================================================
# SELF-CHECK (NEW IN v2.0.2)
# ============================================================================
# A cell that stops half way is the worst failure mode in a notebook: the
# names defined above the stop survive, the ones below do not, and the error
# appears later somewhere unrelated. This check runs last, so reaching it at
# all proves the cell completed, and it verifies that the cells this one
# depends on have been run too.

def _eir_self_check(verbose=True):
    """Confirm this cell completed and its dependencies are present."""
    exports = [
        'clean_cif_for_gemmi', 'calculate_bonds_from_positions',
        'parse_cif_file', 'extract_initial_site_data', 'auto_select_bond_types',
        'auto_analyze_coordination', 'auto_assign_valences',
        'perform_calculations_and_report', 'calculate_global_instability_index',
        'process_single_cif', 'batch_process_directory', 'run_batch_analysis',
        'guarded_shell_cn', 'full_shell_cn', 'group_bond_counts',
        'cn_fault', 'econ_bound', 'VALIDATE_CN', 'db_manager',
    ]
    # Names this cell uses but does not define. CELL 1 supplies most;
    # anion_bvs_for_structure comes from the anion-BVS cell and is needed only
    # at run time, so it is reported separately rather than as a hard failure.
    from_cell1 = [
        'EIR_VERSION', 'ANION_ANION_MIN', 'ANION_FORMERS', 'SHANNON_CN6_VIZ',
        'IMPLAUSIBLE_CONTACT_FRACTION', 'IMPLAUSIBLE_BVS_DISTANCE_FRACTION',
        'JT_OCTAHEDRAL_ELEMENTS', 'MAX_JT_AXIAL_RATIO', '_ANION_SET',
        '_EIR_GEOMETRY', '_SIGN_INVERSIONS', '_dbg', 'ge',
        'assess_site_geometry', 'anion_anion_contacts', 'classify_bond_pair',
        'analyze_mixed_occupancy_at_site', 'calculate_discrepancy_factor',
        'assign_confidence_flag', 'whatif_analysis',
        'check_minority_species', 'check_mixed_tm_site', 'flag_all_error_types',
        'flag_same_source_entries', 'flag_oxidation_state_issues',
        'extract_icsd_metadata', 'extract_per_atom_valences',
        'extract_site_occupancies', 'extract_cif_oxidation_states',
        'get_oxidation_states_for_analysis', 'parse_chemical_formula',
        'parse_atom_type_with_valence', 'is_anion', 'auto_detect_config_file',
        'load_oxidation_states_from_config', 'visualize_problematic_site_3d',
    ]
    g = globals()
    missing_here = [n for n in exports if n not in g]
    missing_dep = [n for n in from_cell1 if n not in g]
    deferred = [n for n in ['anion_bvs_for_structure'] if n not in g]

    if missing_here:
        raise NameError(
            f"This cell did not finish: {len(missing_here)} of its own names "
            f"are undefined ({', '.join(missing_here[:4])}...). Scroll up for "
            f"the real error and re-run this cell in full.")
    if missing_dep:
        raise NameError(
            f"{len(missing_dep)} names this cell needs are missing: "
            f"{', '.join(missing_dep[:6])}"
            f"{'...' if len(missing_dep) > 6 else ''}. These live in the "
            f"helper cell above (CELL 1 and the geometry core). Run that cell, "
            f"then this one.")
    if verbose:
        print('Eir 2.0.2 main cell loaded and self-checked.')
        print(f'  {len(exports)} exports defined, {len(from_cell1)} dependencies present.')
        if deferred:
            print(f"  note: {', '.join(deferred)} comes from the anion-BVS cell "
                  f"below; run it before batch processing.")
        print('  Coordination shell guard active: the gap-analysis CN is checked')
        print('  against ECoN before the bond valence sum is computed, and the sum')
        print('  is reported over both the chosen and the full distance shell.')
        print('  Run VALIDATE_CN() to check the guard, then Cell 8 to batch process.')
    return True


_eir_self_check()

In [ ]:
# CELL 3: ANION-CENTRED BVS ENGINE   (Eir 2.0.13)
#
# CHANGED IN 2.0.2
#   Each anion site now records the site multiplicity and which cations
#   coordinate it, and the structure-level summary identifies the worst site
#   rather than only its deviation. Two reasons.
#
#   First, the mean was unweighted across sites. In an ordered pyrochlore the
#   48f oxygen has six times the multiplicity of the 8b, so weighting them
#   equally overstates the mean anion sum by roughly 0.04 v.u. The
#   multiplicity-weighted mean is now reported alongside.
#
#   Second, without knowing WHICH site deviates the number cannot be
#   interpreted. In Fd-3m pyrochlore the 8b oxygen has no free positional
#   parameter: it sits in the A4 tetrahedron at a*sqrt(3)/8, so its bond
#   valence is fixed by the lattice parameter and the A cation cannot relax
#   into it. Measured across 309 ordered RE2B2O7 entries its overbonding
#   tracks rA/rB at r = +0.82 and changes sign at the pyrochlore-fluorite
#   boundary, while the cation GII over the same range is flat. The ligand
#   composition recorded here is what distinguishes that site (four A cations)
#   from the 48f oxygen (two A, two B) without needing a Wyckoff assignment.
# =============================================================================
# ANION-CENTRED BOND VALENCE SUMS
#
# WHAT IT DOES
#   For each crystallographic anion site, sums the bond valence contributions
#   from every neighbouring cation. The bond valence of an M-X bond is the same
#   quantity whichever end you sum it at, so this reuses the existing R0 and b
#   lookup with no new parameters.
#
# WHY IT MATTERS
#   A cation-only BVS cannot tell a bond valence parameter offset from real
#   structural strain. The anion sum can: if the cation deviations cancel at
#   the shared anion, the parameters are mis-calibrated relative to each other
#   and the structure is fine. If the anion also deviates, the problem is real.
#
# TWO DELIBERATE DIFFERENCES FROM THE CATION PATH, both explained below:
#   1. Contributions are weighted by the neighbouring cation's site occupancy.
#   2. No coordination-number gap cut is applied.
# =============================================================================

import math
import tempfile
import numpy as np

# Alias of PROVISIONAL_ANIONS (CELL 1), where the reasoning lives. This was a
# second hand-written copy of the same nine species.
ANION_ELEMENTS = PROVISIONAL_ANIONS

# Distance tolerance for merging symmetry-generated duplicate images.
# NOT a grid snap. ICSD routinely writes special positions as 0.3333 rather
# than 1/3, which makes symmetry images land ~0.002 A apart; a tolerance of
# 1e-3 A double-counts every neighbour, 0.02 A merges them correctly.
ANION_DEDUP_TOL = 0.02

# v2.0.2. ANION_RMAX was 3.5 A, applied as a hard cut on the summation, and
# its comment claimed it matched the cation-side cutoff (which searches to
# 4.2 A). Neither the claim nor the method held up.
#
# In a tilted GdFeO3-type perovskite each oxygen is coordinated by two B and
# four A cations, and the four A-O bonds split under the tilt so that the
# longest exceeds 3.5 A. Measured over 934 ICSD perovskites, 349 of them
# (37%) had an oxygen summed over five cations instead of six, and the sum
# tracked the miscount directly:
#
#     cations found   n     median anion BVS
#     6 (A:4,B:2)    538          2.026
#     5 (A:3,B:2)    327          1.976
#     4               20          1.795
#
# A bias of that size matters because ANION_CONSISTENT_TOL is 0.20 v.u. and
# anion_balances feeds bvs_diagnosis, so a truncated sum can move a structure
# from cation_and_anion_agree to cation_only_deviates.
#
# The summation now terminates on bond valence rather than distance, which is
# what the comment on the summation loop below always claimed it did. A bond
# is included while its contribution s exceeds ANION_BV_CUTOFF; because s
# falls exponentially, the cut-off distance follows from each pair's own R0
# and b rather than from a global number, and is therefore correct for La-O
# (3.88 A) and Ca-O (3.67 A) alike.

# Contribution below which a bond is not worth counting, in valence units.
# Editable; 0.04 is the default and the reason is a hard constraint, not taste.
#
# The anion sum is only useful because it is compared with the cation sum, and
# a difference between them is read as evidence about the parameters. That
# comparison is only valid if both span the SAME coordination shell. The
# cation side gap-cuts to the first shell, so the anion cut-off has to be
# tight enough to exclude the second.
#
# Rock salt sets the limit. In MgO the second cation shell lies at a*sqrt(3)/2
# and holds eight neighbours; at a = 4.18937 A that is 3.628 A, where
# s = 0.01046 for Mg-O. A cut-off of 0.01 admits all eight, adding 0.084 v.u.,
# so the anion sum reads 2.084 against a cation sum of 2.000 for the same
# bonds. VALIDATE() test 6 fails on exactly that. At 0.04 the second shell is
# excluded (it would have to reach 3.034 A to count) and the two sums agree.
#
#     cut-off   Mg-O reach   MgO 2nd shell (3.628 A)   test 6
#       0.01       3.65 A            included          FAIL
#       0.04       3.03 A            excluded          pass
#
# Nothing is lost at the loose end: on a tilted LaTiO3-type perovskite the
# anion sums are identical at 0.04, at 0.01 and under the old fixed 3.5 A cut,
# because the fourth A-O bond of a strongly tilted structure lies beyond all
# three and is genuinely too long to be a bond.
#
# Lowering this below about 0.02 will start to admit second shells and break
# the cation-anion comparison. Raise it to tighten the first shell.
# v2.0.3: the anion cut-off is no longer independent. s is a property of the
# bond, so a contact admitted to the cation sum must be admitted to the anion
# sum; the previous 0.04 counted bonds in 0.02 <= s < 0.04 at one end only.
#
# MEASURED, not argued. An earlier note here warned that lowering the cut-off
# below about 0.02 would admit second shells. The opposite is true. On 831
# clean ICSD perovskites the modal oxygen coordination is 5 at 0.04 and 6 at
# 0.02, where the structure type requires 2 B + 4 A = 6. Of 491 anion sites
# that gained coordination on the change, EVERY one went from [A:3, B:2] to
# [A:4, B:2]: the B count never moved and nothing exceeded 6. The old cut-off
# was truncating the fourth A-O bond, which is the long bond to a large
# period-4-6 cation that Gagne & Hawthorne (2015) sec 4 name explicitly as
# one to include. 0.02 also gives the anion sum closest to formal (mean
# 1.9990 against 1.9894 at 0.04) and the fewest sites outside
# ANION_CONSISTENT_TOL. Mean |deviation| is flat at 0.090 across all four
# cut-offs tested, so the residual anion scatter is not a shell problem and
# lowering the cut further will not address it.
#
# See BV_SUM_CUTOFF in CELL 1 for the cation-side derivation and the sourcing.
# v2.0.13 NOTE. The cation side now also has an adaptive boundary set from
# each ion's own Lewis acidity and Gagne's measured bond lengths
# (bv_shell_boundary, CELL 1), currently exported but not canonical. When it
# becomes canonical this alias must follow it, or the two ends of the same
# bond will once more be summed under different rules - which is the
# asymmetry v2.0.3 removed by making this an alias in the first place.
ANION_BV_CUTOFF = BV_SUM_CUTOFF

# Neighbour SEARCH radius only, not a summation cut. Must exceed the distance
# at which s falls to ANION_BV_CUTOFF for the largest R0 in use: Ba-O at
# R0 = 2.29 reaches 0.04 v.u. at 3.48 A and 0.01 v.u. at 3.99 A, so 5.0 A
# leaves margin for a lowered cut-off without needing to be changed with it.
ANION_RMAX = 5.0
ANION_CONSISTENT_TOL = 0.20   # |BVS - formal| below which the anion "balances"


def _load_structure_for_anion_bvs(cif_path):
    """Read a CIF into a gemmi SmallStructure, reusing Eir's CIF cleaning."""
    import gemmi
    import os as _os

    cleaned = clean_cif_for_gemmi(cif_path)
    if cleaned is None:
        return None

    with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False,
                                     encoding='utf-8') as tmp:
        tmp.write(cleaned)
        tmp_path = tmp.name
    try:
        st = gemmi.make_small_structure_from_block(gemmi.cif.read(tmp_path).sole_block())
    finally:
        try:
            _os.unlink(tmp_path)
        except OSError:
            pass          # temp file already gone; nothing depends on it

    if st is None or len(st.sites) == 0:
        return None

    # Some ICSD entries leave gemmi unable to resolve the group from the H-M
    # symbol alone; fall back to the International Tables number.
    if st.spacegroup is None:
        import re as _re
        m = (_re.search(r'_space_group_IT_number\s+(\d+)', cleaned)
             or _re.search(r'_symmetry_Int_Tables_number\s+(\d+)', cleaned))
        if m:
            st.spacegroup = gemmi.find_spacegroup_by_number(int(m.group(1)))
    if st.spacegroup is None:
        return None
    return st


def site_multiplicities(st, tol=0.02):
    """
    Number of distinct atoms per site label in one unit cell.

    v2.0.2. Needed to weight the anion sites: reporting an unweighted mean
    over sites treats a 48-fold and an 8-fold oxygen as equal contributors,
    which they are not. Counted from the symmetry operations rather than
    read from a Wyckoff table, so it needs no lookup and works in any setting.
    """
    out = {}
    for s in st.sites:
        seen = []
        for op in st.spacegroup.operations():
            f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
            f = [x - math.floor(x) for x in f]
            if not any(all(min(abs(a - b), 1.0 - abs(a - b)) < tol
                           for a, b in zip(f, g)) for g in seen):
                seen.append(f)
        out[s.label] = len(seen)
    return out


def _expand_cell(st, rmax=None):
    """
    All symmetry images of every site, over enough unit cells to cover rmax.

    v2.0.2: the block is now sized from the search radius and the cell edges
    instead of being fixed at 3x3x3. A fixed one-cell margin covers a 3.5 A
    search only while every edge exceeds 3.5 A; for a simple cubic perovskite
    at a = 3.8 A it was already marginal, and raising the search radius to
    5.0 A would have silently lost neighbours across the cell boundary.
    """
    import gemmi
    n = 1
    if rmax:
        edges = [st.cell.a, st.cell.b, st.cell.c]
        n = max(1, int(math.ceil(rmax / min(e for e in edges if e > 0))))
    span = range(-n, n + 1)
    pts = []
    for s in st.sites:
        for op in st.spacegroup.operations():
            f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
            for i in span:
                for j in span:
                    for k in span:
                        p = st.cell.orthogonalize(
                            gemmi.Fractional(f[0] + i, f[1] + j, f[2] + k))
                        pts.append((s.label, s.element.name, s.occ, p.x, p.y, p.z))
    return pts


def calculate_anion_bvs(cif_path, valences, rmax=ANION_RMAX, verbose=False):
    """
    Anion-centred bond valence sums for one structure.

    Args:
        cif_path : path to the CIF
        valences : {element_or_site_label: valence}, the SAME dict used for the
        cation-side sums. Site labels take precedence over elements, so a
        structure with one element at two valences is summed correctly.
                   calculation, so the two are directly comparable
        rmax     : neighbour cutoff in Angstrom
        verbose  : print the per-bond breakdown

    Returns:
        list of dicts, one per crystallographic anion site:
            anion_label, anion_element, anion_cn, anion_bvs,
            anion_bvs_unweighted, anion_expected, anion_deviation,
            anion_contributions, n_missing_params
        Returns [] if the structure cannot be read.
    """
    st = _load_structure_for_anion_bvs(cif_path)
    if st is None:
        return []

    pts = _expand_cell(st, rmax)
    # v2.0.2: atoms per site label in one cell, for weighting the sites.
    # Never allowed to be fatal. Multiplicity is an enhancement to the
    # reporting; losing it costs a weighted mean, whereas letting it raise
    # costs the whole structure. On the first run of this code exactly that
    # happened to one perovskite of 934, which silently dropped from
    # anion_only_deviates to insufficient_data.
    try:
        _multiplicity = site_multiplicities(st)
    except (NameError, AttributeError, TypeError):
        # A defect in site_multiplicities, not a property of the CIF. The
        # note above is right that losing multiplicity must not cost the
        # structure, but a broken function must not present as a structure
        # that happens to have none.
        raise
    except Exception as _e:
        _multiplicity = {}
        # Not conditional on verbose: a weighted mean silently becoming an
        # unweighted one is exactly the kind of quiet degradation this
        # notebook keeps finding.
        print(f'  [site multiplicities unavailable, mean will be unweighted: '
              f'{type(_e).__name__}: {_e}]')
    P = np.array([[a[3], a[4], a[5]] for a in pts])

    results = []
    for site in st.sites:
        if site.element.name not in ANION_ELEMENTS:
            continue

        o = site.orth(st.cell)
        p0 = np.array([o.x, o.y, o.z])
        d = np.linalg.norm(P - p0, axis=1)
        cand = np.where((d > 0.01) & (d <= rmax))[0]
        cand = cand[np.argsort(d[cand])]

        # Deduplicate symmetry images WITHIN each site label only.
        # Two different labels at the same coordinates are a mixed-occupancy
        # site and BOTH are real neighbours. Deduplicating on position alone
        # silently deletes the minority species: on a set of P2 layered oxides that
        # removed the Ni from every Mn/Ni site and pulled the mean anion sum
        # from 2.11 down to 1.68.
        kept = {}
        neighbours = []
        for idx in cand:
            label, elem, occ = pts[idx][0], pts[idx][1], pts[idx][2]
            if elem in ANION_ELEMENTS:
                continue
            p = P[idx]
            if any(np.linalg.norm(p - q) < ANION_DEDUP_TOL
                   for q in kept.get(label, [])):
                continue
            kept.setdefault(label, []).append(p)
            neighbours.append((float(d[idx]), label, elem, occ))
        neighbours.sort()

        anion_elem = site.element.name
        expected = abs(valences.get(anion_elem, -2))

        total = 0.0
        total_unweighted = 0.0
        n_missing = 0
        n_counted = 0
        _counted_elements = []
        parts = []

        for dist, label, elem, occ in neighbours:
            # v2.0.13: LABEL first, element second. The map is keyed by both;
            # keying it by element alone gave every site of an element the
            # same valence, so in Co3O4 the 8a Co(2+) and the 16d Co(3+)
            # were summed with one R0 chosen by list order. 147 of 1881
            # spinels (7.8%), 10 of 472 pyrochlores and 2 of 938 perovskites
            # carry an element at two valences.
            v = valences.get(label)
            if v is None:
                v = valences.get(elem)
            if v is None:
                n_missing += 1
                continue
            R0, b = get_bv_parameters_fractional(elem, v, anion_elem)
            if R0 is None:
                n_missing += 1
                continue
            s = math.exp((R0 - dist) / b)
            # v2.0.2: terminate on bond valence, not distance. See
            # ANION_BV_CUTOFF. A neighbour inside the search radius whose
            # contribution is below the cut-off is not a bond.
            if s < ANION_BV_CUTOFF:
                continue
            n_counted += 1
            _counted_elements.append(elem)
            # Occupancy weighting. On the cation side the ligands are anions,
            # which are almost always fully occupied, so this rarely matters.
            # On the anion side the ligands are cations, which in layered
            # oxides are routinely 30-70% occupied, so it is essential: the
            # average oxygen sees only the occupied fraction of each cation
            # site. This is the same argument as WAGII, applied per bond.
            total += occ * s
            total_unweighted += s
            parts.append(f"{elem}({label}) {dist:.3f}A occ{occ:.2f} s={s:.3f}")

        # No gap cut is applied. Bond valence falls off exponentially and
        # terminates itself; cutting the shell at the largest geometric gap
        # cannot separate the two halves of a split site, which is exactly
        # where the anion sum is most informative.
        # v2.0.2: which cations coordinate this anion, and how many of each.
        # This is what separates the two oxygen sites of a pyrochlore without
        # needing a Wyckoff assignment: the 8b oxygen is coordinated by four A
        # cations and nothing else, the 48f by two A and two B.
        # Accepted bonds only. Counting the searched neighbours instead
        # reports every cation inside the search radius as a ligand, which
        # after the radius was widened to 5.0 A meant 18 "ligands" on an
        # oxygen with five bonds.
        _lig = {}
        for _el in _counted_elements:
            _lig[_el] = _lig.get(_el, 0) + 1
        _lig_txt = ','.join(f'{e}:{c}' for e, c in
                            sorted(_lig.items(), key=lambda kv: (-kv[1], kv[0])))

        results.append({
            'anion_label': site.label,
            'anion_element': anion_elem,
            # bonds actually counted, not neighbours found inside the
            # search radius; the two differ by the bond-valence cut-off
            'anion_cn': n_counted,
            'anion_neighbours_searched': len(neighbours),
            'anion_bv_cutoff': ANION_BV_CUTOFF,
            'anion_multiplicity': _multiplicity.get(site.label),
            'anion_ligands': _lig_txt or None,
            'anion_n_ligand_species': len(_lig),
            'anion_bvs': round(total, 4),
            'anion_bvs_unweighted': round(total_unweighted, 4),
            'anion_expected': expected,
            'anion_deviation': round(total - expected, 4),
            'anion_contributions': '; '.join(parts) if parts else None,
            'n_missing_params': n_missing,
        })

        if verbose:
            print(f"  {site.label} ({anion_elem}): CN={n_counted}  "
                  f"BVS={total:.3f}  expected {expected:.1f}  "
                  f"dev {total - expected:+.3f}")
            for p in parts:
                print(f"      {p}")

    return results


def summarise_anion_bvs(anion_results, cation_gii=None):
    """
    Collapse per-site anion results into structure-level diagnostic fields.

    anion_balances : True if every anion site sums to its formal charge within
                     ANION_CONSISTENT_TOL. When this is True but the cation GII
                     is large, the cation deviations are cancelling at the
                     shared anion, which points at the bond valence parameters
                     rather than at the structure.
    """
    # v2.0.2: filter ONCE and keep the records alongside their deviations.
    # Previously `devs` was filtered while the per-record lists below were not,
    # so any site with a missing deviation misaligned the index used to pick
    # the worst site, and made the weighted average raise on a length mismatch.
    kept = [r for r in anion_results if r.get('anion_deviation') is not None]
    vals = [r['anion_deviation'] for r in kept]
    if not vals:
        # Every key the normal return provides, so a structure with no usable
        # anion sites yields the same columns rather than absent ones.
        return {'n_anion_sites': 0,
                'anion_bvs_mean': None, 'anion_bvs_mean_weighted': None,
                'anion_deviation_mean': None, 'anion_deviation_mean_weighted': None,
                'anion_deviation_max': None, 'anion_deviation_range': None,
                'anion_worst_label': None, 'anion_worst_cn': None,
                'anion_worst_multiplicity': None, 'anion_worst_ligands': None,
                'anion_worst_bvs': None,
                'anion_balances': None, 'bvs_diagnosis': None}

    devs = np.array(vals)
    balances = bool(np.all(np.abs(devs) <= ANION_CONSISTENT_TOL))

    diagnosis = None
    if cation_gii is not None:
        if cation_gii > 0.30 and balances:
            diagnosis = 'parameter_offset'
        elif cation_gii > 0.30 and not balances:
            diagnosis = 'structure_or_data_problem'
        elif cation_gii <= 0.30 and balances:
            diagnosis = 'consistent'
        else:
            diagnosis = 'check_anion'

    # v2.0.2: multiplicity-weighted mean alongside the unweighted one. The
    # unweighted mean treats every crystallographic site as one contributor
    # regardless of how many atoms sit on it; for a pyrochlore that gives the
    # 8-fold oxygen the same weight as the 48-fold one.
    _bvs = [r['anion_bvs'] for r in kept]
    _mult = [r.get('anion_multiplicity') for r in kept]
    if _mult and all(m for m in _mult) and len(_mult) == len(devs):
        _w = np.array(_mult, float)
        _bvs_w = round(float(np.average(_bvs, weights=_w)), 4)
        _dev_w = round(float(np.average(devs, weights=_w)), 4)
    else:
        _bvs_w = _dev_w = None

    # v2.0.2: which site deviates most, and what coordinates it. A deviation
    # with no site attached cannot be interpreted; with the ligand composition
    # it usually can.
    _worst = kept[int(np.argmax(np.abs(devs)))]

    return {
        'n_anion_sites': len(devs),
        'anion_bvs_mean': round(float(np.mean(_bvs)), 4),
        'anion_bvs_mean_weighted': _bvs_w,
        'anion_deviation_mean': round(float(devs.mean()), 4),
        'anion_deviation_mean_weighted': _dev_w,
        'anion_deviation_max': round(float(devs[np.argmax(np.abs(devs))]), 4),
        'anion_deviation_range': round(float(devs.max() - devs.min()), 4),
        'anion_worst_label': _worst.get('anion_label'),
        'anion_worst_cn': _worst.get('anion_cn'),
        'anion_worst_multiplicity': _worst.get('anion_multiplicity'),
        'anion_worst_ligands': _worst.get('anion_ligands'),
        'anion_worst_bvs': _worst.get('anion_bvs'),
        'anion_balances': balances,
        'bvs_diagnosis': diagnosis,
    }


print("Anion-centred BVS loaded: calculate_anion_bvs(), summarise_anion_bvs()")


# =============================================================================
# SELF-TEST
#
# Paste into a NEW cell below and run. It writes two small CIFs to a temporary
# directory, so it needs no files from you and leaves nothing behind.
#
# Tests 1 and 3 are parameter-independent: they check the crystallography, and
# they must pass. Test 2 prints a number that depends on your R0 and b values.
# =============================================================================

def _test_anion_bvs():
    import os, tempfile, textwrap

    nacro2 = textwrap.dedent("""\
        data_test_NaCrO2
        _cell_length_a 2.9719
        _cell_length_b 2.9719
        _cell_length_c 15.9710
        _cell_angle_alpha 90.
        _cell_angle_beta 90.
        _cell_angle_gamma 120.
        _space_group_name_H-M_alt 'R -3 m H'
        _space_group_IT_number 166
        loop_
        _atom_site_label
        _atom_site_type_symbol
        _atom_site_fract_x
        _atom_site_fract_y
        _atom_site_fract_z
        _atom_site_occupancy
        Na1 Na1+ 0. 0. 0. 1.
        Cr1 Cr3+ 0. 0. 0.5 1.
        O1 O2- 0. 0. 0.2325 1.
        """)

    # Mn and Ni deliberately share (0,0,0), as in ICSD 208051.
    shared = textwrap.dedent("""\
        data_test_shared_site
        _cell_length_a 2.8810
        _cell_length_b 2.8810
        _cell_length_c 11.2060
        _cell_angle_alpha 90.
        _cell_angle_beta 90.
        _cell_angle_gamma 120.
        _space_group_name_H-M_alt 'P 63/m m c'
        _space_group_IT_number 194
        loop_
        _atom_site_label
        _atom_site_type_symbol
        _atom_site_fract_x
        _atom_site_fract_y
        _atom_site_fract_z
        _atom_site_occupancy
        Na1 Na1+ 0.6667 0.3333 0.25 0.35
        Mn1 Mn3+ 0. 0. 0. 0.667
        Ni1 Ni3+ 0. 0. 0. 0.333
        O1 O2- 0.6667 0.3333 0.0905 1.
        """)

    d = tempfile.mkdtemp()
    pa = os.path.join(d, 'nacro2.cif'); open(pa, 'w').write(nacro2)
    pb = os.path.join(d, 'shared.cif'); open(pb, 'w').write(shared)

    ok = True

    print("TEST 1 - symmetry expansion and duplicate merging (NaCrO2)")
    r1 = calculate_anion_bvs(pa, {'Na': 1, 'Cr': 3, 'O': -2})
    if not r1:
        print("   FAIL: no anion sites returned"); return False
    a = r1[0]
    n_cr = a['anion_contributions'].count('Cr(')
    n_na = a['anion_contributions'].count('Na(')
    print(f"   oxygen CN = {a['anion_cn']}  ({n_cr} Cr + {n_na} Na)   expected 6 (3 + 3)")
    if not (a['anion_cn'] == 6 and n_cr == 3 and n_na == 3):
        print("   FAIL"); ok = False
    else:
        print("   pass")

    print("\nTEST 2 - anion sum (depends on your R0 and b values)")
    print(f"   anion BVS = {a['anion_bvs']:.3f}, expected 2.000, deviation {a['anion_deviation']:+.3f}")
    if 1.90 <= a['anion_bvs'] <= 2.05:
        print("   pass - consistent with Gagne & Hawthorne parameters")
    else:
        print("   NOTE - outside 1.90-2.05. Not necessarily wrong, but check")
        print("          which parameter set the database is supplying.")

    print("\nTEST 3 - mixed-occupancy site (Mn and Ni both at 0,0,0)")
    r2 = calculate_anion_bvs(pb, {'Na': 1, 'Mn': 3, 'Ni': 3, 'O': -2})
    if not r2:
        print("   FAIL: no anion sites returned"); return False
    c = r2[0]['anion_contributions'] or ''
    has_mn, has_ni = 'Mn(' in c, 'Ni(' in c
    print(f"   Mn present: {has_mn}   Ni present: {has_ni}   (both must be True)")
    if not (has_mn and has_ni):
        print("   FAIL - the minority species is being deleted by deduplication"); ok = False
    else:
        print("   pass")

    print("\nTEST 4 - occupancy weighting is active")
    w, u = r2[0]['anion_bvs'], r2[0]['anion_bvs_unweighted']
    print(f"   weighted {w:.3f}  vs unweighted {u:.3f}")
    if u > w * 1.05:
        print("   pass")
    else:
        print("   FAIL - weighting does not appear to be applied"); ok = False

    print("\n" + ("ALL STRUCTURAL TESTS PASSED." if ok else "SOMETHING FAILED - see above."))
    return ok


_test_anion_bvs()

In [ ]:
# CELL 4: ANION BVS WIRING   (Eir 2.0.13)
#
# Hands the anion-centred engine the same valences the cation sums used, so
# the two sides of every bond are computed under one set of assumptions.

# Default valences for the species treated as anions before anything is known
# about the structure. Derived from the CELL 1 tables rather than typed out:
# this was a fifth hand-written copy of the same nine species, after
# _ANION_SET, ANION_ELEMENTS, the set inside extract_anion_parameter, and
# PROVISIONAL_ANIONS itself.
DEFAULT_ANION_VALENCES = {el: v for el, v in
                          {**ALWAYS_ANION, **AMPHOTERIC_ANION}.items()
                          if el in PROVISIONAL_ANIONS}


def anion_bvs_for_structure(cif_path, sites, cation_gii=None):
    """
    Run the anion-centred BVS for one structure, using the same valences that
    were reported for its cation sites so the two sums are directly
    comparable.

    The map is keyed by SITE LABEL and by element. Keying it by element alone
    silently gave every site of an element whichever valence appeared last in
    the site list: in Co3O4 the 8a Co(2+) and the 16d Co(3+) were then summed
    with a single R0. That affects 147 of 1881 spinels, 10 of 472 pyrochlores
    and 2 of 938 perovskites - the mixed-valence entries, which are exactly
    the ones an anion-side check is most useful for.
    """
    valences = {}
    _conflict = {}
    for s in sites:
        el, lab, v = s.get('element'), s.get('site_label'), s.get('valence')
        if v is None:
            continue
        if lab:
            valences[lab] = v
        if el:
            if el in valences and valences[el] != v:
                _conflict.setdefault(el, set()).add(valences[el])
                _conflict[el].add(v)
            valences.setdefault(el, v)
    if _conflict:
        # Reported, because the element-level fallback is now ambiguous for
        # these and only the site labels disambiguate them.
        for _el, _vs in _conflict.items():
            print(f'    [{_el} appears at {sorted(_vs)}; anion sums use the '
                  f'per-site valence where the label matches]')
    for el, dv in DEFAULT_ANION_VALENCES.items():
        valences.setdefault(el, dv)

    try:
        res = calculate_anion_bvs(cif_path, valences)
    except (NameError, AttributeError, TypeError, IndexError, KeyError):
        # A defect in the engine, not a property of the CIF. Swallowed, it
        # returns an empty anion result that reads as 'this structure has no
        # anion sites'.
        raise
    except Exception as e:
        print(f'    (anion BVS failed: {type(e).__name__}: {e})')
        res = []

    return res, summarise_anion_bvs(res, cation_gii=cation_gii)


print("anion_bvs_for_structure() ready")


In [ ]:
# CELL 5: GII SELF-TEST   (Eir 2.0.13)
#
# Four hand-calculated cases. No CIF files, no database, no I/O.
#
# The three indices are different quantities and the arithmetic is set out
# below, so a failure says which one moved:
#
#   GII   = sqrt( mean( d_i^2 ) )                  d_i = BVS_i - V_i
#   WAGII = sqrt( sum(occ_i * d_i^2) / sum(occ_i) )
#   bvs_discrepancy = sum(|d_i|) / sum(V_i)        pre-1.7.5, kept only for
#                                                  comparison with old output
#
# v2.0.13: a FAILING test now raises. It used to print 'FAIL - check the
# paste' and return False, with nothing reading the return value, so a
# broken instability index would have run a whole batch and reported
# numbers. That is the fault VALIDATE_SHELL had when bv_shell_cn was
# deleted under three passing checks.
#
# v2.0.13: case 3 added. WAGII exists for sites SHARED between species and
# no case tested one - every site in the old fixture held a single element.
# Case 4 covers zero occupancy, which must be excluded outright rather than
# weighted to nearly nothing.


def _test_gii():
    """Hand-calculated GII cases. Raises AssertionError on any mismatch."""
    problems = []

    def _check(name, got, want, tol=5e-4):
        if got is None:
            problems.append(f'{name}: None (expected {want:.4f})')
        elif abs(got - want) > tol:
            problems.append(f'{name}: {got:.4f} (expected {want:.4f})')

    # ---- cases 1 and 2: three ordered sites ----------------------------
    # deviations by hand: Na1 +0.15, Cr1 -0.15, Na2 -0.10 v.u.
    #   GII  = sqrt((0.0225 + 0.0225 + 0.0100)/3)          = 0.13540
    #   disc = (0.15 + 0.15 + 0.10) / (1 + 3 + 1)          = 0.08000
    #   WAGII at occ 0.5/1.0/0.5:
    #        = sqrt((0.5*0.0225 + 0.0225 + 0.5*0.0100)/2)  = 0.13919
    fake = {'test_block': {
        (0.0, 0.0, 0.00): {'elements': ['Na'], 'labels': ['Na1'],
                           'element_data': {'Na': {'valence': 1, 'bvs': 1.15}}},
        (0.0, 0.0, 0.50): {'elements': ['Cr'], 'labels': ['Cr1'],
                           'element_data': {'Cr': {'valence': 3, 'bvs': 2.85}}},
        (0.0, 0.0, 0.25): {'elements': ['Na'], 'labels': ['Na2'],
                           'element_data': {'Na': {'valence': 1, 'bvs': 0.90}}},
    }}
    full = {'Na1': {'occupancy': 1.0}, 'Cr1': {'occupancy': 1.0},
            'Na2': {'occupancy': 1.0}}
    r1 = calculate_global_instability_index(
        fake, site_occupancies=full)['test_block']
    _check('case 1 GII', r1.get('gii'), 0.1354)
    _check('case 1 bvs_discrepancy', r1.get('bvs_discrepancy'), 0.0800)
    if r1.get('wagii') is None or abs(r1['wagii'] - r1['gii']) > 1e-12:
        problems.append('case 1: WAGII must equal GII exactly when every '
                        'occupancy is 1.0')

    part = {'Na1': {'occupancy': 0.5}, 'Cr1': {'occupancy': 1.0},
            'Na2': {'occupancy': 0.5}}
    r2 = calculate_global_instability_index(
        fake, site_occupancies=part)['test_block']
    _check('case 2 GII', r2.get('gii'), 0.1354)
    _check('case 2 WAGII', r2.get('wagii'), 0.1392)

    # ---- case 3: one site shared between two species -------------------
    # This is what WAGII is FOR. A 5% Mn impurity on a 95% Fe site is asked
    # what its sum would be in a cavity the Fe sizes, and answers with
    # something impossible; unweighted, that phantom carries the same
    # weight as the majority it sits among.
    #   d(Fe) = -0.10, d(Mn) = +2.00
    #   GII   = sqrt((0.0100 + 4.0000)/2)                  = 1.41598
    #   WAGII = sqrt((0.95*0.0100 + 0.05*4.0000)/1.00)     = 0.45771
    shared = {'test_block': {
        (0.0, 0.0, 0.0): {'elements': ['Fe', 'Mn'], 'labels': ['Fe1', 'Mn1'],
                          'element_data': {
                              'Fe': {'valence': 3, 'bvs': 2.90},
                              'Mn': {'valence': 2, 'bvs': 4.00}}},
    }}
    occ3 = {'Fe1': {'occupancy': 0.95}, 'Mn1': {'occupancy': 0.05}}
    r3 = calculate_global_instability_index(
        shared, site_occupancies=occ3)['test_block']
    _check('case 3 GII', r3.get('gii'), 1.4160)
    _check('case 3 WAGII', r3.get('wagii'), 0.4577)
    if r3.get('gii') and r3.get('wagii') and r3['gii'] <= r3['wagii']:
        problems.append('case 3: GII should exceed WAGII on a shared site; '
                        'if it does not, the occupancy weighting is not '
                        'reaching the minority species')

    # ---- case 4: a site refined to zero occupancy ----------------------
    # There is no atom there, so its sum describes an atom that does not
    # exist. It must be EXCLUDED, not weighted to nearly nothing.
    zero = {'test_block': {
        (0.0, 0.0, 0.0): {'elements': ['Na'], 'labels': ['Na1'],
                          'element_data': {'Na': {'valence': 1, 'bvs': 1.15}}},
        (0.0, 0.0, 0.5): {'elements': ['Yb'], 'labels': ['Yb1'],
                          'element_data': {'Yb': {'valence': 3, 'bvs': 6.87}}},
    }}
    occ4 = {'Na1': {'occupancy': 1.0}, 'Yb1': {'occupancy': 0.0}}
    r4 = calculate_global_instability_index(
        zero, site_occupancies=occ4)['test_block']
    _check('case 4 GII (absent atom excluded)', r4.get('gii'), 0.1500)
    if r4.get('n_sites_zero_occupancy') != 1:
        problems.append('case 4: n_sites_zero_occupancy '
                        f"{r4.get('n_sites_zero_occupancy')} (expected 1)")

    # ---- verdict -------------------------------------------------------
    print('GII SELF-TEST')
    print(f"  case 1  ordered, occ 1.0    GII {r1.get('gii'):.4f}  "
          f"WAGII {r1.get('wagii'):.4f}  disc {r1.get('bvs_discrepancy'):.4f}")
    print(f"  case 2  ordered, occ 0.5    GII {r2.get('gii'):.4f}  "
          f"WAGII {r2.get('wagii'):.4f}")
    print(f"  case 3  shared site 95:5    GII {r3.get('gii'):.4f}  "
          f"WAGII {r3.get('wagii'):.4f}   <- why WAGII exists")
    print(f"  case 4  one site at occ 0   GII {r4.get('gii'):.4f}  "
          f"({r4.get('n_sites_zero_occupancy')} absent atom excluded)")
    if problems:
        raise AssertionError('GII self-test FAILED:\n  - '
                             + '\n  - '.join(problems))
    print('  PASSED')
    return True


_test_gii()


In [ ]:
# CELL 6: BUILT-IN VALIDATION SUITE   (Eir 2.0.15)
# =============================================================================
# Reference structures with analytically known answers, generated on the fly.
#
# The point is to test the machinery - CIF parsing, symmetry expansion,
# neighbour finding, coordination assignment, the bond valence sum, the GII
# formula, occupancy weighting - independently of whether any particular R0 or
# b is the right one.
#
# The tests are therefore SELF-CALIBRATING. Rather than hardcoding a lattice
# parameter and asserting a GII, each test reads the parameters Eir will
# actually use, solves for the lattice parameter that must give the target, and
# then checks that Eir returns it. A test that hardcoded numbers would break
# every time the parameter file was updated, and would be testing the
# parameters rather than the code.
#
# Run VALIDATE() after any change to the analysis path.

import contextlib
import io
import math
import os
import tempfile

# Rocksalt, Fm-3m. One cation site, one anion site, both octahedral, and the
# cation-anion distance is exactly a/2. Everything about it is known by hand.
_ROCKSALT_CIF = """data_eir_validation
_cell_length_a {a:.6f}
_cell_length_b {a:.6f}
_cell_length_c {a:.6f}
_cell_angle_alpha 90.
_cell_angle_beta 90.
_cell_angle_gamma 90.
_space_group_IT_number 225
_symmetry_space_group_name_H-M 'F m -3 m'
loop_
_atom_site_label
_atom_site_type_symbol
_atom_site_symmetry_multiplicity
_atom_site_Wyckoff_symbol
_atom_site_fract_x
_atom_site_fract_y
_atom_site_fract_z
_atom_site_B_iso_or_equiv
_atom_site_occupancy
{cat}1 {cat}{cv}+ 4 a 0. 0. 0. 0.5 1.
O1 O2- 4 b 0.5 0.5 0.5 0.5 {oocc:.4f}
"""

# The same structure as ICSD writes it when the depositor leaves the oxidation
# state unset: an _atom_type_oxidation_number loop of zeros, and type symbols
# reading "M0+" and "O0-". ICSD 206948 (NaCl) and 616052 (BaS) are real
# examples. Taking those zeros literally makes the bond valence sum look up
# parameters for M(0+), find none, and return no result at all - no bond
# length, no radius, no sum - for a structure that is otherwise unremarkable.
# On the 613-structure rock salt survey this silently removed 73 sites in 40
# ordinary compounds, which is enough to move a published validation rate.
_ROCKSALT_ZERO_VALENCE_CIF = """data_eir_validation_zero
_cell_length_a {a:.6f}
_cell_length_b {a:.6f}
_cell_length_c {a:.6f}
_cell_angle_alpha 90.
_cell_angle_beta 90.
_cell_angle_gamma 90.
_space_group_IT_number 225
_symmetry_space_group_name_H-M 'F m -3 m'
loop_
_atom_type_symbol
_atom_type_oxidation_number
{cat}0+ 0
O0+ 0
loop_
_atom_site_label
_atom_site_type_symbol
_atom_site_symmetry_multiplicity
_atom_site_Wyckoff_symbol
_atom_site_fract_x
_atom_site_fract_y
_atom_site_fract_z
_atom_site_B_iso_or_equiv
_atom_site_occupancy
{cat}1 {cat}0+ 4 a 0. 0. 0. 0.5 1.
O1 O0+ 4 b 0.5 0.5 0.5 0.5 1.
"""

import gemmi

VALIDATION_CATION = 'Mg'
VALIDATION_VALENCE = 2


def _lattice_for_bvs(target_bvs, R0, b, cn=6):
    """Cubic a that puts a rocksalt cation at exactly `target_bvs` v.u."""
    return 2.0 * (R0 - b * math.log(target_bvs / cn))


def _run_cif(text):
    """
    Write a CIF to a temporary file, analyse it, return the results dict.

    process_single_cif is deliberately chatty, which is right when you are
    inspecting one structure and wrong here, so its output is swallowed. If a
    test fails and you want to see why, call process_single_cif yourself on the
    same CIF text.
    """
    global PY3DMOL_AVAILABLE
    d = tempfile.mkdtemp()
    p = os.path.join(d, 'eir_validation.cif')
    # Several reference structures deviate by more than 0.30 v.u. by design,
    # which is exactly the condition that triggers a 3D view. Those render
    # through IPython's display machinery rather than stdout, so redirecting
    # stdout alone does not stop them and the test output fills with
    # <py3Dmol.view ...> objects. Turn the viewer off for the duration.
    _viz_was = globals().get('PY3DMOL_AVAILABLE', False)
    try:
        PY3DMOL_AVAILABLE = False
        with open(p, 'w', encoding='utf-8') as f:
            f.write(text)
        with contextlib.redirect_stdout(io.StringIO()):
            return process_single_cif(p)
    finally:
        PY3DMOL_AVAILABLE = _viz_was
        try:
            os.unlink(p)
            os.rmdir(d)
        except OSError:
            pass


def VALIDATE(verbose=True, tol_gii=0.002, tol_len=0.001):
    """
    Run the built-in reference tests. Returns True if all pass.

    Each test states what it is checking and why, so a failure points at the
    part of the pipeline that broke rather than just reporting a number.
    """
    cat, cv = VALIDATION_CATION, VALIDATION_VALENCE
    results = []

    def report(name, ok, detail):
        results.append(ok)
        if verbose:
            print(f"  {'PASS' if ok else 'FAIL'}  {name}")
            print(f"        {detail}")

    R0, b = db_manager.get_bv_parameters(cat, cv, 'O')
    if R0 is None:
        print(f"  Cannot validate: no bond valence parameters for "
              f"{cat}{cv}+-O in the database.")
        return False

    if verbose:
        print("=" * 70)
        print("EIR VALIDATION SUITE")
        print("=" * 70)
        print(f"  Reference structure : rocksalt {cat}O, Fm-3m")
        print(f"  Parameters in use   : R0 = {R0}, b = {b}")
        print(f"  Targets are solved from those parameters, so this tests the")
        print(f"  code and not the parameter values.")
        print()

    # -- 1. geometry and coordination, no parameters involved ----------------
    a = 4.20
    r = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
    if not r.get('success'):
        report("1. geometry and coordination", False,
               f"analysis failed: {r.get('error')}")
        # Deliberate early exit: every test below reads geometry, so running
        # them after this one fails produces ten more failures that all say
        # the same thing.
        raise AssertionError(
            "Eir validation stopped at test 1: the reference rocksalt could "
            f"not be analysed ({r.get('error')}). Nothing below it can be "
            "meaningful.")
    site = next((s for s in r['sites'] if s.get('element') == cat), None)
    ok = (site is not None and site.get('cn') == 6
          and abs(site.get('avg_bond_length', 0) - a / 2) < tol_len)
    report("1. geometry and coordination",
           ok,
           f"a = {a}, so every {cat}-O bond must be exactly {a/2:.4f} A with CN 6. "
           f"Got CN {site.get('cn') if site else None}, "
           f"d = {site.get('avg_bond_length') if site else None}")

    # -- 2-4. GII at known values -------------------------------------------
    for target in (0.000, 0.200, 0.400):
        a = _lattice_for_bvs(cv + target, R0, b)
        r = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
        if not r.get('success'):
            report(f"{len(results)+1}. GII = {target:.3f}", False,
                   f"analysis failed: {r.get('error')}")
            continue
        got = r.get('gii')
        ok = got is not None and abs(got - target) < tol_gii
        report(f"{len(results)+1}. GII = {target:.3f}",
               ok,
               f"a = {a:.5f} A puts the cation at {cv + target:.3f} v.u., so GII "
               f"must be {target:.3f}. Got {got:.5f}" if got is not None
               else "no GII returned")

    # -- 5. GII is the RMS form, not the old normalised mean ----------------
    target = 0.400
    a = _lattice_for_bvs(cv + target, R0, b)
    r = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
    gii, legacy = r.get('gii'), r.get('bvs_discrepancy')
    ok = (gii is not None and legacy is not None
          and abs(gii - target) < tol_gii
          and abs(legacy - target / cv) < tol_gii)
    report("5. GII uses the published RMS definition",
           ok,
           f"one site off by {target:.3f} v.u. on a formal valence of {cv}: the "
           f"RMS form gives {target:.3f}, the older normalised form gives "
           f"{target/cv:.3f}. Got gii={gii}, bvs_discrepancy={legacy}")

    # -- 6. the sum rule ----------------------------------------------------
    # Every bond contributes once to a cation sum and once to an anion sum, so
    # the totals must agree. This checks that the cation and anion engines see
    # the same bonds. In rocksalt both sites have multiplicity 4.
    a = _lattice_for_bvs(cv, R0, b)
    r = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
    site = next((s for s in r['sites'] if s.get('element') == cat), None)
    anion = (r.get('anion_summary') or {}).get('anion_bvs_mean')
    ok = (site is not None and anion is not None
          and abs(site['bvs'] - anion) < 0.005)
    report("6. cation and anion sums agree",
           ok,
           f"equal multiplicities, so the cation total must equal the anion "
           f"total. Cation {site['bvs'] if site else None}, "
           f"anion {anion}")

    # -- 7. occupancy weighting --------------------------------------------
    # Halving the anion occupancy must halve the bond valence sum. This is the
    # A8 behaviour and it is exact, not approximate.
    a = _lattice_for_bvs(cv, R0, b)
    full = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
    half = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=0.5))
    sf = next((s for s in full['sites'] if s.get('element') == cat), None)
    sh = next((s for s in half['sites'] if s.get('element') == cat), None)
    ok = (sf and sh and sf.get('bvs') and sh.get('bvs')
          and abs(sh['bvs'] - 0.5 * sf['bvs']) < 0.005)
    report("7. occupancy weighting is exact",
           ok,
           f"halving the O occupancy must halve the sum. Full {sf.get('bvs') if sf else None}, "
           f"half {sh.get('bvs') if sh else None}, "
           f"expected {0.5*sf['bvs']:.4f}" if sf and sf.get('bvs') else "no result")

    # -- 8. GII is the RMS of the per-site deviations -----------------------
    # Recompute GII from the sites Eir itself reported. This catches a GII that
    # has drifted away from the sites it is supposed to summarise.
    a = _lattice_for_bvs(cv + 0.35, R0, b)
    r = _run_cif(_ROCKSALT_CIF.format(a=a, cat=cat, cv=cv, oocc=1.0))
    devs = [s['bvs'] - abs(s['valence'])
            for s in r.get('sites', [])
            if s.get('bvs') is not None and s.get('valence')]
    recomputed = math.sqrt(sum(d * d for d in devs) / len(devs)) if devs else None
    ok = (recomputed is not None and r.get('gii') is not None
          and abs(recomputed - r['gii']) < 0.002)
    report("8. GII matches its own per-site output",
           ok,
           f"RMS of the reported deviations = {recomputed:.5f}, "
           f"reported GII = {r.get('gii')}" if recomputed is not None
           else "no sites returned")

    # -- 9. the candidate search radius is wide enough ----------------------
    # The first shell in a compact rocksalt sits at ~2.1 A, well inside any
    # plausible cutoff, so tests 1-8 pass even with a badly truncated search.
    # This one puts the bonds at 3.8 A deliberately. The lattice is
    # unphysically large, which does not matter: the point is to check that
    # the neighbour search reaches far enough to find a real coordination
    # shell. A cutoff of 3.5 A - the value that was wrong for 122 sites in the
    # layered-oxide survey - fails here.
    a_wide = 7.60
    r = _run_cif(_ROCKSALT_CIF.format(a=a_wide, cat=cat, cv=cv, oocc=1.0))
    site = (next((s for s in r.get('sites', []) if s.get('element') == cat), None)
            if r.get('success') else None)
    ok = site is not None and site.get('cn') == 6
    report("9. neighbour search reaches a distant shell",
           ok,
           f"a = {a_wide} puts every {cat}-O bond at {a_wide/2:.2f} A. CN must "
           f"still be 6. Got "
           f"{site.get('cn') if site else 'no ' + cat + ' site returned'}")

    # -- 10. a stated valence of zero must not be believed -----------------
    # This is a regression guard, not a property of bond valence theory. It
    # exists because a correct fix - letting per-atom CIF valences reach the
    # calculation - had the side effect of believing the zeros, and that
    # stopped Eir reproducing its own published rock salt analysis. The
    # failure was silent: coordination numbers still appeared, so nothing
    # looked wrong until the validation rate was compared against the paper.
    a = _lattice_for_bvs(cv, R0, b)
    r = _run_cif(_ROCKSALT_ZERO_VALENCE_CIF.format(a=a, cat=cat))
    site = (next((s for s in r.get('sites', []) if s.get('element') == cat), None)
            if r.get('success') else None)
    ok = (site is not None
          and site.get('valence') not in (None, 0, 0.0)
          and site.get('bvs') is not None)
    report("10. a CIF valence of zero is not taken literally",
           ok,
           f"the CIF declares {cat}0+ and O0+, which is how ICSD writes an "
           f"unset oxidation state. The valence must fall back to the "
           f"database, not stay at zero. Got valence="
           f"{site.get('valence') if site else None}, "
           f"bvs={site.get('bvs') if site else None}, "
           f"source={site.get('valence_source') if site else None}")

    # -- 11. the same thing by the route the BATCH actually takes -----------
    # Test 10 calls process_single_cif directly, with no oxidation-state
    # argument. batch_process_directory does something different: it reads the
    # CIF's own oxidation-state loop and passes it in as an override, which
    # outranks the database fallback. A file declaring zeros therefore reaches
    # the calculation as a user override of zero.
    #
    # This distinction is not academic. Test 10 passed for a full round while
    # every batch run still returned no bond valence sums at all, because the
    # two paths supply different inputs to the same function. A test that only
    # exercises the convenient path is not testing what anyone runs.
    _zero_cif = _ROCKSALT_ZERO_VALENCE_CIF.format(a=a, cat=cat)
    _d = tempfile.mkdtemp()
    _p = os.path.join(_d, 'eir_validation_batchpath.cif')
    try:
        with open(_p, 'w', encoding='utf-8') as _f:
            _f.write(_zero_cif)
        _viz_was = globals().get('PY3DMOL_AVAILABLE', False)
        try:
            PY3DMOL_AVAILABLE = False
            with contextlib.redirect_stdout(io.StringIO()):
                _ox, _src, _t7 = get_oxidation_states_for_analysis(_p, '', None)
                r2 = process_single_cif(_p, _ox)
        finally:
            PY3DMOL_AVAILABLE = _viz_was
    finally:
        try:
            os.unlink(_p)
            os.rmdir(_d)
        except OSError:
            pass

    site2 = (next((s for s in r2.get('sites', []) if s.get('element') == cat), None)
             if r2.get('success') else None)
    ok = (site2 is not None
          and site2.get('valence') not in (None, 0, 0.0)
          and site2.get('bvs') is not None)
    report("11. the same, via the batch's oxidation-state override",
           ok,
           f"batch_process_directory passes the CIF's own states in as an "
           f"override; here that is {_ox}. A zero override must not outrank "
           f"the database either. Got valence="
           f"{site2.get('valence') if site2 else None}, "
           f"bvs={site2.get('bvs') if site2 else None}")

    # -- 12. the coordination shell rules ------------------------------------
    # VALIDATE_SHELL has existed since v2.0.3 and nothing called it. That is
    # how bv_shell_cn and econ_weighted_bvs came to be deleted as a side
    # effect of inserting the adaptive shell beside them: syntax valid, no
    # name duplicated, and the only test that exercised them never ran.
    try:
        _shell_ok = VALIDATE_SHELL(verbose=False)
        _shell_detail = 'the shell cases in CELL 1'
    except Exception as _e:
        _shell_ok = False
        _shell_detail = f'{type(_e).__name__}: {_e}'
    report('12. coordination shell rules', bool(_shell_ok), _shell_detail)

    # -- 13. normaliser translations are derived, not assumed ----------------
    # The seven half-lattice vectors were hard-coded until v2.0.6. They are
    # the normaliser translations of Pnma and of nothing else: for Fd-3m the
    # cubic rotations forbid them, which is why the shift screen fired zero
    # times on 472 pyrochlores.
    _pnma = set(normaliser_translations(
        gemmi.find_spacegroup_by_name('P n m a')))
    _seven = {(0.5, 0, 0), (0, 0.5, 0), (0, 0, 0.5), (0.5, 0.5, 0),
              (0.5, 0, 0.5), (0, 0.5, 0.5), (0.5, 0.5, 0.5)}
    _fd3m = normaliser_translations(
        gemmi.find_spacegroup_by_name('F d -3 m'))
    _p1 = normaliser_translations(gemmi.find_spacegroup_by_name('P 1'))
    report('13. normaliser translations',
           _pnma == _seven and _fd3m == [(0.5, 0.5, 0.5)] and _p1 == [],
           f'Pnma must give the seven half-lattice vectors (got {len(_pnma)}), '
           f'Fd-3m only (1/2,1/2,1/2) (got {_fd3m}), and P1 none because it '
           f'admits every translation (got {len(_p1)})')

    # -- 14. the origin-choice offset is derived from the two settings -------
    _off = origin_choice_offset(gemmi.find_spacegroup_by_name('F d -3 m'))
    _none = origin_choice_offset(gemmi.find_spacegroup_by_name('P n m a'))
    report('14. origin-choice offset',
           _off is not None and all(abs(x - 0.375) < 1e-9 for x in _off)
           and _none is None,
           f'Fd-3m has two origins offset by (3/8,3/8,3/8) (got {_off}); '
           f'Pnma has one, so None (got {_none})')

    # -- 15. the two packing floors fire on an impossible cell ---------------
    # Rocksalt at a = 4.20 has anion-anion and cation-cation contacts of
    # a/sqrt(2) = 2.970 A, above both floors. Compressed to a = 3.00 they
    # fall to 2.121 A, below both. Neither floor looks outward from a cation,
    # so this is the one pair of screens a bond valence sum cannot stand in
    # for: ICSD 85235 has a GII of 0.0047 and two oxygens 2.191 A apart.
    _r_ok = _run_cif(_ROCKSALT_CIF.format(a=4.20, cat=cat, cv=cv, oocc=1.0))
    _r_bad = _run_cif(_ROCKSALT_CIF.format(a=3.00, cat=cat, cv=cv, oocc=1.0))
    _ok_aa, _ok_cc = (_r_ok.get('anion_packing_plausible'),
                      _r_ok.get('cation_packing_plausible'))
    _bad_aa, _bad_cc = (_r_bad.get('anion_packing_plausible'),
                        _r_bad.get('cation_packing_plausible'))
    report('15. anion and cation packing floors',
           _ok_aa is True and _ok_cc is True
           and _bad_aa is False and _bad_cc is False,
           f'at a = 4.20 both contacts are 2.970 A and must pass '
           f'(anion {_ok_aa}, cation {_ok_cc}); at a = 3.00 both are '
           f'2.121 A and must fail (anion {_bad_aa}, cation {_bad_cc})')

    # -- 16. measured bond lengths resolve and name their source -------------
    _bl = expected_bond_length(cat, cv, 6, 'O')
    _li = expected_bond_length('Li', 1, 4, 'O')
    report('16. measured bond lengths',
           _bl is not None and _li is not None
           and _li.get('tier') == 1 and _li.get('min') is not None,
           f"{cat}({cv}+) at CN 6 must resolve (source "
           f"{_bl.get('source') if _bl else None}); Li(1+) at CN 4 must be "
           f"tier 1 with a usable floor (tier {_li.get('tier') if _li else None}, "
           f"min {_li.get('min') if _li else None})")
    if verbose:
        print()
        print("=" * 70)
        n_pass = sum(results)
        if all(results):
            print(f"  ALL {len(results)} TESTS PASSED")
        else:
            print(f"  {n_pass} of {len(results)} passed - "
                  f"{len(results)-n_pass} FAILED")
        print("=" * 70)
    if not all(results):
        # v2.0.13: a failing suite RAISES. It used to return False with
        # nothing reading the value, so a broken pipeline printed FAIL and
        # then ran the batch anyway and reported numbers.
        _failed = [i + 1 for i, x in enumerate(results) if not x]
        raise AssertionError(
            f'Eir validation FAILED: test(s) {_failed} of {len(results)}. '
            f'Re-run VALIDATE(verbose=True) for the detail.')
    return True


VALIDATE()


# ---------------------------------------------------------------------------
# v2.0.15, item 4.7: Fd-3m:S against Fd-3m:Z.
#
# Lives here rather than in CELL 1c because normaliser_translations and
# origin_choice_offset are defined in CELL 2, below that cell.
#
# The open question was whether the two origin settings of Fd-3m give the same
# normaliser output. They do. Both return [(0.5, 0.5, 0.5)] and nothing else,
# and origin_choice_offset returns (3/8, 3/8, 3/8) from either. The cache is
# keyed on xhm(), which carries the :1 / :2 suffix, so the two never collide.
#
# The thing that DID need checking is the naming. ICSD writes the origin choice
# as a trailing S or Z - 'F d -3 m S', 'F d -3 m Z' - and gemmi does not
# recognise either as a space group name: find_spacegroup_by_name returns None
# for both. Read on 522 real ICSD files, Eir resolves them correctly anyway,
# S to :1 and Z to :2, 366 and 146 of them respectively, plus I 41/a m d S and
# Z. So the S/Z convention is handled, but by a route that nothing was testing.
# It is tested here now, because it is one silent regression away from reading
# every Z-setting spinel on origin-choice-1 operations.
def VALIDATE_SETTINGS(verbose=True):
    """Assert that the two Fd-3m settings agree, and that S/Z are handled."""
    results = []

    def report(name, ok, detail=''):
        results.append(bool(ok))
        if verbose:
            print(f"  {'pass' if ok else 'FAIL'}  {name}"
                  + (f"   {detail}" if detail else ''))

    if verbose:
        print("\nFd-3m SETTING CHECK (item 4.7)")

    s1 = gemmi.find_spacegroup_by_name('Fd-3m:1')
    s2 = gemmi.find_spacegroup_by_name('Fd-3m:2')
    report('1. both Fd-3m settings resolve', s1 is not None and s2 is not None)
    if s1 is None or s2 is None:
        return False

    t1, t2 = normaliser_translations(s1), normaliser_translations(s2)
    report('2. normaliser output is setting-independent', t1 == t2,
           f'{t1} against {t2}')
    report('3. and is the cubic half-lattice vector alone',
           t1 == [(0.5, 0.5, 0.5)], str(t1))
    report('4. the two settings are distinct groups', s1.xhm() != s2.xhm(),
           f'{s1.xhm()} against {s2.xhm()}')
    o1, o2 = origin_choice_offset(s1), origin_choice_offset(s2)
    report('5. origin offset agrees from either setting', o1 == o2, str(o1))

    # The ICSD convention. If gemmi ever starts accepting these, the CIF
    # cleaner's translation must still be right, so assert the CURRENT state
    # rather than the desired one and let the test fail loudly if it moves.
    sS = gemmi.find_spacegroup_by_name('Fd-3m:S')
    sZ = gemmi.find_spacegroup_by_name('Fd-3m:Z')
    report('6. gemmi does not know the ICSD S/Z names, so Eir must translate',
           sS is None and sZ is None)

    ok = all(results)
    print(f"  {'ALL SETTING TESTS PASSED' if ok else 'SOMETHING FAILED'}"
          f"  ({sum(results)}/{len(results)})")
    if not ok:
        raise AssertionError(
            'Fd-3m setting check failed. A test whose failure has no '
            'consequence is not a test.')
    return ok


VALIDATE_SETTINGS()


In [ ]:
# CELL 7: COORDINATION QUALITY, ECoN AND OBSERVATION  (Eir 2.0.13)
# Post-processor. Reads the batch CSV and the CIFs, writes an enriched CSV.
# Does not modify the anion-BVS pipeline in the cells above.
#
# CHANGED IN 2.0.2
# ----------------
# 1. `_cn_fault` tested only `cn > cn_rel + 0.5`. It therefore caught
#    duplicate symmetry images and never caught an amputated coordination
#    shell. Under-counting is the more damaging direction: it lowers the bond
#    valence sum and so imitates genuine underbonding, whereas over-counting
#    raises it and is usually obvious. On 934 ICSD perovskites the one-sided
#    test flagged 133 of 133 over-counted sites and 0 of 330 under-counted
#    ones, the latter carrying a median BVS deviation of -10.4% against -1.4%
#    elsewhere. The closure is replaced by the symmetric `cn_fault` defined in
#    CELL 2, which also consults geo_cn where present.
#
# 2. A structure-level radii-ratio verdict is now written.
#    `core_geometry_plausible` in the batch CSV is a PER-SITE column, while
#    `structure_bvs_geometry_plausible` and `anion_packing_plausible` are per
#    structure. Anything reading all three the same way is wrong:
#    groupby('filename').first() misses the failure whenever the bad site is
#    not the first row. Eir 2.0.2 exports
#    `structure_core_geometry_plausible` directly; for CSVs written by earlier
#    versions this cell now rolls it up correctly, so downstream code has one
#    column to read either way.
#
# 3. `cn_reference` and `cn_disagreement` are written so that a downstream
#    filter does not have to recompute which coordination estimate won.

# =============================================================================
# A1.2 CHUNK 3 - COORDINATION QUALITY, ECoN, AND OBSERVATIONAL DIAGNOSIS
# =============================================================================

import os, math, glob
import numpy as np
import pandas as pd

try:
    import gemmi
    _HAVE_GEMMI = True
except ImportError:
    _HAVE_GEMMI = False


# --- configuration -----------------------------------------------------------

CN_RELATIVE_CUTOFF   = 1.30
ECON_REGULAR_MIN     = 0.95
IMPLAUSIBLE_FRACTION = 0.85
NEIGHBOUR_SEARCH_MAX = 5.0
ANION_TOLERANCE      = 0.20
CATION_GII_TOLERANCE = 0.30

SHANNON_CN6 = {
    'Li': 0.76, 'Na': 1.02, 'K': 1.38, 'Mg': 0.72, 'Ca': 1.00,
    'Al': 0.535, 'Sc': 0.745, 'Ti': 0.605, 'V': 0.58, 'Cr': 0.615,
    'Mn': 0.645, 'Fe': 0.645, 'Co': 0.61, 'Ni': 0.60, 'Cu': 0.73,
    'Zn': 0.74, 'Ga': 0.62, 'Y': 0.90, 'Zr': 0.72, 'Nb': 0.64,
    'Mo': 0.65, 'Ru': 0.62, 'Rh': 0.665, 'Pd': 0.86, 'Ag': 1.15,
    'Cd': 0.95, 'In': 0.80, 'Sn': 0.69, 'Sb': 0.60, 'Te': 0.56,
    'La': 1.03, 'Ce': 1.01, 'Hf': 0.71, 'Ta': 0.64, 'W': 0.62,
    'Re': 0.63, 'Os': 0.63, 'Ir': 0.68, 'Pt': 0.80, 'Au': 0.85,
    'Bi': 1.03, 'B': 0.27, 'Si': 0.40, 'Ge': 0.53, 'P': 0.38,
}
ANION_RADIUS = {'O': 1.40, 'F': 1.33, 'S': 1.84, 'Cl': 1.81,
                'Se': 1.98, 'Br': 1.96, 'N': 1.46, 'I': 2.20,
                'Te': 2.21}


# --- geometry ----------------------------------------------------------------

def effective_coordination_number(distances, max_iter=25, tol=1e-9):
    """
    Hoppe's effective coordination number.

    Returns (ECoN, weighted_mean_distance). ECoN is fractional: it equals the
    integer CN for a regular polyhedron and falls below it as the polyhedron
    distorts, because long bonds are down-weighted by exp(1-(d/d_av)**6).

    Reference: Hoppe, Z. Kristallogr. 150 (1979) 23.
    """
    ds = sorted(float(d) for d in distances if d and d > 0)
    if not ds:
        return 0.0, float('nan')
    d_av = ds[0]
    for _ in range(max_iter):
        w = [math.exp(1.0 - (d / d_av) ** 6) for d in ds]
        sw = sum(w)
        if sw <= 0:
            break
        new = sum(wi * d for wi, d in zip(w, ds)) / sw
        if abs(new - d_av) < tol:
            d_av = new
            break
        d_av = new
    econ = sum(math.exp(1.0 - (d / d_av) ** 6) for d in ds)
    return econ, d_av


def cn_relative(distances, factor=CN_RELATIVE_CUTOFF):
    """Count contacts within `factor` x the shortest. Distortion-insensitive."""
    ds = sorted(float(d) for d in distances if d and d > 0)
    if not ds:
        return 0
    return sum(1 for d in ds if d <= factor * ds[0])


# ---------------------------------------------------------------------------
# A-SITE HANDLING (resolved in v2.0.2)
# ---------------------------------------------------------------------------
# CELL 8 used to redefine geometry_plausible below this cell, so the version in
# force was not the one written here. That override made the verdict ADVISORY
# (None, "not judged") for A-site elements and for any site with CN >= 8, but
# ONLY when the ratio failed: a passing A-site still passed. A one-sided test,
# so an A-site could never fail the radii screen.
#
# It was introduced when the radius lookup was fixed at six-coordinate, which
# did penalise dispersed high-CN cations. Since v1.8.9 the radius comes from
# get_ionic_radius_fractional() at the site's own valence AND coordination
# number, and the justification no longer holds. Measured over 1886 sites in
# the 934-structure ICSD perovskite set:
#
#                       median ratio   1st pctile   below 0.85
#     A-site elements       0.922        0.765         3.1%
#     B-site and others     0.974        0.763         2.9%
#
# The two distributions are indistinguishable, so there is no basis for
# treating A-sites differently at all. The override is therefore OFF by
# default. What it had been hiding:
#
#     ICSD 220506  LaCrO3      La at 1.008 A   ratio 0.415   BVS +769%
#     ICSD 121777  YVO3         Y at 1.403 A   ratio 0.610   BVS +125%
#     ICSD  74087  NdFeO3      Nd at 1.507 A   ratio 0.632   BVS  +66%
#     ICSD 132626  CaTiO3      Ca at 1.631 A   ratio 0.680   BVS  +63%
#     ICSD  33482  La0.8MnO3   La at 1.658 A   ratio 0.682   BVS +294%
#
# The mechanism is kept and is editable. Set WEAK_CATION_ADVISORY_FLOOR to a value
# below IMPLAUSIBLE_FRACTION to restore a narrow advisory band for weakly bonded
# cations: a ratio between the floor and the threshold returns None, and
# anything below the floor still fails. 0.80 is the defensible choice if you
# want one, since it leaves all five entries above untouched. None disables
# the concession, which is the default.
WEAK_CATION_ADVISORY_FLOOR = None

# What counts as a large, weakly bonded cation.
#
# v2.0.13: this was A_SITE_ELEMENTS, a 25-element list, plus
# A_SITE_CN_THRESHOLD = 8. Both are perovskite vocabulary in a tool that also
# runs on pyrochlores (16c and 16d), spinels (8a and 16d) and rock salts, none
# of which has an A site. A hand-maintained element list is also the same
# failure mode as the four copies of the anion set: it goes stale silently and
# says nothing about why a species is on it.
#
# Lewis acidity, valence / coordination number, is the quantity actually
# meant, it is structure-agnostic, and it is the same measure that sets the
# bond-valence shell boundary (BV_CUT_A, CELL 1). It separates the two
# populations cleanly with no list at all:
#
#     Na(1+) CN12  0.083     Ti(4+) CN6  0.667
#     Ca(2+) CN12  0.167     Fe(3+) CN6  0.500
#     Ba(2+) CN12  0.167     Cr(3+) CN6  0.500
#     La(3+) CN12  0.250     Zn(2+) CN4  0.500
#                            Y(3+)  CN8  0.375
#
# 0.25 is also the top of the band in which Gagne's longest observed bonds
# sit at s = 0.0189, the measurement that set BV_SUM_CUTOFF.
WEAK_CATION_LEWIS_MAX = 0.25


def geometry_plausible(element, anion, d_shortest,
                       fraction=IMPLAUSIBLE_FRACTION,
                       valence=None, cn=None):
    """
    True if the shortest cation-anion contact is credible against Shannon radii.

    Returns (is_plausible, expected_distance, ratio). If no radius can be
    found, returns (None, nan, nan) rather than guessing.

    v1.8.9: the cation radius is taken at the site's own valence and
    coordination number via get_ionic_radius_fractional(). The previous
    SHANNON_CN6 lookup was fixed at six-coordinate and blind to valence,
    which overstated the expected contact for every tetrahedral site by
    7-8% and biased them towards failing. SHANNON_CN6 remains as the
    fallback for combinations absent from the tabulation.
    """
    r_c = None
    if valence is not None and cn:
        try:
            v = float(valence)
            if v == v:                     # not NaN
                r_c, _src = get_ionic_radius_fractional(element, v, int(cn))
        except (TypeError, ValueError):
            r_c = None
    if r_c is None:
        r_c = SHANNON_CN6.get(element)
    r_a = ANION_RADIUS.get(anion)
    if r_c is None or r_a is None or not d_shortest:
        return None, float('nan'), float('nan')
    expected = r_c + r_a
    ratio = d_shortest / expected

    if ratio >= fraction:
        return True, expected, ratio

    # Failing. Only an explicitly configured advisory band may withhold that
    # verdict, and never below its floor. Contrast the old behaviour, which
    # withheld it for every A-site failure however severe.
    if WEAK_CATION_ADVISORY_FLOOR is not None:
        _lewis = None
        try:
            if valence is not None and cn:
                _lewis = abs(float(valence)) / float(cn)
        except (TypeError, ValueError, ZeroDivisionError):
            _lewis = None
        if (_lewis is not None and _lewis <= WEAK_CATION_LEWIS_MAX
                and ratio >= WEAK_CATION_ADVISORY_FLOOR):
            return None, expected, ratio

    return False, expected, ratio


# v2.0.2: an explicit ownership tag. CELL 8 checks this to confirm nothing
# further down the notebook has redefined the function. An attribute survives
# exec, import and %run, whereas inspect.getsource() does not always resolve a
# function defined in a notebook cell, and a check that cannot see its evidence
# would raise a false alarm.
geometry_plausible._eir_cell = 'CELL 7'
geometry_plausible._eir_version = '2.0.2'


# --- CIF loading (mirrors Eir's cleaning: UTF-8 repair + HR symop fix) --------

import re as _re, tempfile as _tempfile

def load_structure(path):
    """
    Read a CIF into a gemmi SmallStructure.

    v2.0.3: uses clean_cif_for_gemmi rather than a local copy of it. CELL 7
    carried its own _clean_cif_text, a hand-copy of the CELL 1 cleaner that had
    since diverged from it. Every coordination-quality column in the output was
    therefore computed from an unrepaired parse, and nine sites across five
    structures came back NaN with nothing said, because the only report of the
    failure sits behind `verbose` in the caller.
    """
    txt = clean_cif_for_gemmi(path)
    if txt is None:
        raise RuntimeError(f'could not clean {os.path.basename(path)}')
    with _tempfile.NamedTemporaryFile('w', suffix='.cif', delete=False,
                                      encoding='utf-8') as t:
        t.write(txt); tp = t.name
    try:
        st = gemmi.make_small_structure_from_block(
            gemmi.cif.read(tp).sole_block())
    finally:
        try:
            os.unlink(tp)
        except OSError:
            pass
    if st.spacegroup is None:
        m = (_re.search(r'_space_group_IT_number\s+(\d+)', txt) or
             _re.search(r'_symmetry_Int_Tables_number\s+(\d+)', txt))
        if m:
            st.spacegroup = gemmi.find_spacegroup_by_number(int(m.group(1)))
    return st


def _expand(st):
    pts = []
    for s in st.sites:
        for op in st.spacegroup.operations():
            f = op.apply_to_xyz([s.fract.x, s.fract.y, s.fract.z])
            for i in (-1, 0, 1):
                for j in (-1, 0, 1):
                    for k in (-1, 0, 1):
                        p = st.cell.orthogonalize(
                            gemmi.Fractional(f[0]+i, f[1]+j, f[2]+k))
                        pts.append((s.element.name, p.x, p.y, p.z))
    return pts


def neighbour_distances(cif_path, rmax=NEIGHBOUR_SEARCH_MAX, dedup_tol=0.02):
    """
    Per asymmetric-unit cation site, the sorted distances to each anion species.

    Neighbours are deduplicated by Cartesian position, because the symmetry
    expansion generates the same physical atom once per operation that maps
    onto it. Without this every distance is counted several times and both CN
    and ECoN come out as multiples of the true value.

    Returns {(frac_coords, element): {anion_element: [distances...]}}
    """
    st = load_structure(cif_path)
    if st.spacegroup is None:
        raise RuntimeError("no space group")
    pts = _expand(st)
    els = np.array([p[0] for p in pts])
    P = np.array([[p[1], p[2], p[3]] for p in pts])

    out = {}
    for s in st.sites:
        el = s.element.name
        if el in ANION_RADIUS:
            continue
        o = s.orth(st.cell)
        p0 = np.array([o.x, o.y, o.z])
        d = np.linalg.norm(P - p0, axis=1)
        m = np.where((d > 0.01) & (d <= rmax))[0]
        m = m[np.argsort(d[m])]

        per_anion = {}
        kept_pos = []
        for idx in m:
            ael = els[idx]
            if ael not in ANION_RADIUS:
                continue
            pos = P[idx]
            if any(np.linalg.norm(pos - kp) <= dedup_tol for kp in kept_pos):
                continue
            kept_pos.append(pos)
            per_anion.setdefault(ael, []).append(float(d[idx]))
        for a in per_anion:
            per_anion[a].sort()

        fkey = (round(s.fract.x, 4), round(s.fract.y, 4), round(s.fract.z, 4))
        out[(fkey, el)] = per_anion
    return out


def _parse_coords(s):
    """
    Fractional coordinates from a CSV cell. None if unparseable.

    Returning None makes the row fail to match its site, so a broadened
    except here loses a site silently rather than loudly.
    """
    try:
        t = tuple(float(x) for x in str(s).strip('() ').split(','))
        return tuple(round(v, 4) for v in t)
    except (TypeError, ValueError, AttributeError):
        return None


def _distances_for_row(table, coords_str, element, tol=0.02):
    """Match a CSV row to its site by fractional coordinates, not by element. Returns per-anion mapping"""
    target = _parse_coords(coords_str)
    cands = [(k, v) for k, v in table.items() if k[1] == element]
    if not cands:
        return {} 
    if target is not None:
        best, bestd = None, 1e9
        for (fkey, _el), per_anion in cands:
            d = sum(min(abs(a - b), 1 - abs(a - b)) ** 2
                    for a, b in zip(fkey, target)) ** 0.5
            if d < bestd:
                best, bestd = per_anion, d
        if best is not None and bestd <= tol:
            return best
    return cands[0][1]


# --- diagnosis ---------------------------------------------------------------

# observational_diagnosis was removed at v2.0.13. It derived the same labels
# by recomputing the anion side from an aggregate deviation, and was
# superseded by observational_diagnosis_from_flags, which takes the verified
# per-structure flag instead. Two functions producing the same column from
# different inputs is how a producer and its consumer drift apart.

def observational_diagnosis_from_flags(cation_gii, anion_balances,
                                       gii_tol=CATION_GII_TOLERANCE):
    """
    Same labels, but takes the verified per-structure `anion_balances` flag
    from chunk 1 rather than recomputing the anion side from an aggregate.
    """
    if cation_gii is None or (isinstance(cation_gii, float) and np.isnan(cation_gii)):
        return 'insufficient_data'
    if anion_balances is None or (isinstance(anion_balances, float) and np.isnan(anion_balances)):
        return 'insufficient_data'
    an_ok = str(anion_balances).strip().lower() in ('true', '1', 'yes')
    cat_off = cation_gii > gii_tol
    if not cat_off and an_ok:
        return 'cation_and_anion_agree'
    if cat_off and an_ok:
        return 'cation_only_deviates'
    if cat_off and not an_ok:
        return 'both_deviate'
    return 'anion_only_deviates'


# --- GII including anion sites, and the refined diagnosis --------------------

def gii_all_sites(cation_deviations, anion_deviations):
    """
    RMS deviation over cations AND anions (the Salinas-Sanchez / Brown form).

    Kept separate from Eir's `gii`, which is cation-only. Pooling the anions
    dilutes the index in exactly the parameter-offset cases the cation/anion
    comparison is designed to detect, so the two are reported side by side
    rather than one replacing the other.
    """
    d = [x for x in list(cation_deviations) + list(anion_deviations)
         if x is not None and not (isinstance(x, float) and np.isnan(x))]
    if not d:
        return float('nan')
    return float(np.sqrt(np.mean(np.square(d))))


def refine_diagnosis(label, any_distorted):
    """
    Split `both_deviate` on whether any cation polyhedron is distorted.

    A regular polyhedron that still deviates points at the structure or the
    data. A distorted one is usually real: a Jahn-Teller active ion has one
    averaged geometry standing in for an elongated environment, and BVS
    handles that badly. Everything else passes through unchanged.
    """
    if label != 'both_deviate':
        return label
    if any_distorted is None:
        return label
    return 'both_deviate_distorted' if any_distorted else 'both_deviate_regular'


# The order the diagnosis labels are reported in: agreement first, then the
# cases that point at the parameters, then those that point at the structure.
# Defined once here and read by the dataset summary in CELL 14. Until v2.0.13
# this list existed twice - here, unread, and again in CELL 14 as
# DIAGNOSIS_ORDER_REPORT, byte-identical.
DIAGNOSIS_ORDER = [
    'cation_and_anion_agree',
    'cation_only_deviates',
    'both_deviate_regular',
    'both_deviate_distorted',
    'both_deviate',
    'anion_only_deviates',
    'insufficient_data',
]


# --- main entry point --------------------------------------------------------

def _require_cn_fault():
    """
    `cn_fault` lives in CELL 2 from Eir 2.0.2 onwards. Fail here, with the
    remedy, rather than three lines later with a bare NameError, or worse,
    silently falling back to a one-sided test that reports no faults at all.
    """
    if 'cn_fault' not in globals():
        raise NameError(
            'cn_fault is not defined. It moved to CELL 2 in Eir 2.0.2; '
            're-run the main cell (CELL 2) before this one. Running with the '
            'old one-sided test would report zero amputated coordination '
            'shells rather than none existing.')


def add_coordination_quality(batch_csv_path, cif_directory,
                             output_csv_path=None, verbose=True):
    """
    Post-process an Eir batch CSV: add ECoN, coordination-quality and
    geometry-plausibility columns, and replace bvs_diagnosis with the
    observational labels.

    Leaves the existing pipeline untouched; reads the CSV, writes a new one.
    """
    _require_cn_fault()
    df = pd.read_csv(batch_csv_path)
    if verbose:
        print(f"Loaded {batch_csv_path}")
        print(f"  {len(df)} rows, {df['filename'].nunique()} structures")

    cache = {}
    econ_v, ratio_v, dav_v, cnrel_v = [], [], [], []
    plaus_v, expd_v, dratio_v, regular_v = [], [], [], []
    canion_v = []

    for _, row in df.iterrows():
        fn = row.get('filename')
        el = row.get('element')
        path = os.path.join(cif_directory, fn) if isinstance(fn, str) else None

        if path and path not in cache:
            try:
                cache[path] = neighbour_distances(path)
            except Exception as e:
                # v2.0.3: not conditional on verbose. A structure that cannot be
                # read here loses every coordination column silently, and the
                # NaNs are indistinguishable from a site that genuinely has no
                # neighbours.
                print(f"  ! coordination quality unavailable for {fn}: "
                      f"{type(e).__name__}: {e}")
                cache[path] = {}

        table = cache.get(path, {})
        per_anion = _distances_for_row(table, row.get('coords'), el)
        ds = [d for dl in per_anion.values() for d in dl]

        if not ds:
            econ_v.append(np.nan); ratio_v.append(np.nan)
            dav_v.append(np.nan);  cnrel_v.append(np.nan)
            plaus_v.append(None);  expd_v.append(np.nan)
            dratio_v.append(np.nan); regular_v.append(None)
            canion_v.append(None)
            continue

        cn_r = cn_relative(ds)
        shell = sorted(ds)[:cn_r] if cn_r else sorted(ds)
        e, dav = effective_coordination_number(shell)

        # v1.8.9: test each anion species against its own radius and keep the
        # worst contact. Assuming oxygen made the screen inoperative for
        # sulfides, where a normal M-S bond looks long against r(O).
        plaus, expd, dr, contact_anion = None, np.nan, np.nan, None
        for _ael, _dl in per_anion.items():
            _p, _ex, _r = geometry_plausible(el, _ael, min(_dl),
                                             valence=row.get('valence'), cn=cn_r)
            if _r == _r and (dr != dr or _r < dr):
                plaus, expd, dr, contact_anion = _p, _ex, _r, _ael

        econ_v.append(round(e, 4))
        cnrel_v.append(cn_r)
        ratio_v.append(round(e / cn_r, 4) if cn_r else np.nan)
        dav_v.append(round(dav, 4))
        plaus_v.append(plaus)
        expd_v.append(round(expd, 3) if expd == expd else np.nan)
        dratio_v.append(round(dr, 4) if dr == dr else np.nan)
        regular_v.append(bool(e / cn_r >= ECON_REGULAR_MIN) if cn_r else None)
        canion_v.append(contact_anion)

    df['econ'] = econ_v
    df['econ_d_av'] = dav_v
    df['cn_relative'] = cnrel_v
    df['econ_cn_ratio'] = ratio_v
    # v2.0.2: symmetric fault detection, from CELL 2. The closure that used
    # to live here tested only the over-counting direction. geo_cn is passed
    # as a second opinion where the batch CSV carries it.
    _geo = (df['geo_cn'] if 'geo_cn' in df.columns else pd.Series([None] * len(df)))
    df['cn_fault'] = [cn_fault(c, r, g) for c, r, g in
                      zip(df['cn'], df['cn_relative'], _geo)]

    # The most generous independent estimate, and how far the bond valence
    # coordination number falls short of it. Written out so a downstream
    # filter does not have to recompute which estimate won.
    _refs = [pd.to_numeric(df[c], errors='coerce')
             for c in ('cn_relative', 'geo_cn') if c in df.columns]
    if _refs:
        df['cn_reference'] = pd.concat(_refs, axis=1).max(axis=1)
        df['cn_disagreement'] = (df['cn_reference']
                                 - pd.to_numeric(df['cn'], errors='coerce'))
    _amp = df['cn_fault'].astype(str).str.startswith('cn_below_shell')
    if _amp.any():
        _n_s = df.loc[_amp, 'filename'].nunique()
        print(f"\n  ! {int(_amp.sum())} sites in {_n_s} structures have a bond "
              f"valence sum computed over an amputated coordination shell.")
        print(f"    The gap analysis cut the shell short of every independent "
              f"estimate, so the sum is low for a reason that is not chemistry.")
        print(f"    Filter on cn_fault before quoting any deviation or "
              f"validation rate. Reported, not repaired: cn_gap_shell holds "
              f"the gap answer and cn the canonical one, so the "
              f"disagreement is visible per site.")
    df['polyhedron_regular'] = regular_v
    df['shortest_contact_ratio'] = dratio_v
    df['expected_contact'] = expd_v
    df['geometry_plausible'] = plaus_v
    df['contact_anion'] = canion_v

    # structure-level plausibility: one bad contact condemns the structure
    bad = (df.groupby('filename')['geometry_plausible']
             .apply(lambda s: (s == False).any()))
    df['structure_geometry_plausible'] = ~df['filename'].map(bad).fillna(False).astype(bool)

    # v2.0.2: the same roll-up for the core radii-ratio screen. Eir 2.0.2
    # exports structure_core_geometry_plausible directly; for a CSV written by
    # an earlier version the column is absent and core_geometry_plausible is
    # per SITE, so it is rolled up here on the same rule. Taking .first()
    # instead, as summarise_dataset did, reads whichever site happens to come
    # first and misses the failure whenever the bad site is not row one.
    if 'structure_core_geometry_plausible' not in df.columns:
        if 'core_geometry_plausible' in df.columns:
            _core = df['core_geometry_plausible']
            if _core.dtype == object:
                _core = _core.astype(str).str.lower().map(
                    {'true': True, '1': True, '1.0': True,
                     'false': False, '0': False, '0.0': False})
            _bad_core = (df.assign(_c=_core).groupby('filename')['_c']
                           .apply(lambda s: (s == False).any()))
            df['structure_core_geometry_plausible'] = ~df['filename'].map(
                _bad_core).fillna(False).astype(bool)
            _nf = int((~df.drop_duplicates('filename')
                       ['structure_core_geometry_plausible']).sum())
            print(f"  rolled up core_geometry_plausible from sites: "
                  f"{_nf} structures flagged on the radii-ratio screen")
        else:
            df['structure_core_geometry_plausible'] = None

    if 'gii' in df.columns and 'anion_balances' in df.columns:
        df['bvs_diagnosis'] = [
            observational_diagnosis_from_flags(g, b)
            for g, b in zip(df['gii'], df['anion_balances'])
        ]

    # refined diagnosis: split both_deviate on polyhedron regularity
    _dist = (df.groupby('filename')['polyhedron_regular']
               .apply(lambda s: (False in set(s.dropna().tolist()))
                      if s.notna().any() else None))
    df['bvs_diagnosis_detail'] = [
        refine_diagnosis(lab, _dist.get(fn))
        for lab, fn in zip(df['bvs_diagnosis'], df['filename'])
    ]

    # GII including anion sites, for literature comparability
    _gii_all = {}
    if 'calculate_anion_bvs' in globals():
        for fn, g in df.groupby('filename'):
            path = os.path.join(cif_directory, fn)
            # v2.0.13: keyed by SITE LABEL as well as element. Keying it by
            # element alone gave every site of an element whichever valence
            # came last, so in Co3O4 the 8a Co(2+) and the 16d Co(3+) were
            # summed with one R0. This is the same fault as in
            # anion_bvs_for_structure, in a second path that calls
            # calculate_anion_bvs directly and so did not inherit that fix.
            # 147 of 1881 spinels carry an element at two valences.
            vals = {}
            for _, rr in g.iterrows():
                _v = rr.get('valence')
                if _v != _v:
                    continue
                if rr.get('site_label'):
                    vals[rr['site_label']] = _v
                if rr.get('element'):
                    vals.setdefault(rr['element'], _v)
            for _ael, _dv in DEFAULT_ANION_VALENCES.items():
                vals.setdefault(_ael, _dv)
            try:
                ares = globals()['calculate_anion_bvs'](path, vals)
                adev = [a.get('anion_deviation') for a in ares]
            except (NameError, AttributeError, TypeError) as _e:
                # A defect in the anion engine, not a property of the CIF.
                raise
            except Exception as _e:
                # gii_all_sites would otherwise be computed over the cations
                # alone and reported under a name that says it includes the
                # anions.
                adev = []
                print(f'  [gii_all_sites for {fn}: anion side unavailable, '
                      f'index is cation-only. {type(_e).__name__}: {_e}]')
            cdev = (g['bvs'] - g['valence']).dropna().tolist()
            _gii_all[fn] = gii_all_sites(cdev, adev)
    # v2.0.16: ONE DEFINITION. This column is computed here and must not
    # arrive already populated with something else. v2.0.15 wrote a different
    # quantity under this name in CELL 2 and this line silently overwrote it,
    # so the collision was invisible until someone read the CSV and found the
    # numbers were not the ones the changelog described.
    #
    # Comparing rather than merely testing presence, so re-running CELL 7 on
    # an already-enriched frame stays legal: what is forbidden is disagreement.
    _incoming = df['filename'].map(_gii_all)
    if 'gii_all_sites' in df.columns:
        _prev = pd.to_numeric(df['gii_all_sites'], errors='coerce')
        _new = pd.to_numeric(_incoming, errors='coerce')
        _both = _prev.notna() & _new.notna()
        if _both.any() and not np.allclose(_prev[_both], _new[_both],
                                           rtol=0, atol=1e-9):
            raise AssertionError(
                'gii_all_sites arrived already populated with different '
                'values. Two definitions of one column name; see v2.0.16.')
    df['gii_all_sites'] = _incoming

    # GII position within this run, since absolute thresholds are
    # family-dependent and 0.20 is only a rule of thumb
    _per_gii = df.groupby('filename')['gii'].first()
    _pct = _per_gii.rank(pct=True) * 100.0
    df['gii_percentile_in_run'] = df['filename'].map(_pct).round(1)

    # self-describing configuration
    df['cfg_cn_relative_cutoff'] = CN_RELATIVE_CUTOFF
    df['cfg_econ_regular_min'] = ECON_REGULAR_MIN
    df['cfg_implausible_fraction'] = IMPLAUSIBLE_FRACTION
    df['cfg_anion_tolerance'] = ANION_TOLERANCE
    df['cfg_cation_gii_tolerance'] = CATION_GII_TOLERANCE
    df['cfg_gii_convention'] = 'cation_only'
    df['cfg_anion_anion_min_A'] = ANION_ANION_MIN
    df['cfg_cation_cation_min_A'] = CATION_CATION_MIN
    df['cfg_bv_sum_cutoff'] = BV_SUM_CUTOFF

    # ------------------------------------------------------------------
    # PGRM INTEGRITY GATES
    #
    # Applied HERE, before the CSV is written, so the file on disk and the
    # frame in memory carry the same columns. Classifying afterwards in the
    # calling cell would give a DataFrame with pgrm_* and a CSV without.
    #
    # Geometry decides severity before any chemistry is read. See
    # classify_pgrm for the four facets and for why the previous scheme -
    # which derived a tier FROM the bond valence sum - could not see the
    # dominant fault class in any family with positional freedom.
    # ------------------------------------------------------------------
    if 'classify_pgrm' in globals():
        df = classify_pgrm(df)
        df = apply_pgrm_suppression(df)
        if verbose:
            _st = df.drop_duplicates('filename')
            _t = _st['structure_pgrm_tier'].value_counts()
            print()
            print('  PGRM integrity gates')
            for _k in ('ACCEPT', 'FLAG', 'NOT_ASSESSED', 'EXPUNGE'):
                if _k in _t:
                    print(f'    {_k:14} {_t[_k]:6}  '
                          f'({100 * _t[_k] / len(_st):.1f}%)')
            _ex = _st[_st['structure_pgrm_tier'] == 'EXPUNGE']
            if len(_ex):
                print('    expunged for:',
                      dict(_ex['structure_pgrm_reason'].value_counts()))
                if not PGRM_SUPPRESS:
                    print('    PGRM_SUPPRESS is False, so their bond valence',
                          'results are still reported. Read pgrm_reason')
                    print('    before quoting any number from them.')
    else:
        print('  [classify_pgrm not defined; no integrity gates applied]')

    if output_csv_path is None:
        base, ext = os.path.splitext(batch_csv_path)
        output_csv_path = f"{base}_quality{ext}"
    df.to_csv(output_csv_path, index=False)

    if verbose:
        per = df.groupby('filename').first()
        print(f"\nWritten: {output_csv_path}")
        n_imp = int((~per['structure_geometry_plausible'].astype(bool)).sum())
        print(f"  implausible-geometry structures: {n_imp} / {len(per)}")
        print(f"\n  bvs_diagnosis:")
        for k, v in df.groupby('filename').first()['bvs_diagnosis'].value_counts().items():
            print(f"    {k:28s} {v}")
        reg = df['polyhedron_regular'].dropna()
        if len(reg):
            print(f"\n  polyhedron_regular: {int(reg.sum())} / {len(reg)} sites")
    return df

In [ ]:
# CELL 7b: PGRM INTEGRITY GATES   (Eir 2.0.13)
# ===========================================================================
# PGRM INTEGRITY GATES
#
# Four facets, asked in order. The first failure stops the assessment.
#
#   P  Parse      can the file be read?
#   G  Geometry   can the atoms be where it says?
#   R  Reference  can we assess it?
#   M  Model      is the depositor's description unambiguous?
#
# The point of the ordering is that a structure whose atoms cannot be where
# the file puts them has nothing to say about its chemistry. Eir previously
# decided severity FROM the bond valence sum, so a broken structure was judged
# by the number it produced: ICSD 220506 has a La-O contact of 1.008 A and
# reported a GII of 17.29 at the top of the triage; ICSD 85235 has two oxygens
# 0.56 A too close and reported a GII of 0.0047, the finest in the perovskite
# set. Both numbers are arithmetic performed on impossible coordinates.
#
# This replaces the six-type taxonomy of Reeves-McLaren (2026) Dalton Trans.
# 55, 2868, and contains it: Type 2 -> R1, Type 3 -> M1, Type 4 -> M3,
# Type 5 -> M2, Type 6A -> M6, Type 7 -> M4. Nothing is lost. What is added is
# the G facet, which that taxonomy had no category for and could not have had:
# it was derived on rock salts, where every atom sits on a special position
# with no free positional parameter, so a coordinate error is impossible by
# construction. Perovskites, pyrochlores and spinels all have positional
# freedom, and geometry faults are the dominant class in all three.
#
# Type 4 is implemented here for the first time. It was declared in 2016 and
# never assigned; the heterovalent test finds 3 pyrochlore and 309 spinel
# sites with a valence span of 2 or more on one Wyckoff position.
#
# PGRM_SUPPRESS controls whether an EXPUNGE actually withholds the chemistry.
# It ships False so the classification can be read for a run before it starts
# nulling columns, the same staging as 4a/4b.
# ===========================================================================
PGRM_SUPPRESS = False
PGRM_PREMODERN_YEAR = 1980
PGRM_HETEROVALENT_DV = 2

# The two packing floors come from CELL 1, where they are set from the
# measured distributions. Re-declaring them here would be a second copy that
# drifts: the structural self-check flagged exactly that when this cell first
# defined CATION_CATION_MIN with a fallback default.
_PGRM_ANION_FLOOR = ANION_ANION_MIN
_PGRM_CATION_FLOOR = CATION_CATION_MIN
PGRM_SUPPRESSED_COLUMNS = (
    'bvs', 'bvs_percent_diff', 'bvs_unweighted', 'bvs_full_shell',
    'bvs_gap_shell', 'bvs_bv_adaptive', 'bvs_econ_weighted',
    'gii', 'wagii', 'gii_site_averaged', 'gii_recommended',
    'bl_delta_mean', 'bl_delta_min', 'bl_ratio_min')

_GATE_TIER = {'P': 'EXPUNGE', 'G': 'EXPUNGE', 'R': 'NOT_ASSESSED', 'M': 'FLAG',
              '-': 'ACCEPT'}

def _pgrm_f(x, default=np.nan):
    try:
        v = float(x)
        return v if v == v else default
    except (TypeError, ValueError):
        return default

def _pgrm_is_false(x):
    return str(x).strip().lower() == 'false'

def classify_site(row, struct):
    """
    Return (code, reason, evidence) for the most severe gate this site fails.
    Facets are asked in order and the first failure stops the assessment:
    a structure whose atoms cannot be where the file says has nothing to say
    about its chemistry.
    """
    # ---- P : can the file be read? -------------------------------------
    # Most P failures never reach this table; a file that cannot be parsed
    # appears in the failed-files CSV instead. What survives to here is a
    # site that parsed but has no position.
    _coords = row.get('coords')
    # A NaN reads as a float here, which is how a missing coordinate arrives
    # from pandas. Both cases mean the same thing: the site parsed but has no
    # position, so nothing below can be computed for it.
    if _coords is None or isinstance(_coords, float) or _coords == '':
        return 'P1', 'no_position', 'site parsed without coordinates'

    # ---- G : can the atoms be where it says? ----------------------------
    if _pgrm_is_false(row.get('contact_above_floor')):
        d = _pgrm_f(row.get('min_bond_length'))
        return ('G1', 'contact_below_physical_floor',
                f'shortest contact {d:.3f} A, floor {CATION_ANION_ABSOLUTE_MIN} A')
    cn = _pgrm_f(row.get('cn'))
    if cn == 0 or str(row.get('cn_fault')) == 'no_shell_found':
        return 'G3', 'no_coordination_shell', 'no neighbour within the search radius'
    if row.get('proposed_shift') not in (None, '') and pd.notna(row.get('proposed_shift')):
        s0, s1 = _pgrm_f(row.get('shift_shell_spread')), _pgrm_f(row.get('proposed_spread'))
        _ev = f'first-shell spread {s0:.2f} -> {s1:.2f} at {row.get("proposed_shift")}'

        # G4 is the only gate that makes a claim about WHY. G1, G2 and G5 say
        # a contact is impossible, which one number settles. G4 says the
        # cation is on the wrong origin, which is a hypothesis - and CELL 13b
        # tests it by applying the shift to a copy and re-running everything.
        # Where that test has run, its verdict decides the wording, because
        # 'strongly supported' and 'marginal' are different claims and only
        # the first is worth writing to a database about.
        _fall_through = False
        _sv = row.get('shift_verdict')
        _sv = None if (_sv is None or _sv != _sv) else str(_sv)
        if _sv:
            _ev = str(row.get('shift_evidence') or _ev)
            if _sv == 'not diagnostic':
                # Nothing was failing before the shift, so the re-run could
                # not adjudicate. A hypothesis that was never testable is not
                # grounds to expunge; the split shell is still reported.
                return ('M7', 'split_shell_not_diagnostic', _ev)
            if _sv in ('strongly supported', 'supported'):
                return ('G4', 'cation_on_wrong_origin', _ev)
            if _sv == 'partially supported':
                return ('G4', 'cation_position_suspect', _ev)
            # marginal or NOT supported: the split shell is real but the
            # origin hypothesis failed. Fall through to the packing floors,
            # which judge on the measurement rather than on a rejected claim.
            if _sv in ('marginal', 'NOT supported', 'inconclusive'):
                _fall_through = True
            else:
                return ('G4', 'split_shell_cause_unknown', _ev)
        if not _fall_through:
            return ('G4', 'cation_on_wrong_origin', _ev + '; not yet tested')

    # ORDER WITHIN G. The packing floors are asked LAST, after the tested
    # origin hypothesis, because a supported shift EXPLAINS them.
    #
    # A cation half a lattice vector from its site lands next to another
    # cation: 22 of the 26 perovskite structures carrying a proposed shift
    # also have a cation-cation contact below the floor, at 1.90 to 1.92 A.
    # Asked first, G5 reported 'cation packing impossible' for all of them,
    # which is the symptom. G4 has the diagnosis, and it is a tested one:
    # move the cation and the contact resolves.
    #
    # Where the shift is NOT supported, or has not been tested, the packing
    # floors still fire below and the structure is still condemned - on the
    # measurement rather than on an untested claim.
    aa = _pgrm_f(struct.get('anion_anion_min'))
    if aa == aa and aa < _PGRM_ANION_FLOOR:
        return ('G2', 'anion_packing_impossible',
                f'shortest anion-anion {aa:.3f} A, floor {_PGRM_ANION_FLOOR} A')
    cc = _pgrm_f(struct.get('cation_cation_min'))
    if cc == cc and cc < _PGRM_CATION_FLOOR:
        return ('G5', 'cation_packing_impossible',
                f'shortest cation-cation {cc:.3f} A, floor {_PGRM_CATION_FLOOR} A')
    # ---- R : can we assess it? -----------------------------------------
    if row.get('bvs') is None or _pgrm_f(row.get('bvs')) != _pgrm_f(row.get('bvs')):
        return ('R1', 'no_bond_valence_parameters',
                f'{row.get("element")}({row.get("valence")}) to {row.get("ligand_types")}')
    if _pgrm_f(row.get('cn_shannon_min')) != _pgrm_f(row.get('cn_shannon_min')):
        return ('R2', 'no_ionic_radius',
                f'{row.get("element")}({row.get("valence")}) not tabulated')
    if _pgrm_f(row.get('bl_tier')) in (4.0,) or _pgrm_f(row.get('bl_tier')) != _pgrm_f(row.get('bl_tier')):
        return ('R3', 'no_measured_bond_lengths', 'expected distance from additive radii')

    # ---- M : is the depositor's model unambiguous? ----------------------
    occ = _pgrm_f(row.get('occupancy'), 1.0)
    if occ == 0.0:
        return 'M5', 'zero_occupancy', 'site refined to no atom'

    # M2/M3 mean TWO OR MORE SPECIES SHARE A POSITION, which is an ambiguity in
    # the depositor's model. They do NOT mean partial occupancy: in a spinel
    # inversion is the defining feature of the structure type, and firing on
    # occupancy alone flagged 4732 of 6657 spinel sites and 1420 of 1881
    # structures, which is a statement about spinels rather than about the
    # entries. A vacancy on an otherwise single-species site is M1.
    n_sp = _pgrm_f(struct.get('_site_species_count'), 1.0)
    if n_sp >= 2:
        dv = _pgrm_f(struct.get('_site_valence_span'), 0.0)
        if dv >= PGRM_HETEROVALENT_DV:
            return ('M3', 'shared_site_heterovalent',
                    f'{int(n_sp)} species on one position, valence span {dv:.0f}')
        return ('M2', 'shared_site_homovalent',
                f'{int(n_sp)} species on one position, occupancy {occ:.3f}')
    if 0.0 < occ < 0.98:
        return 'M1', 'partial_occupancy', f'occupancy {occ:.3f}'
    v = _pgrm_f(row.get('valence'))
    if v == v and abs(v - round(v)) > 0.05:
        return 'M4', 'mixed_valence', f'valence {v:.2f}'
    yr = _pgrm_f(struct.get('year'))
    if yr == yr and yr < PGRM_PREMODERN_YEAR:
        return 'M6', 'pre_modern_determination', f'published {int(yr)}'
    return '-', None, None


def classify_pgrm(df):
    """Add pgrm_code / gate / reason / evidence / tier, per site and structure."""
    df = df.copy()

    # v2.0.13a: IDEMPOTENT. apply_shift_verdicts calls this a second time
    # to re-classify with the tested shift verdicts, and the merges below
    # append _x/_y suffixes when the columns already exist - so the second
    # call produced structure_pgrm_code_x and a KeyError on the name
    # everything reads. Drop any previous classification first.
    _prev = [c for c in df.columns
             if c.startswith('pgrm_') or c.startswith('structure_pgrm_')
             or c in ('_site_species_count', '_site_valence_span')]
    if _prev:
        df = df.drop(columns=_prev)

    st = df.drop_duplicates('filename').set_index('filename')

    # valence span per shared Wyckoff position, for M3
    nsp = (df.groupby(['filename', 'coords'])['element']
             .nunique().rename('_site_species_count').reset_index())
    df = df.merge(nsp, on=['filename', 'coords'], how='left')

    span = (df.groupby(['filename', 'coords'])['valence']
              .agg(lambda s: (pd.to_numeric(s, errors='coerce').max()
                              - pd.to_numeric(s, errors='coerce').min()))
              .rename('_site_valence_span').reset_index())
    df = df.merge(span, on=['filename', 'coords'], how='left')

    out = []
    for _, r in df.iterrows():
        s = st.loc[r['filename']]
        s = s.iloc[0] if isinstance(s, pd.DataFrame) else s
        ctx = dict(s)
        ctx['_site_valence_span'] = r.get('_site_valence_span')
        ctx['_site_species_count'] = r.get('_site_species_count')
        out.append(classify_site(r, ctx))
    df['pgrm_code'] = [o[0] for o in out]
    df['pgrm_reason'] = [o[1] for o in out]
    df['pgrm_evidence'] = [o[2] for o in out]
    df['pgrm_gate'] = df['pgrm_code'].str[0]
    df['pgrm_tier'] = df['pgrm_gate'].map(_GATE_TIER)

    sev = {'ACCEPT': 0, 'FLAG': 1, 'NOT_ASSESSED': 2, 'EXPUNGE': 3}

    # Within the worst tier, several sites may fail DIFFERENT gates, and one
    # of them has to be the structure's headline reason. Taking the last row
    # picks by row order: on the perovskites that reported
    # cation_packing_impossible for 17 structures whose actual diagnosis was
    # a misplaced cation, because the OTHER site of the same structure
    # inherits the structure-level packing column and is equally severe.
    #
    # The tie is broken on EXPLANATORY POWER. A tested origin hypothesis
    # explains a short cation-cation contact - move the cation and the
    # contact resolves - so G4 outranks G5 even though both condemn. A
    # site-specific fault outranks a structure-level symptom for the same
    # reason: it says WHERE.
    _code_rank = {'P1': 0, 'P2': 0, 'P3': 0,
                  'G4': 1,      # tested: the cation is on the wrong origin
                  'G1': 2,      # this contact, at this site, is impossible
                  'G3': 3,      # this site has no shell at all
                  'G5': 4,      # the cation sublattice is too close somewhere
                  'G2': 5,      # the anion sublattice is too close somewhere
                  'R1': 6, 'R2': 7, 'R3': 8,
                  'M5': 9, 'M3': 10, 'M2': 11, 'M7': 12, 'M4': 13,
                  'M1': 14, 'M6': 15, '-': 99}
    _w = df.assign(_s=df['pgrm_tier'].map(sev),
                   _r=df['pgrm_code'].map(_code_rank).fillna(50))
    worst = (_w.sort_values(['_s', '_r'], ascending=[True, False])
               .groupby('filename')
               .last()[['pgrm_code', 'pgrm_reason', 'pgrm_evidence', 'pgrm_tier']])
    worst.columns = ['structure_' + c for c in worst.columns]
    return df.merge(worst, left_on='filename', right_index=True, how='left')


def apply_pgrm_suppression(df):
    """
    Withhold derived chemistry from structures that failed a P or G gate.

    A geometry failure is not a poor result, it is the absence of a result:
    the numbers were computed from coordinates that cannot be right, and
    publishing them invites exactly the tuning this project keeps catching
    itself doing. The evidence stays; the chemistry goes.
    """
    if not PGRM_SUPPRESS or 'pgrm_gate' not in df.columns:
        return df
    df = df.copy()
    mask = df['pgrm_gate'].isin(['P', 'G'])
    cols = [c for c in PGRM_SUPPRESSED_COLUMNS if c in df.columns]
    df.loc[mask, cols] = np.nan
    n = int(mask.sum())
    if n:
        print(f"  PGRM: chemistry withheld from {n} site(s) in "
              f"{df.loc[mask, 'filename'].nunique()} structure(s) that failed "
              f"a P or G gate. The evidence is in pgrm_reason and "
              f"pgrm_evidence; no bond valence result is reported for them.")
    return df


In [ ]:
# CELL 8: RETIRED OVERRIDE, AND THE SHADOWING CHECK   (Eir 2.0.13)
# ============================================================================
# This cell used to redefine geometry_plausible(), shadowing the definition in
# CELL 7. The override is now IN CELL 7, beside its only caller, and this cell
# defines nothing. What remains is the check that no cell redefines it again.
#
# WHAT THE OVERRIDE DID
#
#   It made the radius-ratio verdict advisory (None, 'not judged') for large
#   weakly bonded cations and for any site with CN >= 8 - but only when the
#   ratio FAILED. A passing site still passed, so the test was one-sided and
#   such a cation could never fail the radii screen.
#
#   It was written when the radius lookup was fixed at six-coordinate, which
#   genuinely did penalise dispersed high-coordination cations. Since v1.8.9
#   the radius comes from get_ionic_radius_fractional() at the site's own
#   valence and coordination number. Over 1886 sites in the 934-structure ICSD
#   perovskite set the two ratio distributions are then indistinguishable
#   (median 0.922 against 0.974, 1st percentile 0.765 against 0.763, 3.1%
#   against 2.9% below the 0.85 threshold), so the concession had nothing left
#   to correct for.
#
#   Meanwhile it was suppressing five entries whose contacts are impossible:
#   LaCrO3 220506 at ratio 0.415 with a bond valence sum 769% above its formal
#   valence, YVO3 121777 at 0.610, NdFeO3 74087 at 0.632, CaTiO3 132626 at
#   0.680 and La0.8MnO2.8 33482 at 0.682.
#
# v2.0.13: the concession was gated on a 25-element list called A_SITE_ELEMENTS
# plus a coordination threshold. That is perovskite vocabulary in a tool that
# also runs on pyrochlores, spinels and rock salts, none of which has an A
# site. It is now gated on WEAK_CATION_LEWIS_MAX - valence divided by
# coordination number - which is structure-agnostic, needs no list, and is the
# same measure that sets the bond-valence shell boundary.
#
# To restore a concession, set WEAK_CATION_ADVISORY_FLOOR in CELL 7 (0.80 is
# the defensible value; it leaves all five entries above failing). Do not
# reinstate a redefinition here: two definitions of one function in a notebook
# is how the behaviour in force stopped being the behaviour anyone was reading.
# ============================================================================


def _eir_check_no_shadowing(verbose=True):
    """
    Confirm that the functions this notebook relies on are defined once.

    A redefinition later in a notebook silently wins and nothing announces it.
    That is how the override went unnoticed: CELL 7 was read, CELL 8 was what
    ran.

    This complements the structural self-check in CELL 1c, which scans the
    saved SOURCE for duplicate definitions. This one inspects the LIVE object,
    so it also catches a redefinition made interactively after the notebook
    was saved.
    """
    expected_owner = {
        'geometry_plausible': 'CELL 7',
        'add_coordination_quality': 'CELL 7',
        'cn_fault': 'CELL 2',
        'guarded_shell_cn': 'CELL 2',
        'core_radii_ratio': 'CELL 2',
        'calculate_anion_bvs': 'CELL 3',
        'summarise_anion_bvs': 'CELL 3',
    }
    g = globals()
    missing = [f'{n} ({o})' for n, o in expected_owner.items() if g.get(n) is None]
    if missing:
        raise NameError(
            'Not defined: ' + ', '.join(missing) +
            '. Run the owning cells before this one.')

    # The specific regression this cell exists to prevent. Checked by an
    # attribute set in CELL 7 rather than by reading the source: an attribute
    # survives exec, import and %run, whereas inspect.getsource() cannot always
    # resolve a function defined in a notebook cell, and a check that cannot
    # see its own evidence would raise a false alarm.
    fn = g['geometry_plausible']
    owner = getattr(fn, '_eir_cell', None)
    if owner != 'CELL 7':
        raise AssertionError(
            'geometry_plausible() in force is not the CELL 7 version '
            f'(owner tag: {owner!r}). Another cell is redefining it. Find and '
            'remove that definition: while it stands, radius-ratio failures on '
            'weakly bonded cations are suppressed and the geometry screen '
            'reports a pass rate it has not measured.')

    floor = g.get('WEAK_CATION_ADVISORY_FLOOR', 'undefined')
    lmax = g.get('WEAK_CATION_LEWIS_MAX', 'undefined')
    if verbose:
        print('Eir 2.0.13 CELL 8: the radii-ratio override is retired; '
              'nothing is defined here.')
        print('  geometry_plausible() in force is the CELL 7 version.')
        print(f'  WEAK_CATION_ADVISORY_FLOOR = {floor}'
              + ('  (no concession; every cation is judged on the same '
                 'footing)' if floor is None else
                 f'  (advisory between this and the threshold, for cations '
                 f'with valence/CN <= {lmax})'))
        print(f'  {len(expected_owner)} shared functions present and owned '
              f'as expected.')
    return True


_eir_check_no_shadowing()


In [ ]:
# CELL 9: RUN BATCH ANALYSIS   (Eir 2.0.13)
#
# Must sit after the anion-BVS cells, because process_single_cif calls
# anion_bvs_for_structure(). Keeping the runner here means Restart & Run All
# works top to bottom.

# Names the run depends on. A missing one here is a cell that was not run, or
# was run and raised part-way; either way the message should say which,
# because the alternative is a NameError several hundred structures in.
for _name in ('calculate_anion_bvs', 'anion_bvs_for_structure',
              'add_coordination_quality', 'expected_bond_length',
              'normaliser_translations', 'classify_pgrm',
              'reset_read_failures', 'report_read_failures',
              'may_visualise', 'reset_visualisations'):
    if _name not in globals():
        raise NameError(f"{_name}() is not defined. Run the cells above first.")

EIR_CIF_DIRECTORY = select_directory()

if EIR_CIF_DIRECTORY is None:
    df = None
    print('\n\u2717 Analysis cancelled')
else:
    # v2.0.13: the read-failure register is per-run. Without the reset a
    # second batch in the same session reports the first batch's failures.
    reset_read_failures()
    reset_visualisations()
    df = batch_process_directory(EIR_CIF_DIRECTORY)

if df is not None and len(df):
    print('\n' + '=' * 78)
    print('BATCH PROCESSING COMPLETE')
    print('=' * 78)
    print(f"  Structures analysed : {df['filename'].nunique()}")
    print(f'  Sites analysed      : {len(df)}')

    # v2.0.13: anything that could not be READ, as opposed to anything that
    # was read and had nothing to report. Five metadata extractors used to
    # wrap their whole file read in a bare except and return an empty result,
    # so a failed read was indistinguishable from absent metadata - which is
    # why measurement_pressure_GPa appeared on 10 of 934 perovskites with no
    # way to tell which it meant.
    report_read_failures()

    print(f"\n  Results saved to: "
          f"{os.path.join(EIR_CIF_DIRECTORY, 'outputs')}")
    print('  Next: the post-processing cell, which adds coordination quality',
          'and the PGRM integrity gates.')
elif df is not None:
    print('\n\u2717 The batch produced no rows. Check the CIF directory and the',
          'failed-files CSV in outputs/.')
else:
    print('\n\u2717 Analysis failed')


In [ ]:
# CELL 10: PROVENANCE AND OXIDATION-STATE ALTERNATIVES   (Eir 2.0.13)
#
# WHAT THIS CELL USED TO DO, AND WHY IT NO LONGER DOES IT
#
# Until v2.0.10 this cell carried a second verdict layer: an A-to-F quality
# tier and an INCLUDE / CAUTION / EXCLUDE recommendation, derived from the
# bond valence sum and the GII. It has been removed, not repaired, because it
# was answering the question the wrong way round and it contradicted the PGRM
# gates on the same page.
#
# The demonstration is ICSD 85235, TbCrO3. Its chromium octahedron is 0.0004 A
# from the mean of every Cr(3+)-O bond in Gagne's survey, its GII is 0.0047,
# the finest in the 938-structure perovskite set, and two of its oxygens are
# 2.191 A apart against a family median of 2.750. The old layer ranked it
# INCLUDE, tier A, high confidence: the single best structure in the batch.
# PGRM expunges it on the anion packing floor. Both statements were printed in
# the same run.
#
# The reason bond valence could not see it is not an oversight. Chromium sits
# at Wyckoff 4b, a special position invariant under the reflection relating
# the deposited oxygen sublattice to the correct one, so the six Cr-O
# distances are mathematically identical for the right structure and the
# wrong one. A method that sums only cation-anion distances is not merely
# insensitive here; it is provably blind.
#
# Across the 30-structure test batch the two layers disagreed on 6 of 30. Five
# were structures the old layer rejected for being DISORDERED - shared sites
# give a poor per-species sum, which is the phantom-minority problem WAGII
# exists to handle - and one, 85235, it accepted while PGRM expunged it. It
# also rejected ICSD 16666 for being published in 1956, with sound geometry.
#
# Its element-consensus table was worse than its verdicts: it averaged bond
# valence sums across expunged structures, reporting a cobalt consensus of
# 44.026 +/- 62.186 v.u. from two entries whose oxygens are 0.02 and 0.91 A
# apart, and a lithium consensus at CN 16.
#
# Three things here were worth keeping and are kept below: the measurement
# conditions distribution, the publication metadata, and the 'what if'
# best-matching integer oxidation state for shared sites, which has no
# equivalent in PGRM.
#
# For the verdict, read structure_pgrm_tier.

if 'dfq' not in globals() or dfq is None:
    print("Run the post-processing cell first (it creates dfq).")
else:
    _d = dfq
    _st = _d.drop_duplicates('filename')

    # ---------------------------------------------------------------
    # 1. MEASUREMENT CONDITIONS
    #    Bond valence parameters are fitted to ambient-condition
    #    structures, so a high-pressure entry is being assessed against a
    #    reference that does not describe it.
    # ---------------------------------------------------------------
    print("=" * 78)
    print("PROVENANCE")
    print("=" * 78)
    if 'measurement_conditions' in _st.columns:
        _c = _st['measurement_conditions'].fillna('unknown').value_counts()
        print("\n  Measurement conditions")
        for k, v in _c.items():
            print(f"    {str(k):22} {v:5}  ({100 * v / len(_st):.1f}%)")
        _n_unknown = int(_c.get('unknown', 0))
        if _n_unknown > 0.5 * len(_st):
            print(f"\n    {100 * _n_unknown / len(_st):.0f}% of entries record no temperature or")
            print("    pressure. The ambient-conditions assumption rests on the")
            print("    database search filter, not on the deposited files.")

    # ---------------------------------------------------------------
    # 2. PUBLICATION METADATA, grouped by PGRM tier
    # ---------------------------------------------------------------
    if 'structure_pgrm_tier' in _st.columns:
        print("\n  Entries by integrity tier, oldest first")
        for _tier in ('EXPUNGE', 'NOT_ASSESSED', 'FLAG', 'ACCEPT'):
            _sub = _st[_st['structure_pgrm_tier'] == _tier]
            if not len(_sub):
                continue
            print(f"\n    {_tier}  ({len(_sub)})")
            _sub = _sub.sort_values('year', na_position='last')
            for _, _r in _sub.head(12).iterrows():
                _y = _r.get('year')
                _y = int(_y) if _y == _y else '????'
                print(f"      {str(_r.get('icsd_code')):>7}  {_y}  "
                      f"{str(_r.get('chemical_formula'))[:22]:22} "
                      f"{str(_r.get('structure_pgrm_reason') or '')[:28]:28} "
                      f"{str(_r.get('journal'))[:34]}")
            if len(_sub) > 12:
                print(f"      ... and {len(_sub) - 12} more")

    # ---------------------------------------------------------------
    # 3. WHAT IF: the best-matching integer oxidation state
    #    For a site whose assigned valence is a partition the depositor
    #    could not verify from diffraction, this asks which integer state
    #    the observed geometry actually supports. It is a question about
    #    the ASSIGNMENT, not about the structure, which is why it belongs
    #    with the M facet rather than with a verdict.
    # ---------------------------------------------------------------
    if 'pgrm_gate' in _d.columns:
        _m = _d[(_d['pgrm_gate'] == 'M') & _d['bvs'].notna()].copy()
        if len(_m):
            print("\n" + "=" * 78)
            print("WHAT IF: integer oxidation states for ambiguous sites")
            print("=" * 78)
            print("\n  Sites where the depositor's partitioning cannot be verified")
            print("  from diffraction. Which integer state does the geometry support?\n")
            print(f"    {'ICSD':>7} {'site':>6} {'assigned':>9} {'BVS':>7} "
                  f"{'nearest':>8} {'|dev|':>7}  reason")
            _rows = 0
            for _, _r in _m.iterrows():
                _b = float(_r['bvs'])
                _v = _r.get('valence')
                try:
                    _v = float(_v)
                except (TypeError, ValueError):
                    continue
                _near = round(_b)
                if _near < 1:
                    continue
                _dev = abs(_b - _near) / _near * 100
                if abs(_v - _near) < 0.5 or _dev > 15:
                    continue          # already integer, or no state fits well
                print(f"    {str(_r.get('icsd_code')):>7} {str(_r.get('element')):>6} "
                      f"{_v:9.2f} {_b:7.3f} {_near:8d} {_dev:6.1f}%  "
                      f"{str(_r.get('pgrm_reason') or '')}")
                _rows += 1
                if _rows >= 20:
                    print("    ... truncated at 20")
                    break
            if not _rows:
                print("    No ambiguous site is better described by a different")
                print("    integer state than the one assigned.")

    print("\n" + "=" * 78)
    print("For the integrity verdict, read structure_pgrm_tier. This cell")
    print("reports provenance and alternatives; it does not judge.")
    print("=" * 78)


In [ ]:
# CELL 11: POST-PROCESS - COORDINATION QUALITY AND INTEGRITY GATES   (2.0.13)
#
# Adds ECoN, coordination quality and geometry-plausibility columns, replaces
# bvs_diagnosis with the observational labels, and applies the PGRM integrity
# gates. Writes a new CSV alongside the batch output; the original is left
# untouched.

import glob as _glob

if 'EIR_CIF_DIRECTORY' not in globals() or EIR_CIF_DIRECTORY is None:
    raise RuntimeError('Run the batch cell first.')

_outdir = os.path.join(EIR_CIF_DIRECTORY, 'outputs')
_csvs = sorted(_glob.glob(os.path.join(_outdir, 'eir_batch_results_*.csv')),
               key=os.path.getmtime)
_csvs = [c for c in _csvs if not c.endswith('_quality.csv')]

if not _csvs:
    raise FileNotFoundError(f'No batch CSV found in {_outdir}')

EIR_BATCH_CSV = _csvs[-1]
print(f'Post-processing: {os.path.basename(EIR_BATCH_CSV)}\n')

dfq = add_coordination_quality(EIR_BATCH_CSV, EIR_CIF_DIRECTORY)

# v2.0.13: name the file that was written. The runner reported the directory
# and the post-processor reported nothing, so which of the two CSVs in it
# carried the quality columns had to be inferred from the timestamps.
_qcsv = os.path.splitext(EIR_BATCH_CSV)[0] + '_quality.csv'
if os.path.exists(_qcsv):
    print(f'\n  Written: {os.path.basename(_qcsv)}')
    print(f'  {len(dfq)} rows x {dfq.shape[1]} columns')
    print('  Read structure_pgrm_tier for the integrity verdict and',
          'pgrm_reason for the evidence.')


In [ ]:
# CELL 12: WITHIN-ELEMENT OUTLIERS AND THE PER-STRUCTURE TABLE   (2.0.13)
# ===========================================================================
# Two jobs.
#
# 1. The within-element outlier pass: is this site's shortest bond unusual
#    FOR THIS ELEMENT in this run? Robust z on the median and MAD, so a few
#    broken structures cannot move the reference.
#
# 2. The per-structure table. Everything in the quality CSV is per SITE, so
#    every structure-level fact - the integrity verdict, the screen results,
#    the instability index, the packing minima - has had to be recovered with
#    drop_duplicates('filename') by every consumer: the triage, the summary,
#    the shift test and every ad-hoc analysis. That table now exists.
#
# WHERE THE LINE WITH HUGINN NOW FALLS
#
# Huginn 4.0 exists to make sense of what Eir kicks out, and a large part of
# what it does is REDUCTION: rolling per-site rows up to structures, which it
# has to do because Eir shipped only a per-site table. Its own documentation
# records the trap - taking groupby('filename').first() of a per-site screen
# column finds 20 of 48 failing perovskites and misses 28.
#
# With build_structure_table below, that reduction is Eir's job and is done
# once, with the invariant asserted rather than assumed. What should stay in
# Huginn is everything that is not reduction:
#
#   - pooling several Eir runs into one analysis and tracking what was
#     filtered out of each
#   - family-specific interpretation, which Huginn handles with modules
#   - publication figures and journal styling
#   - provenance analysis across decades and journals
#
# Eir should answer 'is this entry sound, and what is the evidence'. Huginn
# should answer 'what does this pool of entries tell us'. Until now Eir could
# not answer its own question without an interpreter, because the answer was
# spread across 225 per-site columns.
#
# RETIRED at v2.0.13: eir_2_0_1_geometry_for_huginn.csv. Every one of its 26
# columns was already in the quality CSV, its filename hard-coded a version
# three releases old, and nothing read it by name - it existed to make the
# native geometry available downstream after the pymatgen pass was dropped,
# which the quality CSV does. Worse, the three within-element columns were
# written ONLY to it: they were attached to dfq in memory and the quality CSV
# was never re-saved, so they were absent from the 225-column file everything
# else reads. The quality CSV is now re-saved with them.
# ===========================================================================
import os as _os
import numpy as _np
import pandas as _pd

MIN_N_WITHIN_ELEMENT = 20    # bonds of an element needed before judging one
WITHIN_ELEMENT_Z = 3.0       # robust z beyond which the bond is unusual


def add_within_element_outliers(df):
    """Robust z of each site's shortest bond against its own element."""
    if 'geo_bond_min' not in df.columns:
        return df
    _b = df.dropna(subset=['geo_bond_min'])
    stats = {}
    for el, grp in _b.groupby('element')['geo_bond_min']:
        med = float(grp.median())
        mad = float((grp - med).abs().median()) or 1e-9
        stats[el] = (med, mad, len(grp))

    def _z(r):
        st = stats.get(r['element'])
        v = r.get('geo_bond_min', _np.nan)
        if not st or st[2] < MIN_N_WITHIN_ELEMENT or v != v:
            return (None, st[2] if st else None, None)
        med, mad, n = st
        z = 0.6745 * (v - med) / mad
        return (round(z, 3), n, bool(abs(z) > WITHIN_ELEMENT_Z))

    df[['within_element_z', 'within_element_n', 'within_element_flag']] = \
        df.apply(_z, axis=1, result_type='expand')
    return df


def build_structure_table(df):
    """
    One row per structure: identity, integrity verdict, screen results,
    instability indices, packing minima and the measured-distance summary.

    Per-site columns are reduced explicitly rather than by taking the first
    row, because 'first' is only correct for columns that are genuinely
    constant within a structure, and several are not.
    """
    g = df.groupby('filename', sort=False)

    # constant within a structure: take the first non-null
    _const = ['icsd_code', 'chemical_formula', 'space_group', 'year', 'journal',
              'authors', 'measurement_conditions', 'gii', 'wagii',
              'gii_site_averaged', 'gii_recommended', 'gii_all_sites',
              'gii_percentile_in_run', 'anion_anion_min', 'cation_cation_min',
              'has_antisite_disorder', 'n_antisite_species', 'n_anion_sites',
              # the anion side, which the cation/anion diagnosis and its figure
              # both need and which was omitted from the first version of this
              # table: the figure reported its columns absent rather than
              # failing, which is the right behaviour but hid the gap.
              'anion_bvs_mean', 'anion_bvs_mean_weighted',
              'anion_deviation_mean', 'anion_deviation_mean_weighted',
              'anion_deviation_max', 'anion_worst_label', 'anion_worst_bvs',
              'anion_worst_ligands', 'anion_balances',
              'shift_verdict', 'shift_evidence',
              'n_sites_shift_proposed', 'bvs_diagnosis', 'bvs_diagnosis_detail',
              'structure_pgrm_tier', 'structure_pgrm_code',
              'structure_pgrm_reason', 'structure_pgrm_evidence',
              'structure_core_geometry_plausible',
              'structure_bvs_geometry_plausible', 'anion_packing_plausible',
              'cation_packing_plausible', 'structure_cn_plausible',
              'u_parameter', 'u_deviation', 'pyrochlore_48f_x',
              'pyrochlore_fluorite_limit', 'origin_choice']
    # Every column above is taken with .first(), which is only correct if it
    # is genuinely constant within a structure. That is ASSERTED, not assumed.
    #
    # Eir carries both granularities under names differing by one prefix:
    # core_geometry_plausible is PER SITE, structure_core_geometry_plausible
    # is per structure, and they sit adjacent in a 225-column CSV. Taking
    # .first() of the per-site column finds 20 of 48 failing perovskites and
    # misses 28, because the failing site is usually not row one. Huginn 4.0
    # documents the same trap from the other side.
    #
    # Anything that turns out to vary is rolled up with ANY-FALSE-CONDEMNS
    # instead, and named in _rolled_up so a reader can see it happened.
    cols, rolled = {}, []
    for c in _const:
        if c not in df.columns:
            continue
        # nunique() ignoring NaN, because a column that is populated on ONE
        # site by design - shift_evidence, proposed_shift - is not varying,
        # it is sparse. Counting NaN as a value made every such column look
        # inconsistent and sent it through the any-False rule, which is
        # meaningless on a string.
        _varies = int((g[c].nunique(dropna=True) > 1).sum())
        if _varies == 0:
            cols[c] = g[c].first()          # first NON-NULL, pandas default
            continue
        _isfalse = df[c].astype(str).str.lower().eq('false')
        _istrue = df[c].astype(str).str.lower().eq('true')
        if (_isfalse | _istrue).any():
            # a boolean screen that genuinely differs between sites
            rolled.append(c)
            cols[c] = ~_isfalse.groupby(df['filename']).any()
        else:
            # not boolean, so any-False says nothing. Take the first non-null
            # and record that the column was not single-valued.
            rolled.append(c + ' (first non-null)')
            cols[c] = g[c].first()
    out = _pd.DataFrame(cols)
    if rolled:
        _shown = ', '.join(rolled[:5]) + (' ...' if len(rolled) > 5 else '')
        print(f'  [{len(rolled)} column(s) varied within a structure and were '
              f'rolled up any-False rather than taken from the first row: '
              f'{_shown}]')
    out.attrs['rolled_up'] = rolled

    # aggregated across the sites
    out['n_sites'] = g.size()
    for src, name, how in [
            ('bl_delta_mean', 'bl_delta_mean_median', 'median'),
            ('bl_ratio_min', 'bl_ratio_min_min', 'min'),
            ('min_bond_length', 'shortest_cation_anion', 'min'),
            ('econ_cn_ratio', 'econ_cn_ratio_min', 'min')]:
        if src in df.columns:
            out[name] = getattr(_pd.to_numeric(df[src], errors='coerce')
                                .groupby(df['filename']), how)()
    for src, name in [('within_element_flag', 'n_within_element_flagged'),
                      ('cn_fault', 'n_cn_faults')]:
        if src in df.columns:
            out[name] = df[src].notna().groupby(df['filename']).sum()
    if 'bl_tier' in df.columns:
        _t = _pd.to_numeric(df['bl_tier'], errors='coerce')
        out['n_bl_measured'] = (_t <= 2).groupby(df['filename']).sum()

    # how many independent screens failed. Computed here because every
    # consumer was rebuilding it, and they did not all use the same five.
    _screens = ['structure_core_geometry_plausible',
               'structure_bvs_geometry_plausible', 'anion_packing_plausible',
               'cation_packing_plausible', 'structure_cn_plausible']
    _have = [c for c in _screens if c in out.columns]
    if _have:
        _fail = sum(out[c].astype(str).str.lower().eq('false').astype(int)
                    for c in _have)
        if 'n_sites_shift_proposed' in out.columns:
            _fail = _fail + (_pd.to_numeric(out['n_sites_shift_proposed'],
                                            errors='coerce').fillna(0) > 0).astype(int)
        out['n_screens_failed'] = _fail
    return out.reset_index()


if 'dfq' in dir() and dfq is not None:
    dfq = add_within_element_outliers(dfq)

    # Re-save the per-site table so the three outlier columns are not stranded
    # in memory. This is the same file add_coordination_quality wrote; it is
    # rewritten rather than duplicated.
    if 'EIR_BATCH_CSV' in dir():
        _sites_csv = _os.path.splitext(EIR_BATCH_CSV)[0] + '_quality.csv'
        dfq.to_csv(_sites_csv, index=False)

        dfs = build_structure_table(dfq)
        _struct_csv = _os.path.splitext(EIR_BATCH_CSV)[0] + '_structures.csv'
        dfs.to_csv(_struct_csv, index=False)

        print('OUTPUTS')
        print(f'  per site      {_os.path.basename(_sites_csv)}'
              f'   {len(dfq)} rows x {dfq.shape[1]} cols')
        print(f'  per structure {_os.path.basename(_struct_csv)}'
              f'   {len(dfs)} rows x {dfs.shape[1]} cols')
        _nf = int(dfq['within_element_flag'].fillna(False).sum())
        _ng = int(dfq['geometry'].notna().sum()) if 'geometry' in dfq else 0
        print(f'  within-element outliers {_nf}  |  sites with geometry {_ng}')
        if 'structure_pgrm_tier' in dfs.columns:
            _ex = dfs[dfs['structure_pgrm_tier'] == 'EXPUNGE']
            if len(_ex):
                print(f'  {len(_ex)} structure(s) at EXPUNGE. The reportable-error',
                      'list is this table filtered to that tier;')
                print('  pgrm_reason and pgrm_evidence carry the measurement.')
    else:
        dfs = build_structure_table(dfq)
        print('EIR_BATCH_CSV not set, so nothing was written. dfs holds the',
              'per-structure table.')
else:
    print('dfq not found. Run the post-processing cell first.')


In [ ]:
# CELL 13: TRIAGE   (Eir 2.0.13)
# ============================================================================
# Rewritten. This cell used to print three raw dumps: a count of every
# coordination number, an ECoN table in which most rows read 1.0000, and a
# bare list of flagged filenames. Between them they said that something was
# unusual without saying what, why, or what to do, which produces concern
# rather than understanding.
#
# It now answers three questions instead:
#
#   1. Which sites have an unusual coordination, and WHY is each one unusual?
#      Every entry is assigned a cause, and the causes are ranked by what they
#      would mean if true. A cation half a lattice vector from its site is a
#      different problem from a Jahn-Teller elongation, and both look like a
#      low coordination number until they are separated.
#
#   2. Which structures are flagged, by WHICH screen, and what does that
#      screen see? A filename on its own cannot be acted on.
#
#   3. What is worth looking at first?
#
# Nothing here recomputes anything. It reads the columns the screens already
# wrote and arranges them so they can be acted on.

def triage(dfq, max_listed=12, verbose=True):
    """Turn the screening columns into a report that can be acted on."""
    L = []
    def out(s=""):
        L.append(s)

    if dfq is None or not len(dfq):
        print("Run the post-processing cell first.")
        return None

    n_struct, n_sites = dfq["filename"].nunique(), len(dfq)

    def num(c):
        return pd.to_numeric(dfq[c], errors="coerce") if c in dfq.columns else pd.Series(dtype=float)

    def is_false(frame, c):
        return frame[c].astype(str).str.lower().eq("false") if c in frame.columns else pd.Series(False, index=frame.index)

    cn_rel, cn_ = num("cn_relative"), num("cn")

    # Which instability index to show. Section 2 works out that a structure
    # with anti-site exchange must be read on wagii, and this cell used to go
    # on printing gii in every table underneath it: ICSD 257082 appeared at
    # 1.411 and 8352 at 1.489 on the same page that said to quote 0.919 and
    # 0.976. A report that contradicts itself is worse than one that says less,
    # so every table below now shows the recommended index and names it.
    if "gii_recommended" in dfq.columns:
        rec = dfq["gii_recommended"].astype(str)
        gii = pd.Series(np.where(rec.eq("wagii"), num("wagii"), num("gii")),
                        index=dfq.index)
        _gii_label = "gii*"
        _n_rec = int(rec.eq("wagii").sum())
    else:
        gii = num("gii")
        _gii_label = "gii"
        _n_rec = 0
    dfq = dfq.copy()
    dfq["gii_shown"] = gii
    dev = num("bvs_percent_diff").abs()

    out("=" * 78)
    out("EIR TRIAGE")
    out("=" * 78)
    # v2.0.13: the reduction to one row per structure is done ONCE, by
    # build_structure_table, which asserts that every column it takes with
    # .first() is genuinely constant within a structure and rolls up
    # any-False otherwise. drop_duplicates here was a second implementation
    # of the same reduction; it happened to read the structure_* columns and
    # so was correct, but reading the unprefixed per-site column instead
    # would have found 20 of 48 failing perovskites and missed 28.
    if 'build_structure_table' in globals():
        per = build_structure_table(dfq).set_index('filename')
    else:
        per = dfq.drop_duplicates('filename').set_index('filename')
        out('  [build_structure_table unavailable; the structure-level rows '
            'were taken from the first site of each file]')
    out(f"  {n_struct} structures, {n_sites} cation sites")
    if _n_rec:
        out(f"  {_gii_label} = the index gii_recommended names for each structure: "
            f"wagii on {per['gii_recommended'].eq('wagii').sum()} with anti-site")
        out(f"  exchange, gii on the rest. See section 3.")
    out()

    # ---------------------------------------------------------------- 0
    # The verdict first. Everything below it is the evidence for it.
    if "structure_pgrm_tier" in per.columns:
        out("-" * 78)
        out("1. INTEGRITY VERDICT")
        out("-" * 78)
        out()
        out("  Four facets, asked in order, the first failure stopping the")
        out("  assessment: can the file be READ, can the atoms be where it")
        out("  says, can we ASSESS it, is the deposited MODEL unambiguous.")
        out("  Geometry decides severity before any chemistry is read.")
        out()
        _tiers = per["structure_pgrm_tier"].value_counts()
        _mean = {"EXPUNGE": "atoms cannot be where the file says",
                 "NOT_ASSESSED": "no reference data for at least one site",
                 "FLAG": "the deposited model is ambiguous",
                 "ACCEPT": "nothing found"}
        for _k in ("EXPUNGE", "NOT_ASSESSED", "FLAG", "ACCEPT"):
            if _k in _tiers:
                out(f"  {_k:14} {_tiers[_k]:5}  "
                    f"({100.0 * _tiers[_k] / n_struct:5.1f}%)   {_mean[_k]}")
        out()
        _ex = per[per["structure_pgrm_tier"] == "EXPUNGE"]
        if len(_ex):
            out("  No bond valence result should be quoted for an expunged")
            out("  structure. The number is arithmetic performed on impossible")
            out("  coordinates, and it is often a GOOD number: ICSD 85235 has")
            out("  two oxygens 2.191 A apart and a GII of 0.0047, the finest")
            out("  in the 938-structure perovskite set.")
            out()
            out("  Expunged for:")
            for _r, _n in _ex["structure_pgrm_reason"].value_counts().items():
                out(f"      {str(_r):34} {_n:4}")
            out()
            _cols = [c for c in ["icsd_code", "chemical_formula", "year",
                                 "structure_pgrm_evidence"]
                     if c in _ex.columns]
            for _line in _ex[_cols].head(max_listed).to_string(
                    index=False).splitlines():
                out("      " + _line)
            if len(_ex) > max_listed:
                out(f"      ... and {len(_ex) - max_listed} more")
            out()
            out("  This list IS the reportable-error set: the per-structure")
            out("  table filtered to EXPUNGE, each row carrying the")
            out("  measurement that condemned it.")
        out()
    # ---------------------------------------------------------------- 1
    out("-" * 78)
    out("2. UNUSUAL COORDINATION: WHAT IS CAUSING IT")
    out("-" * 78)
    out()
    out("  A coordination number well away from the family norm is a symptom,")
    out("  not a diagnosis. The same low number arises from a cation on the")
    out("  wrong site, from bonds that are genuinely absent, and from ordinary")
    out("  Jahn-Teller chemistry. Each is separated below.")
    out()
    out("  These causes read the same evidence as the PGRM G facet, per site")
    out("  rather than per structure: a proposed shift is G4, a contact below")
    out("  the floor is G1, an absent shell is G3. They cannot disagree with")
    out("  section 1, because they are one measurement read two ways.")
    out()

    common = cn_rel.value_counts()
    common = set(common[common >= 0.05 * n_sites].index)
    odd = dfq[cn_rel.notna() & ~cn_rel.isin(common)].copy()
    out(f"  Coordination numbers seen on 5% or more of sites: "
        f"{sorted(int(c) for c in common)}")
    out(f"  Sites outside that set: {len(odd)} of {n_sites} "
        f"({100.0 * len(odd) / n_sites:.1f}%)")
    out()

    if len(odd):
        causes = []
        for idx, r in odd.iterrows():
            g = pd.to_numeric(pd.Series([r.get("gii_shown")]), errors="coerce").iloc[0]
            if pd.notna(r.get("proposed_shift")):
                cause = "split coordination"
            elif str(r.get("cn_plausible")).lower() == "false":
                cause = "bonds missing"
            elif str(r.get("core_geometry_plausible")).lower() == "false" or \
                 str(r.get("bvs_geometry_plausible")).lower() == "false":
                cause = "contact too short"
            elif pd.notna(g) and g < 0.25:
                cause = "probably genuine"
            else:
                cause = "unexplained"
            causes.append(cause)
        odd["cause"] = causes

        EXPLAIN = {
            "split coordination":
                "cation about half a lattice vector from where the anions "
                "expect it; a shift is proposed, nothing is changed",
            "bonds missing":
                "coordination below anything Shannon tabulates for the ion, "
                "so the bond valence sum is incomplete",
            "contact too short":
                "a bond shorter than the ions can physically allow",
            "probably genuine":
                "low coordination but the structure is otherwise sound "
                "(Jahn-Teller, lone pair, square planar)",
            "unexplained":
                "no screen fires; worth a look by eye",
        }
        order = ["split coordination", "bonds missing", "contact too short",
                 "unexplained", "probably genuine"]
        for cause in order:
            sub = odd[odd.cause == cause]
            if not len(sub):
                continue
            g = pd.to_numeric(sub["gii_shown"], errors="coerce")
            out(f"  {cause.upper()}  -  {len(sub)} sites in "
                f"{sub['filename'].nunique()} structures, median "
                f"{_gii_label} {g.median():.3f}")
            out(f"      {EXPLAIN[cause]}")
            cols = [c for c in ["icsd_code", "chemical_formula", "element",
                                "cn", "cn_relative", "gii_shown"] if c in sub.columns]
            extra = ("proposed_shift" if cause == "split coordination"
                     else "cn_shannon_min" if cause == "bonds missing" else None)
            if extra and extra in sub.columns:
                cols = cols + [extra]
            show = sub.sort_values("gii_shown", ascending=False)[cols].head(max_listed)
            for line in show.to_string(index=False).splitlines():
                out("      " + line)
            if len(sub) > max_listed:
                out(f"      ... and {len(sub) - max_listed} more")
            out()

    # ---------------------------------------------------------------- 1b
    out("-" * 78)
    out("3. WHICH INSTABILITY INDEX TO QUOTE")
    out("-" * 78)
    out()
    wag, sav = num("wagii"), num("gii_site_averaged")
    if "has_antisite_disorder" in per.columns:
        anti = per["has_antisite_disorder"].astype(str).str.lower().eq("true")
    elif "n_antisite_species" in per.columns:
        anti = pd.to_numeric(per["n_antisite_species"], errors="coerce").fillna(0) > 0
    else:
        anti = pd.Series(False, index=per.index)
    perg = per.assign(_g=pd.to_numeric(per.get("gii"), errors="coerce"),
                      _w=pd.to_numeric(per.get("wagii"), errors="coerce"),
                      _s=pd.to_numeric(per.get("gii_site_averaged"), errors="coerce"),
                      _a=anti.values)
    n_anti = int(perg["_a"].sum())
    out(f"  Structures with anti-site exchange: {n_anti} of {n_struct}")
    out()
    out("  Eir reports one bond valence sum per SPECIES per site. Where a site")
    out("  is shared, the minority species is asked what its sum would be in a")
    out("  cavity the majority sizes; it answers with something impossible, and")
    out("  gii then weights that phantom equally with the majority it sits")
    out("  among. wagii weights by occupancy; gii_site_averaged combines the")
    out("  sums at the site before taking a deviation, so the phantom never")
    out("  exists separately.")
    out()
    if n_anti:
        a, b = perg[perg["_a"]], perg[~perg["_a"]]
        out(f"      {'':22} {'gii':>8} {'wagii':>8} {'site-avg':>9}")
        out(f"      {'with anti-site (' + str(len(a)) + ')':22} "
            f"{a._g.median():>8.3f} {a._w.median():>8.3f} {a._s.median():>9.3f}")
        out(f"      {'ordered (' + str(len(b)) + ')':22} "
            f"{b._g.median():>8.3f} {b._w.median():>8.3f} {b._s.median():>9.3f}")
        out()
        ratio = (b._g / b._w).median()
        out(f"  On the structures with no shared site gii/wagii is {ratio:.3f}.")
        out("  The two are the same quantity when every occupancy is 1.0 -")
        out("  NOT, as this line claimed until v2.0.13, whenever nothing is")
        out("  shared. WAGII weights each site by its occupancy, so a vacancy")
        out("  separates them just as a shared site does. Full occupancy gives")
        out("  gii == wagii on 869 of 871 perovskites, 355 of 355 pyrochlores")
        out("  and 335 of 338 spinels, while 1543 partially occupied spinels")
        out("  sit off the line. That is what makes the weighted form safe to")
        out("  quote throughout, and the reason to quote it more widely than")
        out("  the anti-site count alone would suggest.")
        out()
        out("  QUOTE wagii (or gii_site_averaged) for the anti-site structures.")
        out("  gii_recommended carries this per structure.")
        worst = a.assign(_r=a._g / a._w).nlargest(min(8, len(a)), "_r")
        cols = [c for c in ["icsd_code", "chemical_formula"] if c in worst.columns]
        if cols:
            out()
            out("  Where the choice matters most:")
            show = worst[cols].copy()
            show["gii"] = worst._g.round(3).values
            show["wagii"] = worst._w.round(3).values
            show["site_avg"] = worst._s.round(3).values
            for line in show.to_string(index=False).splitlines():
                out("      " + line)
    else:
        out("  No anti-site exchange found; gii, wagii and gii_site_averaged")
        out("  will agree and gii can be quoted directly. State which was used")
        out("  regardless, because on a disordered set they diverge sharply.")
    out()

    # ---------------------------------------------------------------- 2
    out("-" * 78)
    out("4. WHICH SCREEN FLAGGED WHAT")
    out("-" * 78)
    out()
    SCREENS = [
        ("radii ratio", "structure_core_geometry_plausible",
         "shortest contact against the sum of the ionic radii"),
        ("bond-valence distance", "structure_bvs_geometry_plausible",
         "shortest contact against what the ion's own parameters imply"),
        ("anion-anion packing", "anion_packing_plausible",
         "two anions closer than their electron clouds allow; the only test "
         "that does not look outward from a cation"),
        ("cation-cation packing", "cation_packing_plausible",
         "two cations closer than they can be; edge- and face-sharing "
         "polyhedra reach 2.6-3.0 A and no closer"),
        ("coordination", "structure_cn_plausible",
         "bonds missing rather than too short"),
    ]
    sets = {}
    for name, col, _ in SCREENS:
        sets[name] = set(per.index[is_false(per, col)]) if col in per.columns else set()
    if "n_sites_shift_proposed" in per.columns:
        sets["half-lattice shift"] = set(
            per.index[pd.to_numeric(per["n_sites_shift_proposed"],
                                    errors="coerce").fillna(0) > 0])
    union = set().union(*sets.values()) if sets else set()

    for name, col, what in SCREENS:
        s = sets.get(name, set())
        excl = s - set().union(*[v for k, v in sets.items() if k != name]) if s else set()
        out(f"  {name:<22} {len(s):4d} structures   {len(excl):3d} that no other "
            f"screen catches")
        out(f"      {what}")
    if "half-lattice shift" in sets:
        s = sets["half-lattice shift"]
        excl = s - set().union(*[v for k, v in sets.items() if k != "half-lattice shift"])
        out(f"  {'half-lattice shift':<22} {len(s):4d} structures   {len(excl):3d} that no "
            f"other screen catches")
        out("      cation on the wrong origin; a candidate position is proposed")
    out()
    out(f"  Flagged by at least one screen: {len(union)} of {n_struct} "
        f"({100.0 * len(union) / n_struct:.0f}%)")
    out()
    out("  A screen catching nothing exclusive is redundant with the others,")
    out("  not necessarily wrong. A screen with exclusive catches is earning")
    out("  its place and should be reported separately.")
    out()

    # ---------------------------------------------------------------- 3
    out("-" * 78)
    out("5. WHAT TO LOOK AT FIRST")
    out("-" * 78)
    out()
    prio = per.loc[sorted(union)].copy() if union else per.head(0).copy()
    if len(prio):
        prio["_gii"] = pd.to_numeric(prio.get("gii_shown"), errors="coerce")
        # v2.0.13: n_screens_failed is computed once in build_structure_table.
        # Every consumer used to rebuild it and they did not all use the same
        # screens, which is why the counts drifted between reports.
        if "n_screens_failed" in prio.columns:
            prio["_n"] = pd.to_numeric(prio["n_screens_failed"],
                                       errors="coerce").fillna(0).astype(int)
        else:
            prio["_n"] = 0
            for name, s in sets.items():
                prio["_n"] += prio.index.isin(s).astype(int)
        prio = prio.sort_values(["_n", "_gii"], ascending=[False, False])
        cols = [c for c in ["icsd_code", "chemical_formula", "year", "gii_shown"]
                if c in prio.columns]
        out(f"  Ranked by number of independent screens, then by GII. A structure")
        out(f"  failing several screens is failing for several reasons.")
        out()
        show = prio[cols].head(max_listed).copy()
        show.insert(0, "screens", prio["_n"].head(max_listed).values)
        for line in show.to_string(index=False).splitlines():
            out("      " + line)
        out()
        multi = int((prio["_n"] >= 2).sum())
        out(f"  {multi} structures fail two or more screens; {len(prio) - multi} "
            f"fail exactly one.")
    else:
        out("  Nothing flagged.")
    out()

    # repeat offenders: several entries of one composition, all flagged
    if "chemical_formula" in per.columns and union:
        grp = per.groupby("chemical_formula")
        rows = []
        for formula, g in grp:
            if len(g) < 3:
                continue
            n_bad = len(set(g.index) & union)
            if n_bad >= 2:
                rows.append((formula, n_bad, len(g)))
        if rows:
            rows.sort(key=lambda r: (-r[1] / r[2], -r[2]))
            out("  Compositions with several entries flagged. One bad structure")
            out("  is an accident; several of the same compound is a pattern,")
            out("  and often traces to one paper.")
            out()
            for formula, n_bad, n_tot in rows[:8]:
                out(f"      {str(formula)[:32]:<32} {n_bad} of {n_tot} flagged")
            out()

    out("=" * 78)
    txt = "\n".join(L)
    if verbose:
        print(txt)
    return txt


if "dfq" in globals() and dfq is not None:
    _triage_text = triage(dfq)
else:
    print("Run the post-processing cell first.")

In [ ]:
# CELL 13b: TEST THE PROPOSED SHIFTS   (Eir 2.0.13)
# ============================================================================
# The half-lattice screen proposes a candidate position for a cation whose
# coordination sphere is split. Up to here that proposal is a column: a spread
# that would improve and a bond valence sum that would move. This cell tests
# it properly, by applying the shift to a COPY of each CIF, running the whole
# analysis again, and writing a parallel dataset.
#
# What that buys, that the per-site numbers do not:
#
#   - every downstream quantity is recomputed, not just the shifted site. The
#     GII, the anion sums, the diagnosis and all five screens are evaluated on
#     the shifted structure, so a shift that fixes one cation and breaks the
#     rest is visible.
#   - the comparison is like for like, because the shifted CIF goes through
#     exactly the same code path as the original.
#   - it can DISAGREE. A proposal that leaves the structure no better is
#     reported as such, which is what makes the others worth believing.
#
# WHAT THIS IS NOT
# It is not a correction. The originals are never modified; the shifted files
# go to a separate directory, each carrying a header recording what was moved
# and why, and the output is a separate CSV to be read ALONGSIDE the quality
# CSV, never in place of it. A shift that improves the geometry is evidence of
# a mixed-origin deposition, not proof of one, and cannot be proven from a
# single deposited CIF. What this cell provides is the strongest evidence
# obtainable without going back to the diffraction data.

import os
import re
import shutil
from datetime import datetime

import pandas as pd


def _shift_cif_text(cif_text, site_label, shift, note=None):
    """
    Return the CIF with one site's fractional coordinates translated.

    Edits the atom_site loop in place rather than round-tripping through a
    parser, so everything else in the file survives untouched, including the
    fields Eir reads for provenance. Standard uncertainties are preserved: a
    translation does not change them, so 0.0250(8) shifted by 1/2 becomes
    0.5250(8).

    Returns (new_text, applied) where applied is False if the site or the
    coordinate columns could not be located, so a silent no-op is impossible.
    """
    lines = cif_text.split('\n')

    # locate the atom_site loop and the column order within it
    tags, start = [], None
    for i, ln in enumerate(lines):
        t = ln.strip()
        if t.startswith('_atom_site_'):
            if start is None or not tags:
                start = i
            tags.append(t.split()[0])
        elif tags and t and not t.startswith('_'):
            break
    if not tags:
        return cif_text, False
    try:
        ix = tags.index('_atom_site_label')
        cx = tags.index('_atom_site_fract_x')
        cy = tags.index('_atom_site_fract_y')
        cz = tags.index('_atom_site_fract_z')
    except ValueError:
        return cif_text, False

    data_start = start + len(tags)

    def _shift_value(txt, delta):
        """Add delta to a CIF number, keeping any standard uncertainty."""
        # CIFs write '1.' and '0.' as well as '0.2500'; a pattern requiring a
        # digit after the point rejects them. ICSD 420711 has Ga at fract_x
        # '1.' and was silently skipped for exactly that reason.
        m = re.match(r'^\s*([+-]?(?:\d+\.?\d*|\.\d+))(\(\d+\))?\s*$', txt)
        if not m:
            return None
        v = (float(m.group(1)) + delta) % 1.0
        digits = len(m.group(1).split('.')[1]) if '.' in m.group(1) else 4
        return f'{v:.{max(digits, 4)}f}' + (m.group(2) or '')

    applied = False
    for i in range(data_start, len(lines)):
        row = lines[i].strip()
        if not row or row.startswith('_') or row.startswith('loop_') \
                or row.startswith('#') or row.startswith('data_'):
            if applied:
                break
            continue
        parts = lines[i].split()
        if len(parts) < len(tags) or parts[ix] != site_label:
            continue
        new = list(parts)
        ok = True
        for col, d in ((cx, shift[0]), (cy, shift[1]), (cz, shift[2])):
            s = _shift_value(parts[col], d)
            if s is None:
                ok = False
                break
            new[col] = s
        if not ok:
            continue
        lines[i] = ' '.join(new)
        applied = True
        break

    if not applied:
        return cif_text, False

    header = [
        '# ---------------------------------------------------------------',
        '# MODIFIED COPY - NOT THE DEPOSITED STRUCTURE',
        f'# Site {site_label} translated by ({shift[0]}, {shift[1]}, {shift[2]})',
        '# by Eir 2.0.2, to TEST a suspected mixed-origin deposition.',
        '# This is a hypothesis under test, not a correction. The deposited',
        '# entry is unchanged; see the original CIF for the data of record.',
        '#',
        '# Only the fractional coordinates were touched. Any Wyckoff symbol,',
        '# site multiplicity or symmetry-equivalent count in this file still',
        '# describes the ORIGINAL position and is now inconsistent with the',
        '# coordinates. That is deliberate: rewriting them would assert the',
        '# shift is right, which is the question rather than the answer.',
    ]
    if note:
        header.append(f'# {note}')
    header += [f'# generated {datetime.now().isoformat(timespec="seconds")}',
               '# ---------------------------------------------------------']
    return '\n'.join(header + lines), True


def test_proposed_shifts(dfq, cif_directory, output_dir=None, verbose=True):
    """
    Apply every proposed shift to a copy, re-analyse, and write the results.

    v2.0.3: the verdict is decided by the SCREENS first and the GII second.

    The previous ladder was ordered on the GII, with the screens acting only as
    a veto and as a gate on the top two tiers. A shift could therefore clear
    every screen it had been failing and still be reported as not supported.
    ICSD 209078 (CaTiO3) settles it: Ti is deposited with a first-shell spread
    of 1.79 A and bond lengths of 1.65, 2.20 and 2.97 A, the shift takes the
    spread to 1.02 A and clears the one screen that was failing, and the GII
    rises from 0.207 to 0.422. The old rule returned NOT supported.

    The GII rises because the deposited Ti sits at CN 4 and sums to 3.890 v.u.,
    2.75% from formal: a short four-coordinate sum lands where a normal six-
    coordinate sum would. Bond valence is degenerate between too few bonds too
    short and the right number at the right length, so it cannot arbitrate a
    question about geometry. The screens can, and the verdict now keys on them.

    Where the two disagree the disagreement is REPORTED rather than resolved,
    in gii_direction and screens_gii_disagree. Those structures are the
    informative ones and should not be collapsed into a single word.

    Args:
        dfq: the post-processed quality DataFrame
        cif_directory: where the original CIFs live
        output_dir: where to write; defaults to <cif_directory>/outputs

    Returns the comparison DataFrame, one row per tested structure and variant.
    """
    if dfq is None or 'proposed_shift' not in dfq.columns:
        print('No proposed_shift column. Run the batch under Eir 2.0.2 first.')
        return None

    prop = dfq[dfq['proposed_shift'].notna()].copy()
    if not len(prop):
        print('No shifts proposed; nothing to test.')
        return None

    output_dir = output_dir or os.path.join(cif_directory, 'outputs')
    shifted_dir = os.path.join(output_dir, 'shifted_cifs')
    os.makedirs(shifted_dir, exist_ok=True)

    _SCREENS_BEFORE = ('anion_packing_plausible',
                       'structure_core_geometry_plausible',
                       'structure_bvs_geometry_plausible',
                       'structure_cn_plausible')
    _SCREENS_AFTER = ('anion_packing_after', 'core_geometry_after',
                      'bvs_geometry_after', 'cn_plausible_after')

    def _num(v):
        """A float, or None. Distinguishes a genuine zero from a missing value."""
        x = pd.to_numeric(pd.Series([v]), errors='coerce').iloc[0]
        return None if x != x else float(x)

    def _verdict(before, after, gb, ga):
        """
        Screens first, GII second.

        The screens test the physical claim the proposal makes: that the ions
        sit where the anion sublattice expects them. The GII is a derived index
        that cannot distinguish a deficient coordination sphere from a sound
        one, so it ranks within a screen outcome rather than across outcomes.
        """
        if ga is None:
            return 'inconclusive'
        if after > before:
            # More screens failing than before is disqualifying however much
            # the GII improved: the shift has moved the problem, not fixed it.
            return 'NOT supported'
        if before == 0 and after == 0:
            # Nothing was failing, so the re-run cannot adjudicate the proposal
            # either way. Reporting this as an outcome is honest; calling it
            # marginal on a GII change is not.
            return 'not diagnostic'
        if after == 0:
            return 'strongly supported' if before >= 2 else 'supported'
        if after < before:
            return 'partially supported'
        # after == before > 0: the shift changed nothing the screens can see
        return 'marginal' if (gb is not None and ga < gb) else 'NOT supported'

    print('=' * 78)
    print(f'TESTING {len(prop)} PROPOSED SHIFTS IN {prop["filename"].nunique()} STRUCTURES')
    print('=' * 78)
    print('  Originals are not modified. Shifted copies go to:')
    print(f'    {shifted_dir}')
    print()

    rows, written, skipped = [], [], []
    for fn, grp in prop.groupby('filename'):
        src = os.path.join(cif_directory, fn)
        if not os.path.exists(src):
            skipped.append((fn, 'CIF not found'))
            rows.append({'filename': fn, 'verdict': 'not tested: CIF not found'})
            continue
        try:
            with open(src, 'r', encoding='utf-8', errors='replace') as fh:
                original = fh.read()
        except OSError as e:
            skipped.append((fn, f'unreadable: {e}'))
            rows.append({'filename': fn, 'verdict': f'not tested: {e}'})
            continue

        try:
            _vec_str = str(grp.iloc[0]['proposed_shift'])
            shift_vec = [float(x) for x in _vec_str.split(',')]
        except ValueError:
            skipped.append((fn, 'unparseable shift'))
            continue

        # Two hypotheses, not one.
        #
        #   single_site  moves only the cations the screen named. Right when a
        #                lone cation was deposited on the wrong Wyckoff site.
        #   all_cations  moves every cation by the same vector. Right when the
        #                cation and anion sublattices as a whole sit on
        #                different origins, which a single-site move cannot
        #                describe.
        #
        # Both are tested and both are reported. Picking one silently would
        # hide the case where neither works, which is itself informative.
        named = [r.get('site_label') for _, r in grp.iterrows()]
        all_cats = sorted(set(dfq.loc[dfq['filename'] == fn, 'site_label']
                              .dropna().astype(str)))
        variants = [('single_site', named), ('all_cations', all_cats)]
        if set(named) == set(all_cats):
            variants = [('single_site', named)]

        for variant, labels in variants:
            text, detail, applied_any = original, [], False
            for lbl in labels:
                text, ok = _shift_cif_text(
                    text, lbl, shift_vec,
                    note=str(grp.iloc[0].get('shift_evidence') or ''))
                if ok:
                    applied_any = True
                    # v2.0.3: the label alone. Concatenating the vector with a
                    # '+' produced strings like 'Mn1+(0.5,0.0,0.0)', which read
                    # as manganese(I) rather than as site Mn1 plus a vector, in
                    # a column that ships as supplementary data.
                    detail.append(lbl)
                else:
                    skipped.append((fn, f'could not locate site {lbl}'))
            if not applied_any:
                rows.append({'filename': fn, 'variant': variant,
                             'icsd_code': grp.iloc[0].get('icsd_code'),
                             'chemical_formula': grp.iloc[0].get('chemical_formula'),
                             'verdict': 'not tested: site not found in CIF'})
                continue

            stem, ext = os.path.splitext(fn)
            dst = os.path.join(shifted_dir, f'{stem}__{variant}{ext}')
            with open(dst, 'w', encoding='utf-8') as fh:
                fh.write(text)
            written.append(dst)

            before = grp.iloc[0]
            n_screens_before = sum(1 for k in _SCREENS_BEFORE
                                   if str(before.get(k)).lower() == 'false')

            res = process_single_cif(dst)
            if not res.get('success'):
                rows.append({'filename': fn, 'variant': variant,
                             'icsd_code': before.get('icsd_code'),
                             'chemical_formula': before.get('chemical_formula'),
                             'sites_shifted': ', '.join(detail),
                             'shift_vector': f"({_vec_str})",
                             'shift_applied': f"{', '.join(detail)} by ({_vec_str})",
                             'gii_before': _num(before.get('gii')),
                             'gii_after': None,
                             'n_screens_failed_before': n_screens_before,
                             'n_screens_failed_after': None,
                             'gii_ratio': None,
                             'gii_direction': None,
                             'screens_gii_disagree': False,
                             'verdict': f"re-analysis failed: {res.get('error')}"})
                continue

            # v2.0.3: an explicit None test throughout. `x or default` treats a
            # genuine GII or WAGII of exactly 0.0 as missing, which is the one
            # value a perfectly regular structure can legitimately return.
            g0 = _num(before.get('gii'))
            g1 = _num(res.get('gii'))
            wa = _num(res.get('wagii'))

            row = {
                'filename': fn,
                'variant': variant,
                'icsd_code': before.get('icsd_code'),
                'chemical_formula': before.get('chemical_formula'),
                'shift_applied': '; '.join(detail),
                'gii_before': round(g0, 4) if g0 is not None else None,
                'gii_after': round(g1, 4) if g1 is not None else None,
                'wagii_after': round(wa, 4) if wa is not None else None,
                'anion_packing_after': res.get('anion_packing_plausible'),
                'core_geometry_after': res.get('core_geometry_plausible'),
                'bvs_geometry_after': res.get('bvs_geometry_plausible'),
                'cn_plausible_after': res.get('cn_plausible'),
                'n_shift_proposed_after': res.get('n_sites_shift_proposed'),
                'anion_bvs_after': (res.get('anion_summary') or {}).get('anion_bvs_mean'),
            }
            row['gii_ratio'] = (round(g0 / g1, 2)
                                if g0 is not None and g1 not in (None, 0.0)
                                else None)
            row['n_screens_failed_after'] = sum(
                1 for k in _SCREENS_AFTER if row.get(k) is False)
            row['n_screens_failed_before'] = n_screens_before

            row['verdict'] = _verdict(n_screens_before,
                                      row['n_screens_failed_after'], g0, g1)

            # Reported alongside the verdict, not folded into it. Where the
            # geometry improves and the GII does not, that is a result about
            # bond valence rather than about the structure.
            if g0 is None or g1 is None:
                row['gii_direction'] = None
            elif g1 < g0:
                row['gii_direction'] = 'improves'
            elif g1 > g0:
                row['gii_direction'] = 'worsens'
            else:
                row['gii_direction'] = 'unchanged'
            row['screens_gii_disagree'] = bool(
                row['verdict'] in ('strongly supported', 'supported',
                                   'partially supported')
                and row['gii_direction'] == 'worsens')

            rows.append(row)

            if verbose:
                mark = '   [GII disagrees]' if row['screens_gii_disagree'] else ''
                print(f"  {fn}  [{variant}]")
                print(f"    {row['shift_applied']}")
                print(f"    screens {row['n_screens_failed_before']} -> "
                      f"{row['n_screens_failed_after']}   "
                      f"GII {row['gii_before']} -> {row['gii_after']}   "
                      f"[{row['verdict']}]{mark}")

    out = pd.DataFrame(rows)
    if len(out):
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        path = os.path.join(output_dir, f'eir_shift_test_{stamp}.csv')
        out.to_csv(path, index=False)
        print()
        print('-' * 78)
        if 'variant' in out.columns and out['variant'].nunique() > 1:
            print(pd.crosstab(out['variant'], out['verdict']).to_string())
            print()
            _needed = {'n_screens_failed_after', 'gii_after'}
            _scored = (out.dropna(subset=list(_needed))
                       if _needed <= set(out.columns) else out.iloc[0:0])
            if len(_scored):
                best = (_scored.sort_values(['n_screens_failed_after', 'gii_after'])
                               .drop_duplicates('filename'))
                print('  Better hypothesis per structure:')
                print('   ', best['variant'].value_counts().to_dict())
            else:
                print('  No structure was re-analysed successfully, so the two')
                print('  hypotheses cannot be compared. The verdicts above are')
                print('  reporting a failure to run, not a failure of the shift.')
        else:
            print(out['verdict'].value_counts().to_string())

        dis = out[out.get('screens_gii_disagree', False) == True]
        if len(dis):
            print()
            print(f'  SCREENS AND GII DISAGREE on {len(dis)} of {len(out)}:')
            print('  the shift clears or reduces screen failures while the GII')
            print('  worsens. Bond valence cannot distinguish a deficient')
            print('  coordination sphere from a sound one, so these are')
            print('  reported rather than resolved. Look at them by eye.')
            print()
            for _, r in dis.iterrows():
                print(f"    {r['icsd_code']}  {r['chemical_formula']:<18} "
                      f"[{r['variant']}]  screens "
                      f"{r['n_screens_failed_before']} -> "
                      f"{r['n_screens_failed_after']}   "
                      f"GII {r['gii_before']} -> {r['gii_after']}")

        print('-' * 78)
        print(f'  {len(written)} shifted CIFs written to {shifted_dir}')
        print(f'  comparison written to {path}')
        print()
        print('  Read this ALONGSIDE the quality CSV, not in place of it. A')
        print('  supported shift is evidence that the deposited cation and')
        print('  anion sublattices sit on different origins; it is not a')
        print('  correction, and the deposited entry remains the data of')
        print('  record until the source says otherwise.')
    if skipped:
        print()
        print(f'  {len(skipped)} not tested:')
        for fn, why in skipped[:10]:
            print(f'    {fn}: {why}')
    return out



def apply_shift_verdicts(dfq, shift_results, structures_csv=None,
                         sites_csv=None, verbose=True):
    """
    Write the tested verdicts back onto the site table and re-classify.

    The half-lattice screen proposes; this cell tests; PGRM must then say
    what the test found. Without this the G4 reason reads
    cation_on_wrong_origin for all 25 perovskite proposals, whether the
    shift cleared every screen it was failing (20 of them) or changed
    nothing the screens can see (1). Those are different claims and the
    reportable-error list should not flatten them.

    Only the single_site variant is used. all_cations is the CONTROL: it
    describes a whole cation sublattice on the wrong origin, and on the
    perovskites it is NOT supported 23 times of 25, which is what says
    these are single misplaced cations rather than a sublattice error.
    """
    if shift_results is None or not len(shift_results):
        return dfq
    ss = shift_results[shift_results.get("variant") == "single_site"]
    if not len(ss):
        return dfq

    _v, _e = {}, {}
    for _, r in ss.iterrows():
        fn = r.get("filename")
        if not fn:
            continue
        _v[fn] = r.get("verdict")
        _b, _a = r.get("n_screens_failed_before"), r.get("n_screens_failed_after")
        _gb, _ga = r.get("gii_before"), r.get("gii_after")
        _e[fn] = (f'shift {r.get("proposed_shift", "?")} on '
                  f'{r.get("shift_applied", r.get("sites_shifted", "?"))}: '
                  f'screens {_b} -> {_a}, GII {_gb} -> {_ga} '
                  f'[{r.get("verdict")}]')

    dfq = dfq.copy()
    _mask = dfq["proposed_shift"].notna() if "proposed_shift" in dfq else None
    dfq["shift_verdict"] = dfq["filename"].map(_v)
    dfq["shift_evidence"] = dfq["filename"].map(_e)
    if _mask is not None:
        # Only the site that carries the proposal gets the verdict; the
        # other sites of the same structure were not shifted.
        dfq.loc[~_mask, ["shift_verdict", "shift_evidence"]] = None

    if "classify_pgrm" in globals():
        dfq = classify_pgrm(dfq)
        if "apply_pgrm_suppression" in globals():
            dfq = apply_pgrm_suppression(dfq)
        if verbose:
            _st = dfq.drop_duplicates("filename")
            _g4 = _st[_st["structure_pgrm_code"].astype(str).str.startswith("G4")
                      | _st["structure_pgrm_reason"].astype(str).isin(
                          ["cation_on_wrong_origin", "cation_position_suspect",
                           "split_shell_cause_unknown",
                           "split_shell_not_diagnostic"])]
            print()
            print("  PGRM re-classified with the tested verdicts:")
            for _r, _n in _g4["structure_pgrm_reason"].value_counts().items():
                print(f"      {str(_r):32} {_n:4}")

    # Re-save so the reportable-error list is self-contained rather than
    # needing a join to eir_shift_test_*.csv.
    if sites_csv:
        dfq.to_csv(sites_csv, index=False)
    if structures_csv and "build_structure_table" in globals():
        build_structure_table(dfq).to_csv(structures_csv, index=False)
    if verbose and (sites_csv or structures_csv):
        print("  site and structure tables rewritten with the verdicts.")
    return dfq

def resolve_cif_directory(dfq, explicit=None, ask=True):
    """
    Find the folder holding the original CIFs, rather than asking for it.

    The notebook already knows: the batch cell picked it, the post-processing
    cell was handed it, and the outputs sit inside it. Asking again is friction
    for something already determined, so this looks in the obvious places and
    verifies each candidate the only way that settles it -- by checking that a
    CIF named in dfq is actually there.

    Order: an explicit argument, then any global holding a plausible path,
    then the parents of any 'outputs' directory among those globals, then the
    working directory and its subdirectories. Only if all of that fails, and
    only if ask is True, does it open a picker.
    """
    if dfq is None or 'filename' not in dfq.columns or not len(dfq):
        return explicit
    probe = str(dfq['filename'].iloc[0])

    def _ok(d):
        try:
            return bool(d) and os.path.exists(os.path.join(str(d), probe))
        except (TypeError, ValueError):
            return False

    if _ok(explicit):
        return explicit

    candidates = []
    g = globals()
    for name, val in list(g.items()):
        if not isinstance(val, str) or len(val) < 3 or name.startswith('__'):
            continue
        if not any(k in name.lower() for k in
                   ('dir', 'path', 'folder', 'cif', 'csv', 'directory')):
            continue
        candidates.append(val)
        # a results CSV or an outputs folder sits one level inside the CIFs
        if val.lower().endswith('.csv'):
            candidates.append(os.path.dirname(os.path.dirname(val)))
        base = os.path.basename(os.path.normpath(val)).lower()
        if base == 'outputs':
            candidates.append(os.path.dirname(os.path.normpath(val)))
    candidates += [os.getcwd(), os.path.dirname(os.getcwd())]
    try:
        for entry in os.listdir(os.getcwd()):
            full = os.path.join(os.getcwd(), entry)
            if os.path.isdir(full):
                candidates.append(full)
    except OSError:
        pass

    seen = set()
    for c in candidates:
        c = str(c)
        if c in seen:
            continue
        seen.add(c)
        if _ok(c):
            print(f'  CIF directory found: {c}')
            return c

    if not ask:
        return None
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        try:
            d = filedialog.askdirectory(
                title='Select the folder holding the original CIFs',
                mustexist=True)
        finally:
            root.quit()
            root.destroy()
        if _ok(d):
            return d
        if d:
            print(f'  {probe} is not in {d}; wrong folder?')
    except Exception as e:
        print(f'  could not open a folder picker ({e})')
    return None


# ---------------------------------------------------------------------------
CIF_DIRECTORY = None    # normally left as None; resolved automatically below

if 'dfq' not in globals() or dfq is None:
    print('Run the post-processing cell first.')
else:
    _cif_dir = resolve_cif_directory(dfq, CIF_DIRECTORY)
    if not _cif_dir:
        print('Could not locate the original CIFs. Set CIF_DIRECTORY '
              'explicitly and re-run.')
    else:
        _shift_results = test_proposed_shifts(dfq, _cif_dir)

        # v2.0.13: feed the verdicts back, so the G4 reason says what the
        # test found rather than that a proposal existed, and the
        # reportable-error list needs no join to the shift CSV.
        _sites_csv = _struct_csv = None
        if 'EIR_BATCH_CSV' in globals():
            _base = os.path.splitext(EIR_BATCH_CSV)[0]
            _sites_csv = _base + '_quality.csv'
            _struct_csv = _base + '_structures.csv'
        dfq = apply_shift_verdicts(dfq, _shift_results,
                                   structures_csv=_struct_csv,
                                   sites_csv=_sites_csv)
        print()
        print('  Re-run the triage cell to see the refined verdicts.')

In [ ]:
# CELL 13c: FIGURES   (Eir 2.0.13)
# ============================================================================
# Three diagnostics that Eir needs to answer its own question. Everything
# pool-level - GII against publication year, deviation by element, provenance
# by decade - stays in Huginn, which is where cross-run analysis belongs.
#
# CONVENTIONS, and they are not negotiable in a submitted figure:
#
#   - Axis labels are QUANTITY CALCULUS: "GII / v.u.", never "GII (v.u.)".
#     The label names a pure number, so the unit divides the quantity.
#   - Single column is 3.35 in and double is 6.9 in. Three panels crammed
#     across a double column gives each 2.3 in, which is a poster habit.
#   - No gridlines, no top or right spine, no frame on the legend, no
#     chartjunk. Nothing is drawn that does not carry information.
#   - Clipped values are COUNTED in an annotation, never piled into the end
#     bin: a clipped tail draws a spike that is an artefact of the clip.
#   - An absent column is not a zero. A screen that was never computed is
#     omitted rather than drawn as a bar of length nothing.
#   - 600 dpi raster; save_figure() also writes PDF, which is what to submit.
#
# Colours are the University of Sheffield brand palette. Brand violet against
# coral separates on lightness as well as hue, so it survives greyscale and
# the common colour-vision deficiencies; the categorical cycle is the
# accessibility-optimised eight.

import matplotlib
import matplotlib.pyplot as plt

SHEFFIELD = {
    'brand_violet': '#440099', 'dark_violet': '#24125E',
    'electric_violet': '#7000FF', 'powder_violet': '#A78CFA',
    'pale_violet': '#C8BBFF', 'wave_white': '#F8F8F9',
    'midnight_black': '#131E29', 'coral': '#E7004C', 'flamingo': '#FF6371',
    'peach': '#FF9664', 'pink': '#FF79E1', 'purple': '#981F92',
    'lavender': '#DAA8E2', 'teal': '#005A8F', 'aqua': '#00BBCC',
    'powder_blue': '#9ADBE8', 'peak_green': '#005750',
    'spearmint': '#3BD4AE', 'pastel_green': '#A1DED2',
}
CAT8 = [SHEFFIELD[k] for k in ('brand_violet','coral','teal','peak_green',
                               'peach','aqua','purple','spearmint')]

def set_style():
    plt.rcParams.update({
        'figure.dpi': 150, 'savefig.dpi': 600,
        'font.family': 'sans-serif',
        'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
        'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
        'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
        'ytick.major.width': 0.6, 'xtick.direction': 'out',
        'ytick.direction': 'out',
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.grid': False, 'legend.frameon': False,
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
        # v2.0.13a: NOT bbox='tight'. Cropping to content gives every figure a
        # different size, because the crop follows the longest axis label and
        # the number of legend rows. constrained_layout fits the content
        # inside a FIXED canvas instead, so a set of figures can sit side by
        # side in a manuscript without rescaling.
        'figure.constrained_layout.use': True,
        'figure.constrained_layout.h_pad': 0.04,
        'figure.constrained_layout.w_pad': 0.04,
        'axes.prop_cycle': plt.cycler(color=CAT8),
    })

W1, W2 = 3.35, 6.9      # single- and double-column inches
H1 = 3.1                # standard height, so a set of figures matches


def _axis_window(values, reference=None, pad=1.15, floor=0.05):
    """
    An upper limit that shows the DATA rather than the outliers.

    A percentile of the whole set is useless when n is small and the tail is
    long: on a 30-structure run one GII of 75.7 put the 97th percentile at
    33, so every real point collapsed into the corner and the figure read as
    blank. The window is therefore set from the REFERENCE population - the
    structures that passed every gate - and everything outside it is counted
    in an annotation rather than compressed into the frame.
    """
    v = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    ref = v if reference is None else pd.to_numeric(
        pd.Series(reference), errors='coerce').dropna()
    if not len(ref):
        ref = v
    if not len(ref):
        return floor
    return max(floor, float(ref.max()) * pad)

def fig_screen_overlap(per, ax=None):
    screens = [('anion packing','anion_packing_plausible'),
               ('cation packing','cation_packing_plausible'),
               ('bond-valence dist.','structure_bvs_geometry_plausible'),
               ('radii ratio','structure_core_geometry_plausible'),
               ('coordination','structure_cn_plausible')]
    sets={}
    for n,c in screens:
        if c in per.columns:            # absent is not the same as zero
            sets[n]=set(per.index[per[c].astype(str).str.lower().eq('false')])
    if 'n_sites_shift_proposed' in per:
        sets['wrong origin']=set(per.index[pd.to_numeric(
            per['n_sites_shift_proposed'],errors='coerce').fillna(0)>0])
    names=list(sets); tot=[len(sets[n]) for n in names]
    exc=[len(sets[n]-set().union(*[v for k,v in sets.items() if k!=n])) for n in names]
    o=np.argsort(tot)[::-1]
    names=[names[i] for i in o]; tot=[tot[i] for i in o]; exc=[exc[i] for i in o]
    if ax is None: _,ax=plt.subplots(figsize=(W1,2.3))
    y=np.arange(len(names))
    ax.barh(y,tot,color=SHEFFIELD['pale_violet'],height=0.62,zorder=2)
    ax.barh(y,exc,color=SHEFFIELD['brand_violet'],height=0.62,zorder=3)
    for i,(t,e) in enumerate(zip(tot,exc)):
        ax.text(t+max(tot)*0.015,i,f'{t}',va='center',ha='left',fontsize=7,
                color=SHEFFIELD['midnight_black'])
    ax.set_yticks(y); ax.set_yticklabels(names)
    ax.invert_yaxis(); ax.set_xlabel('Structures flagged')
    ax.set_xlim(0,max(tot)*1.14)
    import matplotlib.patches as _mp
    ax.legend(handles=[_mp.Patch(color=SHEFFIELD['brand_violet'],
                                 label='this screen alone'),
                       _mp.Patch(color=SHEFFIELD['pale_violet'],
                                 label='also caught elsewhere')],
              loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2,
              handlelength=1.0, columnspacing=1.2, borderaxespad=0.0)
    return ax

def fig_gii_distribution(per, ax=None, tier_col='structure_pgrm_tier'):
    g=pd.to_numeric(per.get('gii_recommended_value',per.get('gii')),errors='coerce')
    ok=per[tier_col].eq('ACCEPT') if tier_col in per else pd.Series(True,index=per.index)
    if ax is None: _,ax=plt.subplots(figsize=(W1,2.3))
    lo, hi = 0.0, 0.60
    bins = np.linspace(lo, hi, 31)
    a, b = g[ok].dropna(), g[~ok].dropna()
    # Values beyond the axis are COUNTED IN THE ANNOTATION, not piled into the
    # last bin: a clipped tail draws a spike that is an artefact of the clip.
    ax.hist([a[a <= hi], b[b <= hi]], bins=bins, stacked=True,
            color=[SHEFFIELD['brand_violet'], SHEFFIELD['coral']],
            label=[f'passed every gate  n = {len(a)}',
                   f'flagged or expunged  n = {len(b)}'], zorder=3)
    ax.set_xlabel('Global instability index / v.u.')
    ax.set_ylabel('Structures')
    ax.set_xlim(lo, hi)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=1,
              handlelength=1.1, borderaxespad=0.0)
    _over = int((a > hi).sum()), int((b > hi).sum())
    if sum(_over):
        ax.text(0.97, 0.95, f'beyond {hi:g} v.u.:  {_over[0]} + {_over[1]}',
                transform=ax.transAxes, ha='right', va='top', fontsize=6.5,
                color=SHEFFIELD['midnight_black'])
    return ax

def fig_cation_vs_anion(per, ax=None, tier_col='structure_pgrm_tier',
                        gii_tol=0.30, anion_tol=0.20):
    """
    The cation and anion sides of the same bonds, on one pair of axes.

    Every bond contributes to one cation sum and one anion sum, so a cation
    discrepancy that CANCELS at the shared oxygen is a property of the
    parameters and one that does not is a property of the structure. The four
    quadrants are the four bvs_diagnosis labels; the guide lines are the
    tolerances that define them, so the figure and the column cannot disagree.
    """
    _yc = next((c for c in ('anion_deviation_mean_weighted',
                            'anion_deviation_mean') if c in per.columns), None)
    if _yc is None or 'gii' not in per.columns:
        return None
    x = pd.to_numeric(per['gii'], errors='coerce')
    y = pd.to_numeric(per[_yc], errors='coerce')
    if ax is None:
        _, ax = plt.subplots(figsize=(W1, H1))
    ok = (per[tier_col].eq('ACCEPT') if tier_col in per
          else pd.Series(True, index=per.index))

    # Axis window set from the bulk, with everything outside COUNTED rather
    # than squashed against the frame: three structures with a GII of 17 make
    # the other 935 unreadable.
    xhi = max(gii_tol * 2, _axis_window(x, reference=x[ok], floor=gii_tol * 2))
    _yr = y[ok].dropna()
    yhi = max(anion_tol * 2,
              (float(_yr.max()) * 1.15 if len(_yr) else anion_tol * 2))
    ylo = min(-anion_tol * 2,
              (float(_yr.min()) * 1.15 if len(_yr) else -anion_tol * 2))
    inside = x.le(xhi) & y.between(ylo, yhi)

    ax.axvline(gii_tol, color=SHEFFIELD['midnight_black'], lw=0.5, ls=':', zorder=1)
    for _h in (anion_tol, -anion_tol):
        ax.axhline(_h, color=SHEFFIELD['midnight_black'], lw=0.5, ls=':', zorder=1)
    ax.scatter(x[ok & inside], y[ok & inside], s=7, lw=0, alpha=0.7,
               c=SHEFFIELD['brand_violet'], zorder=3, label='passed every gate')
    ax.scatter(x[~ok & inside], y[~ok & inside], s=7, lw=0, alpha=0.7,
               c=SHEFFIELD['coral'], zorder=2, label='flagged or expunged')
    ax.set_xlim(0, xhi)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel('Cation GII / v.u.')
    ax.set_ylabel('Mean anion deviation / v.u.')

    _lab = dict(fontsize=6.5, color=SHEFFIELD['midnight_black'], zorder=4,
                style='italic')
    ax.text(0.03, 0.97, 'anions only', transform=ax.transAxes,
            ha='left', va='top', **_lab)
    ax.text(0.97, 0.97, 'both deviate', transform=ax.transAxes,
            ha='right', va='top', **_lab)
    ax.text(0.03, 0.03, 'both balance', transform=ax.transAxes,
            ha='left', va='bottom', **_lab)
    ax.text(0.97, 0.03, 'cations only', transform=ax.transAxes,
            ha='right', va='bottom', **_lab)
    _n_out = int((~inside).sum())
    _extra = f'   {_n_out} beyond the axes' if _n_out else ''
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2,
              handlelength=0.9, columnspacing=1.2, borderaxespad=0.0,
              title=_extra.strip() or None, title_fontsize=6)
    return ax


def fig_gii_vs_wagii(per, ax=None):
    # v2.0.13a: the signature was (per, dfq=None, ax=None) and dfq was never
    # used, so eir_figures calling _fn(per, _ax) bound the AXIS to dfq. The
    # function then made its own figure, drew on that, and returned it - and
    # the axis that was saved stayed empty. The figure was written blank and
    # the call reported success.
    """
    The unweighted and occupancy-weighted instability indices against each
    other, with the identity line.

    They are the SAME quantity when every occupancy is 1.0 - not, as the
    reports long said, when there is no anti-site disorder. WAGII weights each
    site by its occupancy, so a vacancy separates them just as a shared site
    does. Measured across three families, full occupancy gives gii == wagii on
    869 of 871 perovskites, 355 of 355 pyrochlores and 335 of 338 spinels,
    while 1543 partially occupied spinels sit off the line.

    Distance from the line is therefore a direct measure of how much the
    unweighted index is inflated by sites that are not fully there, and the
    far outliers are worth opening: Li(0.93)Ti2O4 (ICSD 73020) reports a GII
    of 0.871 against a WAGII of 0.203, a factor of 4.3.
    """
    if 'gii' not in per.columns or 'wagii' not in per.columns:
        return None
    g = pd.to_numeric(per['gii'], errors='coerce')
    w = pd.to_numeric(per['wagii'], errors='coerce')
    ok = g.notna() & w.notna()
    if not ok.any():
        return None
    if ax is None:
        _, ax = plt.subplots(figsize=(W1, H1))

    # three causes, in order of severity of the effect
    anti = (per.get('has_antisite_disorder', pd.Series(False, index=per.index))
            .astype(str).str.lower().isin(['true', '1']))
    same = (g - w).abs() < 1e-9
    partial = ~same & ~anti

    # Window from the structures that PASSED THE GATES, not from those where
    # the two indices agree. A broken structure can be fully occupied and so
    # sit exactly on the line: ICSD 220506 has a La-O contact of 1.008 A, a
    # GII of 17.29, and gii == wagii. Using agreement as the reference let it
    # set the window at 19.9 while every other point sat below 1, and the
    # figure read as blank.
    _ref_mask = (per['structure_pgrm_tier'].eq('ACCEPT')
                 if 'structure_pgrm_tier' in per.columns else same)
    _ref = pd.concat([g[ok & _ref_mask], w[ok & _ref_mask]])
    hi = _axis_window(pd.concat([g[ok], w[ok]]), reference=_ref)
    ax.plot([0, hi], [0, hi], color=SHEFFIELD['midnight_black'], lw=0.6,
            ls='--', zorder=1)
    for _m, _c, _l in ((same, SHEFFIELD['brand_violet'], 'every site fully occupied'),
                       (partial, SHEFFIELD['aqua'], 'partial occupancy'),
                       (anti, SHEFFIELD['coral'], 'site shared between species')):
        _s = _m & ok & g.le(hi) & w.le(hi)
        if _s.any():
            ax.scatter(w[_s], g[_s], s=8, lw=0, alpha=0.75, c=_c,
                       label=f'{_l}  n = {int((_m & ok).sum())}', zorder=3)
    ax.set_xlim(0, hi)
    ax.set_ylim(0, hi)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('WAGII, occupancy weighted / v.u.')
    ax.set_ylabel('GII, unweighted / v.u.')
    ax.text(0.97, 0.06, 'on the line: the two agree', transform=ax.transAxes,
            ha='right', fontsize=6.5, style='italic',
            color=SHEFFIELD['midnight_black'])
    _out = int((ok & ((g > hi) | (w > hi))).sum())
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.26), ncol=1,
              handlelength=0.9, borderaxespad=0.0,
              title=(f'{_out} beyond the axes' if _out else None),
              title_fontsize=6)
    return ax


def save_figure(fig, path_stem, formats=("pdf", "png")):
    """Write a figure for submission. PDF is the one to send."""
    _d = os.path.dirname(path_stem)
    if _d:
        os.makedirs(_d, exist_ok=True)
    written = []
    for _f in formats:
        _p = f"{path_stem}.{_f}"
        fig.savefig(_p, format=_f)
        written.append(_p)
    return written


def eir_figures(dfq, output_dir=None, stem="eir"):
    """
    The three diagnostics, each as its own single-column figure.

    Returns {name: figure}. Written to output_dir when one is given.
    """
    set_style()
    per = (build_structure_table(dfq).set_index("filename")
           if "build_structure_table" in globals()
           else dfq.drop_duplicates("filename").set_index("filename"))
    made, out = {}, {}
    for _name, _fn in (("screens", fig_screen_overlap),
                       ("gii", fig_gii_distribution),
                       ("cation_anion", fig_cation_vs_anion),
                       ("gii_vs_wagii", fig_gii_vs_wagii)):
        _f, _ax = plt.subplots(figsize=(W1, H1))
        if _fn(per, ax=_ax) is None:      # BY KEYWORD, deliberately
            plt.close(_f)
            print(f"  {_name}: the columns it needs are not present; skipped")
            continue
        made[_name] = _f
        if output_dir:
            out[_name] = save_figure(_f, os.path.join(output_dir,
                                                      f"{stem}_{_name}"))
    if out:
        print(f"  {len(out)} figure(s) written to {output_dir}")
    return made


print("Figures ready: eir_figures(dfq, output_dir). Palette: Sheffield brand.")


In [ ]:
# MEASUREMENT TOOLS   (Eir 2.0.13)
#
# v2.0.13: _shell_by_gap and _shell_by_distance removed. They were the gap
# and fixed-distance shell rules, kept as comparators in compare_shell_rules
# and left behind when that comparison was settled: the bond-valence shell is
# canonical, the gap shell survives as cn_gap_shell for the CN-fault test, and
# nothing needed a fixed-distance rule any longer.
# Nothing here changes a result. These cells answer questions about the
# analysis, and each one is a question that was settled by measuring rather
# than by arguing. Placed after CELL 13b because they need the CIFs, which are
# not resolved until the batch runs.
#
# ONE copy of the input resolver, shared by all three. If you have loose
# measurement cells pasted elsewhere in this notebook, delete them: two copies
# of a helper is how three copies of JAHN_TELLER_IONS happened.

import numpy as np
import pandas as pd

ANION_SWEEP = [0.08, 0.04, 0.02, 0.01]
_bl_dist_factors = [1.20, 1.30, 1.40]
_bl_bv_cutoffs = [0.01, 0.02, 0.04, 0.08]


def _resolve_eir_inputs(quality_csv=None, cif_directory=None):
    """Find the CIF directory and the newest quality CSV from what is to hand."""
    import glob as _glob

    if not cif_directory:
        cif_directory = globals().get('_cif_dir')
    if not cif_directory and 'resolve_cif_directory' in globals():
        cif_directory = resolve_cif_directory(globals().get('dfq'), None, ask=False)
    if not cif_directory or not os.path.isdir(cif_directory):
        print('Could not locate the CIF directory. Pass cif_directory and re-run.')
        return None, None

    if not quality_csv:
        _hits = sorted(_glob.glob(os.path.join(cif_directory, 'outputs',
                                               '*_quality.csv')),
                       key=os.path.getmtime, reverse=True)
        quality_csv = _hits[0] if _hits else None
    if not quality_csv or not os.path.exists(quality_csv):
        print('No *_quality.csv found. Pass quality_csv and re-run.')
        return None, None

    print(f'  CIF directory: {cif_directory}')
    print(f'  quality CSV  : {os.path.basename(quality_csv)}')
    return quality_csv, cif_directory


def _clean_sites(df):
    """Dominant species in structures that pass every geometry screen."""
    df = df[df['is_dominant_species'].astype(str).eq('True')]
    keep = ~(df['structure_core_geometry_plausible'].astype(str).eq('False')
             | df['structure_bvs_geometry_plausible'].astype(str).eq('False')
             | df['anion_packing_plausible'].astype(str).eq('False')
             | df['structure_cn_plausible'].astype(str).eq('False'))
    return df[keep]


# ---------------------------------------------------------------------------
# 1. MEASURED AGAINST ADDITIVE EXPECTED DISTANCES
# ---------------------------------------------------------------------------
def compare_expected_distances(quality_csv=None, cif_directory=None):
    """
    Recompute the radii-screen ratio against MEASURED bond lengths and report
    the coordination dependence that RADII_CN_SLOPE exists to correct.

    If the expected distances are right at every coordination, that dependence
    should largely vanish and a refitted slope should measure near zero. The
    ratio_min column is the one to watch: it compares the shortest contact in
    this structure against the shortest ever observed for this ion
    configuration, which is like against like and needs no fitted threshold.
    """
    quality_csv, _ = _resolve_eir_inputs(quality_csv, cif_directory)
    if quality_csv is None:
        return None

    d = pd.read_csv(quality_csv)
    d = d[d['is_dominant_species'].astype(str).eq('True')]
    d = d.dropna(subset=['min_bond_length', 'cn', 'valence'])

    rows = []
    for _, r in d.iterrows():
        lig = (r.get('fields_describe_ligand')
               or str(r.get('ligand_types') or 'O').split(',')[0] or 'O')
        e = expected_bond_length(r['element'], r['valence'], r['cn'], lig)
        if not e:
            continue
        rows.append({'icsd': r.get('icsd_code'), 'element': r['element'],
                     'cn': int(r['cn']), 'gii': r.get('gii'),
                     'observed': r['min_bond_length'],
                     'ratio_mean': r['min_bond_length'] / e['mean'],
                     'ratio_min': (r['min_bond_length'] / e['min']
                                   if e['min'] else np.nan),
                     'ratio_shannon': r.get('core_shortest_contact_ratio'),
                     'tier': e['tier'], 'source': e['source']})
    out = pd.DataFrame(rows)
    if not len(out):
        print('  nothing resolved')
        return None
    print(f'  {len(out)} sites resolved of {len(d)}')
    print()
    print('  TIER COVERAGE')
    _lab = {1: 'measured', 2: 'interpolated', 3: 'extrapolated',
            4: 'Shannon additive'}
    for t, g in out.groupby('tier'):
        print(f'    {t}  {_lab[t]:18} {len(g):5}  ({100 * len(g) / len(out):.1f}%)')

    sound = out[pd.to_numeric(out['gii'], errors='coerce') < 0.25]
    print()
    print('  CONTACT RATIO BY COORDINATION, structures with GII < 0.25')
    print(f"  {'CN':>4} {'n':>6} {'vs Shannon':>12} {'vs measured mean':>18} "
          f"{'vs measured min':>17}")
    for cn, g in sound.groupby('cn'):
        if len(g) < 20:
            continue
        print(f'  {cn:4d} {len(g):6d} {g["ratio_shannon"].median():12.3f} '
              f'{g["ratio_mean"].median():18.3f} {g["ratio_min"].median():17.3f}')
    print()
    print('  The Shannon column falls with CN; that gradient is what')
    print('  RADII_CN_SLOPE corrects. If the measured columns are flat, the')
    print('  slope was compensating for a wrong expected distance and can go.')
    return out


# ---------------------------------------------------------------------------
# 2. WHICH CUT-OFF SHOULD DEFINE THE COORDINATION SHELL
# ---------------------------------------------------------------------------
def _shell_by_valence_adaptive(ds, R0, b, valence, cn_seed, element, anion):
    """Measured shell boundary, seeded from the gap shell. See bv_shell_boundary."""
    cn, _ = bv_shell_cn_adaptive(ds, R0, b, valence, cn_seed,
                                 element=element, anion=anion)
    return sorted(ds)[:cn] if cn else []


def _shell_by_valence(ds, R0, b, s_min):
    out = []
    for d in sorted(ds):
        if np.exp((R0 - d) / b) < s_min:
            break
        out.append(d)
    return out


def _bvs_of(ds, R0, b):
    return float(sum(np.exp((R0 - d) / b) for d in ds))


def compare_shell_rules(quality_csv=None, cif_directory=None):
    """
    Rebuild the bond valence sum under several shell rules and report the
    deviation by site type. The B site is the control: an octahedron has one
    shell however you cut it, so any rule that moves it is wrong whatever it
    does to the larger cation. Read the CN columns before the deviations.

    NOTE: the A/B labelling below is PEROVSKITE-SPECIFIC. It splits the
    cations of each structure by radius, which names the two sites of an
    ABO3 correctly and means nothing in a rock salt or a spinel with three
    cations. This is a measurement tool for that family, not a general one.
    """
    quality_csv, cif_directory = _resolve_eir_inputs(quality_csv, cif_directory)
    if quality_csv is None:
        return None

    df = pd.read_csv(quality_csv)
    _before = df['filename'].nunique()
    df = _clean_sites(df)
    df = df[df['contact_above_floor'].astype(str).ne('False')]
    print(f'  {_before - df["filename"].nunique()} structures excluded: '
          f'at least one geometry screen failed')
    print(f'  {len(df)} dominant cation sites in {df["filename"].nunique()} structures')

    rad = pd.to_numeric(df['cation_radius'], errors='coerce')
    df = df.assign(_rad=rad)
    df['_site_type'] = np.where(
        df['_rad'] >= df.groupby('filename')['_rad'].transform('max') - 1e-9,
        'A', 'B')

    cache, rows = {}, []
    for _, r in df.iterrows():
        fn = r['filename']
        path = os.path.join(cif_directory, fn)
        if path not in cache:
            try:
                cache[path] = neighbour_distances(path)
            except Exception as e:
                print(f'  ! {fn}: {type(e).__name__}: {e}')
                cache[path] = {}
        per_anion = _distances_for_row(cache[path], r.get('coords'), r.get('element'))
        if not per_anion:
            continue
        el, v = r.get('element'), r.get('valence')
        params = {}
        for ael in per_anion:
            p = get_bv_parameters_fractional(el, v, ael)
            if p and p[0] is not None:
                params[ael] = p
        if not params:
            continue

        rec = {'filename': fn, 'icsd_code': r.get('icsd_code'),
               'element': el, 'valence': v, 'site_type': r['_site_type']}

        def total(rule, *args):
            s, n = 0.0, 0
            for ael, dl in per_anion.items():
                if ael not in params:
                    continue
                R0, b = params[ael]
                if rule is _shell_by_valence:
                    sh = _shell_by_valence(dl, R0, b, *args)
                elif rule is _shell_by_valence_adaptive:
                    sh = _shell_by_valence_adaptive(dl, R0, b, args[0], args[1],
                                                    el, ael)
                else:
                    sh = rule(dl, *args)
                s += _bvs_of(sh, R0, b)
                n += len(sh)
            return s, n

        rec['bvs_gap'], rec['cn_gap'] = total(_shell_by_gap, r.get('cn_gap_shell'))
        rec['bvs_adapt'], rec['cn_adapt'] = total(
            _shell_by_valence_adaptive, v, r.get('cn_gap_shell'))
        for f in _bl_dist_factors:
            rec[f'bvs_d{f}'], rec[f'cn_d{f}'] = total(_shell_by_distance, f)
        for s_min in _bl_bv_cutoffs:
            rec[f'bvs_s{s_min}'], rec[f'cn_s{s_min}'] = total(_shell_by_valence, s_min)
        rows.append(rec)

    out = pd.DataFrame(rows)
    if not len(out):
        print('  nothing measured')
        return None

    rules = ['gap'] + [f'd{f}' for f in _bl_dist_factors] + \
            [f's{s}' for s in _bl_bv_cutoffs] + ['adapt']
    formal = out['valence'].abs()
    print()
    print('  deviation from formal valence (%), by site type')
    print(f"  {'rule':10} {'A mean':>9} {'A med':>9} {'A CN':>7} "
          f"{'B mean':>9} {'B med':>9} {'B CN':>7}")
    for k in rules:
        dev = (out[f'bvs_{k}'] - formal) / formal * 100
        line = f'  {k:10}'
        for t in ('A', 'B'):
            m = out['site_type'].eq(t)
            line += (f'{dev[m].mean():9.2f}{dev[m].median():9.2f}'
                     f'{out.loc[m, f"cn_{k}"].mean():7.1f}')
        print(line)
    print()
    print('  structure GII (cation sites only), and the 0.20 pass rate')
    print(f"  {'rule':10} {'median':>9} {'mean':>9} {'<=0.20':>9}")
    for k in rules:
        dd = out[f'bvs_{k}'] - formal
        g = (dd.pow(2).groupby(out['filename']).mean()) ** 0.5
        print(f'  {k:10}{g.median():9.4f}{g.mean():9.4f}'
              f'{100 * (g <= 0.20).mean():8.1f}%')
    print()
    print('  A rule that fixes the larger cation by summing twenty bonds has')
    print('  not fixed anything; the B-site column is the control.')
    return out


# ---------------------------------------------------------------------------
# 3. DOES THE CUT-OFF HAVE TO DIFFER AT THE TWO ENDS OF A BOND
# ---------------------------------------------------------------------------
def sweep_anion_cutoff(quality_csv=None, cif_directory=None, limit=None):
    """
    Rebind ANION_BV_CUTOFF across a sweep and restore it in a finally block.

    s is a property of the bond, so a contact admitted to the cation sum must be
    admitted to the anion sum. The control here is COORDINATION: an oxygen in a
    Pbnm perovskite is 2 B + 4 A. A cut-off that inflates that is admitting a
    second shell whatever it does to the deviations.
    """
    global ANION_BV_CUTOFF

    quality_csv, cif_directory = _resolve_eir_inputs(quality_csv, cif_directory)
    if quality_csv is None:
        return None

    df = _clean_sites(pd.read_csv(quality_csv))
    valences = {fn: dict(zip(g['element'], g['valence']))
                for fn, g in df.groupby('filename')}
    files = sorted(valences)
    if limit:
        files = files[:limit]
    print(f'  {len(files)} structures, {len(ANION_SWEEP)} cut-offs, '
          f'{len(files) * len(ANION_SWEEP)} analyses')

    original, rows, failed = ANION_BV_CUTOFF, [], 0
    try:
        for cut in ANION_SWEEP:
            ANION_BV_CUTOFF = cut
            print(f'    cut-off {cut} ...', end='', flush=True)
            for fn in files:
                try:
                    sites = calculate_anion_bvs(os.path.join(cif_directory, fn),
                                                valences[fn])
                except Exception:
                    failed += 1
                    continue
                for s in sites:
                    rows.append({'cut': cut, 'filename': fn,
                                 'label': s.get('anion_label'),
                                 'cn': s.get('anion_cn'),
                                 'bvs': s.get('anion_bvs'),
                                 'dev': s.get('anion_deviation'),
                                 'ligands': s.get('anion_ligands')})
            print(' done')
    finally:
        ANION_BV_CUTOFF = original
        print(f'  ANION_BV_CUTOFF restored to {ANION_BV_CUTOFF}')
    if failed:
        print(f'  {failed} analyses could not be run')

    out = pd.DataFrame(rows)
    if not len(out):
        print('  nothing measured')
        return None
    tol = globals().get('ANION_CONSISTENT_TOL', 0.20)

    print()
    print('  THE CONTROL: coordination number')
    print(f"  {'cut-off':>8} {'mean CN':>9} {'median CN':>10} {'CN modes':>24}")
    for cut, g in out.groupby('cut'):
        modes = g['cn'].value_counts(normalize=True).head(3)
        txt = ', '.join(f'{int(k)}:{100 * v:.0f}%' for k, v in modes.items())
        print(f'  {cut:8.2f}{g["cn"].mean():9.2f}{g["cn"].median():10.1f}{txt:>25}')
    print()
    print('  THEN the sums')
    print(f"  {'cut-off':>8} {'BVS mean':>10} {'BVS med':>10} {'|dev|>tol':>11}")
    for cut, g in out.groupby('cut'):
        print(f'  {cut:8.2f}{g["bvs"].mean():10.4f}{g["bvs"].median():10.4f}'
              f'{100 * (g["dev"].abs() > tol).mean():10.1f}%')
    return out


print('Measurement tools loaded: compare_expected_distances(), '
      'compare_shell_rules(), sweep_anion_cutoff()')
print('None of them changes a result.')


In [ ]:
# CELL 14: DATASET SUMMARY REPORT   (Eir 2.0.13)
#
# Produces a plain-language summary of a post-processed Eir run, reporting
# every headline figure twice: once over all structures, and once over only
# those whose geometry passed the plausibility screen.
#
# CHANGED IN 2.0.2: which GII to quote is now decided by the data and stated
# explicitly. On a disordered dataset the per-species GII is not a meaningful
# quantity, and reporting it as the headline overstates the instability of the
# whole family by a factor of several. See section 2.
#
# Usage:  summarise_dataset(dfq)                  # print to screen
#         summarise_dataset(dfq, save_to='...')   # also write a text file

import os
import re
import numpy as np
import pandas as pd


DIAGNOSIS_MEANING = {
    'cation_and_anion_agree':
        "cations and oxygens both balance - no evidence of a problem",
    'cation_only_deviates':
        "cations deviate but oxygens balance - points to the bond valence\n"
        "        parameters rather than the structure",
    'both_deviate':
        "cations and oxygens both deviate - a structural or data problem, or\n"
        "        genuine strain (check polyhedron_regular to tell these apart)",
    'anion_only_deviates':
        "oxygens deviate but cations balance - usually the assumed anion\n"
        "        valence is wrong",
    'both_deviate_regular':
        "both deviate and the polyhedra are regular - a structural or data\n"
        "        problem rather than strain",
    'both_deviate_distorted':
        "both deviate and at least one polyhedron is distorted - usually real,\n"
        "        e.g. a Jahn-Teller active ion, which bond valence handles poorly",
    'insufficient_data':
        "not enough information to judge",
}

# Alias of DIAGNOSIS_ORDER (CELL 7), which is where the list lives. This was
# a byte-identical second copy under a different name, so the definition in
# CELL 7 read as dead code and either could have drifted.
DIAGNOSIS_ORDER_REPORT = DIAGNOSIS_ORDER


def _fmt(value, dp=4):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "n/a"
    return f"{value:.{dp}f}"


def _two_column(label, all_val, sub_val, dp=4, width=34):
    return f"  {label:<{width}} {_fmt(all_val, dp):>14} {_fmt(sub_val, dp):>16}"


def summarise_dataset(dfq, save_to=None, gii_threshold=0.20,
                      shift_results=None):
    """
    Summarise a post-processed Eir dataset.

    dfq : DataFrame returned by add_coordination_quality()
    """
    lines = []
    def out(s=""):
        lines.append(s)

    per = dfq.groupby('filename').first()

    # A structure is treated as sound only if it passes ALL the geometric
    # tests present in the file. They are independent: on the reference set
    # each catches structures the others miss, and none is caught by all
    # three, so taking only the first would understate the failure rate.
    # v2.0.2: five screens, not three, and read from the STRUCTURE-level
    # columns.
    #
    # 'core_geometry_plausible' in the CSV is the PER-SITE radii verdict; Eir
    # exports its structure-level counterpart separately. Taking .first() of
    # the per-site column reads whichever site happens to come first and
    # misses the failure whenever the bad site is not row one. On the
    # 934-structure ICSD perovskite set that reported 39 flagged structures
    # where the correct count is 63.
    #
    # The coordination and half-lattice screens were added in 2.0.2 and were
    # simply absent here, so 'passed every test' was measured against three
    # tests out of five and came out 91% where the true figure is 88%.
    _screens = [('radii ratio', 'structure_core_geometry_plausible'),
                ('bond-valence distance', 'structure_bvs_geometry_plausible'),
                ('anion-anion packing', 'anion_packing_plausible'),
                # v2.0.13: the cation floor, measured from the distributions
                # of all three reference families. 19 exclusive catches,
                # including two perovskites that sat in the triage's
                # UNEXPLAINED bucket.
                ('cation-cation packing', 'cation_packing_plausible'),
                ('coordination', 'structure_cn_plausible')]
    _screens = [(n, c) for n, c in _screens if c in per.columns]
    if 'core_geometry_plausible' in per.columns and             'structure_core_geometry_plausible' not in per.columns:
        # older CSV: roll the per-site column up rather than reading row one
        _roll = (dfq.assign(_c=dfq['core_geometry_plausible']
                            .astype(str).str.lower())
                    .groupby('filename')['_c']
                    .apply(lambda s: not (s == 'false').any()))
        per = per.copy()
        per['structure_core_geometry_plausible'] = per.index.map(_roll)
        _screens.insert(0, ('radii ratio', 'structure_core_geometry_plausible'))

    # The half-lattice screen reports a count, not a boolean.
    _shift_col = ('n_sites_shift_proposed'
                  if 'n_sites_shift_proposed' in per.columns else None)
    if _shift_col is not None:
        per = per.copy()
        per['_shift_ok'] = ~(pd.to_numeric(per[_shift_col], errors='coerce')
                             .fillna(0) > 0)
        _screens.append(('half-lattice shift', '_shift_ok'))
    plaus_mask = pd.Series(True, index=per.index)
    for _n, _c in _screens:
        plaus_mask &= (per[_c] != False)
    good = per[plaus_mask]
    n_all, n_good = len(per), len(good)
    n_bad = n_all - n_good

    sites_all = len(dfq)
    good_files = set(good.index)
    sites_good = int(dfq['filename'].isin(good_files).sum())

    out("=" * 78)
    out("EIR DATASET SUMMARY")
    out("=" * 78)
    out()
    out(f"  Structures analysed          {n_all}")
    out(f"  Crystallographic sites       {sites_all}")
    out()

    # ---------------------------------------------------------------- 1
    out("-" * 78)
    out("1. GEOMETRY SCREENING")
    out("-" * 78)
    out()
    out("  Bond valence is calculated from bond lengths, so a length that cannot")
    out("  be right makes every number built on it meaningless. Each structure")
    out("  is therefore tested before anything is interpreted.")
    out()
    out("  The tests are independent and look at different things:")
    out()
    out("    radii ratio            shortest cation-anion contact against the")
    out("                           sum of the Shannon ionic radii, flagged")
    out("                           below 85%")
    out("    bond-valence distance  the same contact against the distance the")
    out("                           cation's own bond valence parameters imply,")
    out("                           flagged below 90%. Needs no radii table and")
    out("                           no assumption about the cation's role.")
    out("    coordination           whether the ion is observed at this")
    out("                           coordination at all. The only test that")
    out("                           asks whether bonds are MISSING rather")
    out("                           than too short.")
    out("    half-lattice shift     whether translating the cation by half a")
    out("                           lattice vector gives a better polyhedron,")
    out("                           which suggests it was deposited on the")
    out("                           wrong origin. Detected and proposed only;")
    out("                           nothing is corrected.")
    out(f"    anion-anion packing    shortest contact between two anions. The")
    out("                           cation-centred tests look outward from a")
    out("                           cation and cannot see it. ICSD 85235 has a")
    out(f"                           GII of 0.0047 and two oxygens 2.191 A")
    out(f"                           apart. Flagged below {ANION_ANION_MIN} A.")
    out("    cation-cation packing  the same for the cation sublattice.")
    out("                           Edge- and face-sharing polyhedra reach")
    out(f"                           2.6-3.0 A and no closer; flagged below")
    out(f"                           {CATION_CATION_MIN} A.")
    out()
    out("  These are the EVIDENCE. The verdict that follows from them is the")
    out("  PGRM integrity tier, reported below and carried per structure in")
    out("  structure_pgrm_tier.")
    out()
    # v2.0.15 (4.4): flagged AND exclusive. Reporting the first alone
    # invites the reading that each screen is catching what it flags. On 938
    # perovskites radii ratio flags 48 and catches 0 of them that nothing else
    # catches; the one structure it flags and bond-valence distance does not,
    # ICSD 129898, is already EXPUNGE on anion packing. The 30-structure test
    # set says the same, and so did the earlier perovskite run: three datasets.
    # It is corroboration - independent of the bond valence parameters, which
    # is worth something - and it is not a catch.
    _flags = {_n: (per[_c] == False) for _n, _c in _screens}
    out(f"    {'screen':<24} {'flagged':>7}  {'exclusive':>9}")
    for _n, _c in _screens:
        _f = _flags[_n]
        _others = pd.Series(False, index=per.index)
        for _m, _g in _flags.items():
            if _m != _n:
                _others |= _g
        out(f"    {_n:<24} {int(_f.sum()):7d}  {int((_f & ~_others).sum()):9d}")
    out()
    out("    'exclusive' is the number a screen flags that no other screen")
    out("    flags. A screen with none is corroborating, not catching, and")
    out("    should not be the reason a structure is condemned.")
    out()
    if 'structure_pgrm_tier' in per.columns:
        out()
        out("  INTEGRITY VERDICT")
        out()
        _t = per['structure_pgrm_tier'].value_counts()
        _why = {'EXPUNGE': 'the atoms cannot be where the file says',
                'NOT_ASSESSED': 'no reference data for at least one site',
                'FLAG': 'the deposited model is ambiguous',
                'ACCEPT': 'nothing found'}
        for _k in ('EXPUNGE', 'NOT_ASSESSED', 'FLAG', 'ACCEPT'):
            if _k in _t:
                out(f"    {_k:<14}{_t[_k]:5}  ({100.0 * _t[_k] / n_all:5.1f}%)"
                    f"   {_why[_k]}")
        _ex = per[per['structure_pgrm_tier'] == 'EXPUNGE']
        if len(_ex):
            out()
            out("    Quote no bond valence result for an expunged structure.")
            out("    The number is arithmetic on impossible coordinates and is")
            out("    often a GOOD number. Reasons:")
            for _r, _n in _ex['structure_pgrm_reason'].value_counts().items():
                out(f"      {str(_r):<34}{_n:4}")
        out()

    pct = 100.0 * n_good / n_all if n_all else 0
    out(f"  Passed every test            {n_good} of {n_all}  ({pct:.0f}%)")
    out(f"  Failed at least one          {n_bad} of {n_all}  ({100-pct:.0f}%)")
    out()
    if n_bad and len(_screens) > 1:
        _nfail = sum((per[_c] == False).astype(int) for _n, _c in _screens)
        _one = int((_nfail == 1).sum())
        _multi = int((_nfail > 1).sum())
        _all3 = int((_nfail == len(_screens)).sum())
        out(f"  Of the {n_bad} flagged, {_one} fail exactly one test and {_multi} fail")
        out(f"  more than one" + (f", with {_all3} failing all {len(_screens)}." if _all3
                                  else f". None fails all {len(_screens)}."))
        out("  The tests are therefore picking up different faults rather than")
        out("  restating the same one, which is why all are reported.")
        out("")
        # v2.0.2: built from _screens rather than hard-coded. The header
        # listed three columns while the loop below computed marks for all of
        # them, so the coordination and half-lattice verdicts were calculated
        # and then dropped on the floor.
        _abbr = {'radii ratio': 'radii', 'bond-valence distance': 'BVS dist',
                 'anion-anion packing': 'anion pack', 'coordination': 'coord',
                 'half-lattice shift': 'shift'}
        _w = 11
        out('      ' + f"{'ICSD':<32}"
            + ''.join(f'{_abbr.get(_n, _n):<{_w}}' for _n, _c in _screens)
            + f"{'shortest anion-anion':<20}")
        for f in sorted(per.index[~plaus_mask]):
            short = f
            # keep the collection code, drop the dataset prefix whatever it is
            short = re.sub(r'^.*?CollCode', '', short).replace('.cif', '') or short
            marks = []
            for _n, _c in _screens:
                marks.append('FAIL' if per.loc[f, _c] == False else '  . ')
            _aa = per.loc[f, 'anion_anion_min'] if 'anion_anion_min' in per.columns else None
            _aatxt = f"{_aa:.3f} A" if _aa == _aa and _aa is not None else ""
            out('      ' + f'{short:<32}'
                + ''.join(f'{m:<{_w}}' for m in marks)
                + f'{_aatxt:<20}')
        out("")

    # ---------------------------------------------------------------- 2
    out("-" * 78)
    out("2. BOND VALENCE RESULTS")
    out("-" * 78)
    out()
    out("  The global instability index (GII) measures how far the bond valence")
    out("  sums depart from the expected valences across a whole structure.")
    out()
    out("  A word of caution on thresholds. The often-quoted figure of 0.20 v.u.")
    out("  is a rule of thumb drawn from relatively simple ordered structures.")
    out("  Families whose connectivity over-constrains the geometry - spinels")
    out("  and many perovskites among them - routinely exceed it while being")
    out("  perfectly stable, because the strain is built into the structure")
    out("  type rather than indicating an unstable compound. Read the numbers")
    out("  below against an expectation for this family, not against 0.20.")
    out()
    out(f"  {'':<34} {'all structures':>14} {'passed all tests':>16}")
    out(f"  {'':<34} {'-'*14:>14} {'-'*16:>16}")
    out(_two_column("Structures", n_all, n_good, 0))
    out(_two_column("Mean GII (v.u.)", per['gii'].mean(), good['gii'].mean()))
    out(_two_column("Median GII (v.u.)", per['gii'].median(), good['gii'].median()))
    out(_two_column("  25th percentile", per['gii'].quantile(.25), good['gii'].quantile(.25)))
    out(_two_column("  75th percentile", per['gii'].quantile(.75), good['gii'].quantile(.75)))
    out(_two_column("  minimum", per['gii'].min(), good['gii'].min()))
    out(_two_column("  maximum", per['gii'].max(), good['gii'].max()))
    if 'gii_site_averaged' in per.columns and per['gii_site_averaged'].notna().any():
        out(_two_column("Mean GII, site-averaged (v.u.)",
                        per['gii_site_averaged'].mean(), good['gii_site_averaged'].mean()))
        out(_two_column("  median", per['gii_site_averaged'].median(),
                        good['gii_site_averaged'].median()))
    if 'wagii' in per.columns:
        out(_two_column("Mean WAGII (v.u.)", per['wagii'].mean(), good['wagii'].mean()))
    if 'gii_all_sites' in per.columns and per['gii_all_sites'].notna().any():
        out(_two_column("Mean GII incl. anions (v.u.)",
                        per['gii_all_sites'].mean(), good['gii_all_sites'].mean()))
    if 'anion_bvs_mean' in per.columns:
        out(_two_column("Mean anion BVS (expect 2.00)",
                        per['anion_bvs_mean'].mean(), good['anion_bvs_mean'].mean()))

    # ---------------------------------------------------------------- 2a
    # WHICH GII TO QUOTE (new in 2.0.2)
    #
    # Eir reports one row per species per site, so a site shared by two
    # elements produces two bond valence sums at the same geometry. The
    # minority species is asked what its sum would be in a site whose size the
    # majority sets, and answers with something absurd: in ICSD 119329,
    # Nd2Zr2O7 refined with 2% anti-site mixing, the 2% Nd on the zirconium
    # site returns 9.22 v.u. against an expected 3, and that single phantom row
    # carries the same weight as the 98% it sits among. Structure GII 3.75;
    # site-averaged 1.91; the ordered members of the same family sit near 0.19.
    #
    # gii            per species. Correct for an ordered structure, and the
    #                right quantity when the question is about one species.
    # gii_site_averaged
    #                one deviation per crystallographic site, from the
    #                occupancy-weighted combination of the per-species sums.
    #                The right headline for a dataset containing disorder.
    # wagii          occupancy-weighted across sites; down-weights minority
    #                species but still reports them separately.
    #
    # The choice is made here from the measured disorder fraction rather than
    # left to the reader, and the reasoning is printed either way.
    # v2.0.2: prefer the anti-site verdict, which names the actual reason a
    # per-species GII misleads, over the raw disorder fraction. A site shared
    # by two species that appear nowhere else is a solid solution and the sums
    # are meaningful; a species that is a minority here and a majority on
    # another site is anti-site exchange, and its sum is a phantom.
    _anti = None
    if 'has_antisite_disorder' in per.columns:
        _anti = per['has_antisite_disorder'].astype(str).str.lower().eq('true')
    elif 'n_antisite_species' in per.columns:
        _anti = pd.to_numeric(per['n_antisite_species'],
                              errors='coerce').fillna(0) > 0
    if _anti is not None and _anti.any():
        _a, _b = per[_anti.values], per[~_anti.values]
        out()
        out(f"  Anti-site exchange: {int(_anti.sum())} of {n_all} structures.")
        out()
        out("  Eir reports one bond valence sum per SPECIES per site. Where a")
        out("  site is shared, the minority species is asked what its sum would")
        out("  be in a cavity the majority sizes; it answers with something")
        out("  impossible, and the per-species GII weights that phantom equally")
        out("  with the majority it sits among.")
        out()
        _cols = [c for c in ('gii', 'wagii', 'gii_site_averaged')
                 if c in per.columns]
        out(f"      {'':24}" + ''.join(f'{c:>18}' for c in _cols))
        for _lbl, _sub in ((f'with anti-site ({len(_a)})', _a),
                           (f'ordered ({len(_b)})', _b)):
            out(f"      {_lbl:24}" + ''.join(
                f"{pd.to_numeric(_sub[c], errors='coerce').median():>18.4f}"
                for c in _cols))
        out()
        if {'gii', 'wagii'} <= set(_cols) and len(_b):
            _r = (pd.to_numeric(_b['gii'], errors='coerce')
                  / pd.to_numeric(_b['wagii'], errors='coerce')).median()
            out(f"  On the structures with no shared site gii/wagii is {_r:.3f}.")
            out("  The two are the same quantity when every occupancy is 1.0,")
            out("  not merely when nothing is shared: WAGII weights each site")
            out("  by its occupancy, so a vacancy separates them too. Full")
            out("  occupancy gives gii == wagii on 869 of 871 perovskites and")
            out("  355 of 355 pyrochlores; 1543 partially occupied spinels do")
            out("  not. That is what makes the weighted form safe to quote")
            out("  throughout.")
            out()
        out("  The column gii_recommended carries the choice per structure.")
        out("  Which index to quote for the DATASET is decided once, below,")
        out("  under 'Which GII to quote'.")
        out()

    _disorder_frac = None
    if 'has_mixed_occupancy' in dfq.columns:
        _mx = dfq['has_mixed_occupancy'].astype(str).str.lower().eq('true')
        _dis_struct = dfq.assign(_m=_mx).groupby('filename')['_m'].any()
        _disorder_frac = float(_dis_struct.mean())
        _site_frac = float(_mx.mean())

    if _disorder_frac is not None:
        out()
        out("  Which GII to quote.")
        out()
        out(f"  {100 * _disorder_frac:.0f}% of structures here contain at least one site")
        out(f"  shared by two or more species ({100 * _site_frac:.0f}% of all sites).")
        out()
        if _disorder_frac >= 0.10:
            _g_all = per['gii'].median()
            _g_sa = (per['gii_site_averaged'].median()
                     if 'gii_site_averaged' in per.columns else None)
            out("  That is enough for the per-species GII above to be misleading as a")
            out("  headline figure. Eir reports one row per species per site, so a")
            out("  minority species is asked what its bond valence sum would be in a")
            out("  site whose size the majority sets. It returns something impossible,")
            out("  and it is then weighted equally with the majority it sits among.")
            out()
            if _g_sa is not None:
                out(f"      median GII, per species     {_g_all:.4f} v.u.")
                out(f"      median GII, site-averaged   {_g_sa:.4f} v.u.")
                out()
            out("  QUOTE THE SITE-AVERAGED FIGURE for this dataset, and say so.")
            out("  The per-species column remains the right one for a question")
            out("  about a particular species, and where every occupancy is 1.0")
            out("  the three indices agree.")
            out()
            out("  This is the ONLY recommendation in this report. Until v2.0.13")
            out("  the section above also carried one, naming WAGII, so two")
            out("  instructions naming different columns sat 30 lines apart on")
            out("  the same page.")
        else:
            out("  That is low enough that the per-species and site-averaged figures")
            out("  will agree closely, and the per-species GII above can be quoted")
            out("  directly. State which you used regardless: on a dataset with more")
            out("  disorder the two diverge sharply.")
        out()
        out("  Whichever is quoted, state the geometry screening alongside it. A")
        out("  bond valence sum built on an impossible bond length is not evidence")
        out("  of instability, it is evidence of a bad structure, and the two are")
        out("  not distinguishable from the number alone.")
    out()
    delta = per['gii'].mean() - good['gii'].mean()
    if abs(delta) > 0.005:
        out(f"  Excluding the {n_bad} implausible structures lowers the mean GII by")
        out(f"  {abs(delta):.4f} v.u. Quote the right-hand column, and state the")
        out("  screening, rather than reporting the unfiltered figure alone.")
        out()

    # --- ordered vs shared sites ----------------------------------------
    # One row per crystallographic SITE, using the occupancy-weighted
    # deviation. Reporting one row per species instead counts a shared site
    # twice and asks each species what its sum would be if it owned a
    # geometry set by the other, which is a counterfactual rather than a
    # measurement.
    sub_ok = dfq[dfq['filename'].isin(good_files)].copy()
    if 'site_deviation_averaged' in sub_ok.columns and 'coords' in sub_ok.columns:
        _sites = sub_ok.drop_duplicates(subset=['filename', 'coords'])
        _sites = _sites[_sites['site_deviation_averaged'].notna()].copy()
        _sites['_dev'] = _sites['site_deviation_averaged'].abs()
        out("  Deviation per crystallographic site, structures passing every test:")
        out()
        out(f"      {'site type':<34}{'sites':>8}{'mean |BVS - expected|':>24}")
        if 'n_elements_at_site' in _sites.columns:
            _ne = _sites['n_elements_at_site'].fillna(1).astype(int)
            _ordered = _sites[_ne == 1]['_dev']
            _shared = _sites[_ne >= 2]['_dev']
            out(f"      {'one element at site':<34}{len(_ordered):>8}{_ordered.mean():>24.4f}")
            out(f"      {'shared by two elements':<34}{len(_shared):>8}{_shared.mean():>24.4f}")
        else:
            _ordered = _shared = pd.Series(dtype=float)
        _fr = (_sites['valence'] - _sites['valence'].round()).abs() > 1e-6
        out(f"      {'integer assumed valence':<34}{int((~_fr).sum()):>8}"
            f"{_sites.loc[~_fr, '_dev'].mean():>24.4f}")
        out(f"      {'fractional assumed valence':<34}{int(_fr.sum()):>8}"
            f"{_sites.loc[_fr, '_dev'].mean():>24.4f}")
        out()
        if len(_ordered) and len(_shared) and _ordered.mean() > 0:
            _ratio = _shared.mean() / _ordered.mean()
            out(f"  A shared site deviates about {_ratio:.1f} times as much as an ordered")
            out("  one. That is a statement about precision, not validity: bond valence")
            out("  on a site holding two species is a less precise measurement, by a")
            out("  factor that can be quoted, and the figures above are how to quote it.")
            out()
            out("  Two things about the earlier, larger figure for this gap. Reporting")
            out("  one row per species rather than per site inflates it, because a")
            out("  minority species is asked what its sum would be in a site whose")
            out("  geometry the majority sets: a 1.6% anti-site impurity was carrying")
            out("  the same weight as the 98% it sits among. And several structures")
            out("  contributing most of the apparent gap turned out to fail a geometry")
            out("  test, so the effect was partly bad data wearing a disguise.")
            out()

    out("-" * 78)
    out("3. WHERE THE DEVIATION COMES FROM")
    out("-" * 78)
    out()
    out("  Every bond contributes to one cation sum and one anion sum, so a")
    out("  cation discrepancy that cancels at the shared oxygen is a property")
    out("  of the parameters, not of the structure. Comparing the two sides")
    out("  separates those cases.")
    out()
    out(f"  {'':<34} {'all structures':>14} {'passed all tests':>16}")
    out(f"  {'':<34} {'-'*14:>14} {'-'*16:>16}")
    _dcol = ('bvs_diagnosis_detail'
             if 'bvs_diagnosis_detail' in per.columns else 'bvs_diagnosis')
    va = per[_dcol].value_counts()
    vg = good[_dcol].value_counts()
    for k in DIAGNOSIS_ORDER_REPORT:
        if k in va.index or k in vg.index:
            out(_two_column(k, va.get(k, 0), vg.get(k, 0), 0))
    out()
    for k in DIAGNOSIS_ORDER_REPORT:
        if k in va.index and k in DIAGNOSIS_MEANING:
            out(f"    {k}")
            out(f"        {DIAGNOSIS_MEANING[k]}")
    out()

    # ---------------------------------------------------------------- 4
    out("-" * 78)
    out("4. COORDINATION QUALITY")
    out("-" * 78)
    out()
    # Half-lattice proposals, and what testing them showed.
    if 'n_sites_shift_proposed' in per.columns:
        _np = pd.to_numeric(per['n_sites_shift_proposed'],
                            errors='coerce').fillna(0)
        _ns = int((_np > 0).sum())
        if _ns:
            out(f"  {_ns} structures carry a proposed half-lattice shift: a")
            out("  cation whose coordination sphere is SPLIT, short on one side")
            out("  and long on the other, which every short-contact test passes")
            out("  because nothing is individually too short. Where the bond")
            out("  valence sum also passes, as it often does because the two")
            out("  halves cancel, only the geometry gives it away.")
            out()
            out("  Nothing is corrected. CELL 13b applies each proposal to a")
            out("  COPY and re-runs the whole analysis, so the proposal can be")
            out("  judged on what it gives rather than on what it claims.")
            out()

            # What testing them showed. Picked up from CELL 13b automatically
            # so that CELL 15 needs no argument; passing shift_results
            # explicitly overrides it.
            _sr = shift_results
            if _sr is None:
                _sr = globals().get('_shift_results')
            if _sr is not None and len(_sr):
                _sr = _sr.copy()
                if 'variant' not in _sr.columns:
                    # a run from before the variant column existed: the two
                    # hypotheses alternate in file order
                    _sr['variant'] = (_sr.groupby('filename').cumcount()
                                      .map({0: 'single_site', 1: 'all_cations'}))
                _done = _sr.dropna(subset=['gii_after'])
                out("  RESULT OF TESTING THEM")
                out()
                out("  Two hypotheses per structure: moving only the cation the")
                out("  screen named, and moving every cation by the same vector.")
                out("  The second describes a cation sublattice deposited on the")
                out("  wrong origin, which the first cannot represent.")
                out()
                _tab = pd.crosstab(_sr['variant'], _sr['verdict'])
                for _line in _tab.to_string().splitlines():
                    out('      ' + _line)
                out()
                if len(_done):
                    _best = (_done.sort_values('gii_after')
                                  .drop_duplicates('filename'))
                    _wins = _best['variant'].value_counts().to_dict()
                    out(f"  Better hypothesis per structure: {_wins}")
                    _ss = _done[_done['variant'] == 'single_site']
                    if len(_ss) and {'n_screens_failed_before',
                                     'n_screens_failed_after'} <= set(_ss.columns):
                        _b = pd.to_numeric(_ss['n_screens_failed_before'],
                                           errors='coerce').sum()
                        _a = pd.to_numeric(_ss['n_screens_failed_after'],
                                           errors='coerce').sum()
                        _clean = int((pd.to_numeric(
                            _ss['n_screens_failed_after'],
                            errors='coerce') == 0).sum())
                        out(f"  Moving the named cation alone takes screen failures")
                        out(f"  across these structures from {int(_b)} to {int(_a)},")
                        out(f"  and leaves {_clean} of {len(_ss)} passing every test.")
                    out()
                    _sup = _done[_done['verdict'].astype(str)
                                 .str.contains('supported', case=False, na=False)
                                 & ~_done['verdict'].astype(str)
                                 .str.startswith('NOT')]
                    if len(_sup):
                        _sup = _sup.assign(_r=pd.to_numeric(_sup['gii_before'],
                                                            errors='coerce')
                                           / pd.to_numeric(_sup['gii_after'],
                                                           errors='coerce'))
                        _cols = [c for c in ('icsd_code', 'chemical_formula',
                                             'shift_applied', 'gii_before',
                                             'gii_after', 'verdict')
                                 if c in _sup.columns]
                        out("  Best supported:")
                        for _line in (_sup.nlargest(min(8, len(_sup)), '_r')[_cols]
                                      .to_string(index=False).splitlines()):
                            out('      ' + _line)
                        out()
                out("  A supported shift is evidence that the deposited cation")
                out("  sits on the wrong origin. It is not a correction, and the")
                out("  deposited entry remains the data of record until the")
                out("  source says otherwise.")
                out()

    if 'cn_tie_extended' in dfq.columns:
        _tie = dfq['cn_tie_extended'].astype(str).str.lower().eq('true')
        if _tie.any():
            out(f"  Coordination numbers are the count of bonds actually summed. On")
            out(f"  {int(_tie.sum())} sites the shell boundary fell inside a set of")
            out(f"  symmetry-equivalent bonds, so every one of them was included and")
            out(f"  the reported CN is larger than the gap analysis alone would give.")
            out(f"  That is correct: the boundary is a distance, and equal bonds")
            out(f"  cannot be separated by one.")
            out()
    if 'cn_revised' in dfq.columns:
        _rev = dfq['cn_revised'].astype(str).str.lower().eq('true')
        if _rev.any():
            out(f"  On {int(_rev.sum())} sites the coordination shell from gap analysis")
            out(f"  disagreed with ECoN and was extended; see cn_gap, econ_shell and")
            out(f"  cn_revision_reason for each. Gap analysis cuts at the largest gap")
            out(f"  in the bond list, which is the shell boundary for a compact")
            out(f"  polyhedron and an internal gap for a broad one.")
            out()
    out("  The effective coordination number (ECoN, Hoppe 1979) weights each")
    out("  bond by its length. For a regular polyhedron ECoN equals the integer")
    out("  coordination number; it falls below it as the polyhedron distorts.")
    out("  A low ratio on a Jahn-Teller active ion is a chemical result, not an")
    out("  error, and distinguishes real strain from a bad structure.")
    out()
    sub = dfq[dfq['filename'].isin(good_files)]
    reg = sub['polyhedron_regular'].dropna()
    if 'bl_tier' in dfq.columns and dfq['bl_tier'].notna().any():
        _bt = pd.to_numeric(dfq['bl_tier'], errors='coerce')
        _bd = pd.to_numeric(dfq['bl_delta_mean'], errors='coerce')
        _br = pd.to_numeric(dfq['bl_ratio_min'], errors='coerce')
        out("  MEASURED BOND LENGTHS")
        out()
        out("  bl_expected_mean is the mean bond length observed for this exact")
        out("  ion configuration across Gagne's survey of refined inorganic")
        out("  structures; bl_delta_mean is this site's mean minus that, in")
        out("  angstroms, and bl_n_bonds is the sample it rests on.")
        out()
        _lab = {1: 'measured', 2: 'interpolated', 3: 'extrapolated',
                4: 'Shannon additive'}
        for _t in (1, 2, 3, 4):
            _n = int((_bt == _t).sum())
            if _n:
                out(f"      tier {_t}  {_lab[_t]:18} {_n:6}  "
                    f"({100 * _n / int(_bt.notna().sum()):.1f}%)")
        if _bd.notna().any():
            out()
            out(f"  Observed minus expected, median {_bd.median():+.4f} A "
                f"over {int(_bd.notna().sum())} sites.")
        out()
        out("  These are REPORTED, not screened. A range test - is any bond")
        out("  outside anything ever observed for this ion - was built and")
        out("  rejected: an extreme needs a large sample, Gagne's")
        out("  high-coordination entries are the small ones, and on the")
        out("  perovskites it condemned 224 sites in structures that pass")
        out("  every other test, some with GII below 0.02. Quote")
        out("  bl_delta_mean with bl_n_bonds beside it and let the reader")
        out("  weigh it.")
        out()
        if _br.notna().any():
            out(f"  bl_ratio_min, this site's shortest contact against the")
            out(f"  shortest ever observed for this ion configuration, is")
            out(f"  available on {int(_br.notna().sum())} of {len(dfq)} sites, "
                f"median {_br.median():.3f}.")
            out("  Like against like, so it carries no coordination dependence.")
            out("  Null means the tail of the reference distribution is not")
            out("  characterised, which is a fact about the reference data.")
            out()

    if len(reg):
        out(f"  Sites in plausible structures     {sites_good}")
        out(f"  Regular polyhedra                 {int(reg.sum())} of {len(reg)}")
        out(f"  Distorted polyhedra               {int((~reg.astype(bool)).sum())} of {len(reg)}")
    out()
    six = sub[sub['cn_relative'] == 6]
    if len(six):
        g = (six.groupby('element')['econ_cn_ratio']
                .agg(['count', 'mean', 'min']))
        g = g[g['count'] >= 5].sort_values('mean')
        out("  Mean ECoN/CN by element (six-coordinate sites only):")
        out()
        out(f"      {'element':<10}{'sites':>7}{'mean':>10}{'lowest':>10}")
        for el, r in g.iterrows():
            out(f"      {el:<10}{int(r['count']):>7}{r['mean']:>10.4f}{r['min']:>10.4f}")
        out()
        # v2.0.13: only sites that are actually distorted. nsmallest on the
        # whole set lists the five lowest ratios even when all of them are
        # above ECON_REGULAR_MIN, so a run of symmetry-perfect octahedra was
        # reported under the heading 'Most distorted sites' at ratios of
        # 0.999.
        _dist = six[pd.to_numeric(six['econ_cn_ratio'],
                                  errors='coerce') < ECON_REGULAR_MIN]
        worst = _dist.nsmallest(5, 'econ_cn_ratio')
        if len(worst):
            out(f"  Most distorted sites (ECoN/CN below {ECON_REGULAR_MIN}):")
            out()
            for _, r in worst.iterrows():
                # v2.0.13: was .replace('dataset_CollCode', ''), one
                # dataset's file-naming convention hard-coded into a general
                # report. Prefer the ICSD code the CSV already carries.
                fn = (str(int(r['icsd_code']))
                      if r.get('icsd_code') == r.get('icsd_code')
                      else re.sub(r'^.*CollCode|\.cif$', '',
                                  str(r['filename'])))
                out(f"      {fn:<12} {r['element']:<4} "
                    f"ECoN {r['econ']:.2f} of 6   ratio {r['econ_cn_ratio']:.3f}")
            out()

    # ---------------------------------------------------------------- 5
    out("-" * 78)
    out("5. SETTINGS USED")
    out("-" * 78)
    out()
    cfg = [c for c in dfq.columns if c.startswith('cfg_')]
    for c in sorted(cfg):
        out(f"  {c[4:]:<32} {dfq[c].iloc[0]}")
    # the two screens added after the cfg_ columns were defined
    try:
        out(f"  {'bvs_distance_fraction':<32} {IMPLAUSIBLE_BVS_DISTANCE_FRACTION}")
        out(f"  {'anion_anion_min_A':<32} {ANION_ANION_MIN}")
    except NameError:
        pass
    out()
    out("=" * 78)

    text = "\n".join(lines)
    print(text)

    if save_to:
        with open(save_to, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"\nSaved to: {save_to}")

    return text

In [ ]:
# CELL 15: PRINT THE DATASET SUMMARY   (Eir 2.0.13)

if 'dfq' in globals() and dfq is not None:
    _summary_path = os.path.join(EIR_CIF_DIRECTORY, 'outputs', 'eir_summary.txt')
    _txt = summarise_dataset(dfq, save_to=_summary_path)

    # v2.0.13: the figures too, so a run ends with everything a reader needs
    # rather than a wall of numbers that then needs an interpreter.
    if 'eir_figures' in globals():
        _figs = eir_figures(dfq,
                            output_dir=os.path.join(EIR_CIF_DIRECTORY,
                                                    'outputs'))
    print()
    print('A RUN NOW PRODUCES')
    print(f"  {os.path.basename(_summary_path)}                  the narrative")
    print('  ..._quality.csv                     one row per site')
    print('  ..._structures.csv                  one row per structure;')
    print('                                      filter to EXPUNGE for the')
    print('                                      reportable-error list')
    print('  eir_screens / gii / cation_anion / gii_vs_wagii .pdf and .png')
    print()
    print('  Read structure_pgrm_tier for the verdict and pgrm_reason with')
    print('  pgrm_evidence for what was measured. Every figure and every')
    print('  count in the summary derives from those same columns.')
else:
    print('Run the post-processing cell first.')


In [ ]:
# CELL 16: LOOK AT ONE STRUCTURE   (Eir 2.0.13)
# ============================================================================
# Put an ICSD code in and get everything Eir knows about that entry: the
# verdict and the measurement behind it, every site, and - if you ask for it
# - the coordination polyhedron in 3D.
#
# This sits at the END of the reporting on purpose. The summary, the triage
# and the figures tell you WHICH structures to look at; this is where you
# then look. Re-run it as often as you like: it draws at most a handful of
# viewers and resets its own counter each call, so it cannot do what the
# automatic per-site visualisation used to do to a browser.


def look_at(code, dfq=None, cif_directory=None, show_3d=False, max_views=4):
    """
    Everything Eir knows about one entry.

    code           ICSD collection code, or a filename, or part of one
    show_3d        draw the coordination polyhedra. Off by default; each
                   viewer is a live WebGL context and a browser holds few.
    max_views      hard cap on viewers for this call
    """
    _df = dfq if dfq is not None else globals().get("dfq")
    if _df is None:
        print("No results in memory. Run the post-processing cell first.")
        return None

    _key = str(code).strip()
    _m = _df[_df["icsd_code"].astype(str).str.strip() == _key]
    if not len(_m):
        _m = _df[_df["filename"].astype(str).str.contains(_key, case=False,
                                                          na=False)]
    if not len(_m):
        print(f"Nothing matching {_key!r}. Try an ICSD code or part of a "
              f"filename.")
        return None
    if _m["filename"].nunique() > 1:
        _n = _m["filename"].nunique()
        print(f"{_key!r} matches {_n} files. Be more specific:")
        for _f in sorted(_m["filename"].unique())[:8]:
            print(f"    {_f}")
        if _n > 8:
            print(f"    ... and {_n - 8} more")
        return None

    r0 = _m.iloc[0]
    _line = "=" * 78
    print(_line)
    print(f"  {r0['chemical_formula']}   ICSD {r0['icsd_code']}"
          f"   {r0.get('space_group', '?')}")
    print(f"  {r0.get('year', '?')}  {str(r0.get('journal', ''))[:52]}")
    if r0.get("authors") == r0.get("authors"):
        print(f"  {str(r0.get('authors'))[:70]}")
    print(_line)

    # ---- the verdict ---------------------------------------------------
    _tier = r0.get("structure_pgrm_tier")
    if _tier == _tier and _tier is not None:
        print(f"\n  VERDICT   {_tier}")
        if r0.get("structure_pgrm_reason") == r0.get("structure_pgrm_reason"):
            print(f"            {r0['structure_pgrm_reason']}")
        if r0.get("structure_pgrm_evidence") == r0.get("structure_pgrm_evidence"):
            print(f"            {r0['structure_pgrm_evidence']}")
        if _tier == "EXPUNGE":
            print("\n  Quote no bond valence result for this entry. The")
            print("  numbers below are computed from coordinates that")
            print("  cannot be right, and are shown so you can see why.")

    # ---- the screens ---------------------------------------------------
    print("\n  SCREENS")
    for _n, _c in (("radii ratio", "structure_core_geometry_plausible"),
                   ("bond-valence distance", "structure_bvs_geometry_plausible"),
                   ("anion-anion packing", "anion_packing_plausible"),
                   ("cation-cation packing", "cation_packing_plausible"),
                   ("coordination", "structure_cn_plausible")):
        if _c in r0.index:
            _v = str(r0[_c]).lower()
            _mark = "FAIL" if _v == "false" else ("pass" if _v == "true" else "--")
            print(f"    {_n:<24} {_mark}")
    for _n, _c in (("shortest anion-anion", "anion_anion_min"),
                   ("shortest cation-cation", "cation_cation_min")):
        if _c in r0.index and r0[_c] == r0[_c]:
            print(f"    {_n:<24} {r0[_c]:.3f} A")

    # ---- the sites -----------------------------------------------------
    print("\n  SITES")
    _cols = [c for c in ("site_label", "element", "valence", "occupancy", "cn",
                         "cn_gap_shell", "bvs", "bvs_percent_diff",
                         "bl_delta_mean", "bl_n_bonds", "min_bond_length",
                         "econ_cn_ratio", "geometry")
             if c in _m.columns]
    for _l in _m[_cols].to_string(index=False).splitlines():
        print("    " + _l)

    print()
    for _c, _lab in (("gii", "GII"), ("wagii", "WAGII"),
                     ("gii_site_averaged", "GII site-averaged"),
                     ("gii_recommended", "recommended")):
        if _c in r0.index and r0[_c] == r0[_c]:
            _v = r0[_c]
            _txt = f"{_v:.4f}" if isinstance(_v, float) else str(_v)
            print(f"    {_lab:<20} {_txt}")

    if r0.get("proposed_shift") == r0.get("proposed_shift") or (
            "shift_verdict" in r0.index and r0["shift_verdict"] == r0["shift_verdict"]):
        print("\n  PROPOSED SHIFT")
        for _c in ("proposed_shift", "shift_shell_spread", "proposed_spread",
                   "shift_verdict", "shift_evidence"):
            _s = _m[_c].dropna() if _c in _m.columns else []
            if len(_s):
                print(f"    {_c:<22} {_s.iloc[0]}")

    # ---- 3D, on request only -------------------------------------------
    if show_3d:
        if not globals().get("PY3DMOL_AVAILABLE", False):
            print("\n  [3D unavailable: py3Dmol is not installed]")
        elif cif_directory is None and "EIR_CIF_DIRECTORY" not in globals():
            print("\n  [3D needs the CIFs: pass cif_directory=...]")
        else:
            _dir = cif_directory or EIR_CIF_DIRECTORY
            _drawn = 0
            for _, _s in _m.iterrows():
                if _drawn >= max_views:
                    print(f"\n  [stopped at {max_views} viewers; raise "
                          f"max_views if you need more]")
                    break
                try:
                    visualize_problematic_site_3d(
                        os.path.join(_dir, _s["filename"]),
                        _s["element"], _s["site_label"], None,
                        _s.get("deviation"), _s.get("valence"),
                        _s.get("bvs"), filename=_s["filename"])
                    _drawn += 1
                except Exception as _e:
                    print(f"    [3D failed for {_s['site_label']}: "
                          f"{type(_e).__name__}: {_e}]")
    else:
        print("\n  Pass show_3d=True to draw the coordination polyhedra.")

    print()
    return _m


# ---------------------------------------------------------------------------
# Put a code here and re-run this cell.
#
#   look_at(85235)                     TbCrO3: GII 0.0047, oxygens 2.191 A apart
#   look_at(220506)                    LaCrO3: La-O at 1.008 A
#   look_at(11616, show_3d=True)       DyMnO3: cation on the wrong origin
# ---------------------------------------------------------------------------

LOOK_AT = None        # e.g. LOOK_AT = 85235

if LOOK_AT is not None:
    _ = look_at(LOOK_AT)
elif "dfq" in globals() and dfq is not None:
    _n = 0
    if "structure_pgrm_tier" in dfq.columns:
        _n = dfq[dfq["structure_pgrm_tier"] == "EXPUNGE"]["filename"].nunique()
    print(f"look_at(<icsd code>) to investigate one entry."
          + (f"  {_n} structure(s) are at EXPUNGE and worth starting with."
             if _n else ""))
else:
    print("Run the post-processing cell first, then look_at(<icsd code>).")


In [ ]:
# CELL 17: LOOK AT A FAMILY   (Eir 2.0.13)
# ============================================================================
# "Cr1 O3 Pr1  2 of 6 flagged" in the triage is interesting. Seeing the six is
# the next thing you want, and it is the point at which a pattern separates
# from an accident: if the two bad ones are from one paper and the four sound
# ones from four others, that is a different finding from two bad entries
# scattered through six independent determinations.
#
# DOES NOTHING UNLESS ASKED. Set SHOW_FAMILY and re-run.
#
# Matching is on the ELEMENT SET, not the formula string. Eir writes ICSD
# style - "Eu1 O3 V1", not "EuVO3" - and the non-stoichiometric variants are
# usually the ones worth seeing: asking for LaMnO3 should return
# La0.8Mn1O2.8 and La1Mn0.95O3.02 as well, because whether an entry is
# off-stoichiometry is a finding rather than a reason to exclude it.
#
# Selection when a family is larger than the limit: EVERY structure that
# failed something, then sound ones spread evenly across the GII range. The
# flagged ones are why you asked; the sound ones are the control, and taking
# the first few would give you whichever the filesystem listed first.

import re as _re

FAMILY_LIMIT = 20        # rows shown before the selection starts choosing


def _elements_of(text):
    """The element set in a formula, however it is written."""
    if text is None or text != text:
        return frozenset()
    return frozenset(_re.findall(r"[A-Z][a-z]?", str(text)))


def look_at_family(spec, dfq=None, limit=FAMILY_LIMIT, sort_by="gii"):
    """
    Every entry in this compositional family, flagged ones first.

    spec     "LaMnO3", "Cr1 O3 Pr1", "Eu V O" - anything from which element
             symbols can be read. Matching ignores stoichiometry, so the
             off-stoichiometry entries come too.
    limit    rows shown. Everything flagged is kept; the remainder of the
             budget goes to sound structures spread across the GII range.
    sort_by  "gii", "year" or "code"
    """
    _df = dfq if dfq is not None else globals().get("dfq")
    if _df is None:
        print("No results in memory. Run the post-processing cell first.")
        return None

    want = _elements_of(spec)
    if not want:
        print(f"Could not read any element symbols from {spec!r}.")
        return None

    per = (build_structure_table(_df) if "build_structure_table" in globals()
           else _df.drop_duplicates("filename"))
    per = per.copy()
    per["_els"] = per["chemical_formula"].map(_elements_of)
    hit = per[per["_els"] == want]
    if not len(hit):
        _near = per[per["_els"].map(lambda s: want <= s)]
        if len(_near):
            print(f"No entry with exactly {sorted(want)}. "
                  f"{len(_near)} contain those elements plus others:")
            for _f in sorted(_near["chemical_formula"].unique())[:8]:
                print(f"    {_f}")
        else:
            print(f"Nothing with elements {sorted(want)}.")
        return None

    # what counts as flagged
    # "Flagged" here means EXPUNGE or NOT_ASSESSED, OR a failed screen that
    # PGRM does not currently gate on. The radii-ratio and bond-valence
    # distance screens have no G code: they say a contact is short relative to
    # the ions rather than impossible, and whether that should expunge is an
    # open question. 25 perovskites fail only those two and are ACCEPT under
    # PGRM while the triage counts them as flagged, which is why a family can
    # read "1 flagged" here and "5 of 5" there. Counting them keeps the two
    # views consistent until the question is settled.
    _tier = hit.get("structure_pgrm_tier")
    _screen_fail = None
    for _c in ("structure_core_geometry_plausible",
               "structure_bvs_geometry_plausible"):
        if _c in hit.columns:
            _f = hit[_c].astype(str).str.lower().eq("false")
            _screen_fail = _f if _screen_fail is None else (_screen_fail | _f)
    if _tier is not None:
        _t = hit["structure_pgrm_tier"].isin(["EXPUNGE", "NOT_ASSESSED"])
        if _screen_fail is not None:
            _t = _t | _screen_fail
        bad = hit[_t]
        ok = hit[~hit.index.isin(bad.index)]
    elif "n_screens_failed" in hit.columns:
        _n = pd.to_numeric(hit["n_screens_failed"], errors="coerce").fillna(0)
        bad, ok = hit[_n > 0], hit[_n == 0]
    else:
        bad, ok = hit.iloc[0:0], hit

    # selection: all the bad, then sound ones spread across the GII range
    shown, note = hit, ""
    if len(hit) > limit:
        _room = max(0, limit - len(bad))
        if _room and len(ok) > _room:
            _o = ok.assign(_g=pd.to_numeric(ok.get("gii"), errors="coerce"))
            _o = _o.sort_values("_g")
            _idx = np.linspace(0, len(_o) - 1, _room).round().astype(int)
            ok_sel = _o.iloc[sorted(set(_idx))]
        else:
            ok_sel = ok.head(_room)
        shown = pd.concat([bad, ok_sel])
        note = (f"  showing {len(bad)} flagged and {len(ok_sel)} of {len(ok)} "
                f"sound, spread across the GII range. limit={limit}.")

    _by = {"gii": "gii", "year": "year", "code": "icsd_code"}.get(sort_by, "gii")
    if _by in shown.columns:
        shown = shown.sort_values(_by, ascending=(_by != "gii"),
                                  na_position="last")

    print("=" * 78)
    print(f"  {spec}   [{' '.join(sorted(want))}]   {len(hit)} entries, "
          f"{len(bad)} flagged")
    print("=" * 78)
    if note:
        print(note)
    print()
    _cols = [c for c in ("icsd_code", "chemical_formula", "year",
                         "structure_pgrm_tier", "structure_pgrm_reason",
                         "gii", "anion_anion_min", "cation_cation_min")
             if c in shown.columns]
    for _l in shown[_cols].to_string(index=False).splitlines():
        print("  " + _l)

    # the provenance question: is it one paper or many?
    if len(bad) >= 2 and "journal" in hit.columns:
        print()
        _bj = bad["journal"].fillna("?").value_counts()
        _aj = hit["journal"].fillna("?").nunique()
        print(f"  The {len(bad)} flagged entries come from "
              f"{len(_bj)} journal(s), out of {_aj} across the family:")
        for _j, _n in _bj.items():
            print(f"      {_n:3}  {str(_j)[:58]}")
        if len(_bj) == 1 and _aj > 1:
            print("  One source for every flagged entry. That is a pattern,")
            print("  not an accident: check the paper before the database.")
    if "year" in hit.columns and len(bad):
        _by_ = pd.to_numeric(bad["year"], errors="coerce").dropna()
        _ay = pd.to_numeric(hit["year"], errors="coerce").dropna()
        if len(_by_) and len(_ay):
            print()
            print(f"  flagged span {int(_by_.min())}-{int(_by_.max())};  "
                  f"family span {int(_ay.min())}-{int(_ay.max())}")
    print()
    print("  look_at(<code>) for any one of these.")
    print()
    return shown


# ---------------------------------------------------------------------------
# Put a composition here and re-run. Nothing happens until you do.
#
#   SHOW_FAMILY = "EuVO3"        the 5 of 5 flagged case
#   SHOW_FAMILY = "PrCrO3"       2 of 6: is it one paper or two?
#   SHOW_FAMILY = "LaMnO3"       19 entries, including the off-stoichiometry
# ---------------------------------------------------------------------------

SHOW_FAMILY = None

if SHOW_FAMILY is not None:
    _ = look_at_family(SHOW_FAMILY)
else:
    print("look_at_family('LaMnO3') to see a whole compositional family.")
    print("Matching is on the element set, so the off-stoichiometry entries")
    print("come too. Nothing is drawn and nothing is recomputed.")
